###***Time Series Analysis for Residual Chlorine Forecast at Alexander Orr Water Treatment Plant***

Dissertation Title: AI/ML Models to Forecast Chlorine Concentrations in a Water Treatment Plant in the State of Florida, USA

Data set provided by the Alexander Orr Water Treatment Plant, Miami-Dade County, Miami FL, USA

Dissertation By: Angelica Cortes Ortiz

##Dataset description:
Alexander Orr Water Treatment Plant dataset including:
1. Date (month, day, year and hour)
2. Finish water or Flowrate (million gallons per day) (sum of all venturis at exit)
3. Turbidity (NTU)
4. Residual Chlorine (total chlorine)

##The U.S. Environmental Protection Agency establishes a maximum residual disinfectant level (MRDL) of 4.0 mg/L for chlorine and chloramines in public drinking-water systems. In Florida, Rule 62-555.350(6) of the Florida Administrative Code requires water suppliers to maintain free-chlorine residuals between 0.2 and 4.0 mg/L, or combined-chlorine residuals between 0.6 and 4.0 mg/L, throughout the drinking-water distribution system. In this study, the 4.0 mg/L value was used as the upper threshold for identifying and predicting upper-threshold events in the hourly Alexander Orr data. However, an individual treatment-plant SCADA observation at or above this threshold was not independently interpreted as proof that the water was unsafe or that regulatory noncompliance had occurred, because compliance depends on the applicable monitoring location, reporting requirements, and regulatory calculation procedures.

In [ ]:
# Notebook setup
# These installation commands work in notebook environments such as Google Colab.
# Pinning versions helps make the environment more reproducible.

!apt-get install -y graphviz

%pip install -q \
    pymannkendall==1.4.3 \
    pydot==2.0.0 \
    tft-torch==0.0.6 \
    omegaconf==2.3.0


# Standard-library utilities
import copy
import calendar
import gc
import glob
import hashlib
import json
import os
import re
import warnings

from dataclasses import dataclass
from itertools import combinations, product
from math import pi
from pathlib import Path
from typing import Dict, List, Optional, Tuple


# CUDA debugging
# Forces CUDA operations to run synchronously, which helps identify the operation
# responsible for a GPU error. This can slow training and should normally be
# disabled or removed after debugging.
#
# It must be configured before importing TensorFlow or PyTorch.
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


# Data manipulation
import numpy as np
import pandas as pd


# Static and interactive visualization
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import seaborn as sns

# Plotly creates interactive charts with features such as zooming and hovering.
import plotly.graph_objects as go

# Graphviz can be used to create and render model-architecture diagrams.
from graphviz import Digraph


# Statistical functions and hypothesis tests
from scipy.stats import (
    friedmanchisquare,
    jarque_bera,
    norm,
    wilcoxon,
)


# Time-series decomposition and visualization
# Seasonal decomposition separates a series into trend, seasonal, and residual
# components. The remaining functions help visualize seasonal patterns and
# autocorrelation.
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import (
    month_plot,
    plot_acf,
    plot_pacf,
    quarter_plot,
)


# Exponential-smoothing models
# Simple exponential smoothing models a series without trend or seasonality.
# ExponentialSmoothing can include trend and seasonal components.
from statsmodels.tsa.holtwinters import (
    ExponentialSmoothing,
    SimpleExpSmoothing,
)


# ARIMA and stationarity analysis
# The Augmented Dickey-Fuller test evaluates whether a series is stationary.
from statsmodels.tsa.stattools import adfuller

# SARIMAX supports autoregressive, differencing, moving-average, seasonal, and
# optional exogenous-variable components.
from statsmodels.tsa.statespace.sarimax import SARIMAX

# The Mann-Kendall test detects monotonic trends without requiring normal data.
import pymannkendall as mk


# Statistical diagnostics
import statsmodels.api as sm

# These tests examine common model-residual problems:
# - White test: heteroscedasticity
# - ARCH test: conditional heteroscedasticity
# - Ljung-Box test: residual autocorrelation
from statsmodels.stats.diagnostic import (
    acorr_ljungbox,
    het_arch,
    het_white,
)

# A Q-Q plot visually compares a sample distribution with a theoretical one.
from statsmodels.graphics.gofplots import qqplot

# Adjusts p-values when several statistical hypotheses are tested together.
from statsmodels.stats.multitest import multipletests


# Regression evaluation metrics
# These metrics measure different aspects of continuous prediction error.
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
    root_mean_squared_error,
)


# Classification evaluation metrics
# These metrics support class-label, probability-ranking, and threshold analysis.
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)


# Cross-validation and parameter tuning
from sklearn.base import clone
from sklearn.model_selection import (
    ParameterGrid,
    TimeSeriesSplit,
    train_test_split,
)

# ParameterGrid generates combinations from a dictionary of parameter values.
# TimeSeriesSplit preserves temporal ordering during cross-validation.
#
# Be careful with train_test_split for time-series data. Unless shuffle=False is
# used with correctly ordered observations, it can introduce data leakage.


# Feature selection
# SequentialFeatureSelector adds or removes features according to model
# performance until the requested feature subset is obtained.
from sklearn.feature_selection import SequentialFeatureSelector


# Data preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, QuantileTransformer

# ColumnTransformer applies different preprocessing operations to different
# groups of columns. For example, numeric features can be standardized while
# categorical features are one-hot encoded.


# Machine-learning pipelines
# A pipeline ensures that preprocessing and modeling operations are applied in
# a consistent order during both training and prediction.
from sklearn.pipeline import Pipeline


# Classical classification models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier


# TensorFlow and Keras
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import Sequential, layers
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Input, LSTM
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import plot_model

# EarlyStopping can stop training when validation performance no longer
# improves, helping reduce unnecessary computation and overfitting.


# PyTorch and Temporal Fusion Transformer
import torch

from torch.utils.data import DataLoader, Dataset
from omegaconf import OmegaConf
from tft_torch.tft import TemporalFusionTransformer
import tft_torch.loss as tft_loss

# Dataset defines how individual observations are retrieved.
# DataLoader batches, shuffles, and loads observations during training.
# OmegaConf manages the structured configuration used by the TFT model.


# Model persistence
# Joblib is commonly used to save and reload fitted scikit-learn models,
# preprocessing objects, pipelines, and other Python artifacts.
import joblib


# Google Drive access
from google.colab import drive

# Mount Google Drive so the notebook can access persistent files.
# Colab may request authorization when this is first executed.
drive.mount("/content/drive")

# For saving configs
from datetime import datetime, timezone
import sklearn


In [ ]:
# Load the Alexander Orr dataset
# Reads the Excel workbook from Google Drive and stores its first worksheet
# in a pandas DataFrame named `alexanderorr`.
alexanderorr = pd.read_excel(
    "/content/drive/MyDrive/Alexander_Orr_EDA/alexanderorr.xlsx"
)

In [ ]:
# Prepare the datetime index
# Convert the date column to pandas datetime values so that observations can be
# sorted chronologically and calendar features can be extracted.
alexanderorr["date"] = pd.to_datetime(
    alexanderorr["date"],
    errors="raise",
)

# Sort observations from earliest to latest and replace the old row labels with
# a clean sequential index before assigning the date column as the new index.
alexanderorr = (
    alexanderorr
    .sort_values("date")
    .reset_index(drop=True)
    .set_index("date")
)


# Create calendar features
# These features are extracted before any later UTC conversion so they represent
# the calendar date and time recorded by the plant in Miami local time.
alexanderorr["year"] = alexanderorr.index.year
alexanderorr["month"] = alexanderorr.index.month
alexanderorr["day"] = alexanderorr.index.day

# Pandas numbers Monday as 0 and Sunday as 6.
alexanderorr["day_of_week"] = alexanderorr.index.dayofweek

# Extract the hour using the 24-hour clock, where midnight is 0 and 11 p.m. is 23.
alexanderorr["hour"] = alexanderorr.index.hour

# Optional human-readable weekday labels, such as Monday or Tuesday.
# These are easier to interpret but must usually be encoded before modeling.
# alexanderorr["weekday"] = alexanderorr.index.day_name()

In [ ]:
# Configure plot appearance
def setup_times_new_roman():
    """Apply consistent publication-style settings to Matplotlib plots."""

    # Collect the names of all fonts available in the current environment.
    available_fonts = {font.name for font in fm.fontManager.ttflist}

    # Use Times New Roman when installed; otherwise, use Matplotlib's commonly
    # available DejaVu Serif font to preserve a similar serif appearance.
    if "Times New Roman" in available_fonts:
        chosen_font = "Times New Roman"
        print("Using Times New Roman")
    else:
        chosen_font = "DejaVu Serif"
        print(f"Using DejaVu Serif")

    # Update Matplotlib's global settings. These values apply to plots created
    # after this function runs unless an individual plot overrides them.
    plt.rcParams.update(
        {
            # Typography
            "font.family": "serif",
            "font.serif": [chosen_font],
            "font.size": 20,
            "axes.titlesize": 24,
            "axes.labelsize": 18,
            "xtick.labelsize": 16,
            "ytick.labelsize": 16,
            "legend.fontsize": 16,

            # Figure dimensions and resolution
            "figure.figsize": (14, 6),
            "figure.dpi": 300,
            "savefig.dpi": 500,

            # Lines and markers
            "lines.linewidth": 2.5,
            "lines.markersize": 6,

            # Grid appearance
            "axes.grid": True,
            "grid.alpha": 0.3,

            # Automatically adjust spacing to reduce clipped labels and titles.
            "figure.autolayout": True,
        }
    )


# Apply the plotting configuration for the remainder of the notebook.
setup_times_new_roman()


##***Exploratory Data Analysis***

In [ ]:
# Inspect the dataset
# Display the first and last five observations to verify the dataset structure,
# chronological ordering, and date range.
print("First five observations:")
display(alexanderorr.head())

print("\nLast five observations:")
display(alexanderorr.tail())


# Assess missing values
# Count missing values separately for every column.
missing_values = alexanderorr.isna().sum()

print("\nMissing values by column:")
display(missing_values)

# Report the total number of missing cells across the entire DataFrame.
print(f"\nTotal missing values: {missing_values.sum():,}")


# Summarize dataset coverage
# Report the number of observations and derive the date range directly from the
# datetime index so the summary remains accurate if the dataset changes.
print(f"Number of observations: {len(alexanderorr):,}")
print(
    "Date range: "
    f"{alexanderorr.index.min():%A, %B %d, %Y} to "
    f"{alexanderorr.index.max():%A, %B %d, %Y}"
)

In [ ]:
# Check timestamp consistency
# infer_freq() returns a frequency code such as "h" when the entire datetime
# index follows one consistent interval. It returns None if any interval differs.
inferred_frequency = pd.infer_freq(alexanderorr.index)
print(f"Inferred frequency: {inferred_frequency}")


# Check duplicate timestamps
# Duplicate timestamps can cause ambiguous observations during resampling,
# feature engineering, and time-series model training.
duplicate_count = alexanderorr.index.duplicated().sum()
print(f"Duplicate timestamps: {duplicate_count:,}")


# Examine time intervals
# Calculate the elapsed time between every pair of consecutive observations.
time_differences = alexanderorr.index.to_series().diff().dropna()

print("\nMost common time intervals:")
display(time_differences.value_counts().head())


# Identify intervals that differ from the expected hourly frequency.
expected_interval = pd.Timedelta(hours=1)
unexpected_intervals = time_differences[
    time_differences != expected_interval
]

print(f"Unexpected intervals: {len(unexpected_intervals):,}")

# Display the first anomalous intervals and the timestamps at which they end.
if not unexpected_intervals.empty:
    print("\nFirst unexpected intervals:")
    display(unexpected_intervals.head(10))

In [ ]:
# Inspect daylight-saving time transitions

# Fall transition
# When daylight-saving time ends, the clock moves backward from 2 a.m. to
# 1 a.m. As a result, the local 1 a.m. timestamp occurs twice.
duplicate_mask = alexanderorr.index.duplicated(keep=False)
duplicate_timestamps = alexanderorr.index[duplicate_mask]

# Count only the additional occurrences beyond the first observation.
duplicate_count = alexanderorr.index.duplicated(keep="first").sum()

print(f"Repeated timestamp occurrences: {duplicate_count:,}")
print("All observations associated with repeated timestamps:")
display(alexanderorr.loc[duplicate_timestamps])


# Spring transition
# When daylight-saving time begins, the clock moves directly from approximately
# 1 a.m. to 3 a.m. Therefore, no local 2 a.m. observation exists.
time_differences = alexanderorr.index.to_series().diff()
spring_gap_endings = time_differences[
    time_differences == pd.Timedelta(hours=2)
]

print("\nTimestamps immediately after two-hour gaps:")
display(spring_gap_endings)

# Calculate the local timestamps absent from the sequence. The index of each
# result is the first recorded timestamp after the corresponding gap.
missing_local_times = spring_gap_endings.index - pd.Timedelta(hours=1)

print("\nMissing local timestamps:")
display(pd.Index(missing_local_times))

In [ ]:
# Plot daylight-saving time anomalies
def plot_dst_anomalies_by_year(
    data,
    value_col="totalchlorine",
    hours_before=12,
    hours_after=12,
):
    """
    Plot fall and spring daylight-saving time anomalies without modifying the
    supplied DataFrame.

    Fall plots show repeated local timestamps. Spring plots use separate line
    segments to make each missing local hour visibly apparent.
    """

    # Validate the inputs.
    if not isinstance(data.index, pd.DatetimeIndex):
        raise TypeError("The DataFrame index must be a pandas DatetimeIndex.")

    if value_col not in data.columns:
        raise KeyError(f"Column {value_col!r} was not found in the DataFrame.")

    if hours_before < 0 or hours_after < 0:
        raise ValueError("The plotting-window values cannot be negative.")

    # Protect the original DataFrame
    # All work is performed on an independent copy. This function never assigns
    # values to the supplied DataFrame and never uses an in-place operation.
    plot_data = data.copy(deep=True).sort_index()

    # Calculate the intervals between consecutive local timestamps.
    time_differences = plot_data.index.to_series().diff()

    # A zero-hour interval represents a repeated local timestamp during the
    # fall daylight-saving transition.
    fall_event_timestamps = (
        plot_data.index[
            time_differences.eq(pd.Timedelta(0))
        ]
        .unique()
        .sort_values()
    )

    # A two-hour interval represents a missing local hour during the spring
    # daylight-saving transition.
    spring_event_timestamps = (
        plot_data.index[
            time_differences.eq(pd.Timedelta(hours=2))
        ]
        .unique()
        .sort_values()
    )

    print("Repeated fall timestamps:")
    display(fall_event_timestamps)

    print("\nTimestamps immediately after spring gaps:")
    display(spring_event_timestamps)


    # Plot fall transitions
    for event_time in fall_event_timestamps:
        window_start = event_time - pd.Timedelta(hours=hours_before)
        window_end = event_time + pd.Timedelta(hours=hours_after)

        # Create a temporary copy for the current plot.
        plot_subset = plot_data.loc[window_start:window_end].copy(deep=True)

        # Select both observations associated with every repeated timestamp.
        duplicate_mask = plot_subset.index.duplicated(keep=False)

        fig, ax = plt.subplots(figsize=(14, 5))

        ax.plot(
            plot_subset.index,
            plot_subset[value_col],
            marker="o",
            color="tab:blue",
            label="Residual chlorine",
        )

        ax.scatter(
            plot_subset.index[duplicate_mask],
            plot_subset.loc[duplicate_mask, value_col],
            s=120,
            color="darkorange",
            alpha=0.8,
            zorder=3,
            label="Repeated timestamp observations",
        )

        # Shade and mark the repeated local hour.
        ax.axvspan(
            event_time - pd.Timedelta(minutes=30),
            event_time + pd.Timedelta(minutes=30),
            color="orange",
            alpha=0.25,
            label="Repeated-hour window",
        )

        ax.axvline(
            event_time,
            color="darkorange",
            linestyle="--",
            linewidth=2,
        )

        ax.set_title(
            f"Fall DST repeated-timestamp anomaly ({event_time.year})"
        )
        ax.set_xlabel(
            "Original Miami local timestamp before UTC conversion"
        )
        ax.set_ylabel("Residual chlorine (mg/L)")
        ax.tick_params(axis="x", rotation=45)
        ax.legend()

        fig.tight_layout()
        plt.show()


    # Plot spring transitions
    for event_time in spring_event_timestamps:
        window_start = event_time - pd.Timedelta(hours=hours_before)
        window_end = event_time + pd.Timedelta(hours=hours_after)

        # Create a temporary copy for the current plot.
        plot_subset = plot_data.loc[window_start:window_end].copy(deep=True)

        # Find the final recorded timestamp before the two-hour interval.
        observation_before_gap = plot_subset.index[
            plot_subset.index < event_time
        ].max()

        # Calculate the absent local timestamp.
        missing_time = observation_before_gap + pd.Timedelta(hours=1)

        # Split the temporary data into two independent plotting sections.
        # Because they are plotted separately, Matplotlib cannot connect a line
        # across the missing local hour.
        observations_before_gap = plot_subset.loc[
            plot_subset.index <= observation_before_gap
        ].copy(deep=True)

        observations_after_gap = plot_subset.loc[
            plot_subset.index >= event_time
        ].copy(deep=True)

        fig, ax = plt.subplots(figsize=(14, 5))

        # Plot the observations before the missing hour.
        ax.plot(
            observations_before_gap.index,
            observations_before_gap[value_col],
            marker="o",
            color="tab:blue",
            label="Residual chlorine",
        )

        # Plot the observations after the missing hour as a separate line.
        ax.plot(
            observations_after_gap.index,
            observations_after_gap[value_col],
            marker="o",
            color="tab:blue",
        )

        # Highlight the final observation before the gap.
        ax.scatter(
            [observation_before_gap],
            [plot_data.loc[observation_before_gap, value_col]],
            s=120,
            color="darkorange",
            alpha=0.8,
            zorder=3,
            label="Last observation before gap",
        )

        # Highlight the first observation after the gap.
        ax.scatter(
            [event_time],
            [plot_data.loc[event_time, value_col]],
            s=120,
            color="green",
            alpha=0.8,
            zorder=3,
            label="First observation after gap",
        )

        # Shade and mark the local hour missing from the original data.
        ax.axvspan(
            missing_time - pd.Timedelta(minutes=30),
            missing_time + pd.Timedelta(minutes=30),
            color="red",
            alpha=0.2,
            label=f"Missing hour: {missing_time:%H:%M:%S}",
        )

        ax.axvline(
            missing_time,
            color="red",
            linestyle="--",
            linewidth=2,
        )

        ax.set_title(
            f"Spring DST missing-hour gap ({event_time.year})"
        )
        ax.set_xlabel(
            "Date (Miami Local Time)"
        )
        ax.set_ylabel("Residual chlorine (mg/L)")
        ax.tick_params(axis="x", rotation=45)
        ax.legend()

        fig.tight_layout()
        plt.show()


# Generate the plots
# The function receives `alexanderorr` but performs all work on internal copies.
plot_dst_anomalies_by_year(data=alexanderorr)

In [ ]:
# Preserve the Miami-local dataset
# This independent copy retains the original naive Miami clock timestamps for
# local-time inspection, DST plots, and calendar interpretation.
alexanderorr_miami = alexanderorr.copy(deep=True)


# Check daylight-saving time effects
print("Duplicate local timestamps:")
display(
    alexanderorr_miami.index[
        alexanderorr_miami.index.duplicated(keep=False)
    ]
)

local_time_differences = (
    alexanderorr_miami.index
    .to_series()
    .diff()
)

spring_gap_endings = alexanderorr_miami.index[
    local_time_differences.eq(pd.Timedelta(hours=2))
]

print("\nTimestamps immediately after two-hour gaps:")
display(spring_gap_endings)


# Convert the main dataset to UTC
# Assign the Miami timezone to the existing local clock readings.
alexanderorr.index = alexanderorr.index.tz_localize(
    "America/New_York",
    ambiguous="infer",
    nonexistent="raise",
)

# Convert the main DataFrame to a continuous UTC timeline.
alexanderorr = alexanderorr.tz_convert("UTC")


# Add explicit UTC calendar features
# The original year, month, day, day_of_week, and hour columns remain
# Miami-local features for operational EDA. The utc_ columns below are
# created from the normalized UTC index for forecasting feature sets.
alexanderorr["utc_year"] = alexanderorr.index.year
alexanderorr["utc_month"] = alexanderorr.index.month
alexanderorr["utc_day"] = alexanderorr.index.day
alexanderorr["utc_day_of_week"] = alexanderorr.index.dayofweek
alexanderorr["utc_hour"] = alexanderorr.index.hour
alexanderorr["utc_month_year"] = alexanderorr.index.strftime("%Y-%m")
alexanderorr["utc_hour_sin"] = np.sin(2 * np.pi * alexanderorr["utc_hour"] / 24.0)
alexanderorr["utc_hour_cos"] = np.cos(2 * np.pi * alexanderorr["utc_hour"] / 24.0)
alexanderorr["utc_dow_sin"] = np.sin(2 * np.pi * alexanderorr["utc_day_of_week"] / 7.0)
alexanderorr["utc_dow_cos"] = np.cos(2 * np.pi * alexanderorr["utc_day_of_week"] / 7.0)


# Verify the UTC timeline
utc_time_differences = (
    alexanderorr.index
    .to_series()
    .diff()
    .dropna()
)

print("\nAfter timezone handling:")
print(f"Inferred frequency: {pd.infer_freq(alexanderorr.index)}")
print(
    "Duplicate UTC timestamps: "
    f"{alexanderorr.index.duplicated().sum():,}"
)
print(f"Number of observations: {len(alexanderorr):,}")

print("\nMost common UTC intervals:")
display(utc_time_differences.value_counts().head())


# Validate index integrity
assert alexanderorr.index.tz is not None, (
    "The index is not timezone-aware."
)

assert str(alexanderorr.index.tz) == "UTC", (
    "The index was not converted to UTC."
)

assert alexanderorr.index.is_monotonic_increasing, (
    "The UTC index is not chronologically ordered."
)

assert not alexanderorr.index.has_duplicates, (
    "The UTC index contains duplicate timestamps."
)

assert utc_time_differences.eq(pd.Timedelta(hours=1)).all(), (
    "The UTC index is not continuously hourly after DST normalization."
)

print("\nAll UTC index integrity checks passed.")

In [ ]:
# Confirm data and timestamp integrity
# Verify that the UTC conversion did not introduce missing values.
missing_values = alexanderorr.isna().sum()

print("Missing values by column:")
display(missing_values)

print(f"\nTotal missing values: {missing_values.sum():,}")


# Confirm that the UTC index is unique, ordered, and continuously hourly.
utc_time_differences = (
    alexanderorr.index
    .to_series()
    .diff()
    .dropna()
)

print(f"Duplicate UTC timestamps: {alexanderorr.index.duplicated().sum():,}")
print(
    "Non-hourly intervals: "
    f"{utc_time_differences.ne(pd.Timedelta(hours=1)).sum():,}"
)
print(f"Chronologically ordered: {alexanderorr.index.is_monotonic_increasing}")


# Display the first five observations of the UTC-indexed dataset.
print("\nFirst five UTC observations:")
display(alexanderorr.head())

Operational Changes in Alexander Orr:
1. 2022: November 8 to 21

Reference: https://doralfamilyjournal.com/from-next-week-drinking-water-in-miami-dade-will-temporarily-change/

2. 2023: October 16 to 29

Reference: https://www.miamidade.gov/global/news-item.page?Mduid_news=news1632435695536608

3. 2024: September 9 to 22

Reference: https://www.miamidade.gov/global/news-item.page?Mduid_news=news166724138872941

4. 2025: October 13 to 26

Reference: https://www.miamidade.gov/global/news-item.page?Mduid_news=news1753893105003591

About chlorine conversion:
The chlorine conversion is an annual event that is scheduled each year in partnership with the Florida Department of Health in Miami-Dade County and the Miami-Dade Department of Regulatory and Economic Resources (RER), when WASD temporarily changes the method used to chlorinate the drinking water supply at its water treatment plants. Specifically, free chlorine, instead of the standard combined chlorine (chloramine), will be used during the treatment process. Free chlorine is considered an effective method of cleansing water distribution systems.


In [ ]:
# Add the chlorine-conversion indicator
# The published operating periods refer to Miami calendar dates, while the
# DataFrame uses a UTC index. Each local boundary must therefore be converted
# to UTC before it is compared with the DataFrame index.
LOCAL_TIMEZONE = "America/New_York"

conversion_periods_local = [
    ("2022-11-08", "2022-11-21"),
    ("2023-10-16", "2023-10-29"),
    ("2024-09-09", "2024-09-22"),
    ("2025-10-13", "2025-10-26"),
]


# Validate the modeling index
# These checks prevent accidental comparisons between UTC observations and
# naive or differently localized timestamps.
if not isinstance(alexanderorr.index, pd.DatetimeIndex):
    raise TypeError("The alexanderorr index must be a DatetimeIndex.")

if str(alexanderorr.index.tz) != "UTC":
    raise ValueError("The alexanderorr index must use the UTC timezone.")


# Begin with every observation classified as outside a conversion period.
conversion_mask = np.zeros(len(alexanderorr), dtype=bool)

for start_date, end_date in conversion_periods_local:
    # Interpret the published start date as midnight in Miami.
    start_local = pd.Timestamp(start_date, tz=LOCAL_TIMEZONE)

    # Published end dates are inclusive. Adding one day creates an exclusive
    # boundary at midnight immediately following the final local calendar day.
    end_exclusive_local = (
        pd.Timestamp(end_date, tz=LOCAL_TIMEZONE)
        + pd.Timedelta(days=1)
    )

    # Convert both Miami-local boundaries to UTC. This automatically applies
    # the correct daylight-saving offset for the corresponding dates.
    start_utc = start_local.tz_convert("UTC")
    end_exclusive_utc = end_exclusive_local.tz_convert("UTC")

    # Combine this period with all previously identified conversion periods.
    conversion_mask |= (
        (alexanderorr.index >= start_utc)
        & (alexanderorr.index < end_exclusive_utc)
    )


# Store the indicator as 0 or 1. The int8 type is sufficient for a binary
# feature and uses less memory than the default int64 type.
alexanderorr["chlorine_conversion"] = conversion_mask.astype("int8")


# Validate the new feature
print("Chlorine-conversion indicator counts:")
display(
    alexanderorr["chlorine_conversion"]
    .value_counts()
    .sort_index()
    .rename_axis("chlorine_conversion")
    .rename("observations")
)

assert alexanderorr["chlorine_conversion"].isin([0, 1]).all(), (
    "The chlorine-conversion indicator contains values other than 0 and 1."
)

print("\nChlorine-conversion indicator created successfully.")


In [ ]:
# Validate the chlorine-conversion indicator
# Select conversion-period observations without modifying the original data.
flagged = alexanderorr.loc[
    alexanderorr["chlorine_conversion"].eq(1)
].copy(deep=True)

# Convert only the temporary index to Miami time for calendar-date validation.
flagged_local_index = flagged.index.tz_convert(LOCAL_TIMEZONE)

# Create a summary of the observed local dates for each conversion year.
flagged_summary_data = pd.DataFrame(
    {
        "local_year": flagged_local_index.year,
        "local_date": flagged_local_index.date,
    },
    index=flagged.index,
)

conversion_summary = (
    flagged_summary_data
    .groupby("local_year")["local_date"]
    .agg(
        first_flagged_date="min",
        last_flagged_date="max",
        flagged_observations="count",
    )
)

print("Observed conversion periods in Miami local time:")
display(conversion_summary)


# Determine the Miami-local calendar coverage of the complete dataset.
dataset_local_index = alexanderorr.index.tz_convert(LOCAL_TIMEZONE)
dataset_first_date = pd.Timestamp(dataset_local_index.min().date())
dataset_last_date = pd.Timestamp(dataset_local_index.max().date())


# Compare each observed period with the portion of the published period that
# overlaps the available dataset.
for start_date, end_date in conversion_periods_local:
    published_start = pd.Timestamp(start_date)
    published_end = pd.Timestamp(end_date)
    period_year = published_start.year

    expected_start = max(published_start, dataset_first_date)
    expected_end = min(published_end, dataset_last_date)

    # Skip a published period if it does not overlap the available dataset.
    if expected_start > expected_end:
        continue

    assert period_year in conversion_summary.index, (
        f"No flagged observations were found for {period_year}."
    )

    observed_start = pd.Timestamp(
        conversion_summary.loc[period_year, "first_flagged_date"]
    )
    observed_end = pd.Timestamp(
        conversion_summary.loc[period_year, "last_flagged_date"]
    )

    assert observed_start == expected_start, (
        f"{period_year} begins on {observed_start.date()}, "
        f"but {expected_start.date()} was expected."
    )

    assert observed_end == expected_end, (
        f"{period_year} ends on {observed_end.date()}, "
        f"but {expected_end.date()} was expected."
    )

print("\nAll observed conversion periods match the available published dates.")


In [ ]:
# Visualize chlorine-conversion periods
# The published dates are defined using Miami calendar time, while the plotted
# dataset uses UTC. Each shading boundary is therefore converted to UTC.

fig, ax = plt.subplots(figsize=(14, 6))


# Plot residual chlorine
ax.plot(
    alexanderorr.index,
    alexanderorr["totalchlorine"],
    color="tab:blue",
    label="Normal Operation",
)


# Shade chlorine-conversion periods
for period_number, (start_date, end_date) in enumerate(
    conversion_periods_local
):
    # Treat the published start date as local midnight in Miami.
    start_local = pd.Timestamp(
        start_date,
        tz=LOCAL_TIMEZONE,
    )

    # Published end dates are inclusive. The following local midnight provides
    # the exclusive end boundary for the shaded interval.
    end_exclusive_local = (
        pd.Timestamp(end_date, tz=LOCAL_TIMEZONE)
        + pd.Timedelta(days=1)
    )

    # Convert the local boundaries to UTC so they align correctly with the
    # timezone-aware UTC index on the x-axis.
    start_utc = start_local.tz_convert("UTC")
    end_exclusive_utc = end_exclusive_local.tz_convert("UTC")

    ax.axvspan(
        start_utc,
        end_exclusive_utc,
        color="orange",
        alpha=0.3,
        label=(
            "Free chlorine conversion"
            if period_number == 0
            else None
        ),
    )


# Configure the chart
ax.set_title(
    "Residual Chlorine (mg/L)"
)
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("Residual chlorine (mg/L)")
ax.tick_params(axis="x", rotation=45)

# Place the legend inside the chart using a stable axes-relative position.
ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 0.03),
    frameon=True,
)

fig.tight_layout()
plt.show()

In [ ]:
# Visualize each chlorine-conversion period
# The dictionary supports convenient year-specific plots. These published dates
# describe Miami-local calendar periods, not UTC calendar periods.
conversion_periods_by_year = {
    2022: ("2022-11-08", "2022-11-21"),
    2023: ("2023-10-16", "2023-10-29"),
    2024: ("2024-09-09", "2024-09-22"),
    2025: ("2025-10-13", "2025-10-26"),
}

LOCAL_TIMEZONE = "America/New_York"


# Validate the plotting index
if str(alexanderorr.index.tz) != "UTC":
    raise ValueError("The alexanderorr index must use the UTC timezone.")


# Generate one focused chart for each conversion period.
for year, (start_date, end_date) in conversion_periods_by_year.items():
    # Interpret the published boundaries as Miami-local midnight.
    start_local = pd.Timestamp(
        start_date,
        tz=LOCAL_TIMEZONE,
    )

    # The published end date is inclusive, so the exclusive boundary is
    # midnight at the beginning of the following local calendar day.
    end_exclusive_local = (
        pd.Timestamp(end_date, tz=LOCAL_TIMEZONE)
        + pd.Timedelta(days=1)
    )

    # Convert the local boundaries to UTC so they align with the DataFrame index.
    start_utc = start_local.tz_convert("UTC")
    end_exclusive_utc = end_exclusive_local.tz_convert("UTC")

    # Include three weeks before and after the conversion period to provide
    # context for changes in residual chlorine.
    window_start_utc = start_utc - pd.Timedelta(days=21)
    window_end_utc = end_exclusive_utc + pd.Timedelta(days=21)

    # Create a temporary plotting copy. This does not modify `alexanderorr`.
    plot_subset = alexanderorr.loc[
        window_start_utc:window_end_utc
    ].copy(deep=True)

    # Skip the figure if the requested period does not overlap the dataset.
    if plot_subset.empty:
        print(f"No observations are available around the {year} period.")
        continue

    fig, ax = plt.subplots(figsize=(12, 5))

    # Plot the residual-chlorine measurements surrounding the period.
    ax.plot(
        plot_subset.index,
        plot_subset["totalchlorine"],
        color="tab:blue",
        label="Residual chlorine",
    )

    # Shade the complete published Miami-local conversion period after its
    # boundaries have been mapped correctly to UTC.
    ax.axvspan(
        start_utc,
        end_exclusive_utc,
        color="orange",
        alpha=0.3,
        label="Free chlorine conversion",
    )

    ax.set_title(
        f"Residual Chlorine (mg/L) Around the {year} Period"

    )
    ax.set_xlabel("Date (UTC)")
    ax.set_ylabel("Residual chlorine (mg/L)")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()

    fig.tight_layout()
    plt.show()

##Central Tendency and Distribution

In [ ]:
# Summarize residual-chlorine distribution
# describe() reports the nonmissing count, mean, standard deviation, minimum,
# quartiles, and maximum.
print("Residual chlorine (mg/L) descriptive statistics:")
display(alexanderorr["totalchlorine"].describe())


# Calculate the mode
# A dataset can have multiple modes, so pandas returns the result as a Series.
mode1 = alexanderorr["totalchlorine"].mode()

print("\nMode value or values:")
display(mode1)


# Calculate distribution-shape metrics
# Positive skewness indicates a longer right tail, while negative skewness
# indicates a longer left tail.
skew1 = alexanderorr["totalchlorine"].skew()

# Pandas reports excess kurtosis, where zero is comparable to the kurtosis of a
# normal distribution.
kurt1 = alexanderorr["totalchlorine"].kurt()

print(f"\nSkewness: {skew1:.4f}")
print(f"Excess kurtosis: {kurt1:.4f}")

In [ ]:
# Summarize turbidity distribution
# describe() reports the nonmissing count, mean, standard deviation, minimum,
# quartiles, and maximum.
print("Turbidity descriptive statistics:")
display(alexanderorr["turbidity"].describe())


# Calculate the mode
# A dataset can have multiple modes, so pandas returns the result as a Series.
mode3 = alexanderorr["turbidity"].mode()

print("\nMode value or values:")
display(mode3)


# Calculate distribution-shape metrics
# Positive skewness indicates a longer right tail, while negative skewness
# indicates a longer left tail.
skew3 = alexanderorr["turbidity"].skew()

# Pandas reports excess kurtosis, where zero is comparable to the kurtosis of a
# normal distribution.
kurt3 = alexanderorr["turbidity"].kurt()

print(f"\nSkewness: {skew3:.4f}")
print(f"Excess kurtosis: {kurt3:.4f}")

In [ ]:
# Summarize finished-water flow-rate distribution
# describe() reports the nonmissing count, mean, standard deviation, minimum,
# quartiles, and maximum.
print("Finished-water flow-rate descriptive statistics:")
display(alexanderorr["finishwater"].describe())


# Calculate the mode
# A dataset can have multiple modes, so pandas returns the result as a Series.
mode2 = alexanderorr["finishwater"].mode()

print("\nMode value or values:")
display(mode2)


# Calculate distribution-shape metrics
# Positive skewness indicates a longer right tail, while negative skewness
# indicates a longer left tail.
skew2 = alexanderorr["finishwater"].skew()

# Pandas reports excess kurtosis, where zero is comparable to the kurtosis of a
# normal distribution.
kurt2 = alexanderorr["finishwater"].kurt()

print(f"\nSkewness: {skew2:.4f}")
print(f"Excess kurtosis: {kurt2:.4f}")

##Univariate Visualizations: Line Graph

In [ ]:
# Plot residual chlorine over time
# The UTC index provides a continuous hourly timeline without duplicated or
# missing daylight-saving clock hours.
fig, ax = plt.subplots(figsize=(15, 10))

ax.plot(
    alexanderorr.index,
    alexanderorr["totalchlorine"],
    color="tab:blue",
    linewidth=2,
    label="Residual chlorine",
)

ax.set_title("Residual Chlorine Over Time (mg/L)")
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("Residual chlorine (mg/L)")
ax.grid(alpha=0.3)
ax.tick_params(axis="x", rotation=45)
ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
# Plot turbidity over time
# The UTC index provides a continuous hourly timeline without daylight-saving
# time duplicates or missing clock hours.
fig, ax = plt.subplots(figsize=(15, 10))

ax.plot(
    alexanderorr.index,
    alexanderorr["turbidity"],
    color="tab:blue",
    linewidth=2,
    label="Turbidity",
)

ax.set_title("Turbidity (NTU)")
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("Turbidity (NTU)")
ax.grid(alpha=0.3)
ax.tick_params(axis="x", rotation=45)
ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
# Plot finished-water flow rate over time
# The UTC index provides a continuous hourly timeline without daylight-saving
# time duplicates or missing clock hours.
fig, ax = plt.subplots(figsize=(15, 10))

ax.plot(
    alexanderorr.index,
    alexanderorr["finishwater"],
    color="tab:blue",
    linewidth=2,
    label="Flow rate",
)

ax.set_title("Flow Rate (mgd)")
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("Flow Rate (mgd)")
ax.grid(alpha=0.3)
ax.tick_params(axis="x", rotation=45)
ax.legend()

fig.tight_layout()
plt.show()

##Interactive Plots

In [ ]:
# Function definition
def interactive_timeseries_plot(
    df,
    y_col,
    title,
    yaxis_title,
    xaxis_title="Date (UTC)",
    width=1400,
    height=650,
    font_family="Times New Roman",
    base_font_size=32,
    title_font_size=36,
    axis_title_font_size=32,
    tick_font_size=32,
    legend_font_size=26,
    line_width=3,
    show_rangeslider=True
):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df.index,
        y=df[y_col],
        mode='lines',
        name=yaxis_title,
        line=dict(width=line_width)
    ))

    fig.update_layout(
        width=width,
        height=height,

        # Global font
        font=dict(
            family=font_family,
            size=base_font_size
        ),

        title=dict(
            text=title,
            x=0.5,
            xanchor='center',
            font=dict(size=title_font_size, family=font_family)
        ),

        xaxis_title=dict(
            text=xaxis_title,
            font=dict(size=axis_title_font_size, family=font_family)
        ),
        yaxis_title=dict(
            text=yaxis_title,
            font=dict(size=axis_title_font_size, family=font_family)
        ),

        xaxis=dict(
            tickfont=dict(size=tick_font_size, family=font_family),
            rangeslider=dict(visible=show_rangeslider)
        ),
        yaxis=dict(
            tickfont=dict(size=tick_font_size, family=font_family)
        ),

        legend=dict(
            font=dict(size=legend_font_size, family=font_family)
        ),

        hovermode='x unified',
        template='plotly_white'
    )

    return fig

In [ ]:
# Residual Chlorine
fig = interactive_timeseries_plot(
    df=alexanderorr,
    y_col='totalchlorine',
    title='Residual Chlorine (mg/L)',
    yaxis_title='Residual Chlorine (mg/L)'
)
fig.show()

In [ ]:
# Turbidity
fig = interactive_timeseries_plot(
    df=alexanderorr,
    y_col='turbidity',
    title='Turbidity (NTU)',
    yaxis_title='Turbidity (NTU)'
)
fig.show()

In [ ]:
# Flowrate
fig = interactive_timeseries_plot(
    df=alexanderorr,
    y_col='finishwater',
    title='Flowrate (mgd)',
    yaxis_title='Flowrate (mgd)'
)
fig.show()

##Monthly Trends

In [ ]:
# Calculate monthly average residual chlorine
# The year and month columns were extracted before UTC conversion, so they
# continue to represent Miami-local calendar periods.
monthly_avg = (
    alexanderorr
    .groupby(["year", "month"])["totalchlorine"]
    .mean()
    .reset_index()
)

# Create a representative timestamp using the first day of each local calendar
# month. This timestamp is used only to position the monthly averages.
monthly_avg["Sample Date"] = pd.to_datetime(
    monthly_avg[["year", "month"]].assign(day=1)
)


# Plot monthly average residual chlorine
plt.figure(figsize=(10, 5))

plt.plot(
    monthly_avg["Sample Date"],
    monthly_avg["totalchlorine"],
    marker="o"
)

plt.title(
    "Monthly Average Residual Chlorine (mg/L)"
)
plt.ylabel("Average Residual Chlorine (mg/L)")

# Move the y-axis label slightly left and below its default center position.
plt.gca().yaxis.set_label_coords(-0.10, 0.35)

plt.xlabel("Date (Miami Local Time)")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)

# Add left-side space for the repositioned y-axis label.
plt.subplots_adjust(left=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate monthly average turbidity
# The year and month columns were extracted before UTC conversion, so they
# continue to represent Miami-local calendar periods.
monthly_avg = (
    alexanderorr
    .groupby(["year", "month"])["turbidity"]
    .mean()
    .reset_index()
)

# Create a representative timestamp using the first day of each local calendar
# month. This timestamp is used only to position the monthly averages.
monthly_avg["Sample Date"] = pd.to_datetime(
    monthly_avg[["year", "month"]].assign(day=1)
)


# Plot monthly average turbidity
plt.figure(figsize=(10, 5))

plt.plot(
    monthly_avg["Sample Date"],
    monthly_avg["turbidity"],
    marker="o"
)

plt.title("Monthly Average Turbidity (NTU)")
plt.ylabel("Average Turbidity (NTU)")

# Move the y-axis label slightly left and below its default center position.
plt.gca().yaxis.set_label_coords(-0.10, 0.35)

plt.xlabel("Date (Miami Local Time)")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)

# Add left-side space for the repositioned y-axis label.
plt.subplots_adjust(left=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate monthly average finished-water flow rate
# The year and month columns were extracted before UTC conversion, so they
# continue to represent Miami-local calendar periods.
monthly_avg = (
    alexanderorr
    .groupby(["year", "month"])["finishwater"]
    .mean()
    .reset_index()
)

# Create a representative timestamp using the first day of each local calendar
# month. This timestamp is used only to position the monthly averages.
monthly_avg["Sample Date"] = pd.to_datetime(
    monthly_avg[["year", "month"]].assign(day=1)
)


# Plot monthly average finished-water flow rate
plt.figure(figsize=(10, 5))

plt.plot(
    monthly_avg["Sample Date"],
    monthly_avg["finishwater"],
    marker="o"
)

plt.title("Monthly Average Flow Rate (mgd)")
plt.ylabel("Average Flow Rate (mdg)")

# Move the y-axis label slightly left and below its default center position.
plt.gca().yaxis.set_label_coords(-0.10, 0.35)

plt.xlabel("Date (Miami Local Time)")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)

# Add left-side space for the repositioned y-axis label.
plt.subplots_adjust(left=0.25)
plt.tight_layout()
plt.show()


##Distribution Information-Histograms

In [ ]:
# Residual Chlorine
plt.figure(figsize=(8,5))
sns.histplot(alexanderorr['totalchlorine'], bins=100, kde=True, color='teal')
plt.title('Residual Chlorine (mg/L)')
plt.xlabel('Residual Chlorine (mg/L)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Turbidity
plt.figure(figsize=(8,5))
sns.histplot(alexanderorr['turbidity'], bins=100, kde=True, color='teal')
plt.title('Turbidity (NTU)')
plt.xlabel('Turbidity (NTU)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Flowrate
plt.figure(figsize=(8,5))
sns.histplot(alexanderorr['finishwater'], bins=100, kde=True, color='teal')
plt.title('Flowrate (mgd)')
plt.xlabel('Flowrate (mgd)')
plt.ylabel('Frequency')
plt.show()

##Distribution Information-Boxplots

In [ ]:
# Residual Chlorine
plt.figure(figsize=(10,5))
sns.boxplot(x='month', y='totalchlorine', data=alexanderorr)
plt.title('Monthly Variation in Residual Chlorine (mg/L)')
plt.xlabel('Month (Miami Local Time)')
plt.ylabel('Residual Chlorine (mg/L)')
plt.show()

In [ ]:
# Turbidity
plt.figure(figsize=(10,5))
sns.boxplot(x='month', y='turbidity', data=alexanderorr)
plt.title('Monthly Variation in Turbidity (NTU)')
plt.xlabel('Month (Miami Local Time)')
plt.ylabel('Turbidity (NTU)')
plt.show()

In [ ]:
# Turbidity (monthly variation supressing outliers)
plt.figure(figsize=(10,5))
sns.boxplot(x='month', y='turbidity', data=alexanderorr, showfliers=False)

plt.title('Monthly Variation in Turbidity (Outliers Suppressed)')
plt.xlabel('Month (Miami Locla Time)')
plt.ylabel('Turbidity (NTU)')
plt.show()

In [ ]:
# Flowrate
plt.figure(figsize=(10,5))
sns.boxplot(x='month', y='finishwater', data=alexanderorr)
plt.title('Monthly Variation in Flowrate (mgd)')
plt.xlabel('Month (Miami Local Time)')
plt.ylabel('Flowrate (mgd)')
plt.show()

In [ ]:
# Monthly residual-chlorine trends
# Create year-month labels directly from the UTC index.
alexanderorr["month_year"] = alexanderorr.index.strftime("%Y-%m")

plt.figure(figsize=(14, 6))
sns.boxplot(x="month_year", y="totalchlorine", data=alexanderorr)
plt.title("Distribution by Month of Residual Chlorine (mg/L)")
plt.xlabel("Date (Miami Local Time)")
plt.ylabel("Residual Chlorine (mg/L)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Monthly turbidity trends
# Create year-month labels directly from the UTC index.
alexanderorr["month_year"] = alexanderorr.index.strftime("%Y-%m")

plt.figure(figsize=(14, 6))
sns.boxplot(x="month_year", y="turbidity", data=alexanderorr)
plt.title("Distribution by Month of Turbidity (NTU)")
plt.xlabel("Date (Miami Local Time)")
plt.ylabel("Turbidity (NTU)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Monthly turbidity trends with outlier markers hidden
alexanderorr["month_year"] = alexanderorr.index.strftime("%Y-%m")

plt.figure(figsize=(14, 6))
sns.boxplot(
    x="month_year",
    y="turbidity",
    data=alexanderorr,
    showfliers=False
)
plt.title("Distribution by Month of Turbidity (NTU)")
plt.xlabel("Date (Miami Local Time)")
plt.ylabel("Turbidity (NTU)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Monthly flow-rate trends
# Create year-month labels directly from the UTC index.
alexanderorr["month_year"] = alexanderorr.index.strftime("%Y-%m")

plt.figure(figsize=(14, 6))
sns.boxplot(x="month_year", y="finishwater", data=alexanderorr)
plt.title("Distribution by Month of Flow Rate (mgd)")
plt.xlabel("Date (Miami Local Time)")
plt.ylabel("Flow Rate (mgd)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

##Calendar Heatmaps

Grouping all months together

In [ ]:
# Definition for heatmap function
def plot_heatmap(df, value_col, title):
    pivot = df.pivot_table(
        values=value_col,
        index=df.index.hour,
        columns=df.index.month,
        aggfunc='mean'
    )

    plt.figure(figsize=(10, 6))
    sns.heatmap(pivot, cmap='coolwarm', annot=False)
    plt.title(title)
    plt.xlabel('Month (UTC)')
    plt.ylabel('Hour of Day (UTC)')
    plt.tight_layout()
    plt.show()


#Heatmap plots
plot_heatmap(alexanderorr, 'totalchlorine', 'Residual Chlorine (mg/L) Heatmap')
plot_heatmap(alexanderorr, 'turbidity', 'Turbidity (NTU) Heatmap')
plot_heatmap(alexanderorr, 'finishwater', 'Flowrate (mgd) Heatmap')

Visualizing data for each year and each month independently

In [ ]:
#Definition for calendar function

def calendar_heatmap(df, value_col, year, aggfunc='mean', cmap='coolwarm', ylabel=None):
    """Calendar-style heatmap for one variable in one year (expects DatetimeIndex)."""

    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("calendar_heatmap expects df.index to be a DatetimeIndex. Set df = df.set_index('date') first.")

    # Filter to selected year
    df_year = df.loc[df.index.year == year, [value_col]].copy()

    if df_year.empty:
        print(f"No data available for {value_col} in {year}")
        return

    # ISO week and day-of-week from index
    iso_week = df_year.index.isocalendar().week.astype(int)
    dow = df_year.index.dayofweek  # Mon=0 ... Sun=6

    pivot = pd.pivot_table(
        df_year,
        values=value_col,
        index=dow,
        columns=iso_week,
        aggfunc=aggfunc
    )

    plt.figure(figsize=(16, 4))
    sns.heatmap(
        pivot,
        cmap=cmap,
        cbar_kws={'label': ylabel if ylabel else value_col}
    )
    plt.yticks(
        np.arange(7) + 0.5,
        ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
        rotation=0
    )
    plt.title(f'Calendar Heatmap of {ylabel if ylabel else value_col} ({year})')
    plt.xlabel('ISO Week of Year (UTC)')
    plt.ylabel('Day of Week (UTC)')
    plt.tight_layout()
    plt.show()

variables = {
    'totalchlorine': 'Residual Chlorine (mg/L)',
    'turbidity': 'Turbidity (NTU)',
    'finishwater': 'Flowrate (mgd)'
}

for col, label in variables.items():
    for y in sorted(alexanderorr.index.year.unique()):
        calendar_heatmap(alexanderorr, col, y, ylabel=label)


##Anomalies and Outliers

In [ ]:
# Residual Chlorine
alexanderorr['z_totalchlorine'] = (
    alexanderorr['totalchlorine'] - alexanderorr['totalchlorine'].mean()
) / alexanderorr['totalchlorine'].std()

threshold = 3

anomalies_rc = alexanderorr[
    np.abs(alexanderorr['z_totalchlorine']) > threshold
]

plt.figure(figsize=(12,6))
plt.plot(alexanderorr.index, alexanderorr['totalchlorine'], label='Residual Chlorine (mg/L)')
plt.scatter(anomalies_rc.index, anomalies_rc['totalchlorine'], color='red', label='Outliers', zorder=5)
plt.title('Outliers/Anomalies in Residual Chlorine (mg/L)')
plt.ylabel('Residual Chlorine (mg/L)')
plt.xlabel('Date (UTC)')
plt.xticks(rotation=45)
plt.legend()
plt.show()


In [ ]:
# Turbidity
alexanderorr['z_turbidity'] = (
    alexanderorr['turbidity'] - alexanderorr['turbidity'].mean()
) / alexanderorr['turbidity'].std()

anomalies_t = alexanderorr[
    np.abs(alexanderorr['z_turbidity']) > threshold
]

plt.figure(figsize=(12,6))
plt.plot(alexanderorr.index, alexanderorr['turbidity'], label='Turbidity (NTU)')
plt.scatter(anomalies_t.index, anomalies_t['turbidity'], color='red', label='Outliers', zorder=5)
plt.title('Outliers/Anomalies in Turbidity (NTU)')
plt.ylabel('Turbidity (NTU)')
plt.xlabel('Date (UTC)')
plt.xticks(rotation=45)
plt.legend()
plt.show()

In [ ]:
# Flowrate
alexanderorr['z_finishwater'] = (
    alexanderorr['finishwater'] - alexanderorr['finishwater'].mean()
) / alexanderorr['finishwater'].std()

anomalies_fr = alexanderorr[
    np.abs(alexanderorr['z_finishwater']) > threshold
]

plt.figure(figsize=(12,6))
plt.plot(alexanderorr.index, alexanderorr['finishwater'], label='Flowrate (mgd)')
plt.scatter(anomalies_fr.index, anomalies_fr['finishwater'], color='red', label='Outliers', zorder=5)
plt.title('Outliers/Anomalies in Flowrate (mgd)')
plt.ylabel('Flowrate (mgd)')
plt.xlabel('Date (UTC)')
plt.legend()
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Counting the amount of outliers per UTC calendar month

def flag_iqr_anomalies(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

alexanderorr['chlorine_anomaly'] = flag_iqr_anomalies(alexanderorr['totalchlorine'])
alexanderorr['turbidity_anomaly'] = flag_iqr_anomalies(alexanderorr['turbidity'])
alexanderorr['flowrate_anomaly'] = flag_iqr_anomalies(alexanderorr['finishwater'])


chlorine_monthly  = alexanderorr[alexanderorr['chlorine_anomaly']].resample('ME').size()
turbidity_monthly = alexanderorr[alexanderorr["turbidity_anomaly"]].resample('ME').size()
flowrate_monthly  = alexanderorr[alexanderorr['flowrate_anomaly']].resample('ME').size()

print(chlorine_monthly)
print(turbidity_monthly)
print(flowrate_monthly)

print('Chlorine anomalies per month:')
print(chlorine_monthly)

print('\nTurbidity anomalies per month:')
print(turbidity_monthly)

print('\nFlowrate anomalies per month:')
print(flowrate_monthly)

Plot of Outliers per Month

In [ ]:
# Plot of outliers per UTC calendar month
def plot_monthly_anomalies(monthly_series, title, ylabel='Number of Outliers'):
    fig, ax = plt.subplots(figsize=(12, 5))

    ax.bar(monthly_series.index, monthly_series.values, width=20)
    ax.set_title(title)
    ax.set_xlabel('Month (UTC)')
    plt.xticks(rotation=45)
    ax.set_ylabel(ylabel)

    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=60)

    plt.tight_layout()
    plt.show()

plot_monthly_anomalies(chlorine_monthly, 'Monthly Residual Chlorine (mg/L) Outliers')
plot_monthly_anomalies(turbidity_monthly, 'Monthly Turbidity (NTU) Outliers')
plot_monthly_anomalies(flowrate_monthly, 'Monthly Flowrate (mgd) Outliers')

##Residual Chlorine Anomalies According to Upper Threshold

In [ ]:
# Residual Chlorine US EPA Thresholds
lower_limit_rc = 0.2
upper_limit_rc = 4.0

# Flag out of range residual chlorine levels
alexanderorr['chlorine_out_of_range'] = (
    (alexanderorr['totalchlorine'] < lower_limit_rc) |
    (alexanderorr['totalchlorine'] > upper_limit_rc)
)

chlor_num_anomalies = int(alexanderorr['chlorine_out_of_range'].sum())

plt.figure(figsize=(12, 6))

# Line plot
plt.plot(
    alexanderorr.index,
    alexanderorr['totalchlorine'],
    color='tab:blue',
    label='Residual Chlorine (mg/L)'
)

# Scatter of flagged points
outliers = alexanderorr[alexanderorr['chlorine_out_of_range']]

plt.scatter(
    outliers.index,
    outliers['totalchlorine'],
    color='red',
    label='Out of Range',
    zorder=5
)

# Including threshold lines
plt.axhline(lower_limit_rc, color='blue', linestyle='--', label='Lower Limit (0.2 mg/L)')
plt.axhline(upper_limit_rc, color='orange', linestyle='--', label='Upper Limit (4.0 mg/L)')

plt.title('Residual Chlorine Concentrations Outside Established Thresholds')
plt.xlabel('Date (UTC)')
plt.xticks(rotation=45)
plt.ylabel('Residual Chlorine (mg/L)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(chlor_num_anomalies)




In [ ]:
# Turbidity limits
lower_limit_t = 0.0
upper_limit_t = 1.0

# Flag out of range residual chlorine levels
alexanderorr["turbidity_out_of_range"] = (
    (alexanderorr["turbidity"] < lower_limit_t) |
    (alexanderorr["turbidity"] > upper_limit_t)
)

turb_num_anomalies = int(alexanderorr["turbidity_out_of_range"].sum())

plt.figure(figsize=(12, 6))

# Line plot
plt.plot(
    alexanderorr.index,
    alexanderorr["turbidity"],
    color="tab:blue",
    label="Turbidity (NTU)"
)

# Scatter of flagged points
outliers = alexanderorr[alexanderorr["turbidity_out_of_range"]]

plt.scatter(
    outliers.index,
    outliers["turbidity"],
    color="red",
    label="Out of Range",
    zorder=5
)

# Including threshold lines
plt.axhline(lower_limit_t, color="blue", linestyle="--", label="Lower Limit (0.0 NTU)")
plt.axhline(upper_limit_t, color="orange", linestyle="--", label="Upper Limit (1.0 NTU)")

plt.title("Turbidity (NTU) Outside Regulatory Thresholds")
plt.xlabel("Date (UTC)")
plt.xticks(rotation=45)
plt.ylabel("Turbidity (NTU)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(turb_num_anomalies)


#171 values exceed the recommended limit

In [ ]:
# Count values of Residual Chlorine above 4.0 ppm limit per month since the start date of dataset (Time in UTC)

chlorine_num_anomalies = alexanderorr[alexanderorr['chlorine_out_of_range']]
monthly_counts = chlorine_num_anomalies.groupby(chlorine_num_anomalies.index.to_period('M')).size()

print(monthly_counts)

In [ ]:
# Plot of  values of Residual Chlorine above 4.0 mg/L limit per month since the start date of dataset

monthly_counts_dt = monthly_counts.copy()
monthly_counts_dt.index = monthly_counts_dt.index.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 6))

# Plot bars
bars = ax.bar(monthly_counts_dt.index, monthly_counts_dt.values)

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2,
        height + (0.02 * monthly_counts_dt.max()),
        str(int(height)),
        ha='center',
        va='bottom'
    )

# Increase y-axis upper limit (KEY FIX)
max_val = monthly_counts_dt.max()
ax.set_ylim(0, max_val * 1.2)

# X ticks
ax.set_xticks(monthly_counts_dt.index)
ax.set_xticklabels(
    [d.strftime('%Y-%m') for d in monthly_counts_dt.index],
    rotation=45,
    ha='right'
)

# Labels
ax.set_title('Monthly Count of Residual Chlorine Values Exceeding Upper Threshold')
ax.set_xlabel('Date (UTC)')
ax.set_ylabel('Number of Anomalies')

plt.tight_layout()
plt.show()

##Rolling Averages

In [ ]:
# Residual Chlorine andT Turbidity (Visualizing Joint Rolling Averages)
alexanderorr['totalchlorine'].rolling(window=24).mean().plot(figsize=(12,6), legend=True)
alexanderorr['turbidity'].rolling(window=24).mean().plot(secondary_y=True, figsize=(12,6), legend=True)
plt.title('Rolling Average of Residual Chlorine and Turbidity')
plt.xlabel('Date (UTC)')
plt.show()


In [ ]:
# Flowrate
alexanderorr['finishwater'].rolling(window=24).mean().plot(figsize=(12,6))
plt.title('Rolling Average of Flowrate')
plt.xlabel('Date (UTC)')
plt.ylabel('Flowrate (mgd)')

##Percent Changes

In [ ]:
# Residual chlorine
alexanderorr['chlorine_pct_change'] = (alexanderorr['totalchlorine'].pct_change()* 100)

plt.figure(figsize=(12,6))

plt.plot(
    alexanderorr.index,
    alexanderorr['chlorine_pct_change'],
    color='tab:blue'
)
plt.title('Percent Change in Residual Chlorine')
plt.xlabel('Date (UTC)')
plt.ylabel('Residual Chlorine (mg/L) % Change')


In [ ]:
# Turbidity
alexanderorr['turbidity_pct_change'] = (alexanderorr['turbidity'].pct_change()* 100)

plt.figure(figsize=(12,6))

plt.plot(
    alexanderorr.index,
    alexanderorr['turbidity_pct_change'],
    color='tab:blue'
)
plt.title('Percent Change in Turbidity')
plt.xlabel('Date (UTC)')
plt.ylabel('Turbidity (NTU)')


In [ ]:
# Flowrate
alexanderorr['flowrate_pct_change'] = (alexanderorr['finishwater'].pct_change()* 100)

plt.figure(figsize=(12,6))

plt.plot(
    alexanderorr.index,
    alexanderorr['flowrate_pct_change'],
    color='tab:blue'
)
plt.title('Percent Change in Flowrate (mgd)')
plt.xlabel('Date (UTC)')
plt.ylabel('Percent Change (%)')


In [ ]:
# Day to day Residual Chlorine percentage change
daily = alexanderorr['totalchlorine'].resample('D').mean()
daily_pct_change = daily.pct_change()*100

plt.figure(figsize=(12,6))
plt.plot(daily.index, daily_pct_change)
plt.title('Daily Percentage Change in Residual Chlorine (mg/L)')
plt.xlabel('Date (UTC)')
plt.ylabel('Percentage Change (%)')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Day to day Turbidity percentage change
daily = alexanderorr['turbidity'].resample('D').mean()
daily_pct_change = daily.pct_change()*100

plt.figure(figsize=(12,6))
plt.plot(daily.index, daily_pct_change)
plt.title('Daily Percentage Change in Turbidity (NTU)')
plt.xlabel('Date (UTC)')
plt.ylabel('Percentage Change (%)')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Day to day Flowrate percentage change
daily = alexanderorr['finishwater'].resample('D').mean()
daily_pct_change = daily.pct_change()*100

plt.figure(figsize=(12,6))
plt.plot(daily.index, daily_pct_change)
plt.title('Daily Percentage Change in Flowrate (mgd)')
plt.xlabel('Date (UTC)')
plt.ylabel('Percentage Change (%)')
plt.grid(alpha=0.3)
plt.show()

##Correlations

In [ ]:
# Spearman correlation
spearman_corr = alexanderorr[
    ['totalchlorine', 'turbidity', 'finishwater']
].corr(method='spearman')

print(spearman_corr)

# Correlation heatmap
plt.figure(figsize=(6,5))
sns.heatmap(
    spearman_corr,
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.2f'
)
plt.title('Spearman Correlation Matrix')
plt.tight_layout()
plt.show()

# Explore correlation methods that take into account time-dependency

In [ ]:
# Cross-correlation for time dependency

# Lagged correlation
def plot_cross_correlations(
    df,
    col_pairs,
    max_lag=168,
    figsize=(12, 5)
):

    for x_col, y_col in col_pairs:
        lags = list(range(0, max_lag + 1))
        corrs = []

        for lag in lags:
            x = df[x_col]
            y = df[y_col].shift(-lag)   # y at future time

            valid = pd.concat([x, y], axis=1).dropna()
            if len(valid) > 1:
                corr = valid.iloc[:, 0].corr(valid.iloc[:, 1])
            else:
                corr = np.nan

            corrs.append(corr)

        # Plot
        plt.figure(figsize=figsize)
        plt.plot(lags, corrs, marker='o', markersize=3)
        plt.axhline(0, linestyle='--')
        plt.title(f'Cross-Correlation: {x_col} (t) vs {y_col} (t + lag)')
        plt.xlabel('Lag')
        plt.ylabel('Correlation')
        plt.grid(alpha=0.3)
        plt.show()

        # Report strongest lag
        corrs_arr = np.array(corrs, dtype=float)
        if np.isfinite(corrs_arr).any():
            best_idx = np.nanargmax(np.abs(corrs_arr))
            print(
                f'{x_col} -> {y_col}: strongest correlation at lag {lags[best_idx]} '
                f'= {corrs_arr[best_idx]:.4f}'
            )
            print('-' * 60)

pairs = [
    ('turbidity', 'totalchlorine'),
    ('turbidity', 'finishwater'),
    ('finishwater', 'turbidity')
]

plot_cross_correlations(alexanderorr, pairs, max_lag=168)

##Seasonal Observations

In [ ]:
# Residual Chlorine Monthly Seasonality
monthly = alexanderorr['totalchlorine'].resample('ME').mean()
fig, ax = plt.subplots(figsize=(10, 6))

month_plot(monthly, ax=ax)

ax.set_title('Monthly Seasonality of Residual Chlorine (mg/L)')
ax.set_xlabel('Month (UTC)')
ax.set_ylabel('Residual Chlorine (mg/L)')

#Residual Chlorine Quarterly Seasonality
quarterly = alexanderorr['totalchlorine'].resample('QE').mean()
fig, ax = plt.subplots(figsize=(10, 6))
quarter_plot(quarterly, ax=ax)

ax.set_title('Quarterly Seasonality of Residual Chlorine (mg/L)')
ax.set_xlabel('Quarter (UTC)')
ax.set_ylabel('Residual Chlorine (mg/L)')

plt.tight_layout()
plt.show()

In [ ]:
# Turbidity Monthly Seasonality
monthly_turbidity = alexanderorr['turbidity'].resample('ME').mean()

fig, ax = plt.subplots(figsize=(10, 6))
month_plot(monthly_turbidity, ax=ax)

ax.set_title('Monthly Seasonality of Turbidity (NTU)')
ax.set_xlabel(' Month (UTC)')
ax.set_ylabel('Turbidity (NTU)')

plt.tight_layout()
plt.show()

#Turbidity Quarterly Seasonality
quarterly = alexanderorr['turbidity'].resample('QE').mean()
fig, ax = plt.subplots(figsize=(10, 6))
quarter_plot(quarterly, ax=ax)

ax.set_title('Quarterly Seasonality of Turbidity (NTU)')
ax.set_xlabel('Quarter (UTC)')
ax.set_ylabel('Turbidity (NTU)')

plt.tight_layout()
plt.show()

In [ ]:
# Flowrate Monthly Seasonality

monthly_finishwater = alexanderorr['finishwater'].resample('ME').mean()

fig, ax = plt.subplots(figsize=(10, 6))
month_plot(monthly_finishwater, ax=ax)

ax.set_title('Monthly Seasonality of Flowrate (mgd)')
ax.set_xlabel('Month (UTC)')
ax.set_ylabel('Flowrate (mgd)')

plt.tight_layout()
plt.show()

#Flowrate Quarterly Seasonality
quarterly_finishwater = alexanderorr['finishwater'].resample('QE').mean()

fig, ax = plt.subplots(figsize=(10, 6))
quarter_plot(quarterly_finishwater, ax=ax)

ax.set_title('Quarterly Seasonality of Flowrate (mgd)')
ax.set_xlabel('Quarter (UTC)')
ax.set_ylabel('Flowrate (mgd)')

plt.tight_layout()
plt.show()

##Seasonal Decomposition

In [ ]:
# Residual Chlorine

decomposition_chlorine = seasonal_decompose(alexanderorr['totalchlorine'], model='additive', period=2600)
fig = decomposition_chlorine.plot()
fig.suptitle('Seasonal Decomposition of Residual Chlorine (mg/L)', fontsize=60)
fig.set_size_inches(35, 18)
plt.show()


#Periods to check
#daily: 24 hours
#weekly: 168 hours
#yearly: 8760 hours

In [ ]:
# Turbidity
decomposition_turbidity = seasonal_decompose(alexanderorr['turbidity'], model='additive', period=2600)
fig = decomposition_turbidity.plot()
fig.suptitle('Seasonal Decomposition of Turbidity (NTU)', fontsize=60)
fig.set_size_inches(25, 18)
plt.show()

In [ ]:
# Flowrate
decomposition_flowrate = seasonal_decompose(alexanderorr['finishwater'], model='additive', period=2600)
fig = decomposition_flowrate.plot()
fig.suptitle('Seasonal Decomposition of Flowrate (mgd)', fontsize=60)
fig.set_size_inches(25, 18)
plt.show()

##Autocorrelation and Partial Autocorrelation

In [ ]:
# Plot the autocorrelation function (ACF) for Residual Chlorine
plot_acf(alexanderorr['totalchlorine'], lags=500)  #lags 48; 72; 500; 26,000; 6,000; 6,200 (still outside confidence interval, so significant)
plt.title('Residual Chlorine (mg/L) Autocorrelation')
plt.show()


# Periodic trends observed, and values above 0 in close range (e.g., 24 hours)

# Partial autocorrelation
plot_pacf(alexanderorr['totalchlorine'], lags=200)
plt.title('Residual Chlorine (mg/L) Partial Autocorrelation')
plt.show()

In [ ]:
# Plot the autocorrelation function (ACF) for Turdbidity
plot_acf(alexanderorr['turbidity'], lags=162)
plt.title('Turbidity (NTU) Autocorrelation')
plt.show()

# Declining trend observed; driven by short-time disturbances; not seasonal in nature

# Partial autocorrelation
plot_pacf(alexanderorr['turbidity'], lags=162)
plt.title('Turbidity (NTU) Partial Autocorrelation')
plt.show()

In [ ]:
# Plot the autocorrelation function (ACF) for Flowrate
plot_acf(alexanderorr['finishwater'], lags=48)
plt.title('Flowrate (mgd) Autocorrelation')
plt.show()

# Periodic trends observed but not as strong as with turbidity, and values above 0

# Partial autocorrelation
plot_pacf(alexanderorr['finishwater'], lags=48)
plt.title('Flowrate (mgd) Partial Autocorrelation')
plt.show()

#Sample Size Estimation Based on Wilkis (2011)

In [ ]:
# Wilkis (2011) - Sample Size Estimation (Helps to observe how much independent information can be obtained from the data)
y = alexanderorr['totalchlorine'].asfreq('h')

n = len(y)
r1 = y.autocorr(lag=1)
n_eff = n * (1 - r1) / (1 + r1)

print(f"Total observations (n): {n}")
print(f"Lag-1 autocorrelation (r1): {r1:.4f}")
print(f"Effective sample size (n'): {n_eff:.2f}")

# Dickey-Fuller Test to Check if Time Series is Stationary


In [ ]:
# Dickey-Fuller Test to Check if the time series is stationary

result_dfuller=adfuller(y)
print('p-value: %f' % result_dfuller[1])

if result_dfuller[1] > 0.05:
  print('The time series is not stationary')
else:
  print('The time series is stationary')

# Null hypothesis: time series is not stationary; alternative: time series is stationary
# ARIMA(p,0,q)

#Mann-Kendall Test to Check for Monotonic Upward or Downward Trend

In [ ]:
# Evaluating trends

result_mk = mk.seasonal_test(y,
    period=24
)

print(result_mk)

if result_mk.h:  #statistically significant
    if result_mk.trend == 'increasing':
        print(f"Significant increasing trend (p={result_mk.p:.4f})")
    elif result_mk.trend == 'decreasing':
        print(f"Significant decreasing trend (p={result_mk.p:.4f})")
    else:
        print(f"Significant trend detected but direction unclear (p={result_mk.p:.4f})")
else:
    print(f"No statistically significant trend (p={result_mk.p:.4f})")
#Significant increasing trend observed

# Seasonal Differencing to Confirm that Seasonality is Strong in Residual Chlorine

In [ ]:
# Checking seasonal differencing (a lot changes, so D=1)
seasonal_diff = y.diff(24)

# ACF plot on differenced series
plot_acf(seasonal_diff.dropna(), lags=50)
plt.title('ACF After Seasonal Differencing')
plt.show()
#This confirms that the seasonal component is very present

# PACF on differenced series
plot_pacf(seasonal_diff.dropna(), lags=50)
plt.title('PACF After Seasonal Differencing')
plt.show()

# Based on this:
#1. Differencing was useful, so there were this strong cyclical peaks att lag 24, 48, etc. So D = 1.
#2. Short-term nonseasonal structure still present so, I still need to keep AR and MA terms
#3. Small non-seasonal MA component: q = 1
#4. Based on DFT, the series can be seen as stationary: d = 0
#5. Positive values on spikes on PACF, at lags 1 and 2,non-seasonal AR component may be present: p = 1, or 2
#6. Positive spikes remain on PACF at 24, 48, still a seasonal AR structure present: P = 1
#7. Seasonal cycles at seem to be daily, 24 hours, 48 hours, and so on: m = 24
#8. Spikes at 24 and 48: Q = 1, or 2

#Simple model candidates:
#SARIMA(p,d,q)(P,D,Q)m
#SARIMA(1,0,1)(1,1,1)24
#(1,0,1)(1,1,1)m=24; (2,01,1)(1,1,1)m=24; (1,0,1)(1,1,0)m=24; (1,01,1)(1,1,1)m=24

#*AI/ML Models and Baseline Models for Residual Chlorine*

In [ ]:
# Creating the output directory where model weights for transfer learning will be stored

BASE_OUTPUT = "/content/drive/MyDrive/Forecasting_AlexanderOrr" #All model outputs and results will go here

#28 July:
#Create directory to save model configurations for future generalization
SAVED_MODEL_DIR = os.path.join(
    BASE_OUTPUT,
    "Saved_Models"
)

os.makedirs(
    SAVED_MODEL_DIR,
    exist_ok=True
)

print("Saved-model folder:")
print(SAVED_MODEL_DIR)


In [ ]:
# Create and return a structured model-output folder, adding optional configuration and season subfolders when provided.

def make_output_folder(
    model_group,
    model_name,
    config_name=None,
    season=None
):

    parts = [
        PROJECT_DIR,
        model_group,
        model_name
    ]

    if config_name is not None:
        parts.append(config_name)

    if season is not None:
        parts.append(season)

    output_path = os.path.join(*parts)

    os.makedirs(output_path, exist_ok=True)

    return output_path

In [ ]:
# Prepare, save, or reload the canonical UTC master DataFrame used by every
# forecasting and classification model.

MASTER_DATA_OUTPUT_FILE = (
    "/content/drive/MyDrive/"
    "Alexander_Orr_DLML/"
    "alexanderorrdl.xlsx"
)


def prepare_master_dataframe(alexanderorr):
    """
    Create the canonical dataframe shared by all baseline, deep-learning, and
    classification models while preserving every existing EDA column.
    """
    df = alexanderorr.copy(deep=True)

    # Confirm that the earlier UTC timestamp preparation was completed.
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(
            "alexanderorr must have a DatetimeIndex."
        )

    if str(df.index.tz) != "UTC":
        raise ValueError(
            "alexanderorr must already have a UTC index."
        )

    if not df.index.is_monotonic_increasing:
        raise ValueError(
            "The UTC index must already be chronologically ordered."
        )

    if df.index.has_duplicates:
        raise ValueError(
            "Unexpected duplicate UTC timestamps were found."
        )

    # Confirm that every consecutive observation represents exactly one hour.
    time_differences = (
        df.index
        .to_series()
        .diff()
        .dropna()
    )

    if not time_differences.eq(
        pd.Timedelta(hours=1)
    ).all():
        raise ValueError(
            "The UTC index must already be continuously hourly."
        )

    # Confirm that the original measurements required by the models exist.
    required_measurement_columns = [
        "totalchlorine",
        "turbidity",
        "finishwater",
        "chlorine_conversion",
    ]

    missing_measurement_columns = [
        column
        for column in required_measurement_columns
        if column not in df.columns
    ]

    if missing_measurement_columns:
        raise ValueError(
            "Required measurement columns are missing: "
            f"{missing_measurement_columns}"
        )

    # Check missing values only in the original measurements used by models.
    # Temporary EDA columns can contain expected first-row missing values.
    missing_measurement_values = (
        df[required_measurement_columns]
        .isna()
        .sum()
    )

    missing_measurement_values = (
        missing_measurement_values[
            missing_measurement_values > 0
        ]
    )

    if not missing_measurement_values.empty:
        raise ValueError(
            "Missing values remain in required measurements:\n"
            f"{missing_measurement_values}"
        )

    # Create Miami-local calendar features for operational EDA.
    # The canonical dataframe index remains in UTC.
    miami_index = df.index.tz_convert(
        "America/New_York"
    )

    df["year"] = miami_index.year
    df["month"] = miami_index.month
    df["day"] = miami_index.day
    df["day_of_week"] = miami_index.dayofweek

    # Keep the alternate weekday name used by existing code.
    df["dayofweek"] = df["day_of_week"]

    df["hour"] = miami_index.hour
    df["month_year"] = miami_index.strftime("%Y-%m")

    # Create Miami-local cyclical calendar features for operational EDA.
    df["hour_sin"] = np.sin(
        2 * np.pi * df["hour"] / 24.0
    )

    df["hour_cos"] = np.cos(
        2 * np.pi * df["hour"] / 24.0
    )

    df["dow_sin"] = np.sin(
        2 * np.pi * df["dayofweek"] / 7.0
    )

    df["dow_cos"] = np.cos(
        2 * np.pi * df["dayofweek"] / 7.0
    )

    # Create explicit UTC calendar features for forecasting models.
    df["utc_year"] = df.index.year
    df["utc_month"] = df.index.month
    df["utc_day"] = df.index.day
    df["utc_day_of_week"] = df.index.dayofweek
    df["utc_hour"] = df.index.hour
    df["utc_month_year"] = df.index.strftime("%Y-%m")

    # Create UTC cyclical calendar features.
    df["utc_hour_sin"] = np.sin(
        2 * np.pi * df["utc_hour"] / 24.0
    )

    df["utc_hour_cos"] = np.cos(
        2 * np.pi * df["utc_hour"] / 24.0
    )

    df["utc_dow_sin"] = np.sin(
        2 * np.pi * df["utc_day_of_week"] / 7.0
    )

    df["utc_dow_cos"] = np.cos(
        2 * np.pi * df["utc_day_of_week"] / 7.0
    )

    # Add the constant numeric feature required by TFT.
    df["static_dummy"] = 0.0

    # Confirm that every local and UTC calendar feature was created.
    required_calendar_columns = [
        "year",
        "month",
        "day",
        "day_of_week",
        "dayofweek",
        "hour",
        "month_year",
        "hour_sin",
        "hour_cos",
        "dow_sin",
        "dow_cos",
        "utc_year",
        "utc_month",
        "utc_day",
        "utc_day_of_week",
        "utc_hour",
        "utc_month_year",
        "utc_hour_sin",
        "utc_hour_cos",
        "utc_dow_sin",
        "utc_dow_cos",
        "static_dummy",
    ]

    missing_calendar_columns = [
        column
        for column in required_calendar_columns
        if column not in df.columns
    ]

    if missing_calendar_columns:
        raise ValueError(
            "Required calendar features are missing: "
            f"{missing_calendar_columns}"
        )

    # Confirm that model-required calendar features contain no missing values.
    missing_calendar_values = (
        df[required_calendar_columns]
        .isna()
        .sum()
    )

    missing_calendar_values = (
        missing_calendar_values[
            missing_calendar_values > 0
        ]
    )

    if not missing_calendar_values.empty:
        raise ValueError(
            "Missing values remain in calendar features:\n"
            f"{missing_calendar_values}"
        )

    # Confirm that the explicit UTC features match the UTC index.
    utc_feature_checks = {
        "utc_year": df.index.year,
        "utc_month": df.index.month,
        "utc_day": df.index.day,
        "utc_day_of_week": df.index.dayofweek,
        "utc_hour": df.index.hour,
    }

    for column, expected_values in utc_feature_checks.items():
        if not np.array_equal(
            df[column].to_numpy(),
            np.asarray(expected_values)
        ):
            raise ValueError(
                f"{column} does not match the UTC index."
            )

    # Attach explicit hourly frequency metadata after continuity validation.
    # This does not add timestamps because continuity was already confirmed.
    df = df.asfreq("h")

    return df


def validate_loaded_master_dataframe(df):
    """
    Validate a canonical master dataframe loaded from the Excel checkpoint.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(
            "The loaded master dataframe must have a DatetimeIndex."
        )

    if str(df.index.tz) != "UTC":
        raise ValueError(
            "The loaded master dataframe must have a UTC-aware index."
        )

    if not df.index.is_monotonic_increasing:
        raise ValueError(
            "The loaded UTC index is not chronologically ordered."
        )

    if df.index.has_duplicates:
        raise ValueError(
            "The loaded master dataframe contains duplicate timestamps."
        )

    # Confirm continuous hourly timestamps before assigning frequency metadata.
    time_differences = (
        df.index
        .to_series()
        .diff()
        .dropna()
    )

    if not time_differences.eq(
        pd.Timedelta(hours=1)
    ).all():
        raise ValueError(
            "The loaded master dataframe is not continuously hourly."
        )

    required_measurement_columns = [
        "totalchlorine",
        "turbidity",
        "finishwater",
        "chlorine_conversion",
    ]

    required_calendar_columns = [
        "year",
        "month",
        "day",
        "day_of_week",
        "dayofweek",
        "hour",
        "month_year",
        "hour_sin",
        "hour_cos",
        "dow_sin",
        "dow_cos",
        "utc_year",
        "utc_month",
        "utc_day",
        "utc_day_of_week",
        "utc_hour",
        "utc_month_year",
        "utc_hour_sin",
        "utc_hour_cos",
        "utc_dow_sin",
        "utc_dow_cos",
        "static_dummy",
    ]

    required_model_columns = (
        required_measurement_columns
        + required_calendar_columns
    )

    missing_required_columns = [
        column
        for column in required_model_columns
        if column not in df.columns
    ]

    if missing_required_columns:
        raise ValueError(
            "The saved master dataframe is missing required columns: "
            f"{missing_required_columns}"
        )

    # Restore numeric model-column types that Excel may have generalized.
    numeric_columns = [
        "totalchlorine",
        "turbidity",
        "finishwater",
        "chlorine_conversion",
        "year",
        "month",
        "day",
        "day_of_week",
        "dayofweek",
        "hour",
        "hour_sin",
        "hour_cos",
        "dow_sin",
        "dow_cos",
        "utc_year",
        "utc_month",
        "utc_day",
        "utc_day_of_week",
        "utc_hour",
        "utc_hour_sin",
        "utc_hour_cos",
        "utc_dow_sin",
        "utc_dow_cos",
        "static_dummy",
    ]

    for column in numeric_columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="raise"
        )

    # Confirm that required measurements contain no missing values.
    missing_measurement_values = (
        df[required_measurement_columns]
        .isna()
        .sum()
    )

    missing_measurement_values = (
        missing_measurement_values[
            missing_measurement_values > 0
        ]
    )

    if not missing_measurement_values.empty:
        raise ValueError(
            "The saved master dataframe contains missing "
            "measurement values:\n"
            f"{missing_measurement_values}"
        )

    # Confirm that required calendar columns contain no missing values.
    missing_calendar_values = (
        df[required_calendar_columns]
        .isna()
        .sum()
    )

    missing_calendar_values = (
        missing_calendar_values[
            missing_calendar_values > 0
        ]
    )

    if not missing_calendar_values.empty:
        raise ValueError(
            "The saved master dataframe contains missing "
            "calendar values:\n"
            f"{missing_calendar_values}"
        )

    # Confirm that UTC calendar fields match the restored UTC index.
    utc_feature_checks = {
        "utc_year": df.index.year,
        "utc_month": df.index.month,
        "utc_day": df.index.day,
        "utc_day_of_week": df.index.dayofweek,
        "utc_hour": df.index.hour,
    }

    for column, expected_values in utc_feature_checks.items():
        actual_values = (
            pd.to_numeric(
                df[column],
                errors="coerce"
            )
            .to_numpy()
        )

        if not np.array_equal(
            actual_values,
            np.asarray(expected_values)
        ):
            raise ValueError(
                f"{column} does not match the restored UTC index."
            )

    return df


def load_master_dataframe_from_excel(output_file):
    """
    Load the saved Excel checkpoint and restore its UTC-aware hourly index.
    """
    loaded = pd.read_excel(
        output_file
    )

    if "timestamp_utc" not in loaded.columns:
        raise ValueError(
            "The Excel checkpoint does not contain the expected "
            "'timestamp_utc' column."
        )

    # Excel timestamps are timezone-naive, but this column explicitly
    # represents UTC.
    loaded["timestamp_utc"] = pd.to_datetime(
        loaded["timestamp_utc"],
        errors="raise",
        utc=True
    )

    loaded = loaded.set_index(
        "timestamp_utc"
    )

    loaded.index.name = "timestamp_utc"
    loaded = loaded.sort_index()

    # Restore Boolean audit flags that Excel may read as numeric or text.
    audit_flag_columns = [
        column
        for column in loaded.columns
        if column.endswith("_forward_filled")
    ]

    for column in audit_flag_columns:
        if pd.api.types.is_bool_dtype(loaded[column]):
            loaded[column] = (
                loaded[column]
                .fillna(False)
                .astype(bool)
            )
        else:
            loaded[column] = (
                loaded[column]
                .replace({
                    "True": True,
                    "False": False,
                    "TRUE": True,
                    "FALSE": False,
                    1: True,
                    0: False,
                })
                .fillna(False)
                .astype(bool)
            )

    loaded = validate_loaded_master_dataframe(
        loaded
    )

    # Assign explicit hourly frequency metadata. Validation already confirmed
    # that there are no missing hourly timestamps.
    loaded = loaded.asfreq("h")

    return loaded


def export_master_dataframe_to_excel(
    df,
    output_file
):
    """
    Export the canonical dataframe without modifying the timezone-aware
    in-memory dataframe.
    """
    os.makedirs(
        os.path.dirname(output_file),
        exist_ok=True
    )

    # Create an independent copy for Excel-specific changes.
    df_export = df.copy(deep=True)

    # Excel cannot store a timezone-aware DatetimeIndex. Convert explicitly to
    # UTC and remove timezone metadata only from the export copy.
    if (
        isinstance(df_export.index, pd.DatetimeIndex)
        and df_export.index.tz is not None
    ):
        df_export.index = (
            df_export.index
            .tz_convert("UTC")
            .tz_localize(None)
        )

    # The index name records that the timezone-naive Excel values represent UTC.
    df_export.index.name = "timestamp_utc"

    # Excel cannot directly store timezone-aware datetime columns.
    timezone_columns = (
        df_export
        .select_dtypes(include=["datetimetz"])
        .columns
    )

    for column in timezone_columns:
        df_export[column] = (
            df_export[column]
            .dt
            .tz_convert("UTC")
            .dt
            .tz_localize(None)
        )

    df_export.to_excel(
        output_file,
        index=True
    )


# Load the saved checkpoint when it exists.
if os.path.exists(MASTER_DATA_OUTPUT_FILE):

    print("\n" + "=" * 70)
    print("EXISTING MASTER DATAFRAME FOUND")
    print("=" * 70)

    df = load_master_dataframe_from_excel(
        MASTER_DATA_OUTPUT_FILE
    )

    master_dataframe_source = (
        "Loaded from existing Excel checkpoint"
    )

# Create and save the checkpoint when it does not exist.
else:

    print("\n" + "=" * 70)
    print("MASTER DATAFRAME CHECKPOINT NOT FOUND")
    print("=" * 70)

    # The original Alexander Orr dataframe is needed only the first time.
    if "alexanderorr" not in globals():
        raise NameError(
            "alexanderorrdl.xlsx does not exist and `alexanderorr` is "
            "not available. Run the earlier data-loading and preparation "
            "cells once to create the checkpoint."
        )

    df = prepare_master_dataframe(
        alexanderorr
    )

    export_master_dataframe_to_excel(
        df=df,
        output_file=MASTER_DATA_OUTPUT_FILE
    )

    master_dataframe_source = (
        "Created from alexanderorr and exported to Excel"
    )


# Display the canonical modeling-data integrity summary.
print("\n" + "=" * 70)
print("MASTER DATAFRAME READY")
print("=" * 70)

print("Source:", master_dataframe_source)
print("Path:", MASTER_DATA_OUTPUT_FILE)
print("Shape:", df.shape)
print("Timezone:", df.index.tz)
print("Inferred frequency:", pd.infer_freq(df.index))
print("Explicit frequency:", df.index.freq)

print(
    "Duplicate timestamps:",
    int(df.index.duplicated().sum())
)

required_measurement_columns = [
    "totalchlorine",
    "turbidity",
    "finishwater",
    "chlorine_conversion",
]

print(
    "Missing measurement values:",
    int(
        df[required_measurement_columns]
        .isna()
        .sum()
        .sum()
    )
)

# Report expected missing values in temporary EDA-derived columns separately.
eda_columns_with_missing_values = (
    df
    .isna()
    .sum()
    .loc[lambda values: values > 0]
)

print("\nColumns containing missing values:")

if eda_columns_with_missing_values.empty:
    print("None")
else:
    print(eda_columns_with_missing_values)

print("\nCanonical modeling dataframe is ready.")

print(
    "All in-memory modeling timestamps are timezone-aware UTC."
)

if master_dataframe_source == (
    "Loaded from existing Excel checkpoint"
):
    print(
        "Earlier raw-data preparation and EDA cells did not need to be rerun."
    )
else:
    print(
        "A reusable Excel checkpoint was created successfully at:\n"
        f"{MASTER_DATA_OUTPUT_FILE}"
    )

In [ ]:
# Configure reproducibility, hardware selection, and TFT output settings.
SEED = 42

# Set the NumPy random seed.
np.random.seed(SEED)

# Set the TensorFlow random seed when TensorFlow is available.
try:
    tf.random.set_seed(SEED)
except NameError:
    pass

# Set the PyTorch random seed and use a GPU when CUDA is available.
try:
    torch.manual_seed(SEED)
    DEVICE = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
except NameError:
    DEVICE = "cpu"

print("Using device:", DEVICE)

# Control optional TFT diagnostic output and plots.
TFT_SHOW_DIAGNOSTIC_PLOTS = False
TFT_SHOW_FINAL_HOLDOUT_PLOT = False
TFT_PRINT_SKIP_ERRORS = False

In [ ]:
# General forecasting helpers
# General helpers

# Create an output directory and any missing parent folders.
# No error is raised when the directory already exists.
def ensure_output_dir(output_dir):
    os.makedirs(output_dir, exist_ok=True)


# Request Python garbage collection to release objects that are no longer
# referenced. This can help manage memory between model-training runs.
def clean_memory():
    gc.collect()


# Convert a timestamp so that it matches a requested timezone.
def ensure_same_tz(ts, tz):
    # Convert the supplied value into a pandas Timestamp.
    ts = pd.Timestamp(ts)

    # If the target index is timezone-naive, return a timezone-naive timestamp.
    if tz is None:
        if ts.tzinfo is not None:
            return ts.tz_convert(None)
        return ts

    # Assign the requested timezone when the timestamp has no timezone.
    if ts.tzinfo is None:
        return ts.tz_localize(tz)

    # Convert an already timezone-aware timestamp to the requested timezone.
    return ts.tz_convert(tz)


# Assign a meteorological season according to the timestamp's calendar month.
def season_name(ts):
    m = pd.Timestamp(ts).month

    if m in [3, 4, 5]:
        return "Spring"
    elif m in [6, 7, 8]:
        return "Summer"
    elif m in [9, 10, 11]:
        return "Autumn"
    else:
        return "Winter"


# Calculate regression metrics comparing observed and predicted values.
def evaluate_predictions(y_true, y_pred):
    # Convert the inputs to NumPy arrays for consistent calculations.
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Prevent division by zero when actual values are zero or nearly zero.
    epsilon = 1e-6

    # Calculate mean absolute percentage error as a percentage.
    mape = np.mean(
        np.abs(
            (y_true - y_pred)
            / np.maximum(np.abs(y_true), epsilon)
        )
    ) * 100

    # Return error, fit, and percentage metrics in a reusable dictionary.
    return {
        "RMSE": float(
            np.sqrt(mean_squared_error(y_true, y_pred))
        ),
        "MAE": float(
            mean_absolute_error(y_true, y_pred)
        ),
        "R2": float(
            r2_score(y_true, y_pred)
        ),
        "MAPE": float(mape)
    }


# Evaluate model performance separately during normal operation and chlorine
# conversion periods.
def stable_run_id(row_dict: Dict) -> str:
    # Include only settings intended to distinguish one run from another.
    keys = [
        "model_family",
        "eval_month_start",
        "season",
        "train_months",
        "hidden_units",
        "batch_size",
        "num_heads"
    ]

    # Convert each setting to text and use an empty value when it is absent.
    payload = {
        key: str(row_dict.get(key, ""))
        for key in keys
    }

    # Sort keys before hashing so dictionary insertion order cannot change the
    # generated identifier.
    text = json.dumps(
        payload,
        sort_keys=True
    )

    # MD5 is used only to create a compact identifier, not for security.
    return hashlib.md5(
        text.encode("utf-8")
    ).hexdigest()[:12]


# Add empirical prediction intervals using residual quantiles calculated from
# training or calibration errors.
def add_residual_based_intervals(
    pred_df,
    pred_col,
    calibration_residuals,
    alpha=0.05
):
    # Work on a copy so the original predictions are not modified.
    pred_df = pred_df.copy()

    # Remove missing calibration residuals and convert them to a NumPy array.
    residuals = (
        pd.Series(calibration_residuals)
        .dropna()
        .to_numpy()
    )

    # Require enough residuals to estimate the interval quantiles reasonably.
    if residuals.size < 30:
        raise ValueError(
            "At least 30 training/calibration residuals "
            "are required for intervals."
        )

    # For alpha=0.05, calculate the empirical 2.5th and 97.5th percentiles.
    lower_error, upper_error = np.quantile(
        residuals,
        [
            alpha / 2,
            1 - alpha / 2
        ]
    )

    # Add the residual quantiles to every point forecast. This assumes the
    # calibration residuals were defined as actual minus predicted.
    pred_df["lower_ci"] = (
        pred_df[pred_col] + lower_error
    )

    pred_df["upper_ci"] = (
        pred_df[pred_col] + upper_error
    )

    # Record the interval method for later reporting and comparison.
    pred_df["interval_method"] = (
        "empirical_training_residual_quantiles"
    )

    return pred_df


# Add model-based confidence intervals returned by a fitted statsmodels ARIMA,
# SARIMA, or SARIMAX forecast.
def add_sarimax_intervals(
    pred_df,
    conf_int
):
    # Work on a copy so the original predictions are not modified.
    pred_df = pred_df.copy()

    # Extract the lower and upper forecast boundaries from statsmodels.
    pred_df["lower_ci"] = (
        conf_int.iloc[:, 0].values
    )

    pred_df["upper_ci"] = (
        conf_int.iloc[:, 1].values
    )

    # Estimate the forecast standard deviation from a nominal 95% interval.
    pred_df["sigma"] = (
        pred_df["upper_ci"]
        - pred_df["lower_ci"]
    ) / (2 * 1.96)

    return pred_df


# Print a titled console section using consistent separator lines.
def print_table(title, obj):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    print(obj)

In [ ]:
# Forecast plotting and residual-diagnostic helpers
# Plotting and residual diagnostics

# Plot actual and forecast residual-chlorine values without a forecast interval.
def plot_forecast_vs_actual_plain(
    pred_df,
    title,
    actual_col="actual",
    pred_col="forecast",
    ylabel="Residual Chlorine (mg/L)",
    save_path=None,
    show_plot=True
):
    # Create the figure and plotting axes.
    fig, ax = plt.subplots(figsize=(14, 6))

    # Plot the observed and forecast values on the same UTC timeline.
    ax.plot(
        pred_df.index,
        pred_df[actual_col],
        label="Actual"
    )

    ax.plot(
        pred_df.index,
        pred_df[pred_col],
        label="Forecast"
    )

    # Configure the title, axis labels, and legend.
    ax.set_title(title)
    ax.set_xlabel("Forecast Timestamp (UTC)")
    ax.set_ylabel(ylabel)
    ax.legend()

    # Format datetime labels and reduce clipping.
    fig.autofmt_xdate()
    fig.tight_layout()

    # Save a high-resolution copy when an output path is provided.
    if save_path is not None:
        fig.savefig(
            save_path,
            dpi=600
        )

    # Display the plot only when requested.
    if show_plot:
        plt.show()

    # Close the figure to release memory after saving or displaying it.
    plt.close(fig)


# Plot actual and forecast values with an optional forecast interval.
def plot_forecast_vs_actual_interval(
    pred_df,
    title,
    actual_col="actual",
    pred_col="forecast",
    lower_col="lower_ci",
    upper_col="upper_ci",
    ylabel="Residual Chlorine (mg/L)",
    save_path=None,
    show_plot=True
):
    # Create the figure and plotting axes.
    fig, ax = plt.subplots(figsize=(14, 6))

    # Plot the observed and forecast values on the same UTC timeline.
    ax.plot(
        pred_df.index,
        pred_df[actual_col],
        label="Actual"
    )

    ax.plot(
        pred_df.index,
        pred_df[pred_col],
        label="Forecast"
    )

    # Shade the forecast interval only when both boundary columns are available.
    if (
        lower_col in pred_df.columns
        and upper_col in pred_df.columns
    ):
        ax.fill_between(
            pred_df.index,
            pred_df[lower_col],
            pred_df[upper_col],
            alpha=0.20,
            label="Forecast interval"
        )

    # Configure the title, axis labels, and legend.
    ax.set_title(title)
    ax.set_xlabel("Forecast Timestamp (UTC)")
    ax.set_ylabel(ylabel)
    ax.legend()

    # Format datetime labels and reduce clipping.
    fig.autofmt_xdate()
    fig.tight_layout()

    # Save a high-resolution copy when an output path is provided.
    if save_path is not None:
        fig.savefig(
            save_path,
            dpi=600
        )

    # Display the plot only when requested.
    if show_plot:
        plt.show()

    # Close the figure to release memory after saving or displaying it.
    plt.close(fig)


# Save separate plain and interval forecast-comparison plots using consistent
# filenames and output-folder organization.
def save_forecast_plots(
    pred_df,
    output_dir,
    file_stem,
    model_label,
    actual_col="actual",
    pred_col="forecast",
    show_plot=True
):
    # Create the output directory when it does not already exist.
    ensure_output_dir(output_dir)

    # Create and save the plain actual-versus-forecast plot.
    plot_forecast_vs_actual_plain(
        pred_df=pred_df,
        title=f"{model_label}: Forecast vs Actual",
        actual_col=actual_col,
        pred_col=pred_col,
        save_path=(
            f"{output_dir}/"
            f"{file_stem}_forecast_vs_actual_plain.png"
        ),
        show_plot=show_plot
    )

    # Create and save the actual-versus-forecast plot with its interval.
    plot_forecast_vs_actual_interval(
        pred_df=pred_df,
        title=(
            f"{model_label}: Forecast vs Actual "
            "with Forecast Interval"
        ),
        actual_col=actual_col,
        pred_col=pred_col,
        save_path=(
            f"{output_dir}/"
            f"{file_stem}_forecast_vs_actual_interval.png"
        ),
        show_plot=show_plot
    )


# Preserve compatibility with older code that calls plot_forecast_vs_actual()
# while producing both a plain plot and an interval plot.
def plot_forecast_vs_actual(
    pred_df,
    title,
    actual_col="actual",
    pred_col="forecast",
    lower_col="lower_ci",
    upper_col="upper_ci",
    ylabel="Residual Chlorine (mg/L)",
    save_path=None,
    show_plot=True
):
    # Create separate output paths for the plain and interval plots.
    if save_path is not None:
        save_path = str(save_path)

        if save_path.endswith(".png"):
            plain_path = save_path.replace(
                ".png",
                "_plain.png"
            )

            interval_path = save_path.replace(
                ".png",
                "_interval.png"
            )

        else:
            plain_path = save_path + "_plain.png"
            interval_path = save_path + "_interval.png"

    # Keep both paths unset when the plots should not be saved.
    else:
        plain_path = None
        interval_path = None

    # Generate the plain actual-versus-forecast plot.
    plot_forecast_vs_actual_plain(
        pred_df=pred_df,
        title=title,
        actual_col=actual_col,
        pred_col=pred_col,
        ylabel=ylabel,
        save_path=plain_path,
        show_plot=show_plot
    )

    # Generate the actual-versus-forecast plot with its interval.
    plot_forecast_vs_actual_interval(
        pred_df=pred_df,
        title=f"{title} with Forecast Interval",
        actual_col=actual_col,
        pred_col=pred_col,
        lower_col=lower_col,
        upper_col=upper_col,
        ylabel=ylabel,
        save_path=interval_path,
        show_plot=show_plot
    )


# Calculate residuals, export them, and save diagnostic plots for residual
# behavior over time, distribution shape, normality, and autocorrelation.
def save_residual_outputs(
    pred_df,
    actual_col,
    pred_col,
    output_dir,
    file_stem,
    lags=72,
    show_plots=True
):
    # Create the output directory when it does not already exist.
    ensure_output_dir(output_dir)

    # Calculate residuals as actual minus forecast.
    residual_df = pred_df.copy()
    residual_df["residual"] = (
        residual_df[actual_col]
        - residual_df[pred_col]
    )

    # Save timestamped residual values for later statistical analysis.
    residual_df.to_csv(
        f"{output_dir}/{file_stem}_residuals.csv"
    )

    # Remove missing residuals before distribution and correlation diagnostics.
    residuals = (
        residual_df["residual"]
        .dropna()
    )


    # Plot residuals over time to identify trends, changing variance, and
    # periods of systematic overprediction or underprediction.
    fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(
        residual_df.index,
        residual_df["residual"]
    )

    # Mark zero error as the reference line.
    ax.axhline(
        0,
        linestyle="--"
    )

    ax.set_title("Residuals over Time")
    ax.set_xlabel("Forecast Timestamp (UTC)")
    ax.set_ylabel("Residual Chlorine Error (mg/L)")

    fig.autofmt_xdate()
    fig.tight_layout()

    fig.savefig(
        f"{output_dir}/"
        f"{file_stem}_residuals_timeseries.png",
        dpi=1200
    )

    if show_plots:
        plt.show()

    plt.close(fig)


    # Plot the residual distribution to assess symmetry, spread, and extreme
    # forecast errors.
    fig, ax = plt.subplots(figsize=(10, 5))

    ax.hist(
        residuals,
        bins=40
    )

    ax.set_title("Residual Distribution")
    ax.set_xlabel("Residual Chlorine Error (mg/L)")
    ax.set_ylabel("Frequency")

    fig.tight_layout()

    fig.savefig(
        f"{output_dir}/"
        f"{file_stem}_residuals_histogram.png",
        dpi=1200
    )

    if show_plots:
        plt.show()

    plt.close(fig)


    # Create a Q-Q plot to compare the residual distribution with a normal
    # distribution.
    fig = qqplot(
        residuals,
        line="s"
    )

    fig.axes[0].set_title("Residual Q-Q Plot")
    fig.tight_layout()

    fig.savefig(
        f"{output_dir}/"
        f"{file_stem}_residuals_qqplot.png",
        dpi=1200
    )

    if show_plots:
        plt.show()

    plt.close(fig)


    # Plot residual autocorrelation when enough residual observations exist.
    if len(residuals) > 2:
        fig, ax = plt.subplots(figsize=(12, 5))

        # Limit the number of lags so it cannot exceed the available residual
        # observations.
        plot_acf(
            residuals,
            lags=min(
                lags,
                max(1, len(residuals) - 1)
            ),
            ax=ax
        )

        ax.set_title("Residual Autocorrelation")
        ax.set_xlabel("Lag (Hours)")
        ax.set_ylabel("Autocorrelation")

        fig.tight_layout()

        fig.savefig(
            f"{output_dir}/"
            f"{file_stem}_residuals_acf.png",
            dpi=1200
        )

        if show_plots:
            plt.show()

        plt.close(fig)

In [ ]:
# Chronological split helpers

    # Time-based training, validation, and holdout splits

# Split one target series into training and holdout periods using a fixed
# timestamp boundary.
def make_date_holdout_split(
    df,
    target_col,
    holdout_start
):
    # Create a copy so cleaning the target does not modify the original data.
    df_model = df.copy()

    # Convert the target to numeric values and mark invalid entries as missing.
    df_model[target_col] = pd.to_numeric(
        df_model[target_col],
        errors="coerce"
    )

    # Remove rows where the target cannot be used for model evaluation.
    df_model = df_model.dropna(
        subset=[target_col]
    )

    # Make the holdout boundary compatible with the DataFrame index timezone.
    # When the modeling index is UTC, the resulting boundary is also UTC.
    holdout_start = ensure_same_tz(
        holdout_start,
        df_model.index.tz
    )

    # Use every observation before the boundary for model training.
    train = df_model.loc[
        df_model.index < holdout_start,
        target_col
    ].copy()

    # Reserve observations at and after the boundary for final evaluation.
    test = df_model.loc[
        df_model.index >= holdout_start,
        target_col
    ].copy()

    return train, test


# Create chronological training, validation, and test DataFrames using calendar
# month boundaries aligned with the timezone of the supplied DataFrame index.
def split_by_time(
    df,
    train_months,
    eval_start_month,
    val_months=1,
    test_months=1
):
    # Read the timezone from the DataFrame index.
    tz = df.index.tz

    # Treat the evaluation start as the exclusive end of the training period.
    train_end = ensure_same_tz(
        eval_start_month,
        tz
    )

    # Calculate the beginning of the requested training window.
    train_start = (
        train_end
        - pd.DateOffset(months=train_months)
    )


    # Begin validation immediately after the training period.
    val_start = train_end

    # Calculate the requested exclusive validation boundary.
    requested_val_end = (
        val_start
        + pd.DateOffset(months=val_months)
    )

    # Record the final available validation timestamp when the requested
    # validation period extends beyond the dataset.
    val_end = min(
        requested_val_end,
        df.index.max()
    )


    # Begin testing at the requested end of the validation period.
    test_start = requested_val_end

    # Calculate the requested exclusive test boundary.
    requested_test_end = (
        test_start
        + pd.DateOffset(months=test_months)
    )


    # Select the chronological training observations.
    # The start is included and the end is excluded.
    train_df = df[
        (df.index >= train_start)
        & (df.index < train_end)
    ].copy()


    # Select the validation observations.
    # The start is included and the requested end is excluded.
    val_df = df[
        (df.index >= val_start)
        & (df.index < requested_val_end)
    ].copy()


    # Use a complete test period when the requested end boundary is available.
    if requested_test_end <= df.index.max():

        # Include test_start but exclude the next calendar boundary.
        test_end = requested_test_end

        test_df = df[
            (df.index >= test_start)
            & (df.index < test_end)
        ].copy()


    # Use all remaining observations when the dataset ends before the requested
    # test-period boundary.
    else:

        # Set the test end to the final available observation.
        test_end = df.index.max()

        # Include both the test start and final available timestamp.
        test_df = df[
            (df.index >= test_start)
            & (df.index <= test_end)
        ].copy()

        # Warn that the test period contains less data than requested.
        print(
            f"Warning: partial test period. "
            f"Requested end: {requested_test_end}. "
            f"Available data end: {test_end}."
        )


    # Return the split DataFrames and their corresponding boundaries for
    # reporting, diagnostics, and reproducibility.
    return (
        train_df,
        val_df,
        test_df,
        train_start,
        train_end,
        val_start,
        val_end,
        test_start,
        test_end
    )

In [ ]:
# Define one representative model-selection month for each meteorological
# season. These boundaries are interpreted as UTC by the splitting functions.

SEASONAL_MONTH_PLAN = {
    "Winter": {
         # Use January 2025 for winter model selection and tuning.
        "selection_months": ["2025-01-01"],
    },
    "Spring": {
         # Use April 2025 for spring model selection and tuning.
        "selection_months": ["2025-04-01"],
    },
    "Summer": {
         # Use July 2025 for summer model selection and tuning.
        "selection_months": ["2025-07-01"],
    },
    "Autumn": {
         # Use October 2025 for winter model selection and tuning. Since November 2025 truncates, will also do a separate Autumn run with November 2024.
        "selection_months": ["2024-10-01"], #So that I can have a full autumn
    }
}

# Group each season into an independently named processing chunk so seasonal
# experiments can be run separately or iterated in a consistent order.

def get_seasonal_chunks():
    return {
        "chunk_1_winter": ["Winter"],
        "chunk_2_spring": ["Spring"],
        "chunk_3_summer": ["Summer"],
        "chunk_4_autumn": ["Autumn"]
    }

In [ ]:
# Shared configuration for exponential smoothing and ARIMA-family baselines.
# Run this cell before defining or executing any baseline-model functions.

@dataclass
class BaselineConfig:
    model_family: str
    target_col: str
    output_dir: str
    holdout_start: str = "2025-05-01"

    # Optional SARIMAX predictors and their forecast-time availability.
    exog_cols: Optional[List[str]] = None
    future_known_exog_cols: Tuple[str, ...] = (
        "chlorine_conversion",
    )
    allow_oracle_exog: bool = True

    # Optional ARIMA-family candidate orders.
    candidate_models: Optional[
        List[
            Tuple[
                Tuple[int, int, int],
                Tuple[int, int, int, int]
            ]
        ]
    ] = None

    # Exponential-smoothing models evaluated by the smoothing runner.
    smoothing_models: Tuple[str, ...] = (
        "ses",
        "des",
        "holt_winters"
    )

    # Time-series cross-validation and optimizer settings.
    n_splits: int = 5
    cv_test_size: int = 720
    maxiter: int = 500
    alpha: float = 0.05
    show_plots: bool = True

In [ ]:
# Exponential-smoothing fixed-holdout workflow

# Exponential-smoothing fixed-holdout evaluation


# Fit the requested exponential-smoothing model using the training series.
def fit_smoothing_model(
    train,
    model_name
):
    # Simple exponential smoothing models the series level without an explicit
    # trend or seasonal component.
    if model_name == "ses":
        return SimpleExpSmoothing(
            train,
            initialization_method="estimated"
        ).fit(
            optimized=True
        )

    # Double exponential smoothing models the series level and an additive
    # trend without a seasonal component.
    if model_name == "des":
        return ExponentialSmoothing(
            train,
            trend="add",
            seasonal=None,
            initialization_method="estimated"
        ).fit(
            optimized=True
        )

    # Holt-Winters models the level, additive trend, and additive daily
    # seasonality. With hourly observations, a period of 24 represents one day.
    if model_name == "holt_winters":
        return ExponentialSmoothing(
            train,
            trend="add",
            seasonal="add",
            seasonal_periods=24,
            initialization_method="estimated"
        ).fit(
            optimized=True
        )

    # Stop execution when an unsupported model name is provided.
    raise ValueError(
        "model_name must be ses, des, or holt_winters"
    )


# Fit each configured exponential-smoothing model on observations before the
# fixed UTC holdout boundary and evaluate forecasts on the untouched holdout.
def run_smoothing_fixed_holdout(
    df,
    cfg: BaselineConfig
):
    # Create the output directory for predictions, metrics, and plots.
    ensure_output_dir(
        cfg.output_dir
    )

    # Split the target into chronological training and holdout series.
    train, test = make_date_holdout_split(
        df=df,
        target_col=cfg.target_col,
        holdout_start=cfg.holdout_start
    )

    # Store one metric row and one forecast DataFrame for each model.
    results = []
    forecasts = {}

    # Fit and evaluate every smoothing model listed in the configuration.
    for model_name in cfg.smoothing_models:
        # Fit the selected model using only the training period.
        model = fit_smoothing_model(
            train,
            model_name
        )

        # Produce one forecast for every observation in the fixed holdout.
        forecast = model.forecast(
            len(test)
        )

        # Align the forecast with the UTC timestamps of the holdout observations.
        forecast.index = test.index

        # Combine actual and forecast values into one aligned DataFrame.
        pred_df = pd.DataFrame({
            "actual": test,
            "forecast": forecast
        }, index=test.index)

        # Calculate in-sample training residuals as actual minus fitted values.
        # These residuals are used to estimate empirical forecast intervals.
        training_residuals = (
            train
            - model.fittedvalues
        )

        # Add empirical residual-quantile intervals without using holdout errors.
        pred_df = add_residual_based_intervals(
            pred_df=pred_df,
            pred_col="forecast",
            calibration_residuals=training_residuals,
            alpha=cfg.alpha
        )

        # Calculate the primary holdout forecast metrics.
        metrics = evaluate_predictions(
            pred_df["actual"],
            pred_df["forecast"]
        )

        # Record the model name, holdout boundary, and performance metrics.
        row = {
            "model_family": model_name,
            "holdout_start": cfg.holdout_start,
            **metrics
        }

        results.append(row)
        forecasts[model_name] = pred_df

        # Create a consistent filename prefix for this model's outputs.
        file_stem = (
            f"{model_name}_fixed_holdout"
        )

        # Save timestamped actual values, forecasts, and forecast intervals.
        pred_df.to_csv(
            f"{cfg.output_dir}/"
            f"{file_stem}_forecast.csv"
        )

        # Save the model's holdout metrics.
        pd.DataFrame([metrics]).to_csv(
            f"{cfg.output_dir}/"
            f"{file_stem}_metrics.csv",
            index=False
        )

        # Save plain and interval forecast-comparison plots.
        save_forecast_plots(
            pred_df=pred_df,
            output_dir=cfg.output_dir,
            file_stem=file_stem,
            model_label=model_name.upper(),
            show_plot=cfg.show_plots
        )

        # Save residual values and residual-diagnostic plots.
        save_residual_outputs(
            pred_df=pred_df,
            actual_col="actual",
            pred_col="forecast",
            output_dir=cfg.output_dir,
            file_stem=file_stem,
            show_plots=cfg.show_plots
        )

        # Remove the fitted model and request memory cleanup before fitting the
        # next model.
        del model
        clean_memory()

    # Combine all model-performance records into one comparison table.
    results_df = pd.DataFrame(
        results
    )

    # Save the complete fixed-holdout metric comparison.
    results_df.to_csv(
        f"{cfg.output_dir}/"
        "smoothing_fixed_holdout_metrics.csv",
        index=False
    )

    # Display the metric comparison in the notebook.
    print_table(
        "Classical Exponential Smoothing Results",
        results_df
    )

    # Return both the metric table and timestamped model forecasts.
    return results_df, forecasts

In [ ]:
# ARIMA, SARIMA, and SARIMAX fixed-holdout workflows

# ARIMA, SARIMA, and SARIMAX candidate orders, cross-validation, and fixed
# holdout evaluation


# Define nonseasonal ARIMA candidates.
# Each entry contains:
# - Nonseasonal order: (p, d, q)
# - Seasonal order: (P, D, Q, seasonal period)
#
# ARIMA uses (0, 0, 0, 0) because it has no seasonal component.
ARIMA_CANDIDATE_MODELS = [
    ((1, 0, 1), (0, 0, 0, 0)),
    ((2, 0, 1), (0, 0, 0, 0)),
    ((1, 1, 1), (0, 0, 0, 0)),
    ((2, 1, 1), (0, 0, 0, 0))
]


# Define SARIMA candidates with a 24-hour daily seasonal period.
SARIMA_CANDIDATE_MODELS = [
    ((1, 0, 1), (1, 1, 1, 24)),
    ((1, 0, 1), (1, 1, 2, 24)),
    ((2, 0, 1), (1, 1, 1, 24)),
    ((2, 0, 1), (1, 1, 2, 24)),
    ((1, 1, 1), (1, 1, 1, 24)),
    ((1, 1, 1), (1, 1, 2, 24)),
    ((2, 1, 1), (1, 1, 1, 24)),
    ((2, 1, 1), (1, 1, 2, 24)),
]


# Define SARIMAX candidates using the same nonseasonal and seasonal orders as
# SARIMA. SARIMAX additionally accepts external predictor variables.
SARIMAX_CANDIDATE_MODELS = [
    ((1, 0, 1), (1, 1, 1, 24)),
    ((1, 0, 1), (1, 1, 2, 24)),
    ((2, 0, 1), (1, 1, 1, 24)),
    ((2, 0, 1), (1, 1, 2, 24)),
    ((1, 1, 1), (1, 1, 1, 24)),
    ((1, 1, 1), (1, 1, 2, 24)),
    ((2, 1, 1), (1, 1, 1, 24)),
    ((2, 1, 1), (1, 1, 2, 24)),
]


# Return the default candidate-order list for the requested model family.
def get_default_candidate_models(model_family):
    if model_family == "arima":
        return ARIMA_CANDIDATE_MODELS

    if model_family == "sarima":
        return SARIMA_CANDIDATE_MODELS

    if model_family == "sarimax":
        return SARIMAX_CANDIDATE_MODELS

    raise ValueError(
        "model_family must be arima, sarima, or sarimax"
    )


# Evaluate every candidate ARIMA-family order using expanding-window
# chronological cross-validation before the fixed UTC holdout period.
def run_arima_family_cv(
    df,
    cfg: BaselineConfig
):
    # Create the model-output directory.
    ensure_output_dir(cfg.output_dir)

    # Use manually supplied candidate orders when available; otherwise, use the
    # defaults associated with the configured model family.
    candidate_models = (
        cfg.candidate_models
        or get_default_candidate_models(
            cfg.model_family
        )
    )

    # Use external predictors only when they were supplied in the configuration.
    exog_cols = cfg.exog_cols or []

    # Identify predictors whose future values are not declared available.
    unavailable_future = sorted(
        set(exog_cols)
        - set(cfg.future_known_exog_cols)
    )

    # Prevent nondeployable SARIMAX experiments from using observed future
    # predictor values unless oracle evaluation was explicitly enabled.
    if (
        cfg.model_family == "sarimax"
        and unavailable_future
        and not cfg.allow_oracle_exog
    ):
        raise ValueError(
            "SARIMAX would use observed future covariates "
            "unavailable at forecast time: "
            f"{unavailable_future}. Forecast them first or "
            "explicitly mark an oracle experiment."
        )

    # List the target and external predictors needed for the experiment.
    needed_cols = [
        cfg.target_col
    ] + exog_cols

    # Create an independent modeling copy.
    df_model = df.copy()

    # Convert model inputs to numeric values and mark invalid entries as missing.
    for column in needed_cols:
        df_model[column] = pd.to_numeric(
            df_model[column],
            errors="coerce"
        )

    # Remove observations missing any required model input.
    df_model = df_model.dropna(
        subset=needed_cols
    )

    # Align the holdout boundary with the UTC modeling index.
    holdout_start = ensure_same_tz(
        cfg.holdout_start,
        df_model.index.tz
    )

    # Exclude the untouched final holdout from model selection.
    train_pool = df_model.loc[
        df_model.index < holdout_start
    ].copy()

    # Create expanding-window chronological cross-validation folds.
    tscv = TimeSeriesSplit(
        n_splits=cfg.n_splits,
        test_size=cfg.cv_test_size
    )

    # Store successful candidate results and failed model fits separately.
    results = []
    failures = []

    # Evaluate each nonseasonal and seasonal order combination.
    for order, seasonal_order in candidate_models:
        fold_metrics = []
        failed = False

        print(
            f"Testing {cfg.model_family.upper()} "
            f"order={order}, seasonal={seasonal_order}"
        )

        # Fit the candidate separately in each chronological fold.
        for fold, (train_idx, test_idx) in enumerate(
            tscv.split(train_pool),
            start=1
        ):
            fold_train = train_pool.iloc[
                train_idx
            ]

            fold_test = train_pool.iloc[
                test_idx
            ]

            # Separate the target training and validation series.
            y_train = fold_train[
                cfg.target_col
            ]

            y_test = fold_test[
                cfg.target_col
            ]

            # Supply external predictors only for SARIMAX.
            X_train = (
                fold_train[exog_cols]
                if cfg.model_family == "sarimax"
                else None
            )

            X_test = (
                fold_test[exog_cols]
                if cfg.model_family == "sarimax"
                else None
            )

            try:
                # Fit the requested ARIMA-family model using statsmodels.
                model = SARIMAX(
                    y_train,
                    exog=X_train,
                    order=order,
                    seasonal_order=seasonal_order,
                    enforce_stationarity=False,
                    enforce_invertibility=False
                ).fit(
                    maxiter=cfg.maxiter,
                    disp=False
                )

                # Forecast the complete validation fold.
                forecast = model.forecast(
                    steps=len(y_test),
                    exog=X_test
                )

                # Align forecasts with their UTC target timestamps.
                forecast.index = y_test.index

                # Calculate fold-level forecast metrics.
                metrics = evaluate_predictions(
                    y_test,
                    forecast
                )

                # Record metrics and optimizer-convergence status.
                fold_metrics.append({
                    "fold": fold,
                    "RMSE": metrics["RMSE"],
                    "MAE": metrics["MAE"],
                    "R2": metrics["R2"],
                    "MAPE": metrics["MAPE"],
                    "converged": model.mle_retvals.get(
                        "converged",
                        None
                    )
                })

                # Remove fitted objects before processing the next fold.
                del model, forecast
                clean_memory()

            except Exception as error:
                # Record the failed order and stop evaluating its remaining folds.
                failures.append({
                    "order": order,
                    "seasonal_order": seasonal_order,
                    "fold": fold,
                    "error": str(error)
                })

                failed = True
                clean_memory()
                break

        # Summarize only candidates that completed at least one fold without a
        # raised fitting error.
        if not failed and len(fold_metrics) > 0:
            fold_df = pd.DataFrame(
                fold_metrics
            )

            results.append({
                "order": order,
                "seasonal_order": seasonal_order,
                "RMSE": fold_df["RMSE"].mean(),
                "MAE": fold_df["MAE"].mean(),
                "R2": fold_df["R2"].mean(),
                "MAPE": fold_df["MAPE"].mean(),
                "all_folds_converged": bool(
                    fold_df["converged"].all()
                )
            })

    # Create the candidate-comparison table.
    results_df = pd.DataFrame(
        results
    )

    # Rank successful candidates by average cross-validation RMSE.
    if not results_df.empty:
        results_df = (
            results_df
            .sort_values("RMSE")
            .reset_index(drop=True)
        )

    # Create the fitting-failure table.
    failures_df = pd.DataFrame(
        failures
    )

    return results_df, failures_df


# Fit the selected ARIMA-family order on all pre-holdout observations and
# evaluate it once on the untouched fixed UTC holdout.
def fit_arima_family_holdout(
    df,
    cfg: BaselineConfig,
    order,
    seasonal_order
):
    # Create the model-output directory.
    ensure_output_dir(cfg.output_dir)

    # Retrieve configured SARIMAX predictors.
    exog_cols = cfg.exog_cols or []

    # Identify predictors whose future values are not declared available.
    unavailable_future = sorted(
        set(exog_cols)
        - set(cfg.future_known_exog_cols)
    )

    # Prevent accidental use of unavailable observed future predictors.
    if (
        cfg.model_family == "sarimax"
        and unavailable_future
        and not cfg.allow_oracle_exog
    ):
        raise ValueError(
            "SARIMAX would use observed future covariates "
            "unavailable at forecast time: "
            f"{unavailable_future}. Forecast them first or "
            "explicitly mark an oracle experiment."
        )

    # List the target and external predictors required by the model.
    needed_cols = [
        cfg.target_col
    ] + exog_cols

    # Create an independent modeling copy.
    df_model = df.copy()

    # Convert the required columns to numeric values.
    for column in needed_cols:
        df_model[column] = pd.to_numeric(
            df_model[column],
            errors="coerce"
        )

    # Remove rows with unusable target or predictor values.
    df_model = df_model.dropna(
        subset=needed_cols
    )

    # Align the fixed holdout boundary with the UTC index.
    holdout_start = ensure_same_tz(
        cfg.holdout_start,
        df_model.index.tz
    )

    # Use all observations before the boundary for final model fitting.
    train_df = df_model.loc[
        df_model.index < holdout_start
    ].copy()

    # Reserve observations at and after the boundary for final evaluation.
    test_df = df_model.loc[
        df_model.index >= holdout_start
    ].copy()

    # Separate the training and holdout targets.
    y_train = train_df[
        cfg.target_col
    ]

    y_test = test_df[
        cfg.target_col
    ]

    # Supply external predictors only for SARIMAX.
    X_train = (
        train_df[exog_cols]
        if cfg.model_family == "sarimax"
        else None
    )

    X_test = (
        test_df[exog_cols]
        if cfg.model_family == "sarimax"
        else None
    )

    # Fit the selected order using all pre-holdout observations.
    model = SARIMAX(
        y_train,
        exog=X_train,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(
        maxiter=cfg.maxiter,
        disp=False
    )

    # Generate holdout forecasts and model-based forecast intervals.
    forecast_obj = model.get_forecast(
        steps=len(y_test),
        exog=X_test
    )

    forecast = (
        forecast_obj.predicted_mean
    )

    conf_int = forecast_obj.conf_int(
        alpha=cfg.alpha
    )

    # Align forecasts and intervals with the UTC holdout timestamps.
    forecast.index = y_test.index
    conf_int.index = y_test.index

    # Combine actual and forecast values.
    pred_df = pd.DataFrame({
        "actual": y_test,
        "forecast": forecast
    }, index=y_test.index)

    # Add statsmodels forecast-interval boundaries.
    pred_df = add_sarimax_intervals(
        pred_df,
        conf_int
    )

    # Calculate final untouched-holdout metrics.
    metrics = evaluate_predictions(
        pred_df["actual"],
        pred_df["forecast"]
    )

    # Create a consistent filename prefix for saved outputs.
    file_stem = (
        f"{cfg.model_family}_fixed_holdout"
    )

    # Save timestamped holdout forecasts and intervals.
    pred_df.to_csv(
        f"{cfg.output_dir}/"
        f"{file_stem}_forecast.csv"
    )

    # Save final holdout metrics.
    pd.DataFrame([metrics]).to_csv(
        f"{cfg.output_dir}/"
        f"{file_stem}_metrics.csv",
        index=False
    )

    # Save plain and interval forecast-comparison plots.
    save_forecast_plots(
        pred_df=pred_df,
        output_dir=cfg.output_dir,
        file_stem=file_stem,
        model_label=cfg.model_family.upper(),
        show_plot=cfg.show_plots
    )

    # Save residual values and residual-diagnostic plots.
    save_residual_outputs(
        pred_df=pred_df,
        actual_col="actual",
        pred_col="forecast",
        output_dir=cfg.output_dir,
        file_stem=file_stem,
        show_plots=cfg.show_plots
    )

    return model, pred_df, metrics


# Select the best candidate through cross-validation and evaluate that candidate
# once on the untouched fixed holdout.
def run_arima_family_fixed_holdout(
    df,
    cfg: BaselineConfig
):
    # Run candidate-order cross-validation.
    cv_results_df, cv_failures_df = (
        run_arima_family_cv(
            df,
            cfg
        )
    )

    # Save successful candidate results.
    cv_results_df.to_csv(
        f"{cfg.output_dir}/"
        f"{cfg.model_family}_cv_results.csv",
        index=False
    )

    # Save fitting failures for diagnostics.
    cv_failures_df.to_csv(
        f"{cfg.output_dir}/"
        f"{cfg.model_family}_cv_failures.csv",
        index=False
    )

    # Display successful and failed candidate results.
    print_table(
        f"{cfg.model_family.upper()} CV Results",
        cv_results_df
    )

    print_table(
        f"{cfg.model_family.upper()} CV Failures",
        cv_failures_df
    )

    # Stop when no candidate successfully completes cross-validation.
    if cv_results_df.empty:
        raise ValueError(
            f"No successful CV results for "
            f"{cfg.model_family}"
        )

    # Select the candidate with the lowest average cross-validation RMSE.
    best_order = tuple(
        cv_results_df.iloc[0]["order"]
    )

    best_seasonal_order = tuple(
        cv_results_df.iloc[0][
            "seasonal_order"
        ]
    )

    # Fit and evaluate the selected candidate on the final fixed holdout.
    model, pred_df, metrics = (
        fit_arima_family_holdout(
            df=df,
            cfg=cfg,
            order=best_order,
            seasonal_order=best_seasonal_order
        )
    )

    # Display final untouched-holdout performance.
    print_table(
        f"{cfg.model_family.upper()} "
        "Final Holdout Metrics",
        pd.DataFrame([metrics])
    )

    # Return all model-selection and final-evaluation outputs.
    return {
        "cv_results": cv_results_df,
        "cv_failures": cv_failures_df,
        "best_order": best_order,
        "best_seasonal_order": best_seasonal_order,
        "pred_df": pred_df,
        "metrics": metrics
    }


# Route a baseline experiment to exponential smoothing or the appropriate
# ARIMA-family workflow.
def run_baseline_experiment(
    df,
    cfg: BaselineConfig
):
    # Create the configured output directory.
    ensure_output_dir(
        cfg.output_dir
    )

    # Run the exponential-smoothing workflow.
    if cfg.model_family == "smoothing":
        return run_smoothing_fixed_holdout(
            df,
            cfg
        )

    # Run ARIMA, SARIMA, or SARIMAX model selection and holdout evaluation.
    if cfg.model_family in [
        "arima",
        "sarima",
        "sarimax"
    ]:
        return run_arima_family_fixed_holdout(
            df,
            cfg
        )

    # Stop when an unsupported model-family name is provided.
    raise ValueError(
        "Invalid baseline model_family"
    )


In [ ]:
# Store the shared settings used by ANN and LSTM forecasting experiments.
@dataclass
class ForecastConfig:
    # Identify the deep-learning model family as ANN or LSTM.
    model_family: str

    # Identify the continuous measurement being forecast.
    target_col: str

    # List continuous input features supplied to the model.
    numeric_feature_cols: List[str]

    # List categorical input features used by ANN preprocessing.
    categorical_feature_cols: List[str]

    # Store the directory used for models, metrics, predictions, and plots.
    output_dir: str


    # Forecast the target this many hourly steps into the future.
    horizon_steps: int = 12

    # Use this many historical hourly observations as the input window.
    lookback: int = 24

    # List explicit lag lengths used by ANN feature engineering when applicable.
    lag_steps: Optional[List[int]] = None


    # Test models using these alternative training-window lengths in months.
    train_length_options: Tuple[int, ...] = (
        12,
        24
    )

    # Reserve this many calendar months for validation.
    val_months: int = 1

    # Reserve this many calendar months for final testing.
    test_months: int = 1


    # Test these neural-network hidden-layer sizes.
    hidden_units_options: Tuple[int, ...] = (
        64,
        128
    )

    # Test these numbers of observations per training batch.
    batch_size_options: Tuple[int, ...] = (
        16,
        64
    )


    # Set the maximum number of training epochs.
    epochs: int = 200

    # Stop training after this many epochs without validation improvement.
    patience: int = 20


# Store the settings used specifically by Temporal Fusion Transformer models.
@dataclass
class TFTConfig:
    # Identify the model family as TFT.
    model_family: str

    # Identify the continuous measurement being forecast.
    target_col: str

    # List measurements and features available throughout the historical input
    # window.
    historical_feature_cols: List[str]

    # List features known for every timestamp in the future forecast horizon.
    # These should use explicit UTC calendar features when time features are
    # included.
    future_known_feature_cols: List[str]

    # List numeric attributes that remain constant across the time series.
    static_numeric_cols: List[str]

    # Store the directory used for models, metrics, predictions, and plots.
    output_dir: str


    # Use this many historical hourly observations as the TFT encoder window.
    lookback: int = 24

    # Forecast the target this many hourly steps into the future.
    horizon_steps: int = 12


    # Test TFT models using these training-window lengths in months.
    train_length_options: Tuple[int, ...] = (
        12,
        24
    )

    # Reserve this many calendar months for validation.
    val_months: int = 1

    # Reserve this many calendar months for final testing.
    test_months: int = 1


    # Test these TFT hidden-state sizes.
    hidden_units_options: Tuple[int, ...] = (
        16,
        32
    )

    # Test these multi-head attention configurations.
    num_heads_options: Tuple[int, ...] = (
        2,
        4
    )

    # Test these numbers of observations per training batch.
    batch_size_options: Tuple[int, ...] = (
        16,
        32
    )


    # Set the maximum number of TFT training epochs.
    epochs: int = 60

    # Stop training after this many epochs without validation improvement.
    patience: int = 10

    # Set the optimizer learning rate.
    learning_rate: float = 5e-4

    # Estimate lower, median, and upper conditional forecast quantiles.
    quantiles: Tuple[float, ...] = (
        0.1,
        0.5,
        0.9
    )


# Select only the columns required by one model configuration and return a
# chronologically ordered model-specific dataframe.
def build_model_df_for_config(
    df,
    cfg
):
    # Every forecasting model uses the canonical continuous UTC timeline.
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("The modeling dataframe must use a DatetimeIndex.")

    if str(df.index.tz) != "UTC":
        raise ValueError("The modeling dataframe index must use UTC.")

    # Prevent ambiguous Miami-local calendar aliases from entering the TFT
    # future-known branch. Explicit utc_* features are required there.
    if cfg.model_family == "tft":
        ambiguous_calendar_cols = {
            "year", "month", "day", "hour",
            "day_of_week", "dayofweek", "month_year",
            "hour_sin", "hour_cos", "dow_sin", "dow_cos"
        }
        ambiguous_future = sorted(
            set(cfg.future_known_feature_cols) & ambiguous_calendar_cols
        )
        if ambiguous_future:
            raise ValueError(
                "TFT future-known calendar features must use explicit UTC "
                f"column names. Ambiguous columns: {ambiguous_future}"
            )

    # LSTM models use numeric historical inputs and the target.
    if cfg.model_family == "lstm":
        needed_cols = (
            cfg.numeric_feature_cols
            + [cfg.target_col]
        )

    # ANN models use numeric inputs, categorical inputs, and the target.
    elif cfg.model_family == "ann":
        needed_cols = (
            cfg.numeric_feature_cols
            + cfg.categorical_feature_cols
            + [cfg.target_col]
        )

    # TFT models use historical inputs, future-known inputs, static inputs, and
    # the target.
    elif cfg.model_family == "tft":
        needed_cols = (
            cfg.historical_feature_cols
            + cfg.future_known_feature_cols
            + cfg.static_numeric_cols
            + [cfg.target_col]
        )

    # Stop when the configuration contains an unsupported model family.
    else:
        raise ValueError(
            "Unknown model_family"
        )

    # Remove duplicate column names while preserving their original order.
    needed_cols = list(
        dict.fromkeys(needed_cols)
    )

    # Confirm that every configured input exists in the master dataframe.
    missing_cols = [
        column
        for column in needed_cols
        if column not in df.columns
    ]

    if missing_cols:
        raise ValueError(
            "Missing columns required by config: "
            f"{missing_cols}"
        )

    # Create a model-specific copy containing only the configured columns.
    out = (
        df[needed_cols]
        .copy()
        .sort_index()
    )

    # Add the constant TFT placeholder when it was not already supplied.
    if (
        cfg.model_family == "tft"
        and "static_dummy" not in out.columns
    ):
        out["static_dummy"] = 0.0

    return out

#Univariate and Multivariable Model Configuration

In [ ]:
# Deep-learning model configurations

# Deep-learning configurations for univariate and multivariable ANN, LSTM, and
# Temporal Fusion Transformer forecasting experiments.


# Configure a univariate LSTM using only historical residual chlorine.
cfg_uni_lstm = ForecastConfig(
    # Select the LSTM training and sequence-building workflow.
    model_family="lstm",

    # Forecast residual chlorine.
    target_col="totalchlorine",

    # Use historical residual chlorine as the only sequence input.
    numeric_feature_cols=[
        "totalchlorine"
    ],

    # No categorical features are used by this LSTM.
    categorical_feature_cols=[],

    # Save univariate LSTM outputs in a dedicated directory.
    output_dir=(
        f"{BASE_OUTPUT}/"
        "DeepLearning/LSTM/univariate"
    ),

    # Use the previous 24 hourly observations as model input.
    lookback=24,

    # Forecast residual chlorine 12 hours ahead.
    horizon_steps=12,

    # LSTM sequences provide historical context, so separate lag columns are not
    # created.
    lag_steps=None,

    # Compare models trained on 12 and 24 months of historical data.
    train_length_options=(
        12,
        24
    ),

    # Compare LSTM networks with 64 and 128 hidden units.
    hidden_units_options=(
        64,
        128
    ),

    # Compare batch sizes of 16 and 64 sequences.
    batch_size_options=(
        16,
        64
    ),

    # Train for at most 200 epochs.
    epochs=200,

    # Stop after 20 epochs without validation improvement.
    patience=20
)


# Configure a multivariable LSTM using historical chlorine, flow rate,
# turbidity, and the chlorine-conversion operating indicator.
cfg_multi_lstm = ForecastConfig(
    model_family="lstm",
    target_col="totalchlorine",

    # All inputs are historical sequence values rather than future-known values.
    numeric_feature_cols=[
        "totalchlorine",
        "finishwater",
        "turbidity",
        # "chlorine_conversion"
    ],

    categorical_feature_cols=[],

    # Store this multivariable LSTM experiment separately.
    output_dir=(
        f"{BASE_OUTPUT}/"
        "DeepLearning/LSTM/flowrate_turbidity"
    ),

    lookback=24,
    horizon_steps=12,
    lag_steps=None,

    train_length_options=(
        12,
        24
    ),

    hidden_units_options=(
        64,
        128
    ),

    batch_size_options=(
        16,
        64
    ),

    epochs=200,
    patience=20
)


# Configure a univariate ANN using lagged residual-chlorine values.
cfg_uni_ann = ForecastConfig(
    # Select the ANN tabular-feature training workflow.
    model_family="ann",

    # Forecast residual chlorine.
    target_col="totalchlorine",

    # No additional numeric predictors are used in the univariate experiment.
    # Historical chlorine enters through the lag features specified below.
    numeric_feature_cols=[],

    categorical_feature_cols=[],

    # Save univariate ANN outputs in a dedicated directory.
    output_dir=(
        f"{BASE_OUTPUT}/"
        "DeepLearning/ANN/univariate"
    ),

    # Retain the common 24-hour lookback setting in the configuration.
    lookback=24,

    # Forecast residual chlorine 12 hours ahead.
    horizon_steps=12,

    # Create chlorine lags at 1, 2, 3, 6, 12, and 24 hours.
    lag_steps=[
        1,
        2,
        3,
        6,
        12,
        24
    ],

    train_length_options=(
        12,
        24
    ),

    # Test smaller hidden layers appropriate for the tabular ANN.
    hidden_units_options=(
        8,
        16,
        32
    ),

    batch_size_options=(
        16,
        64
    ),

    epochs=200,
    patience=20
)


# Configure a multivariable ANN using lagged chlorine plus turbidity, flow rate,
# and the chlorine-conversion indicator.
cfg_multi_ann = ForecastConfig(
    model_family="ann",
    target_col="totalchlorine",

    # Use the listed plant measurements and operational indicator as additional
    # numeric predictors.
    numeric_feature_cols=[
        "turbidity",
        "finishwater",
        # "chlorine_conversion"
    ],

    # Alternative predictor configurations can be enabled when needed.
    # numeric_feature_cols=["turbidity"]
    # categorical_feature_cols=["chlorine_conversion"]

    # Treat the conversion indicator as numeric in the active configuration.
    categorical_feature_cols=[],

    # Save this ANN experiment in its configured output directory.
    output_dir=(
        f"{BASE_OUTPUT}/"
        "DeepLearning/ANN/flowrate_turbidity"
    ),

    lookback=24,
    horizon_steps=12,

    # Create target and predictor lag features at these hourly offsets.
    lag_steps=[
        1,
        2,
        3,
        6,
        12,
        24
    ],

    train_length_options=(
        12,
        24
    ),

    hidden_units_options=(
        8,
        16,
        32
    ),

    batch_size_options=(
        16,
        64
    ),

    epochs=200,
    patience=20
)


# Configure a univariate TFT using historical residual chlorine and UTC calendar
# features that are deterministically known throughout the forecast horizon.
cfg_uni_tft = TFTConfig(
    # Select the Temporal Fusion Transformer workflow.
    model_family="tft",

    # Forecast residual chlorine.
    target_col="totalchlorine",

    # Use only past residual chlorine as a historical process input.
    historical_feature_cols=[
        "totalchlorine"
    ],

    # UTC hour sine and cosine are known for every future timestamp and preserve
    # the circular relationship between hour 23 and hour 0.
    future_known_feature_cols=[
        "utc_hour_sin",
        "utc_hour_cos"
    ],

    # Supply the constant numeric placeholder required by the TFT structure.
    static_numeric_cols=[
        "static_dummy"
    ],

    # Save univariate TFT outputs in a dedicated directory.
    output_dir=(
        f"{BASE_OUTPUT}/"
        "DeepLearning/TFT/univariate"
    ),

    # Use the previous 24 hours to forecast 12 hours ahead.
    lookback=24,
    horizon_steps=12,

    train_length_options=(
        12,
        24
    ),

    # Compare TFT hidden-state sizes of 16 and 32.
    hidden_units_options=(
        16,
        32
    ),

    # Compare attention configurations with two and four heads.
    num_heads_options=(
        2,
        4
    ),

    batch_size_options=(
        16,
        32
    ),

    # Use fewer maximum epochs than ANN and LSTM because TFT is more expensive.
    epochs=60,

    # Stop after ten epochs without validation improvement.
    patience=10,

    # Set the Adam optimizer learning rate.
    learning_rate=5e-4,

    # Estimate the 10th, 50th, and 90th conditional forecast quantiles.
    quantiles=(
        0.1,
        0.5,
        0.9
    )
)


# Configure a multivariable TFT using historical plant measurements and the
# conversion indicator while keeping future inputs limited to known UTC time.
cfg_multi_tft = TFTConfig(
    model_family="tft",
    target_col="totalchlorine",

    # Historical inputs are taken only from timestamps available before the
    # forecast horizon.
    historical_feature_cols=[
        "totalchlorine",
        "turbidity",
        "finishwater",
        # "chlorine_conversion"
    ],

    # The conversion indicator remains historical-only because its future
    # schedule is not assumed to be known.
    future_known_feature_cols=[
        "utc_hour_sin",
        "utc_hour_cos"
    ],

    static_numeric_cols=[
        "static_dummy"
    ],

    # Save the multivariable TFT as the full-model configuration.
    output_dir=(
        f"{BASE_OUTPUT}/"
        "DeepLearning/TFT/flowrate_turbidity"
    ),

    lookback=24,
    horizon_steps=12,

    train_length_options=(
        12,
        24
    ),

    hidden_units_options=(
        16,
        32
    ),

    num_heads_options=(
        2,
        4
    ),

    batch_size_options=(
        16,
        32
    ),

    epochs=60,
    patience=10,
    learning_rate=5e-4,

    quantiles=(
        0.1,
        0.5,
        0.9
    )
)


# Confirm that the forecast target is never supplied as a future-known TFT
# input, which would expose future chlorine values and cause target leakage.
assert (
    cfg_uni_tft.target_col
    not in cfg_uni_tft.future_known_feature_cols
), "Leakage detected in univariate TFT configuration."

assert (
    cfg_multi_tft.target_col
    not in cfg_multi_tft.future_known_feature_cols
), "Leakage detected in multivariable TFT configuration."


In [ ]:
# Deep-learning prediction-interval helpers

# Prediction-interval interpretation
#
# Each model family estimates uncertainty differently:
#
# - SES, DES, and Holt-Winters:
#   Add empirical lower and upper quantiles from training or calibration
#   residuals to each point forecast.
#
# - ARIMA, SARIMA, and SARIMAX:
#   Use model-based forecast intervals returned by statsmodels. These depend on
#   the fitted-model assumptions and estimated forecast variance.
#
# - ANN and LSTM:
#   Use 100 stochastic inference passes with dropout kept active. The mean of
#   those passes is the point prediction, while the 2.5th and 97.5th
#   percentiles form an approximate 95% uncertainty interval.
#
# - TFT:
#   Directly learns conditional quantile forecasts. q10 and q90 define a nominal
#   central 80% predictive interval rather than a 95% interval.
#
# Because the interval methods and nominal coverage levels differ, interval
# widths should not be compared without also reporting empirical coverage and
# average interval width:
#
# coverage = mean((actual >= lower_ci) & (actual <= upper_ci))
# average_width = mean(upper_ci - lower_ci)
#
# These differences affect uncertainty interpretation only. They do not create
# point-forecast leakage when intervals use training or independent calibration
# information and the final holdout remains untouched.


# Add available TFT quantile bounds or independent residual-based fallback
# bounds to a deep-learning prediction DataFrame.
def add_dl_intervals_to_prediction_df(
    pred_df,
    actual_col,
    pred_col,
    alpha=0.05,
    calibration_residuals=None
):
    # Work on a copy so the original predictions are not modified.
    pred_df = pred_df.copy()

    # Use TFT's learned q10 and q90 forecasts when both are available.
    # These columns form a nominal central 80% predictive interval.
    if (
        "q10" in pred_df.columns
        and "q90" in pred_df.columns
    ):
        pred_df["lower_ci"] = pred_df["q10"]
        pred_df["upper_ci"] = pred_df["q90"]
        pred_df["interval_method"] = (
            "tft_learned_q10_q90"
        )

        return pred_df

    # Require independent calibration errors when learned quantile forecasts are
    # unavailable. Final-holdout residuals must not be used for this purpose.
    if calibration_residuals is None:
        raise ValueError(
            "Independent calibration residuals are required "
            "for fallback DL intervals."
        )

    # Add empirical residual-quantile intervals to the point forecasts.
    return add_residual_based_intervals(
        pred_df=pred_df,
        pred_col=pred_col,
        calibration_residuals=calibration_residuals,
        alpha=alpha
    )


# Generate approximate ANN or LSTM prediction intervals by keeping dropout
# active during repeated stochastic inference passes.
def mc_dropout_predict_keras(
    model,
    X,
    n_passes=100,
    seed=SEED
):
    # Reset the TensorFlow seed immediately before sampling so the complete
    # sequence of stochastic passes is reproducible.
    tf.random.set_seed(
        int(seed)
    )

    # Store one prediction vector from every stochastic inference pass.
    predictions = []

    # training=True keeps dropout active even though the model is not being
    # fitted during these calls.
    for _ in range(n_passes):
        prediction = (
            model(
                X,
                training=True
            )
            .numpy()
            .reshape(-1)
        )

        predictions.append(
            prediction
        )

    # Arrange predictions as:
    # rows = stochastic passes
    # columns = forecast observations
    predictions = np.array(
        predictions
    )

    # Use the average stochastic prediction as the point forecast.
    mean_prediction = predictions.mean(
        axis=0
    )

    # Use the 2.5th percentile as the approximate lower 95% boundary.
    lower_ci = np.quantile(
        predictions,
        0.025,
        axis=0
    )

    # Use the 97.5th percentile as the approximate upper 95% boundary.
    upper_ci = np.quantile(
        predictions,
        0.975,
        axis=0
    )

    return (
        mean_prediction,
        lower_ci,
        upper_ci
    )


# Add previously calculated MC-dropout boundaries and an interval-method label
# to a deep-learning prediction DataFrame.
def add_mc_dropout_interval_columns(
    pred_df,
    lower_ci,
    upper_ci
):
    # Work on a copy so the original predictions are not modified.
    pred_df = pred_df.copy()

    # Store the approximate lower and upper MC-dropout boundaries.
    pred_df["lower_ci"] = lower_ci
    pred_df["upper_ci"] = upper_ci

    # Record the method used to calculate the interval.
    pred_df["interval_method"] = (
        "mc_dropout_approximate_uncertainty"
    )

    return pred_df

#DL Supporting Functions

In [ ]:
# Seasonal configuration-selection helper

# Summarize each hyperparameter configuration by season and select the
# configuration with the lowest average RMSE. Average MAE is used as the
# tie-breaker when configurations have the same RMSE.
def get_best_configuration_per_season(results_df, hyperparam_cols):

    # Define one unique configuration using its season and hyperparameters.
    group_cols = ["season"] + hyperparam_cols

    # Average the evaluation metrics across all runs belonging to the same
    # seasonal hyperparameter configuration.
    config_summary = (
        results_df
        .groupby(group_cols, as_index=False)
        .agg({
            "RMSE": "mean",
            "MAE": "mean",
            "R2": "mean",
            "MAPE": "mean"
        })
    )

    # Rank configurations separately within each season. Lower RMSE is the
    # primary selection criterion, followed by lower MAE as a tie-breaker.
    best_config_per_season = (
        config_summary
        .sort_values(
            ["season", "RMSE", "MAE"],
            ascending=[True, True, True]
        )
        .groupby("season", as_index=False)
        .first()
    )

    # Return both the complete configuration summary and the selected
    # best-performing configuration for each season.
    return config_summary, best_config_per_season

In [ ]:
# LSTM data-preparation and model-building helpers

# LSTM data-preparation and model-building helpers

# Fit the standardization parameters using the training data only.
# Restricting the fit to training data prevents validation/test information
# from leaking into the feature means and standard deviations.
def fit_lstm_scaler(train_df, model_cols):
    scaler = StandardScaler()
    scaler.fit(train_df[model_cols].values)
    return scaler


# Apply the training-fitted scaler to another DataFrame while preserving its
# original index and any columns not included in model_cols.
def apply_lstm_scaler(df_in, scaler, model_cols):
    out = df_in.copy()
    out[model_cols] = scaler.transform(
        df_in[model_cols].values
    )
    return out


# Convert standardized target values back to their original measurement units.
# The correct mean and scale are retrieved from the target column's position
# within the jointly fitted scaler.
def inverse_target_from_joint_scaler(
    y_scaled,
    scaler,
    model_cols,
    target_col
):
    target_idx = model_cols.index(target_col)

    return (
        y_scaled * scaler.scale_[target_idx]
        + scaler.mean_[target_idx]
    )


# Convert a continuous time series into supervised LSTM samples.
# Each input contains `lookback` consecutive observations, while each target
# is the value located `horizon` time steps after the final input observation.
def build_lstm_sequences(
    values,
    timestamps,
    lookback,
    horizon,
    n_input_features,
    target_idx
):
    X = []
    y = []
    y_times = []

    # The first sample ends after enough observations exist for the complete
    # lookback window.
    start_t = lookback - 1

    # Stop early enough for every sample to have its requested future target.
    end_t = len(values) - horizon

    for t in range(start_t, end_t):

        # Select the historical input window ending at position t.
        X.append(
            values[
                t - lookback + 1:t + 1,
                :n_input_features
            ]
        )

        # Select the target value at the specified forecast horizon.
        y.append(values[t + horizon, target_idx])

        # Preserve the target timestamp for aligning forecasts with actuals.
        y_times.append(timestamps[t + horizon])

    return (
        np.array(X, dtype=np.float32),
        np.array(y, dtype=np.float32),
        y_times
    )


# Build training, validation, and test sequences while giving validation and
# test samples access to the preceding historical context. Targets are filtered
# so they remain strictly inside their assigned validation or test period.
def make_lstm_split_sequences(
    train_df,
    val_df,
    test_df,
    input_cols,
    target_col,
    lookback,
    horizon
):
    # Create one ordered column list without duplicating the target when it is
    # already included among the input features.
    model_cols = list(
        dict.fromkeys(
            input_cols + [target_col]
        )
    )

    # Record the target's position in the joint feature matrix.
    target_idx = model_cols.index(target_col)

    # Only columns explicitly listed in input_cols are supplied to the LSTM.
    n_input_features = len(input_cols)

    # Build training sequences entirely from the training period.
    X_train, y_train, _ = build_lstm_sequences(
        train_df[model_cols].to_numpy(),
        train_df.index.to_numpy(),
        lookback,
        horizon,
        n_input_features,
        target_idx
    )

    # Add recent training observations before validation so the first
    # validation forecasts have sufficient historical input context.
    val_context = pd.concat(
        [
            train_df.tail(lookback + horizon),
            val_df
        ],
        axis=0
    )

    # Initially build every valid sequence from the combined context.
    X_val_all, y_val_all, ytime_val_all = build_lstm_sequences(
        val_context[model_cols].to_numpy(),
        val_context.index.to_numpy(),
        lookback,
        horizon,
        n_input_features,
        target_idx
    )

    # Keep only sequences whose target timestamps fall inside validation.
    ytime_val_all = np.asarray(ytime_val_all)
    val_start = val_df.index.min()
    val_end = val_df.index.max()

    val_mask = np.asarray(
        (ytime_val_all >= val_start)
        & (ytime_val_all <= val_end),
        dtype=bool
    )

    X_val = X_val_all[val_mask]
    y_val = y_val_all[val_mask]

    # Add the most recent pre-test observations so the first test forecasts
    # also have enough historical input context.
    test_context = pd.concat(
        [
            val_context.tail(lookback + horizon),
            test_df
        ],
        axis=0
    )

    # Build all possible test-context sequences before filtering by target time.
    X_test_all, y_test_all, ytime_test_all = build_lstm_sequences(
        test_context[model_cols].to_numpy(),
        test_context.index.to_numpy(),
        lookback,
        horizon,
        n_input_features,
        target_idx
    )

    # Keep only sequences whose target timestamps fall inside the test period.
    ytime_test_all = np.asarray(ytime_test_all)
    test_start = test_df.index.min()
    test_end = test_df.index.max()

    test_mask = np.asarray(
        (ytime_test_all >= test_start)
        & (ytime_test_all <= test_end),
        dtype=bool
    )

    X_test = X_test_all[test_mask]
    y_test = y_test_all[test_mask]

    # Retain test target timestamps for constructing the final prediction
    # DataFrame in the correct chronological order.
    ytime_test = list(ytime_test_all[test_mask])

    return (
        X_train,
        y_train,
        X_val,
        y_val,
        X_test,
        y_test,
        ytime_test
    )


# Previous LSTM architecture retained for reference.
# This version did not include dropout and therefore could not produce
# stochastic MC-dropout uncertainty estimates.
# def build_lstm_model(input_shape, hidden_units):
#     model = Sequential([
#         Input(shape=input_shape),
#         LSTM(hidden_units),
#         Dense(1)
#     ])
#     model.compile(
#         optimizer="adam",
#         loss="mse",
#         metrics=["mae"]
#     )
#     return model


# Build and compile a single-layer LSTM regression model.
# Dropout regularizes the input connections during training and can remain
# active during MC-dropout prediction passes for approximate uncertainty.
def build_lstm_model(
    input_shape,
    hidden_units,
    dropout_rate=0.10
):
    model = Sequential([
        Input(shape=input_shape),

        # recurrent_dropout remains zero because nonzero recurrent dropout can
        # substantially slow GPU-accelerated LSTM training.
        LSTM(
            hidden_units,
            dropout=dropout_rate,
            recurrent_dropout=0.0
        ),

        # Produce one continuous forecast for each input sequence.
        Dense(1)
    ])

    # Optimize mean squared error while reporting mean absolute error as an
    # additional training and validation metric.
    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    return model

In [ ]:
# ANN data-preparation and model-building helpers

# ANN data-preparation and model-building helpers

# Create lagged copies of the selected columns.
# For hourly data, lag 1 represents one hour earlier and lag 24 represents
# the value observed 24 hours earlier.
def add_lags(data, cols, lags):
    out = data.copy()

    for col in cols:
        for lag in lags:
            out[f"{col}_lag_{lag}"] = out[col].shift(lag)

    return out


# Create the supervised-learning table used by the ANN.
# Each row contains information available at forecast origin t and the target
# observed `horizon_steps` hours after that forecast origin.
def build_ann_frame(
    df_in,
    target_col,
    numeric_base_cols,
    categorical_cols,
    lag_steps,
    horizon_steps
):
    out = df_in.copy()

    # Extract the UTC hour from the canonical UTC index.
    out["hour_proxy"] = out.index.hour

    # Store the exact UTC timestamp of the value being predicted. This keeps
    # forecasts, actual values, plots, and saved residuals aligned to t+horizon.
    out["target_timestamp"] = (
        pd.Series(out.index, index=out.index)
        .shift(-horizon_steps)
    )

    # Create historical lag features for the numeric predictors and target.
    out = add_lags(
        out,
        numeric_base_cols + [target_col],
        lag_steps
    )

    # Align each row with the future target located `horizon_steps` ahead.
    # The active horizon determines how far ahead the ANN target is located.
    out["target_future"] = out[target_col].shift(
        -horizon_steps
    )

    # Remove rows without sufficient lag history or an available future target.
    out = out.dropna().copy()

    return out


# Create the ordered numeric and complete feature lists used by the ANN.
def get_ann_feature_lists(
    target_col,
    numeric_base_cols,
    categorical_cols,
    lag_steps
):
    lag_cols = []

    # Reconstruct the names of all lagged variables created above.
    for col in numeric_base_cols + [target_col]:
        for lag in lag_steps:
            lag_cols.append(f"{col}_lag_{lag}")

    # Include the current target value observed at forecast origin t, the
    # current numeric predictors, historical lags, and the UTC hour.
    # Current chlorine is observed information and is not the future target.
    numeric_cols = list(
        dict.fromkeys(
            [target_col]
            + numeric_base_cols
            + lag_cols
            + ["hour_proxy"]
        )
    )

    # Add any categorical predictors and remove duplicate column names.
    all_feature_cols = list(
        dict.fromkeys(
            numeric_cols + categorical_cols
        )
    )

    return numeric_cols, all_feature_cols


# Fit the ANN feature preprocessing pipeline using training data only.
# This prevents validation and test information from influencing scaling or
# categorical encoding.
def fit_ann_preprocessor(
    train_df,
    numeric_cols,
    categorical_cols,
    all_feature_cols
):
    if len(categorical_cols) == 0:

        # Standardize numeric predictors when no categorical variables exist.
        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    StandardScaler(),
                    numeric_cols
                )
            ]
        )

    else:

        # Standardize numeric predictors and one-hot encode categorical ones.
        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    StandardScaler(),
                    numeric_cols
                ),
                (
                    "cat",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False
                    ),
                    categorical_cols
                )
            ]
        )

    # Learn preprocessing parameters exclusively from the training period.
    preprocessor.fit(train_df[all_feature_cols])

    return preprocessor


# Fit a separate target scaler using future targets from the training set only.
def fit_ann_target_scaler(train_df):
    scaler = StandardScaler()

    scaler.fit(
        train_df[["target_future"]].values
    )

    return scaler


# Transform ANN predictors using the preprocessing pipeline fitted on training.
def transform_ann_X(
    df_in,
    preprocessor,
    all_feature_cols
):
    return preprocessor.transform(
        df_in[all_feature_cols]
    )


# Standardize the future target using the training-fitted target scaler.
def transform_ann_y(df_in, scaler):
    return scaler.transform(
        df_in[["target_future"]].values
    ).reshape(-1)


# Convert standardized predictions back to residual-chlorine units.
def inverse_ann_target(y_scaled, scaler):
    return scaler.inverse_transform(
        np.asarray(y_scaled).reshape(-1, 1)
    ).reshape(-1)


# Build training, validation, and test ANN tables.
# Historical context from the preceding split is included so the first rows of
# validation and test have enough observations to construct their lag features.
def make_ann_month_tables(
    train_df_raw,
    val_df_raw,
    test_df_raw,
    target_col,
    numeric_base_cols,
    categorical_cols,
    lag_steps,
    horizon_steps
):
    # Attach recent training observations to the beginning of validation.
    val_context_raw = pd.concat(
        [
            train_df_raw.tail(
                max(lag_steps) + horizon_steps
            ),
            val_df_raw
        ],
        axis=0
    )

    # Attach recent pre-test observations to the beginning of the test period.
    test_context_raw = pd.concat(
        [
            val_context_raw.tail(
                max(lag_steps) + horizon_steps
            ),
            test_df_raw
        ],
        axis=0
    )

    # Construct the supervised training table.
    train_df = build_ann_frame(
        train_df_raw,
        target_col,
        numeric_base_cols,
        categorical_cols,
        lag_steps,
        horizon_steps
    )

    # Construct validation sequences using the added historical context.
    val_context = build_ann_frame(
        val_context_raw,
        target_col,
        numeric_base_cols,
        categorical_cols,
        lag_steps,
        horizon_steps
    )

    # Construct test sequences using the added historical context.
    test_context = build_ann_frame(
        test_context_raw,
        target_col,
        numeric_base_cols,
        categorical_cols,
        lag_steps,
        horizon_steps
    )

    # Select rows by their future target timestamps, not by forecast-origin
    # timestamps. This evaluates every target that belongs to validation.
    val_df = val_context[
        (val_context["target_timestamp"] >= val_df_raw.index.min())
        & (val_context["target_timestamp"] <= val_df_raw.index.max())
    ].copy()

    # Apply the same target-time alignment to the untouched test period.
    test_df = test_context[
        (test_context["target_timestamp"] >= test_df_raw.index.min())
        & (test_context["target_timestamp"] <= test_df_raw.index.max())
    ].copy()

    return train_df, val_df, test_df


# Previous ANN architecture retained for reference.
# This version did not contain dropout and therefore could not support the
# stochastic MC-dropout prediction intervals used by the current model.
# def build_ann_model(input_dim, hidden_units):
#     model = keras.Sequential([
#         layers.Input(shape=(input_dim,)),
#         layers.Dense(hidden_units, activation="tanh"),
#         layers.Dense(1, activation="linear")
#     ])
#
#     model.compile(
#         optimizer=keras.optimizers.Nadam(),
#         loss="mse",
#         metrics=["mae"]
#     )
#
#     return model


# Build and compile the ANN regression model.
# Dropout provides training regularization and supports MC-dropout uncertainty
# estimation when predictions are later generated with training=True.
def build_ann_model(
    input_dim,
    hidden_units,
    dropout_rate=0.10
):
    model = keras.Sequential([
        # Define the number of preprocessed input features.
        layers.Input(shape=(input_dim,)),

        # Learn nonlinear relationships between the predictors and target.
        layers.Dense(
            hidden_units,
            activation="tanh"
        ),

        # Randomly deactivate a proportion of hidden units during training.
        layers.Dropout(dropout_rate),

        # Produce one continuous standardized forecast.
        layers.Dense(
            1,
            activation="linear"
        )
    ])

    # Train using mean squared error and report mean absolute error.
    model.compile(
        optimizer=keras.optimizers.Nadam(),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [ ]:
# TFT data-preparation and model-building helpers

# TFT data-preparation, dataset, and model-building helpers

# Attempt to import OmegaConf, which converts the TFT configuration dictionary
# into the configuration-object format expected by tft_torch.
try:
    from omegaconf import OmegaConf

except Exception as e:
    # Report the import problem without stopping the rest of the notebook.
    # TFT functions will not work until OmegaConf is installed successfully.
    print("OmegaConf import issue:", e)


# Attempt to import the TFT model and its accompanying loss functions.
try:
    from tft_torch.tft import TemporalFusionTransformer
    import tft_torch.loss as tft_loss

except Exception as e:
    # Report the problem without stopping non-TFT sections of the notebook.
    # The TFT experiment will still fail if these imports remain unavailable.
    print(
        "tft_torch import issue. If running TFT, install/import "
        "tft_torch as in your original notebook:",
        e
    )


# Provide a PyTorch Dataset that stores each TFT input component in a
# dictionary and returns one dictionary of tensors for every sample.
class DictTensorDataset(Dataset):

    # Convert every numeric array in array_dict to a float32 PyTorch tensor.
    def __init__(self, array_dict):
        self.keys_list = list(array_dict.keys())
        self.data = {}

        for key, values in array_dict.items():
            self.data[key] = torch.tensor(
                values,
                dtype=torch.float32
            )

    # Return the total number of samples using the first stored tensor.
    # All arrays are expected to have the same number of samples.
    def __len__(self):
        return self.data[self.keys_list[0]].shape[0]

    # Return all TFT input components belonging to one sample.
    def __getitem__(self, idx):
        return {
            key: self.data[key][idx]
            for key in self.keys_list
        }


# Convert a scaled time-series DataFrame into the array structure required by
# the TFT. Each sample contains a historical window, future-known predictors
# for the forecast horizon, static features, and the target at the final step.
def build_tft_arrays(
    df_scaled,
    historical_feature_cols,
    future_known_feature_cols,
    static_numeric_cols,
    target_col,
    lookback,
    horizon
):
    # Extract historical variables observed up to the forecast origin.
    hist_values = (
        df_scaled[historical_feature_cols]
        .values
        .astype(np.float32)
    )

    # Extract variables genuinely known throughout the future horizon.
    # In this notebook, these should be deterministic UTC calendar features,
    # rather than future chlorine, turbidity, or flowrate measurements.
    fut_values = (
        df_scaled[future_known_feature_cols]
        .values
        .astype(np.float32)
    )

    # Extract the residual-chlorine target values.
    target_values = (
        df_scaled[target_col]
        .values
        .astype(np.float32)
    )

    # Extract static numeric variables that remain constant for the series.
    static_values = (
        df_scaled[static_numeric_cols]
        .values
        .astype(np.float32)
    )

    # Initialize containers for all generated TFT samples.
    hist_list = []
    fut_list = []
    tgt_list = []
    static_num_list = []
    tgt_times = []

    # Calculate the number of complete windows that can be constructed.
    # Every sample must contain both the full lookback and forecast horizon.
    max_t = len(df_scaled) - lookback - horizon + 1

    for start in range(max_t):
        # Define the beginning and end of the historical input window.
        hist_start = start
        hist_end = start + lookback

        # Define the end of the future-known input window.
        fut_end = hist_end + horizon

        # Select the complete historical lookback window.
        hist_x = hist_values[
            hist_start:hist_end,
            :
        ]

        # Select known future predictors for every step in the horizon.
        fut_x = fut_values[
            hist_end:fut_end,
            :
        ]

        # Use the target at the final forecast-horizon step.
        # For horizon=12, this is the target 12 hours after the final
        # historical observation.
        y_last = target_values[fut_end - 1]

        # Store the historical and future-known inputs.
        hist_list.append(hist_x)
        fut_list.append(fut_x)

        # Store the single final-horizon target as a one-element array.
        tgt_list.append([y_last])

        # Store static features using the final historical observation.
        # This is appropriate only when these values are genuinely static.
        static_num_list.append(
            static_values[hist_end - 1]
        )

        # Preserve the UTC timestamp corresponding to the predicted target.
        tgt_times.append(
            df_scaled.index[fut_end - 1]
        )

    # Return model arrays and target timestamps.
    # tgt_times is metadata for prediction alignment and should not be passed
    # into DictTensorDataset because timestamps are not float tensors.
    return {
        "historical_ts_numeric": np.array(
            hist_list,
            dtype=np.float32
        ),
        "future_ts_numeric": np.array(
            fut_list,
            dtype=np.float32
        ),
        "static_feats_numeric": np.array(
            static_num_list,
            dtype=np.float32
        ),
        "target": np.array(
            tgt_list,
            dtype=np.float32
        ),
        "tgt_times": tgt_times
    }


# Build a Temporal Fusion Transformer using the experiment configuration and
# the current hidden-unit and attention-head hyperparameter combination.
def build_tft_model_from_cfg(
    cfg,
    hidden_units,
    num_heads
):
    # Describe the TFT task, input dimensions, and model architecture using the
    # configuration structure required by tft_torch.
    configuration = {
        # Configure the TFT for continuous-value forecasting.
        "task_type": "regression",

        # Identify the location of the forecast target relative to the
        # historical input window.
        "target_window_start": cfg.horizon_steps,

        # Describe the number and type of variables entering each TFT branch.
        "data_props": {
            # Historical numeric features are observed through forecast origin.
            "num_historical_numeric": len(
                cfg.historical_feature_cols
            ),

            # This implementation does not use historical categorical inputs.
            "num_historical_categorical": 0,
            "historical_categorical_cardinalities": [],

            # Static numeric features describe the series or monitoring site.
            "num_static_numeric": len(
                cfg.static_numeric_cols
            ),

            # This implementation does not use static categorical inputs.
            "num_static_categorical": 0,
            "static_categorical_cardinalities": [],

            # Future numeric features must be known for the complete horizon.
            "num_future_numeric": len(
                cfg.future_known_feature_cols
            ),

            # This implementation does not use future categorical inputs.
            "num_future_categorical": 0,
            "future_categorical_cardinalities": []
        },

        # Define the internal TFT architecture and quantile outputs.
        "model": {
            # Set the number of parallel attention heads.
            "attention_heads": num_heads,

            # Disable dropout inside the TFT architecture.
            "dropout": 0.0,

            # Use one recurrent LSTM layer inside the TFT.
            "lstm_layers": 1,

            # Produce one prediction for each configured quantile.
            # With (0.1, 0.5, 0.9), q50 is the median forecast and q10–q90
            # defines a nominal central 80% prediction interval.
            "output_quantiles": list(cfg.quantiles),

            # Set the common hidden-state dimension used by the TFT.
            "state_size": hidden_units
        }
    }

    # Convert the dictionary into an OmegaConf object and initialize the TFT.
    return TemporalFusionTransformer(
        config=OmegaConf.create(configuration)
    )

# Build TFT train, validation, and holdout arrays with consistent historical
# context. Validation and holdout may use observations that occurred before
# their target period, but targets are retained only when their timestamps fall
# inside the assigned evaluation period. No validation or holdout values fit a
# scaler.
def subset_tft_arrays_by_target_period(
    arrays,
    period_start,
    period_end
):
    target_times = pd.DatetimeIndex(
        arrays["tgt_times"]
    )

    mask = np.asarray(
        (target_times >= period_start)
        & (target_times <= period_end),
        dtype=bool
    )

    if not mask.any():
        raise ValueError(
            "No TFT targets fall inside the requested evaluation period."
        )

    filtered = {}

    for key, values in arrays.items():
        if key == "tgt_times":
            filtered[key] = list(
                target_times[mask]
            )
        else:
            filtered[key] = np.asarray(
                values
            )[mask]

    return filtered


def make_tft_split_arrays_with_context(
    train_scaled,
    val_scaled,
    test_scaled,
    historical_feature_cols,
    future_known_feature_cols,
    static_numeric_cols,
    target_col,
    lookback,
    horizon
):
    # The extra context permits a forecast whose target is the first timestamp
    # of the next split. Filtering below removes context-period targets.
    context_rows = lookback + horizon

    train_arrays = build_tft_arrays(
        train_scaled,
        historical_feature_cols,
        future_known_feature_cols,
        static_numeric_cols,
        target_col,
        lookback,
        horizon
    )

    val_context = pd.concat(
        [
            train_scaled.tail(context_rows),
            val_scaled
        ],
        axis=0
    )

    val_arrays_all = build_tft_arrays(
        val_context,
        historical_feature_cols,
        future_known_feature_cols,
        static_numeric_cols,
        target_col,
        lookback,
        horizon
    )

    val_arrays = subset_tft_arrays_by_target_period(
        val_arrays_all,
        val_scaled.index.min(),
        val_scaled.index.max()
    )

    test_context = pd.concat(
        [
            val_context.tail(context_rows),
            test_scaled
        ],
        axis=0
    )

    test_arrays_all = build_tft_arrays(
        test_context,
        historical_feature_cols,
        future_known_feature_cols,
        static_numeric_cols,
        target_col,
        lookback,
        horizon
    )

    test_arrays = subset_tft_arrays_by_target_period(
        test_arrays_all,
        test_scaled.index.min(),
        test_scaled.index.max()
    )

    val_target_times = pd.DatetimeIndex(
        val_arrays["tgt_times"]
    )

    test_target_times = pd.DatetimeIndex(
        test_arrays["tgt_times"]
    )

    # These assertions make the former 35-hour TFT omission impossible to
    # reintroduce silently when lookback or horizon settings change.
    assert val_target_times.min() == val_scaled.index.min(), (
        "TFT validation targets do not begin at the validation boundary."
    )

    assert val_target_times.max() == val_scaled.index.max(), (
        "TFT validation targets do not cover the complete validation period."
    )

    assert test_target_times.min() == test_scaled.index.min(), (
        "TFT holdout targets do not begin at the holdout boundary."
    )

    assert test_target_times.max() == test_scaled.index.max(), (
        "TFT holdout targets do not cover the complete holdout period."
    )

    return (
        train_arrays,
        val_arrays,
        test_arrays
    )



In [ ]:
# Training diagnostics and model-metadata helpers

# Training diagnostics, model summaries, architecture plots, and metadata

# Save the model's epoch-by-epoch training history as both a CSV file and a
# line plot. TFT uses quantile loss, while ANN and LSTM histories may contain
# loss and MAE values.
def save_training_history_csv_and_plot(
    history_dict,
    output_dir,
    file_stem,
    show_plot=True,
    is_tft=False
):
    # Create the output directory if it does not already exist.
    ensure_output_dir(output_dir)

    # Convert the training-history dictionary into a table where each row
    # represents one training epoch.
    history_df = pd.DataFrame(history_dict)

    # Save the complete numeric training history for later inspection.
    history_df.to_csv(
        f"{output_dir}/{file_stem}_training_history.csv",
        index=False
    )

    fig, ax = plt.subplots(figsize=(12, 6))

    if is_tft:
        # TFT training is evaluated using quantile loss.
        if "loss" in history_df.columns:
            ax.plot(
                history_df.index + 1,
                history_df["loss"],
                label="Training quantile loss"
            )

        if "val_loss" in history_df.columns:
            ax.plot(
                history_df.index + 1,
                history_df["val_loss"],
                label="Validation quantile loss"
            )

        ax.set_ylabel("Quantile loss")

    else:
        # ANN and LSTM training histories may contain both the optimization
        # loss and mean absolute error for training and validation.
        if "loss" in history_df.columns:
            ax.plot(
                history_df.index + 1,
                history_df["loss"],
                label="Training loss"
            )

        if "val_loss" in history_df.columns:
            ax.plot(
                history_df.index + 1,
                history_df["val_loss"],
                label="Validation loss"
            )

        if "mae" in history_df.columns:
            ax.plot(
                history_df.index + 1,
                history_df["mae"],
                label="Training MAE"
            )

        if "val_mae" in history_df.columns:
            ax.plot(
                history_df.index + 1,
                history_df["val_mae"],
                label="Validation MAE"
            )

        ax.set_ylabel("Metric value")

    # Label epochs starting at 1 instead of the DataFrame's zero-based index.
    ax.set_xlabel("Epoch")
    ax.set_title("Training History")
    ax.legend()

    fig.tight_layout()

    # Save the training-history figure.
    fig.savefig(
        f"{output_dir}/{file_stem}_training_history.png",
        dpi=150
    )

    # Display the plot only when requested.
    if show_plot:
        plt.show()

    # Close the figure to release memory during repeated experiments.
    plt.close(fig)


# Save the text produced by Keras model.summary() for documentation and model
# architecture verification.
def save_keras_model_summary(
    model,
    output_dir,
    file_stem
):
    # Create the output directory if necessary.
    ensure_output_dir(output_dir)

    # Capture each printed model-summary line in a list.
    lines = []

    model.summary(
        print_fn=lambda line: lines.append(line)
    )

    # Write the complete summary to a plain-text file.
    with open(
        f"{output_dir}/{file_stem}_model_summary.txt",
        "w",
        encoding="utf-8"
    ) as file:
        file.write("\n".join(lines))


# Save a visual diagram of a Keras model's layers and tensor shapes.
# Graphviz and pydot must be installed for this function to work.
def save_keras_model_architecture_plot(
    model,
    output_dir,
    file_stem
):
    # Create the output directory if necessary.
    ensure_output_dir(output_dir)

    try:
        # Import locally so a missing plotting dependency does not prevent
        # the rest of the modeling code from running.
        from tensorflow.keras.utils import plot_model

        plot_model(
            model,
            to_file=(
                f"{output_dir}/"
                f"{file_stem}_architecture.png"
            ),
            show_shapes=True,
            show_dtype=False,
            show_layer_names=True,
            expand_nested=True,
            dpi=150
        )

        print(
            f"Architecture diagram saved for {file_stem}"
        )

    except Exception as e:
        # Report the plotting problem without terminating model training.
        print(
            f"Could not save architecture diagram "
            f"for {file_stem}: {e}"
        )

        print(
            "In Colab, run: "
            "!apt-get install graphviz -y "
            "&& !pip install pydot"
        )


# Count the total number of trainable parameters in a Keras model.
# Only weights updated during model training are included.
def count_keras_trainable_params(model):
    return int(
        np.sum([
            np.prod(weight.shape)
            for weight in model.trainable_weights
        ])
    )


# Save the selected TFT experiment settings, architecture size, and input
# feature configuration so that the fitted run can be reproduced and audited.
def save_tft_metadata_and_config(
    cfg,
    model,
    output_dir,
    file_stem,
    eval_start_month,
    train_months,
    hidden_units,
    num_heads,
    batch_size,
    best_epoch
):
    # Create the output directory if necessary.
    ensure_output_dir(output_dir)

    # Store the main run metadata in a one-row table.
    metadata_df = pd.DataFrame([{
        "model_family": "tft",

        # Record the evaluation month as a calendar date.
        "eval_month_start": str(
            pd.Timestamp(eval_start_month).date()
        ),

        # Record the selected data and architecture hyperparameters.
        "train_months": train_months,
        "hidden_units": hidden_units,
        "num_heads": num_heads,
        "batch_size": batch_size,
        "best_epoch": best_epoch,

        # Count only parameters that are updated during TFT training.
        "trainable_params": int(
            sum(
                parameter.numel()
                for parameter in model.parameters()
                if parameter.requires_grad
            )
        )
    }])

    # Save run-level metadata in CSV format for comparison across experiments.
    metadata_df.to_csv(
        f"{output_dir}/{file_stem}_model_metadata.csv",
        index=False
    )

    # Save the complete TFT feature and architecture configuration as JSON.
    with open(
        f"{output_dir}/{file_stem}_model_config.json",
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            {
                # Historical features must be observed by forecast origin.
                "historical_feature_cols": (
                    cfg.historical_feature_cols
                ),

                # Future-known features must be available throughout the
                # complete forecast horizon.
                "future_known_feature_cols": (
                    cfg.future_known_feature_cols
                ),

                # Static variables should remain constant for the series.
                "static_numeric_cols": (
                    cfg.static_numeric_cols
                ),

                # Record the temporal window settings.
                "lookback": cfg.lookback,
                "horizon_steps": cfg.horizon_steps,

                # Record the selected model hyperparameters.
                "hidden_units": hidden_units,
                "num_heads": num_heads,
                "batch_size": batch_size,

                # Record the predicted quantiles used for point forecasts and
                # prediction intervals.
                "quantiles": list(cfg.quantiles)
            },
            file,
            indent=2
        )

#TFT Explainability Helper

In [ ]:
# TFT explainability helper

# TFT explainability-output helper

# Summarize and save the TFT variable-selection weights and temporal-attention
# scores collected during test prediction. These outputs describe the model's
# internal weighting behavior and should not be interpreted as causal effects.
def save_tft_explainability(
    historical_weights_all,
    future_weights_all,
    static_weights_all,
    attention_scores_all,
    cfg,
    output_dir,
    file_stem,
    show_plots=False
):
    # Create the output directory if it does not already exist.
    ensure_output_dir(output_dir)

    # Calculate the global mean importance of each variable.
    # This helper assumes that the final tensor dimension corresponds to the
    # variables and averages over samples, time steps, and other dimensions.
    def get_mean_variable_importance(
        weight_batches,
        variable_names,
        weight_type
    ):
        # Combine the weight arrays collected from every prediction batch.
        weights = np.concatenate(
            weight_batches,
            axis=0
        )

        # Print the original shape to support explainability validation.
        print(
            f"{weight_type} raw weight shape: "
            f"{weights.shape}"
        )

        # Preserve the final tensor dimension as the variable dimension and
        # average over every preceding dimension.
        averaging_axes = tuple(
            range(weights.ndim - 1)
        )

        mean_importance = weights.mean(
            axis=averaging_axes
        )

        # Convert the result into a one-dimensional importance array.
        mean_importance = np.asarray(
            mean_importance
        ).reshape(-1)

        # Confirm that the number of extracted weights matches the number of
        # feature names supplied through the TFT configuration.
        if len(mean_importance) != len(variable_names):
            raise ValueError(
                f"{weight_type} explainability mismatch: "
                f"{len(variable_names)} variable names but "
                f"{len(mean_importance)} importance values. "
                f"Raw tensor shape: {weights.shape}."
            )

        return mean_importance


    # Calculate historical-variable importance

    # Retrieve the names of variables observed in the historical input window.
    historical_names = list(
        cfg.historical_feature_cols
    )

    # Average the historical variable-selection weights across all test
    # samples and historical time positions.
    historical_importance = get_mean_variable_importance(
        weight_batches=historical_weights_all,
        variable_names=historical_names,
        weight_type="Historical"
    )

    # Create a table ranked from highest to lowest mean selection weight.
    historical_df = pd.DataFrame({
        "variable": historical_names,
        "importance": historical_importance
    }).sort_values(
        "importance",
        ascending=False
    )

    # Save the historical-variable importance values.
    historical_df.to_csv(
        (
            f"{output_dir}/"
            f"{file_stem}_historical_variable_importance.csv"
        ),
        index=False
    )


    # Calculate known-future-variable importance

    # Retrieve the names of variables known throughout the forecast horizon.
    historical_names = list(
        cfg.historical_feature_cols
    )

    future_names = list(
        cfg.future_known_feature_cols
    )

    # Average the known-future variable-selection weights across test samples
    # and future horizon positions.
    future_importance = get_mean_variable_importance(
        weight_batches=future_weights_all,
        variable_names=future_names,
        weight_type="Future"
    )

    # Create a table ranked from highest to lowest mean selection weight.
    future_df = pd.DataFrame({
        "variable": future_names,
        "importance": future_importance
    }).sort_values(
        "importance",
        ascending=False
    )

    # Save the known-future-variable importance values.
    future_df.to_csv(
        (
            f"{output_dir}/"
            f"{file_stem}_future_variable_importance.csv"
        ),
        index=False
    )


    # Calculate static-variable importance

    # Retrieve the names of numeric variables treated as static by the TFT.
    static_names = list(
        cfg.static_numeric_cols
    )

    # Average the static variable-selection weights across all test samples.
    static_importance = get_mean_variable_importance(
        weight_batches=static_weights_all,
        variable_names=static_names,
        weight_type="Static"
    )

    # Create a table ranked from highest to lowest mean selection weight.
    static_df = pd.DataFrame({
        "variable": static_names,
        "importance": static_importance
    }).sort_values(
        "importance",
        ascending=False
    )

    # Save the static-variable importance values.
    static_df.to_csv(
        (
            f"{output_dir}/"
            f"{file_stem}_static_variable_importance.csv"
        ),
        index=False
    )


    # Calculate global temporal attention

    # Combine the attention arrays collected from every test batch.
    attention_scores = np.concatenate(
        attention_scores_all,
        axis=0
    )

    # Print the raw shape so the assumed temporal dimension can be checked.
    print(
        "Attention raw weight shape: "
        f"{attention_scores.shape}"
    )

    # Preserve the final tensor dimension as the temporal dimension and average
    # over samples, attention heads, and every other preceding dimension.
    attention_averaging_axes = tuple(
        range(attention_scores.ndim - 1)
    )

    mean_attention = attention_scores.mean(
        axis=attention_averaging_axes
    )

    # Convert the global attention result into a one-dimensional array.
    mean_attention = np.asarray(
        mean_attention
    ).reshape(-1)

    # Determine the number of temporal positions represented in the output.
    total_steps = len(
        mean_attention
    )

    # Treat up to `lookback` positions as historical attention positions.
    historical_steps = min(
        cfg.lookback,
        total_steps
    )

    # Treat any remaining positions as future-horizon positions.
    future_steps = (
        total_steps
        - historical_steps
    )

    # Label historical positions from the oldest observation to the most recent.
    attention_labels = [
        f"t-{lag}"
        for lag in range(
            historical_steps,
            0,
            -1
        )
    ]

    # Add labels for attention positions belonging to the forecast horizon.
    attention_labels.extend([
        f"future_step_{step}"
        for step in range(
            1,
            future_steps + 1
        )
    ])

    # Confirm that every extracted attention value has one temporal label.
    if len(attention_labels) != len(mean_attention):
        raise ValueError(
            "Temporal-attention mismatch: "
            f"{len(attention_labels)} labels but "
            f"{len(mean_attention)} attention values. "
            f"Raw tensor shape: {attention_scores.shape}."
        )

    # Create a table containing the global mean attention at each time position.
    attention_df = pd.DataFrame({
        "time_position": attention_labels,
        "attention": mean_attention
    })

    # Save the temporal-attention values.
    attention_df.to_csv(
        (
            f"{output_dir}/"
            f"{file_stem}_temporal_attention.csv"
        ),
        index=False
    )


    # Plot variable-selection importance

    # Combine the three importance tables with their plot titles and filenames.
    importance_plots = [
        (
            historical_df,
            "TFT Historical Variable Importance",
            "historical_variable_importance"
        ),
        (
            future_df,
            "TFT Known-Future Variable Importance",
            "future_variable_importance"
        ),
        (
            static_df,
            "TFT Static Variable Importance",
            "static_variable_importance"
        )
    ]

    for plot_df, title, suffix in importance_plots:
        # Skip a plot if its corresponding importance table has no variables.
        if plot_df.empty:
            continue

        # Sort in ascending order so the most important variable appears at
        # the top of the horizontal bar chart.
        plot_df = plot_df.sort_values(
            "importance",
            ascending=True
        )

        fig, ax = plt.subplots(
            figsize=(10, 6)
        )

        ax.barh(
            plot_df["variable"],
            plot_df["importance"]
        )

        ax.set_title(title)
        ax.set_xlabel("Mean Variable-Selection Weight")
        ax.set_ylabel("Variable")

        fig.tight_layout()

        # Save the variable-importance chart.
        fig.savefig(
            (
                f"{output_dir}/"
                f"{file_stem}_{suffix}.png"
            ),
            dpi=300,
            bbox_inches="tight"
        )

        # Display the figure only when requested.
        if show_plots:
            plt.show()

        # Close the figure to release memory.
        plt.close(fig)


    # Plot global temporal attention

    fig, ax = plt.subplots(
        figsize=(12, 6)
    )

    # Create one x-axis position for every temporal attention value.
    x_positions = np.arange(
        total_steps
    )

    ax.plot(
        x_positions,
        mean_attention,
        marker="o"
    )

    # Mark the boundary between historical and future positions when the
    # attention output contains future positions.
    if future_steps > 0:
        ax.axvline(
            historical_steps - 0.5,
            linestyle="--",
            label="Forecast boundary"
        )

        ax.legend()

    # Display the corresponding historical or future label at each position.
    ax.set_xticks(
        x_positions
    )

    ax.set_xticklabels(
        attention_labels,
        rotation=90
    )

    ax.set_title("TFT Global Temporal Attention")
    ax.set_xlabel("Time Position")
    ax.set_ylabel("Mean Attention Weight")

    fig.tight_layout()

    # Save the global temporal-attention chart.
    fig.savefig(
        (
            f"{output_dir}/"
            f"{file_stem}_temporal_attention.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    # Display the plot only when requested.
    if show_plots:
        plt.show()

    # Close the figure to release memory.
    plt.close(fig)

    print(
        f"Saved TFT explainability outputs: {file_stem}"
    )

In [ ]:
# High-level seasonal deep-learning runners

# Run the retained-winner selection and holdout workflow for one seasonal chunk.
def run_seasonal_chunk(df, cfg, chunk_name):
    season_chunks = get_seasonal_chunks()

    if chunk_name not in season_chunks:
        raise ValueError(
            f"Unknown chunk_name: {chunk_name}. "
            f"Choose from: {list(season_chunks.keys())}"
        )

    seasons_to_run = season_chunks[chunk_name]
    print("Running seasons in chunk:", seasons_to_run)

    return run_seasonal_selection_and_true_holdout(
        df=df,
        cfg=cfg,
        output_tag=chunk_name,
        seasons_to_run=seasons_to_run
    )


# Run one model configuration and seasonal chunk, display the main results,
# save organized combined tables, and freeze the exact retained winner.
def run_and_save_dl_chunk(df, cfg, chunk_name, config_name):
    df_model = build_model_df_for_config(df, cfg)

    assert_no_retraining_seasonal_workflow_active()

    (
        selection_results_df,
        config_summary,
        best_config_per_season,
        final_holdout_results_df,
        final_forecasts_df
    ) = run_seasonal_chunk(
        df=df_model,
        cfg=cfg,
        chunk_name=chunk_name
    )

    print("\n=== Seasonal Validation Configuration Results ===")
    print(selection_results_df.head())
    print("\n=== Best Configuration Per Season ===")
    print(best_config_per_season)
    print("\n=== Final Unseen Seasonal Holdout Metrics ===")
    print(final_holdout_results_df)

    combined_output_dir = os.path.join(cfg.output_dir, "combined_outputs")
    ensure_output_dir(combined_output_dir)

    tables = {
        "validation_configuration_results": selection_results_df,
        "validation_configuration_summary": config_summary,
        "best_config_per_season_from_validation": best_config_per_season,
        "true_holdout_metrics": final_holdout_results_df
    }

    for suffix, table in tables.items():
        table.to_csv(
            os.path.join(combined_output_dir, f"{chunk_name}_{suffix}.csv"),
            index=False
        )

    if not final_forecasts_df.empty:
        final_forecasts_df.to_csv(
            os.path.join(
                combined_output_dir,
                f"{chunk_name}_true_holdout_forecasts_all.csv"
            ),
            index=False
        )

    seasons = get_seasonal_chunks()[chunk_name]
    if len(seasons) != 1:
        raise ValueError("Each saved chunk must contain exactly one season.")

    save_retained_dl_winner(
        model_family=cfg.model_family,
        season=seasons[0],
        config_name=config_name,
        cfg=cfg
    )

    print("\nAll combined seasonal outputs saved to:")
    print(combined_output_dir)

    return (
        selection_results_df,
        config_summary,
        best_config_per_season,
        final_holdout_results_df,
        final_forecasts_df
    )


In [ ]:
# Corrected seasonal workflow: retain the winning fitted model without retraining
# Candidate configurations are compared using validation data. The fitted model
# belonging to the winning configuration is preserved in memory and then used
# directly on its untouched test period.

# Store one fitted validation winner for each model family and season.
# These in-memory artifacts are cleared when the notebook runtime restarts.
FITTED_SEASONAL_WINNERS = {}


# Create a consistent registry key from the model family and season.
def _selection_key(model_family, season):
    return f"{model_family.lower()}::{season}"


# Generate LSTM holdout predictions from the retained fitted model, scaler,
# test sequences, and timestamps without fitting the model again.
def _make_lstm_holdout_from_artifact(artifact, cfg):
    model = artifact["model"]
    scaler = artifact["scaler"]
    model_cols = artifact["model_cols"]
    X_test = artifact["X_test"]
    y_test = artifact["y_test"]
    ytime_test = artifact["ytime_test"]
    y_pred_scaled, lower_scaled, upper_scaled = mc_dropout_predict_keras(
        model=model,
        X=X_test,
        n_passes=100
    )

    y_true = inverse_target_from_joint_scaler(
        y_test, scaler, model_cols, cfg.target_col
    )
    y_pred = inverse_target_from_joint_scaler(
        y_pred_scaled, scaler, model_cols, cfg.target_col
    )
    lower_ci = inverse_target_from_joint_scaler(
        lower_scaled, scaler, model_cols, cfg.target_col
    )
    upper_ci = inverse_target_from_joint_scaler(
        upper_scaled, scaler, model_cols, cfg.target_col
    )

    pred_df = pd.DataFrame({
        "date": ytime_test,
        f"actual_{cfg.target_col}_t_plus_{cfg.horizon_steps}h": y_true,
        f"predicted_{cfg.target_col}_t_plus_{cfg.horizon_steps}h": y_pred
    }).set_index("date")

    pred_df = add_mc_dropout_interval_columns(
        pred_df=pred_df,
        lower_ci=lower_ci,
        upper_ci=upper_ci
    )

    metrics = evaluate_predictions(y_true, y_pred)
    return pred_df, metrics


# Generate ANN holdout predictions from the retained fitted model and its
# previously prepared test data without fitting the model again.
def _make_ann_holdout_from_artifact(artifact, cfg):
    model = artifact["model"]
    X_test = artifact["X_test"]
    y_test = artifact["y_test"]
    y_scaler = artifact["y_scaler"]

    # Recover the prepared ANN holdout table containing target timestamps.
    test_df = artifact["test_df"]

    y_pred_scaled, lower_scaled, upper_scaled = mc_dropout_predict_keras(
        model=model,
        X=X_test,
        n_passes=100
    )

    y_true = inverse_ann_target(y_test, y_scaler)
    y_pred = inverse_ann_target(y_pred_scaled, y_scaler)
    lower_ci = inverse_ann_target(lower_scaled, y_scaler)
    upper_ci = inverse_ann_target(upper_scaled, y_scaler)

    pred_df = pd.DataFrame({
        "date": test_df["target_timestamp"].array,
        f"actual_{cfg.target_col}_t_plus_{cfg.horizon_steps}h": y_true,
        f"predicted_{cfg.target_col}_t_plus_{cfg.horizon_steps}h": y_pred
    }).set_index("date")

    pred_df = add_mc_dropout_interval_columns(
        pred_df=pred_df,
        lower_ci=lower_ci,
        upper_ci=upper_ci
    )

    metrics = evaluate_predictions(y_true, y_pred)
    return pred_df, metrics


# Generate TFT predictions for a retained validation or test split and collect
# the model's available variable-selection and temporal-attention tensors.
def _predict_tft_split_from_artifact(artifact, cfg, split_name):
    model = artifact["model"]
    loader = artifact[f"{split_name}_loader"]
    arrays = artifact[f"{split_name}_arrays"]
    joint_scaler = artifact["joint_scaler"]
    model_cols = artifact["model_cols"]

    model.eval()
    preds_scaled, p10_scaled, p50_scaled, p90_scaled = [], [], [], []
    historical_weights_all, future_weights_all = [], []
    static_weights_all, attention_scores_all = [], []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(batch)
            preds = out["predicted_quantiles"].detach().cpu().numpy()
            historical_key = (
                "historical_selection_weights"
                if "historical_selection_weights" in out
                else "historical_weights"
                if "historical_weights" in out
                else None
            )
            future_key = (
                "future_selection_weights"
                if "future_selection_weights" in out
                else "future_weights"
                if "future_weights" in out
                else None
            )

            if historical_key is not None:
                historical_weights_all.append(
                    out[historical_key].detach().cpu().numpy()
                )
            if future_key is not None:
                future_weights_all.append(
                    out[future_key].detach().cpu().numpy()
                )
            if "static_weights" in out:
                static_weights_all.append(
                    out["static_weights"].detach().cpu().numpy()
                )
            if "attention_scores" in out:
                attention_scores_all.append(
                    out["attention_scores"].detach().cpu().numpy()
                )
            q_idx = {q: i for i, q in enumerate(cfg.quantiles)}
            p10_scaled.extend(preds[:, 0, q_idx[0.1]])
            p50_scaled.extend(preds[:, 0, q_idx[0.5]])
            p90_scaled.extend(preds[:, 0, q_idx[0.9]])
            preds_scaled.extend(preds[:, 0, q_idx[0.5]])

    preds_scaled = np.asarray(preds_scaled).reshape(-1)
    p10_scaled = np.asarray(p10_scaled).reshape(-1)
    p50_scaled = np.asarray(p50_scaled).reshape(-1)
    p90_scaled = np.asarray(p90_scaled).reshape(-1)
    y_true_scaled = arrays["target"][:, 0]

    y_true = inverse_target_from_joint_scaler(
        y_true_scaled, joint_scaler, model_cols, cfg.target_col
    )
    y_pred = inverse_target_from_joint_scaler(
        preds_scaled, joint_scaler, model_cols, cfg.target_col
    )
    y_p10 = inverse_target_from_joint_scaler(
        p10_scaled, joint_scaler, model_cols, cfg.target_col
    )
    y_p50 = inverse_target_from_joint_scaler(
        p50_scaled, joint_scaler, model_cols, cfg.target_col
    )
    y_p90 = inverse_target_from_joint_scaler(
        p90_scaled, joint_scaler, model_cols, cfg.target_col
    )

    pred_df = pd.DataFrame({
        "date": arrays["tgt_times"],
        f"actual_{cfg.target_col}_t_plus_{cfg.horizon_steps}h": y_true,
        f"predicted_{cfg.target_col}_t_plus_{cfg.horizon_steps}h": y_pred,
        "q10": y_p10,
        "q50": y_p50,
        "q90": y_p90
    }).set_index("date")

    pred_df["lower_ci"] = pred_df["q10"]
    pred_df["upper_ci"] = pred_df["q90"]
    pred_df["interval_method"] = "tft_quantile_interval"

    artifact["historical_weights_all"] = historical_weights_all
    artifact["future_weights_all"] = future_weights_all
    artifact["static_weights_all"] = static_weights_all
    artifact["attention_scores_all"] = attention_scores_all

    metrics = evaluate_predictions(y_true, y_pred)
    return pred_df, metrics


# Evaluate every LSTM configuration for one validation month and retain the
# exact fitted model with the lowest validation RMSE and MAE.
def run_one_eval_month_lstm(df, cfg: ForecastConfig, eval_start_month):
    results = []
    best_artifact = None
    best_score = (np.inf, np.inf)
    # Preserve model-column order without duplicating the target column.
    model_cols = list(
        dict.fromkeys(
            cfg.numeric_feature_cols + [cfg.target_col]
        )
    )

    print(f"\nRunning LSTM validation month: {eval_start_month}")

    for train_months, hidden_units, batch_size in product(
        cfg.train_length_options,
        cfg.hidden_units_options,
        cfg.batch_size_options
    ):
        print(
            f"  Trying train_months={train_months}, "
            f"hidden_units={hidden_units}, batch_size={batch_size}"
        )

        (
            train_df, val_df, test_df,
            train_start, train_end,
            val_start, val_end,
            test_start, test_end
        ) = split_by_time(
            df=df,
            train_months=train_months,
            eval_start_month=eval_start_month,
            val_months=cfg.val_months,
            test_months=cfg.test_months
        )

        min_required = cfg.lookback + cfg.horizon_steps + 24
        if any(len(x) < min_required for x in (train_df, val_df, test_df)):
            print("    skipped because split too short")
            continue

        scaler = fit_lstm_scaler(train_df, model_cols)
        train_scaled = apply_lstm_scaler(train_df, scaler, model_cols)
        val_scaled = apply_lstm_scaler(val_df, scaler, model_cols)
        test_scaled = apply_lstm_scaler(test_df, scaler, model_cols)

        (
            X_train, y_train,
            X_val, y_val,
            X_test, y_test,
            ytime_test
        ) = make_lstm_split_sequences(
            train_scaled,
            val_scaled,
            test_scaled,
            input_cols=cfg.numeric_feature_cols,
            target_col=cfg.target_col,
            lookback=cfg.lookback,
            horizon=cfg.horizon_steps
        )

        if any(len(x) == 0 for x in (X_train, X_val, X_test)):
            print("    skipped because no usable sequences")
            continue

        model = build_lstm_model(
            (X_train.shape[1], X_train.shape[2]),
            hidden_units
        )
        early_stopping = EarlyStopping(
            monitor="val_loss",
            patience=cfg.patience,
            restore_best_weights=True,
            mode="min"
        )
        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=cfg.epochs,
            batch_size=batch_size,
            shuffle=False,
            verbose=0,
            callbacks=[early_stopping]
        )

        val_pred_scaled = model.predict(X_val, verbose=0).ravel()
        y_val_real = inverse_target_from_joint_scaler(
            y_val, scaler, model_cols, cfg.target_col
        )
        val_pred_real = inverse_target_from_joint_scaler(
            val_pred_scaled, scaler, model_cols, cfg.target_col
        )
        metrics = evaluate_predictions(y_val_real, val_pred_real)


        row = {
            "model_family": "lstm",
            "evaluation_type": "seasonal_configuration_validation",
            "metric_split": "validation",
            "eval_month_start": ensure_same_tz(eval_start_month, df.index.tz),
            "season": season_name(val_start),
            "train_months": train_months,
            "hidden_units": hidden_units,
            "batch_size": batch_size,
            "num_heads": np.nan,
            "train_start": train_start,
            "train_end": train_end,
            "val_start": val_start,
            "val_end": val_end,
            "test_start": test_start,
            "test_end": test_end,
            "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
            "RMSE": metrics["RMSE"],
            "MAE": metrics["MAE"],
            "R2": metrics["R2"],
            "MAPE": metrics["MAPE"]
        }
        row["run_id"] = stable_run_id(row)
        results.append(row)

        score = (metrics["RMSE"], metrics["MAE"])
        if score < best_score:
            best_score = score
            best_artifact = {
                "model": model,
                "scaler": scaler,
                "model_cols": model_cols,
                "X_test": X_test,
                "y_test": y_test,
                "ytime_test": ytime_test,
                "test_df": test_df,
                "history": history,
                "history_dict": {
                    key: list(values)
                    for key, values in history.history.items()
                },
                "row": row.copy()
            }

    result_df = pd.DataFrame(results)
    if best_artifact is None:
        return result_df

    season = best_artifact["row"]["season"]
    FITTED_SEASONAL_WINNERS[_selection_key("lstm", season)] = best_artifact
    print("  Retained winning fitted LSTM model in memory; no retraining will occur.")
    return result_df


# Evaluate every ANN configuration for one validation month and retain the
# exact fitted model with the lowest validation RMSE and MAE.
def run_one_eval_month_ann(df, cfg: ForecastConfig, eval_start_month):
    results = []
    best_artifact = None
    best_score = (np.inf, np.inf)

    numeric_cols, all_feature_cols = get_ann_feature_lists(
        cfg.target_col,
        cfg.numeric_feature_cols,
        cfg.categorical_feature_cols,
        cfg.lag_steps
    )

    print(f"\nRunning ANN validation month: {eval_start_month}")

    for train_months, hidden_units, batch_size in product(
        cfg.train_length_options,
        cfg.hidden_units_options,
        cfg.batch_size_options
    ):
        try:
            (
                train_df_raw, val_df_raw, test_df_raw,
                train_start, train_end,
                val_start, val_end,
                test_start, test_end
            ) = split_by_time(
                df=df,
                train_months=train_months,
                eval_start_month=eval_start_month,
                val_months=cfg.val_months,
                test_months=cfg.test_months
            )

            train_df, val_df, test_df = make_ann_month_tables(
                train_df_raw,
                val_df_raw,
                test_df_raw,
                target_col=cfg.target_col,
                numeric_base_cols=cfg.numeric_feature_cols,
                categorical_cols=cfg.categorical_feature_cols,
                lag_steps=cfg.lag_steps,
                horizon_steps=cfg.horizon_steps
            )

            if train_df.empty or val_df.empty or test_df.empty:
                continue

            preprocessor = fit_ann_preprocessor(
                train_df,
                numeric_cols,
                cfg.categorical_feature_cols,
                all_feature_cols
            )
            y_scaler = fit_ann_target_scaler(train_df)

            X_train = transform_ann_X(train_df, preprocessor, all_feature_cols)
            y_train = transform_ann_y(train_df, y_scaler)
            X_val = transform_ann_X(val_df, preprocessor, all_feature_cols)
            y_val = transform_ann_y(val_df, y_scaler)
            X_test = transform_ann_X(test_df, preprocessor, all_feature_cols)
            y_test = transform_ann_y(test_df, y_scaler)

            model = build_ann_model(X_train.shape[1], hidden_units)
            early_stopping = keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=cfg.patience,
                restore_best_weights=True
            )
            history = model.fit(
                X_train,
                y_train,
                validation_data=(X_val, y_val),
                epochs=cfg.epochs,
                batch_size=batch_size,
                verbose=0,
                callbacks=[early_stopping]
            )

            val_pred_scaled = model.predict(X_val, verbose=0).reshape(-1)
            y_true = inverse_ann_target(y_val, y_scaler)
            y_pred = inverse_ann_target(val_pred_scaled, y_scaler)
            metrics = evaluate_predictions(y_true, y_pred)


            row = {
                "model_family": "ann",
                "evaluation_type": "seasonal_configuration_validation",
                "metric_split": "validation",
                "eval_month_start": ensure_same_tz(eval_start_month, df.index.tz),
                "season": season_name(val_start),
                "train_months": train_months,
                "hidden_units": hidden_units,
                "batch_size": batch_size,
                "num_heads": np.nan,
                "train_start": train_start,
                "train_end": train_end,
                "val_start": val_start,
                "val_end": val_end,
                "test_start": test_start,
                "test_end": test_end,
                "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
                "RMSE": metrics["RMSE"],
                "MAE": metrics["MAE"],
                "R2": metrics["R2"],
                "MAPE": metrics["MAPE"]
            }
            row["run_id"] = stable_run_id(row)
            results.append(row)

            score = (metrics["RMSE"], metrics["MAE"])
            if score < best_score:
                best_score = score
                best_artifact = {
                    "model": model,
                    "preprocessor": preprocessor,
                    "y_scaler": y_scaler,
                    "X_test": X_test,
                    "y_test": y_test,
                    "test_df": test_df,
                    "history": history,
                    "history_dict": {
                        key: list(values)
                        for key, values in history.history.items()
                    },
                    "row": row.copy()
                }

        except Exception as exc:
            print(f"    skipped because of error: {exc}")

    result_df = pd.DataFrame(results)
    if best_artifact is None:
        return result_df

    season = best_artifact["row"]["season"]
    FITTED_SEASONAL_WINNERS[_selection_key("ann", season)] = best_artifact
    print("  Retained winning fitted ANN model in memory; no retraining will occur.")
    return result_df


# Fit and validate one TFT candidate while retaining its model, scalers,
# loaders, prepared arrays, history, and split metadata for later holdout use.
def run_one_model_tft_retained_candidate(
    df,
    cfg,
    eval_start_month,
    train_months,
    hidden_units,
    num_heads,
    batch_size
):
    (
        train_df, val_df, test_df,
        train_start, train_end,
        val_start, val_end,
        test_start, test_end
    ) = split_by_time(
        df=df,
        train_months=train_months,
        eval_start_month=eval_start_month,
        val_months=cfg.val_months,
        test_months=cfg.test_months
    )

    joint_scale_cols = list(dict.fromkeys(
        cfg.historical_feature_cols
        + cfg.future_known_feature_cols
        + [cfg.target_col]
    ))
    static_scale_cols = cfg.static_numeric_cols

    joint_scaler = StandardScaler().fit(train_df[joint_scale_cols].values)
    static_scaler = StandardScaler().fit(train_df[static_scale_cols].values)

    def _scale(frame, scaler, cols):
        out = frame.copy()
        out[cols] = scaler.transform(frame[cols].values)
        return out

    train_scaled = _scale(train_df, joint_scaler, joint_scale_cols)
    val_scaled = _scale(val_df, joint_scaler, joint_scale_cols)
    test_scaled = _scale(test_df, joint_scaler, joint_scale_cols)
    train_scaled = _scale(train_scaled, static_scaler, static_scale_cols)
    val_scaled = _scale(val_scaled, static_scaler, static_scale_cols)
    test_scaled = _scale(test_scaled, static_scaler, static_scale_cols)

    (
        train_arrays,
        val_arrays,
        test_arrays
    ) = make_tft_split_arrays_with_context(
        train_scaled=train_scaled,
        val_scaled=val_scaled,
        test_scaled=test_scaled,
        historical_feature_cols=cfg.historical_feature_cols,
        future_known_feature_cols=cfg.future_known_feature_cols,
        static_numeric_cols=cfg.static_numeric_cols,
        target_col=cfg.target_col,
        lookback=cfg.lookback,
        horizon=cfg.horizon_steps
    )

    def _loader(arrays):
        ds = DictTensorDataset({
            "historical_ts_numeric": arrays["historical_ts_numeric"],
            "future_ts_numeric": arrays["future_ts_numeric"],
            "static_feats_numeric": arrays["static_feats_numeric"],
            "target": arrays["target"]
        })
        return DataLoader(ds, batch_size=batch_size, shuffle=False)

    train_loader = _loader(train_arrays)
    val_loader = _loader(val_arrays)
    test_loader = _loader(test_arrays)

    model = build_tft_model_from_cfg(cfg, hidden_units, num_heads).to(DEVICE)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.learning_rate
    )
    desired_quantiles = torch.tensor(
        cfg.quantiles,
        dtype=torch.float32,
        device=DEVICE
    )

    best_val_loss = np.inf
    epoch_log = []
    best_state = None
    best_epoch = 0
    patience_counter = 0

    for epoch in range(cfg.epochs):
        model.train()
        train_losses = []
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(batch)
            q_loss, _, _ = tft_loss.get_quantiles_loss_and_q_risk(
                outputs=out["predicted_quantiles"],
                targets=batch["target"],
                desired_quantiles=desired_quantiles
            )
            q_loss.backward()
            optimizer.step()
            train_losses.append(q_loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                out = model(batch)
                q_loss, _, _ = tft_loss.get_quantiles_loss_and_q_risk(
                    outputs=out["predicted_quantiles"],
                    targets=batch["target"],
                    desired_quantiles=desired_quantiles
                )
                val_losses.append(q_loss.item())

        mean_train_loss = float(np.mean(train_losses))
        mean_val_loss = float(np.mean(val_losses))
        epoch_log.append({
            "epoch": epoch + 1,
            "loss": mean_train_loss,
            "val_loss": mean_val_loss
        })
        if mean_val_loss < best_val_loss:
            best_val_loss = mean_val_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            best_epoch = epoch + 1
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= cfg.patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()

    artifact = {
        "model": model,
        "joint_scaler": joint_scaler,
        "static_scaler": static_scaler, #Added 28 July
        "model_cols": joint_scale_cols,
        "val_loader": val_loader,
        "val_arrays": val_arrays,
        "val_df": val_df,
        "test_loader": test_loader,
        "test_arrays": test_arrays,
        "test_df": test_df,
        "history_dict": pd.DataFrame(epoch_log).to_dict(orient="list")
    }
    pred_df, metrics = _predict_tft_split_from_artifact(
        artifact, cfg, "val"
    )

    row = {
        "model_family": "tft",
        "evaluation_type": "seasonal_configuration_validation",
        "metric_split": "validation",
        "eval_month_start": ensure_same_tz(eval_start_month, df.index.tz),
        "season": season_name(val_start),
        "train_months": train_months,
        "hidden_units": hidden_units,
        "batch_size": batch_size,
        "num_heads": num_heads,
        "train_start": train_start,
        "train_end": train_end,
        "val_start": val_start,
        "val_end": val_end,
        "test_start": test_start,
        "test_end": test_end,
        "best_epoch": best_epoch,
        "RMSE": metrics["RMSE"],
        "MAE": metrics["MAE"],
        "R2": metrics["R2"],
        "MAPE": metrics["MAPE"]
    }
    row["run_id"] = stable_run_id(row)
    artifact["row"] = row.copy()
    return row, artifact


# Compare all TFT configurations for one validation month and register the
# exact fitted candidate with the best validation RMSE and MAE.
def run_one_eval_month_tft(df, cfg: TFTConfig, eval_start_month, chunk_name=None):
    results = []
    best_artifact = None
    best_score = (np.inf, np.inf)

    print(f"\nRunning TFT validation month: {eval_start_month}")

    for train_months, hidden_units, num_heads, batch_size in product(
        cfg.train_length_options,
        cfg.hidden_units_options,
        cfg.num_heads_options,
        cfg.batch_size_options
    ):
        try:
            row, artifact = run_one_model_tft_retained_candidate(
                df=df,
                cfg=cfg,
                eval_start_month=eval_start_month,
                train_months=train_months,
                hidden_units=hidden_units,
                num_heads=num_heads,
                batch_size=batch_size
            )
            results.append(row)
            score = (row["RMSE"], row["MAE"])
            if score < best_score:
                best_score = score
                best_artifact = artifact
        except Exception as exc:
            print(f"    skipped because of error: {exc}")

    result_df = pd.DataFrame(results)
    if best_artifact is None:
        return result_df

    season = best_artifact["row"]["season"]
    FITTED_SEASONAL_WINNERS[_selection_key("tft", season)] = best_artifact
    print("  Retained winning fitted TFT model in memory; no retraining will occur.")
    return result_df


# Route one seasonal validation experiment to its model-family runner.
def run_one_eval_month(df, cfg, eval_start_month, chunk_name=None):
    if cfg.model_family == "lstm":
        return run_one_eval_month_lstm(df, cfg, eval_start_month)
    if cfg.model_family == "ann":
        return run_one_eval_month_ann(df, cfg, eval_start_month)
    if cfg.model_family == "tft":
        return run_one_eval_month_tft(df, cfg, eval_start_month, chunk_name)
    raise ValueError("model_family must be lstm, ann, or tft")



# Confirm that a retained winner contains the fitted model, result metadata,
# and complete training history needed to create the final outputs.
def _validate_retained_winner_artifact(artifact, cfg):
    required_common = {"model", "row"}
    missing_common = required_common - set(artifact)
    if missing_common:
        raise RuntimeError(
            f"Retained winner is missing required fields: {sorted(missing_common)}"
        )

    if cfg.model_family in {"lstm", "ann"}:
        if "history_dict" not in artifact:
            raise RuntimeError(
                "Retained Keras winner is missing history_dict; "
                "the winning model training plot cannot be generated."
            )
        history_dict = artifact["history_dict"]
        if not history_dict.get("loss") or not history_dict.get("val_loss"):
            raise RuntimeError(
                "Retained Keras history must contain both loss and val_loss."
            )
        if len(history_dict["loss"]) != len(history_dict["val_loss"]):
            raise RuntimeError(
                "Retained Keras loss and val_loss histories have different lengths."
            )

    elif cfg.model_family == "tft":
        if "history_dict" not in artifact:
            raise RuntimeError(
                "Retained TFT winner is missing history_dict; "
                "the winning model training plot cannot be generated."
            )
        history_dict = artifact["history_dict"]
        if not history_dict.get("loss") or not history_dict.get("val_loss"):
            raise RuntimeError(
                "Retained TFT history must contain both training and validation quantile loss."
            )
        if len(history_dict["loss"]) != len(history_dict["val_loss"]):
            raise RuntimeError(
                "Retained TFT loss and val_loss histories have different lengths."
            )


# Save forecasts, diagnostics, training history, architecture information, and
# metadata from the retained fitted winner without retraining it.
def _save_retained_winner_outputs(artifact, cfg, season, pred_df):
    _validate_retained_winner_artifact(artifact, cfg)
    row = artifact["row"]
    model = artifact["model"]
    validation_month = row["eval_month_start"]
    holdout_month = row["test_start"]
    holdout_date = str(pd.Timestamp(holdout_month).date())
    train_months = int(row["train_months"])
    hidden_units = int(row["hidden_units"])
    batch_size = int(row["batch_size"])
    actual_col = f"actual_{cfg.target_col}_t_plus_{cfg.horizon_steps}h"
    pred_col = f"predicted_{cfg.target_col}_t_plus_{cfg.horizon_steps}h"

    if cfg.model_family == "lstm":
        file_stem = (
            f"lstm_{season.lower()}_{holdout_date}_"
            f"train{train_months}_hidden{hidden_units}_batch{batch_size}"
        )
        save_forecast_plots(
            pred_df=pred_df, output_dir=cfg.output_dir, file_stem=file_stem,
            model_label="LSTM", actual_col=actual_col, pred_col=pred_col,
            show_plot=True
        )
        save_training_history_csv_and_plot(
            history_dict=artifact["history_dict"],
            output_dir=cfg.output_dir, file_stem=file_stem
        )
        save_residual_outputs(
            pred_df=pred_df, actual_col=actual_col, pred_col=pred_col,
            output_dir=cfg.output_dir, file_stem=file_stem
        )
        save_keras_model_summary(model=model, output_dir=cfg.output_dir, file_stem=file_stem)
        save_keras_model_architecture_plot(model=model, output_dir=cfg.output_dir, file_stem=file_stem)
        pd.DataFrame([{
            "model_family": "lstm",
            "validation_month_start": str(pd.Timestamp(validation_month).date()),
            "actual_holdout_start": holdout_date,
            "train_months": train_months,
            "hidden_units": hidden_units,
            "batch_size": batch_size,
            "trainable_params": count_keras_trainable_params(model)
        }]).to_csv(f"{cfg.output_dir}/{file_stem}_model_metadata.csv", index=False)

    elif cfg.model_family == "ann":
        file_stem = (
            f"ann_{season.lower()}_{holdout_date}_"
            f"train{train_months}_hidden{hidden_units}_batch{batch_size}"
        )
        save_forecast_plots(
            pred_df=pred_df, output_dir=cfg.output_dir, file_stem=file_stem,
            model_label="ANN", actual_col=actual_col, pred_col=pred_col,
            show_plot=True
        )
        save_training_history_csv_and_plot(
            history_dict=artifact["history_dict"],
            output_dir=cfg.output_dir, file_stem=file_stem, show_plot=True
        )
        save_residual_outputs(
            pred_df=pred_df, actual_col=actual_col, pred_col=pred_col,
            output_dir=cfg.output_dir, file_stem=file_stem, show_plots=True
        )
        save_keras_model_summary(model=model, output_dir=cfg.output_dir, file_stem=file_stem)
        save_keras_model_architecture_plot(model=model, output_dir=cfg.output_dir, file_stem=file_stem)
        pd.DataFrame([{
            "model_family": "ann",
            "validation_month_start": str(pd.Timestamp(validation_month).date()),
            "actual_holdout_start": holdout_date,
            "train_months": train_months,
            "hidden_units": hidden_units,
            "batch_size": batch_size,
            "trainable_params": count_keras_trainable_params(model)
        }]).to_csv(f"{cfg.output_dir}/{file_stem}_model_metadata.csv", index=False)

    else:
        num_heads = int(row["num_heads"])
        file_stem = (
            f"tft_{season.lower()}_{holdout_date}_"
            f"train{train_months}_hidden{hidden_units}_heads{num_heads}_batch{batch_size}"
        )
        pred_df.to_csv(f"{cfg.output_dir}/{file_stem}_forecast.csv")
        save_forecast_plots(
            pred_df=pred_df, output_dir=cfg.output_dir, file_stem=file_stem,
            model_label="TFT", actual_col=actual_col, pred_col=pred_col,
            show_plot=True
        )
        save_training_history_csv_and_plot(
            history_dict=artifact["history_dict"],
            output_dir=cfg.output_dir, file_stem=file_stem,
            show_plot=True, is_tft=True
        )
        save_residual_outputs(
            pred_df=pred_df, actual_col=actual_col, pred_col=pred_col,
            output_dir=cfg.output_dir, file_stem=file_stem, show_plots=True
        )
        save_tft_metadata_and_config(
            cfg=cfg, model=model, output_dir=cfg.output_dir, file_stem=file_stem,
            eval_start_month=validation_month, train_months=train_months,
            hidden_units=hidden_units, num_heads=num_heads, batch_size=batch_size,
            best_epoch=int(row["best_epoch"])
        )
        if all(
            artifact.get(k)
            for k in [
                "historical_weights_all", "future_weights_all",
                "static_weights_all", "attention_scores_all"
            ]
        ):
            save_tft_explainability(
                historical_weights_all=artifact["historical_weights_all"],
                future_weights_all=artifact["future_weights_all"],
                static_weights_all=artifact["static_weights_all"],
                attention_scores_all=artifact["attention_scores_all"],
                cfg=cfg, output_dir=cfg.output_dir, file_stem=file_stem,
                show_plots=True
            )



# Inspect the active holdout function and stop before a long experiment if an
# older implementation that retrains the selected model is still active.
def assert_no_retraining_seasonal_workflow_active():
    import inspect

    active_source = inspect.getsource(
        apply_best_config_to_true_holdout_months
    )

    assert "FITTED_SEASONAL_WINNERS" in active_source, (
        "The active seasonal holdout function is not using the retained "
        "fitted-winner registry. Rerun the corrected seasonal workflow cell."
    )

    assert "train_one_model(" not in active_source, (
        "The active seasonal holdout function still calls train_one_model() "
        "and would retrain. Rerun the corrected seasonal workflow cell."
    )

    print(
        "Seasonal workflow check passed: the retained fitted winner will "
        "predict the holdout without retraining."
    )


# Select each season's best validation configuration and apply the same retained
# fitted model to its untouched test period without reinitialization or retraining.
def apply_best_config_to_true_holdout_months(
    df,
    cfg,
    selection_results_df,
    output_tag,
    seasons_to_run=None
):
    if cfg.model_family == "tft":
        hyperparam_cols = [
            "train_months", "hidden_units", "num_heads", "batch_size"
        ]
    else:
        hyperparam_cols = ["train_months", "hidden_units", "batch_size"]

    if seasons_to_run is not None:
        selection_results_df = selection_results_df[
            selection_results_df["season"].isin(seasons_to_run)
        ].copy()

    config_summary, best_config_per_season = get_best_configuration_per_season(
        selection_results_df,
        hyperparam_cols
    )

    config_summary.to_csv(
        f"{cfg.output_dir}/{output_tag}_validation_configuration_summary.csv",
        index=False
    )
    best_config_per_season.to_csv(
        f"{cfg.output_dir}/{output_tag}_best_config_per_season_from_validation.csv",
        index=False
    )

    final_holdout_results = []
    final_forecast_frames = []

    for _, config_row in best_config_per_season.iterrows():
        season = config_row["season"]
        key = _selection_key(cfg.model_family, season)

        if key not in FITTED_SEASONAL_WINNERS:
            raise RuntimeError(
                f"No retained fitted winner found for {key}. "
                "Run selection and holdout in the same notebook session."
            )

        artifact = FITTED_SEASONAL_WINNERS[key]
        retained_row = artifact["row"]

        # Strict validation that the retained model is exactly the dataframe winner.
        for col in hyperparam_cols:
            if int(retained_row[col]) != int(config_row[col]):
                raise AssertionError(
                    f"Retained fitted model does not match selected {col}."
                )

        print(f"\n===== TRUE HOLDOUT: {season} =====")
        print("Using the SAME fitted model selected on validation.")
        print("No model initialization and no retraining.")

        if cfg.model_family == "lstm":
            pred_df, metrics = _make_lstm_holdout_from_artifact(
                artifact, cfg
            )
        elif cfg.model_family == "ann":
            pred_df, metrics = _make_ann_holdout_from_artifact(
                artifact, cfg
            )
        else:
            pred_df, metrics = _predict_tft_split_from_artifact(
                artifact, cfg, "test"
            )

        _save_retained_winner_outputs(artifact, cfg, season, pred_df)

        actual_holdout_start = pd.Timestamp(retained_row["test_start"])
        actual_holdout_start_str = actual_holdout_start.strftime("%Y-%m-%d")

        row = {
            "evaluation_type": "true_holdout",
            "model_retrained": False,
            "same_fitted_model_as_validation_winner": True,
            "season": season,
            "eval_month_start": retained_row["eval_month_start"],
            "actual_holdout_start": actual_holdout_start_str,
            "train_months": int(config_row["train_months"]),
            "hidden_units": int(config_row["hidden_units"]),
            "batch_size": int(config_row["batch_size"]),
            "num_heads": (
                int(config_row["num_heads"])
                if cfg.model_family == "tft"
                else np.nan
            ),
            "RMSE": metrics["RMSE"],
            "MAE": metrics["MAE"],
            "R2": metrics["R2"],
            "MAPE": metrics["MAPE"]
        }
        final_holdout_results.append(row)

        pred_df = pred_df.copy()
        pred_df["season"] = season
        pred_df["eval_month_start"] = retained_row["eval_month_start"]
        pred_df["actual_holdout_start"] = actual_holdout_start_str
        pred_df["model_retrained"] = False
        final_forecast_frames.append(pred_df.reset_index())

        forecast_csv = (
            f"{cfg.output_dir}/{output_tag}_{season.lower()}_"
            f"{actual_holdout_start_str}_true_holdout_forecast.csv"
        )
        pred_df.to_csv(forecast_csv)

    final_holdout_results_df = pd.DataFrame(final_holdout_results)
    final_holdout_results_df.to_csv(
        f"{cfg.output_dir}/{output_tag}_true_holdout_metrics.csv",
        index=False
    )

    if final_forecast_frames:
        final_forecasts_df = pd.concat(final_forecast_frames, ignore_index=True)
        final_forecasts_df.to_csv(
            f"{cfg.output_dir}/{output_tag}_true_holdout_forecasts_all.csv",
            index=False
        )
    else:
        final_forecasts_df = pd.DataFrame()

    return (
        config_summary,
        best_config_per_season,
        final_holdout_results_df,
        final_forecasts_df
    )


# Run validation-based seasonal configuration selection followed immediately
# by true-holdout evaluation using the retained fitted winners.
def run_seasonal_selection_and_true_holdout(
    df,
    cfg,
    output_tag,
    seasons_to_run=None
):
    ensure_output_dir(cfg.output_dir)

    # Clear only this model family's retained objects before a new run.
    for key in list(FITTED_SEASONAL_WINNERS):
        if key.startswith(f"{cfg.model_family.lower()}::"):
            del FITTED_SEASONAL_WINNERS[key]

    selection_months = []
    for season, plan in SEASONAL_MONTH_PLAN.items():
        if seasons_to_run is None or season in seasons_to_run:
            selection_months.extend(plan["selection_months"])

    all_selection_results = []

    for eval_month in selection_months:
        month_results = run_one_eval_month(
            df, cfg, eval_month, output_tag
        )
        if not month_results.empty:
            month_results["evaluation_type"] = (
                "seasonal_configuration_validation"
            )
            month_results["metric_split"] = "validation"
            all_selection_results.append(month_results)

    if not all_selection_results:
        raise ValueError(
            f"No successful model runs for model_family={cfg.model_family}"
        )

    selection_results_df = pd.concat(
        all_selection_results,
        ignore_index=True
    )
    selection_results_df.to_csv(
        f"{cfg.output_dir}/{output_tag}_validation_configuration_results.csv",
        index=False
    )

    (
        config_summary,
        best_config_per_season,
        final_holdout_results_df,
        final_forecasts_df
    ) = apply_best_config_to_true_holdout_months(
        df=df,
        cfg=cfg,
        selection_results_df=selection_results_df,
        output_tag=output_tag,
        seasons_to_run=seasons_to_run
    )

    return (
        selection_results_df,
        config_summary,
        best_config_per_season,
        final_holdout_results_df,
        final_forecasts_df
    )


# Validate the result labels and confirm that every final holdout forecast came
# from a retained validation winner that was not retrained.
def validate_no_retraining_seasonal_results(
    selection_results_df,
    final_holdout_results_df
):
    assert set(selection_results_df["metric_split"].dropna().unique()) == {
        "validation"
    }
    assert set(
        selection_results_df["evaluation_type"].dropna().unique()
    ) == {"seasonal_configuration_validation"}
    assert set(
        final_holdout_results_df["evaluation_type"].dropna().unique()
    ) == {"true_holdout"}
    assert (
        final_holdout_results_df["model_retrained"].eq(False).all()
    )
    assert (
        final_holdout_results_df[
            "same_fitted_model_as_validation_winner"
        ].eq(True).all()
    )
    print(
        "VALIDATION PASSED: candidate configurations were compared on "
        "validation, and the same fitted winning model predicted the holdout "
        "without retraining."
    )

In [ ]:
# Save retained deep-learning winners to Google Drive

# Save retained deep-learning winners to Google Drive

# The following information is saved for each winning model:
# - Learned model weights and parameters
# - Winning hyperparameter configuration
# - Fitted Alexander Orr scaler or preprocessor
# - Exact input-feature names and feature order
# - Lookback and forecast-horizon settings
# - Training history
#
# This function saves the fitted winner retained during seasonal validation.
# It does not initialize, fit, or retrain a model.


# Save one retained ANN, LSTM, or TFT seasonal winner and all objects required
# to reconstruct its preprocessing and prediction workflow.
def save_retained_dl_winner(
    model_family,
    season,
    config_name,
    cfg
):
    # Normalize the identifiers used to locate and save the retained winner.
    model_family = model_family.lower()
    season = str(season)

    # Reject unsupported model-family names.
    if model_family not in {
        "ann",
        "lstm",
        "tft"
    }:
        raise ValueError(
            "model_family must be ann, lstm, or tft."
        )

    # Construct the same registry key used when the fitted winner was retained.
    winner_key = _selection_key(
        model_family,
        season
    )

    # The fitted winner must still exist in the current notebook session.
    if winner_key not in FITTED_SEASONAL_WINNERS:
        raise KeyError(
            f"No retained winner found for {winner_key}. "
            "Run that seasonal configuration first."
        )

    # Retrieve the complete fitted artifact and its winning result row.
    artifact = FITTED_SEASONAL_WINNERS[
        winner_key
    ]

    winning_row = artifact["row"]

    # Create a model-family, configuration, and season-specific output folder.
    # SAVED_MODEL_DIR should point to the intended Google Drive model folder.
    save_dir = os.path.join(
        SAVED_MODEL_DIR,
        "DeepLearning",
        model_family.upper(),
        config_name,
        season
    )

    os.makedirs(
        save_dir,
        exist_ok=True
    )


    # Save an ANN winner

    if model_family == "ann":
        # Save the complete fitted Keras ANN architecture and learned weights.
        artifact["model"].save(
            os.path.join(
                save_dir,
                "frozen_model.keras"
            )
        )

        # Save the fitted numeric-scaling and categorical-encoding pipeline.
        joblib.dump(
            artifact["preprocessor"],
            os.path.join(
                save_dir,
                "fitted_preprocessor.joblib"
            )
        )

        # Save the scaler used to standardize and inverse-transform the target.
        joblib.dump(
            artifact["y_scaler"],
            os.path.join(
                save_dir,
                "fitted_target_scaler.joblib"
            )
        )

        # Reconstruct the exact ordered ANN feature lists from the configuration.
        (
            numeric_cols,
            all_feature_cols
        ) = get_ann_feature_lists(
            cfg.target_col,
            cfg.numeric_feature_cols,
            cfg.categorical_feature_cols,
            cfg.lag_steps
        )

        # Record everything required to rebuild the ANN feature table and load
        # the saved winner correctly.
        metadata = {
            "model_family": "ann",
            "season": season,
            "config_name": config_name,
            "target_col": cfg.target_col,
            "numeric_base_cols": list(
                cfg.numeric_feature_cols
            ),
            "categorical_cols": list(
                cfg.categorical_feature_cols
            ),
            "numeric_cols": list(
                numeric_cols
            ),
            "all_feature_cols": list(
                all_feature_cols
            ),
            "lag_steps": [
                int(value)
                for value in cfg.lag_steps
            ],
            "horizon_steps": int(
                cfg.horizon_steps
            ),
            "hidden_units": int(
                winning_row["hidden_units"]
            ),
            "batch_size": int(
                winning_row["batch_size"]
            )
        }


    # Save an LSTM winner

    elif model_family == "lstm":
        # Save the complete fitted Keras LSTM architecture and learned weights.
        artifact["model"].save(
            os.path.join(
                save_dir,
                "frozen_model.keras"
            )
        )

        # Save the scaler fitted on the winning LSTM's training data.
        joblib.dump(
            artifact["scaler"],
            os.path.join(
                save_dir,
                "fitted_scaler.joblib"
            )
        )

        # Record the exact input ordering and temporal settings required to
        # reconstruct the LSTM sequences.
        metadata = {
            "model_family": "lstm",
            "season": season,
            "config_name": config_name,
            "target_col": cfg.target_col,
            "input_cols": list(
                cfg.numeric_feature_cols
            ),
            "model_cols": list(
                artifact["model_cols"]
            ),
            "lookback": int(
                cfg.lookback
            ),
            "horizon_steps": int(
                cfg.horizon_steps
            ),
            "hidden_units": int(
                winning_row["hidden_units"]
            ),
            "batch_size": int(
                winning_row["batch_size"]
            )
        }


    # Save a TFT winner

    else:
        # Save the fitted PyTorch parameter state.
        # Reloading requires rebuilding the same TFT architecture before loading
        # this state dictionary.
        torch.save(
            artifact["model"].state_dict(),
            os.path.join(
                save_dir,
                "frozen_model_state_dict.pt"
            )
        )

        # Save the scaler fitted jointly on historical, future-known, and target
        # variables.
        joblib.dump(
            artifact["joint_scaler"],
            os.path.join(
                save_dir,
                "fitted_joint_scaler.joblib"
            )
        )

        # Save the separate scaler fitted on the static numeric variables.
        joblib.dump(
            artifact["static_scaler"],
            os.path.join(
                save_dir,
                "fitted_static_scaler.joblib"
            )
        )

        # Record the complete TFT feature structure and architecture settings
        # required to reconstruct the model before loading its state dictionary.
        metadata = {
            "model_family": "tft",
            "season": season,
            "config_name": config_name,
            "target_col": cfg.target_col,
            "historical_feature_cols": list(
                cfg.historical_feature_cols
            ),
            "future_known_feature_cols": list(
                cfg.future_known_feature_cols
            ),
            "static_numeric_cols": list(
                cfg.static_numeric_cols
            ),
            "model_cols": list(
                artifact["model_cols"]
            ),
            "lookback": int(
                cfg.lookback
            ),
            "horizon_steps": int(
                cfg.horizon_steps
            ),
            "hidden_units": int(
                winning_row["hidden_units"]
            ),
            "num_heads": int(
                winning_row["num_heads"]
            ),
            "batch_size": int(
                winning_row["batch_size"]
            ),
            "quantiles": [
                float(value)
                for value in cfg.quantiles
            ],
            "learning_rate": float(
                cfg.learning_rate
            ),

            # These values must remain consistent with the architecture created
            # by build_tft_model_from_cfg().
            "dropout": 0.0,
            "lstm_layers": 1
        }


    # Add the winning training-window length to every model family's metadata.
    # Both names are retained for compatibility with code that may expect
    # either metadata field.
    metadata["train_months"] = int(
        winning_row["train_months"]
    )

    metadata["selected_train_months"] = int(
        winning_row["train_months"]
    )


    # Save the complete metadata only after all common fields have been added.
    # This ordering ensures that train_months is included in the JSON file.
    with open(
        os.path.join(
            save_dir,
            "model_metadata.json"
        ),
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            metadata,
            file,
            indent=4
        )


    # Save the complete winning result row, including its selected
    # hyperparameters, validation metrics, split dates, and run identifier.
    with open(
        os.path.join(
            save_dir,
            "winning_configuration.json"
        ),
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            winning_row,
            file,
            indent=4,
            default=str
        )


    # Save the epoch-by-epoch training history when it was preserved in the
    # retained fitted artifact.
    if "history_dict" in artifact:
        pd.DataFrame(
            artifact["history_dict"]
        ).to_csv(
            os.path.join(
                save_dir,
                "training_history.csv"
            ),
            index=False
        )


    # Confirm which fitted winner was saved and where its files were written.
    print("\n" + "=" * 70)
    print("FROZEN DEEP-LEARNING WINNER SAVED")
    print("=" * 70)

    print("Model family:", model_family.upper())
    print("Configuration:", config_name)
    print("Season:", season)
    print("Folder:", save_dir)

In [ ]:
# Audit the retained-winner workflow definition

# Audit the active seasonal retained-winner workflow.
# This check inspects the currently defined functions to confirm that the fitted
# validation winner is reused for holdout evaluation without retraining.
def audit_retained_winner_workflow_definition():
    # Import Python's source-code inspection utilities.
    import inspect

    # Retrieve the source code of the active true-holdout function.
    holdout_source = inspect.getsource(
        apply_best_config_to_true_holdout_months
    )

    # Retrieve the source code of the function that saves the retained winner's
    # forecasts, training history, diagnostics, and metadata.
    saver_source = inspect.getsource(
        _save_retained_winner_outputs
    )

    # Confirm that holdout evaluation retrieves models from the fitted-winner
    # registry created during seasonal validation.
    assert "FITTED_SEASONAL_WINNERS" in holdout_source, (
        "The active holdout workflow is not using the retained-winner registry."
    )

    # Confirm that the holdout function does not call the general training
    # router, which would initialize and fit another model.
    assert "train_one_model(" not in holdout_source, (
        "The active holdout workflow still calls train_one_model() and may "
        "retrain the selected model."
    )

    # Confirm that retained artifacts are validated before their histories,
    # diagnostics, and final model outputs are saved.
    assert "_validate_retained_winner_artifact" in saver_source, (
        "The retained-winner saving function does not validate the fitted "
        "artifact before creating final outputs."
    )

    print(
        "Seasonal workflow audit passed: the fitted validation winner is "
        "retained, its exact training history is used for the epoch plot, "
        "and no retraining occurs before the holdout."
    )


# Run the workflow-definition audit before starting seasonal experiments.
audit_retained_winner_workflow_definition()


# Validation-Only Lookback and Forecast-Horizon Sensitivity Screen

This optional preliminary experiment addresses the committee request without changing the locked production configurations. It evaluates univariate ANN, LSTM, and TFT models for lookbacks of 12 and 24 hours and forecast horizons of 12 and 24 hours using **Summer validation data only**. The final seasonal holdout is neither constructed nor evaluated. Architecture, training length, and batch settings are held fixed so the screen isolates lookback and horizon. Lookback is selected separately within each horizon because 12-hour and 24-hour forecasts are different operational tasks.


In [ ]:
# Validation-only lookback/horizon sensitivity screen

SENSITIVITY_LOOKBACKS = (12, 24)
SENSITIVITY_HORIZONS = (12, 24)
SENSITIVITY_TRAIN_MONTHS = 24
SENSITIVITY_EVAL_START = "2025-07-01"  # Summer validation month

SENSITIVITY_OUTPUT_DIR = os.path.join(
    BASE_OUTPUT,
    "Experiments",
    "Lookback_Horizon_Sensitivity",
    "Summer"
)


def set_sensitivity_seed(seed=SEED):
    """Reset each candidate so grid order does not determine initialization."""
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_sensitivity_train_validation(
    df,
    eval_start_month=SENSITIVITY_EVAL_START,
    train_months=SENSITIVITY_TRAIN_MONTHS
):
    """Return training and validation only; no holdout frame is created."""
    validation_start = ensure_same_tz(
        eval_start_month,
        df.index.tz
    )

    training_start = (
        validation_start
        - pd.DateOffset(months=train_months)
    )

    validation_end = (
        validation_start
        + pd.DateOffset(months=1)
    )

    train_df = df.loc[
        (df.index >= training_start)
        & (df.index < validation_start)
    ].copy()

    validation_df = df.loc[
        (df.index >= validation_start)
        & (df.index < validation_end)
    ].copy()

    if train_df.empty or validation_df.empty:
        raise ValueError(
            "Sensitivity training or validation data is empty."
        )

    return (
        train_df,
        validation_df,
        training_start,
        validation_start,
        validation_end
    )


def make_lstm_train_validation_sequences(
    train_scaled,
    validation_scaled,
    input_cols,
    target_col,
    lookback,
    horizon
):
    model_cols = list(
        dict.fromkeys(
            input_cols + [target_col]
        )
    )
    target_index = model_cols.index(target_col)
    input_count = len(input_cols)

    X_train, y_train, _ = build_lstm_sequences(
        train_scaled[model_cols].to_numpy(),
        train_scaled.index.to_numpy(),
        lookback,
        horizon,
        input_count,
        target_index
    )

    validation_context = pd.concat(
        [
            train_scaled.tail(lookback + horizon),
            validation_scaled
        ],
        axis=0
    )

    X_validation_all, y_validation_all, target_times_all = (
        build_lstm_sequences(
            validation_context[model_cols].to_numpy(),
            validation_context.index.to_numpy(),
            lookback,
            horizon,
            input_count,
            target_index
        )
    )

    target_times_all = pd.DatetimeIndex(
        target_times_all
    )

    validation_mask = np.asarray(
        (target_times_all >= validation_scaled.index.min())
        & (target_times_all <= validation_scaled.index.max()),
        dtype=bool
    )

    return (
        X_train,
        y_train,
        X_validation_all[validation_mask],
        y_validation_all[validation_mask],
        target_times_all[validation_mask]
    )


def make_ann_train_validation_tables(
    train_raw,
    validation_raw,
    target_col,
    lag_steps,
    horizon
):
    train_table = build_ann_frame(
        train_raw,
        target_col=target_col,
        numeric_base_cols=[],
        categorical_cols=[],
        lag_steps=lag_steps,
        horizon_steps=horizon
    )

    validation_context = pd.concat(
        [
            train_raw.tail(max(lag_steps) + horizon),
            validation_raw
        ],
        axis=0
    )

    validation_all = build_ann_frame(
        validation_context,
        target_col=target_col,
        numeric_base_cols=[],
        categorical_cols=[],
        lag_steps=lag_steps,
        horizon_steps=horizon
    )

    validation_table = validation_all.loc[
        (
            validation_all["target_timestamp"]
            >= validation_raw.index.min()
        )
        & (
            validation_all["target_timestamp"]
            <= validation_raw.index.max()
        )
    ].copy()

    return train_table, validation_table


def fit_sensitivity_lstm(
    df,
    lookback,
    horizon,
    hidden_units=64,
    batch_size=64,
    epochs=60,
    patience=10
):
    set_sensitivity_seed()
    train_df, validation_df, train_start, val_start, val_end = (
        get_sensitivity_train_validation(df)
    )

    model_columns = ["totalchlorine"]
    scaler = fit_lstm_scaler(train_df, model_columns)
    train_scaled = apply_lstm_scaler(train_df, scaler, model_columns)
    validation_scaled = apply_lstm_scaler(
        validation_df,
        scaler,
        model_columns
    )

    X_train, y_train, X_validation, y_validation, target_times = (
        make_lstm_train_validation_sequences(
            train_scaled,
            validation_scaled,
            input_cols=["totalchlorine"],
            target_col="totalchlorine",
            lookback=lookback,
            horizon=horizon
        )
    )

    model = build_lstm_model(
        (X_train.shape[1], X_train.shape[2]),
        hidden_units
    )
    callback = EarlyStopping(
        monitor="val_loss",
        patience=patience,
        restore_best_weights=True,
        mode="min"
    )
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=epochs,
        batch_size=batch_size,
        shuffle=False,
        verbose=0,
        callbacks=[callback]
    )

    prediction_scaled = model.predict(
        X_validation,
        verbose=0
    ).reshape(-1)
    actual = inverse_target_from_joint_scaler(
        y_validation,
        scaler,
        model_columns,
        "totalchlorine"
    )
    prediction = inverse_target_from_joint_scaler(
        prediction_scaled,
        scaler,
        model_columns,
        "totalchlorine"
    )

    return {
        "model_family": "LSTM",
        "lookback_hours": lookback,
        "forecast_horizon_hours": horizon,
        "train_months": SENSITIVITY_TRAIN_MONTHS,
        "hidden_units": hidden_units,
        "batch_size": batch_size,
        "num_heads": np.nan,
        "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
        "validation_rows": len(actual),
        "validation_target_start": target_times.min(),
        "validation_target_end": target_times.max(),
        **evaluate_predictions(actual, prediction)
    }


def fit_sensitivity_ann(
    df,
    lookback,
    horizon,
    hidden_units=16,
    batch_size=64,
    epochs=60,
    patience=10
):
    set_sensitivity_seed()
    train_raw, validation_raw, train_start, val_start, val_end = (
        get_sensitivity_train_validation(df)
    )

    lag_lookup = {
        12: [1, 2, 3, 6, 12],
        24: [1, 2, 3, 6, 12, 24]
    }
    lag_steps = lag_lookup[lookback]
    train_table, validation_table = make_ann_train_validation_tables(
        train_raw,
        validation_raw,
        target_col="totalchlorine",
        lag_steps=lag_steps,
        horizon=horizon
    )

    numeric_columns, feature_columns = get_ann_feature_lists(
        "totalchlorine",
        [],
        [],
        lag_steps
    )
    preprocessor = fit_ann_preprocessor(
        train_table,
        numeric_columns,
        [],
        feature_columns
    )
    target_scaler = fit_ann_target_scaler(
        train_table
    )
    X_train = transform_ann_X(
        train_table,
        preprocessor,
        feature_columns
    )
    y_train = transform_ann_y(
        train_table,
        target_scaler
    )
    X_validation = transform_ann_X(
        validation_table,
        preprocessor,
        feature_columns
    )
    y_validation = transform_ann_y(
        validation_table,
        target_scaler
    )

    model = build_ann_model(
        X_train.shape[1],
        hidden_units
    )
    callback = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=patience,
        restore_best_weights=True,
        mode="min"
    )
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_validation, y_validation),
        epochs=epochs,
        batch_size=batch_size,
        shuffle=False,
        verbose=0,
        callbacks=[callback]
    )

    prediction_scaled = model.predict(
        X_validation,
        verbose=0
    ).reshape(-1)
    actual = inverse_ann_target(
        y_validation,
        target_scaler
    )
    prediction = inverse_ann_target(
        prediction_scaled,
        target_scaler
    )

    target_times = pd.DatetimeIndex(
        validation_table["target_timestamp"]
    )

    return {
        "model_family": "ANN",
        "lookback_hours": lookback,
        "forecast_horizon_hours": horizon,
        "train_months": SENSITIVITY_TRAIN_MONTHS,
        "hidden_units": hidden_units,
        "batch_size": batch_size,
        "num_heads": np.nan,
        "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
        "validation_rows": len(actual),
        "validation_target_start": target_times.min(),
        "validation_target_end": target_times.max(),
        **evaluate_predictions(actual, prediction)
    }


def fit_sensitivity_tft(
    df,
    lookback,
    horizon,
    hidden_units=16,
    num_heads=2,
    batch_size=32,
    epochs=40,
    patience=8
):
    set_sensitivity_seed()
    train_df, validation_df, train_start, val_start, val_end = (
        get_sensitivity_train_validation(df)
    )

    temporary_cfg = copy.deepcopy(
        cfg_uni_tft
    )
    temporary_cfg.lookback = lookback
    temporary_cfg.horizon_steps = horizon
    temporary_cfg.epochs = epochs
    temporary_cfg.patience = patience

    joint_columns = list(dict.fromkeys(
        temporary_cfg.historical_feature_cols
        + temporary_cfg.future_known_feature_cols
        + [temporary_cfg.target_col]
    ))
    static_columns = temporary_cfg.static_numeric_cols

    joint_scaler = StandardScaler().fit(
        train_df[joint_columns].values
    )
    static_scaler = StandardScaler().fit(
        train_df[static_columns].values
    )

    def scale_frame(frame):
        scaled = frame.copy()
        scaled[joint_columns] = joint_scaler.transform(
            frame[joint_columns].values
        )
        scaled[static_columns] = static_scaler.transform(
            frame[static_columns].values
        )
        return scaled

    train_scaled = scale_frame(train_df)
    validation_scaled = scale_frame(validation_df)

    train_arrays = build_tft_arrays(
        train_scaled,
        temporary_cfg.historical_feature_cols,
        temporary_cfg.future_known_feature_cols,
        temporary_cfg.static_numeric_cols,
        temporary_cfg.target_col,
        lookback,
        horizon
    )

    validation_context = pd.concat(
        [
            train_scaled.tail(lookback + horizon),
            validation_scaled
        ],
        axis=0
    )
    validation_arrays_all = build_tft_arrays(
        validation_context,
        temporary_cfg.historical_feature_cols,
        temporary_cfg.future_known_feature_cols,
        temporary_cfg.static_numeric_cols,
        temporary_cfg.target_col,
        lookback,
        horizon
    )
    validation_arrays = subset_tft_arrays_by_target_period(
        validation_arrays_all,
        validation_scaled.index.min(),
        validation_scaled.index.max()
    )

    def make_loader(arrays):
        dataset = DictTensorDataset({
            "historical_ts_numeric": arrays["historical_ts_numeric"],
            "future_ts_numeric": arrays["future_ts_numeric"],
            "static_feats_numeric": arrays["static_feats_numeric"],
            "target": arrays["target"]
        })
        return DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False
        )

    train_loader = make_loader(train_arrays)
    validation_loader = make_loader(validation_arrays)
    model = build_tft_model_from_cfg(
        temporary_cfg,
        hidden_units,
        num_heads
    ).to(DEVICE)
    optimizer = torch.optim.Adam(
        filter(lambda parameter: parameter.requires_grad, model.parameters()),
        lr=temporary_cfg.learning_rate
    )
    desired_quantiles = torch.tensor(
        temporary_cfg.quantiles,
        dtype=torch.float32,
        device=DEVICE
    )

    best_state = None
    best_validation_loss = np.inf
    best_epoch = 0
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            batch = {
                key: value.to(DEVICE)
                for key, value in batch.items()
            }
            optimizer.zero_grad()
            output = model(batch)
            loss, _, _ = tft_loss.get_quantiles_loss_and_q_risk(
                outputs=output["predicted_quantiles"],
                targets=batch["target"],
                desired_quantiles=desired_quantiles
            )
            loss.backward()
            optimizer.step()

        model.eval()
        validation_losses = []
        with torch.no_grad():
            for batch in validation_loader:
                batch = {
                    key: value.to(DEVICE)
                    for key, value in batch.items()
                }
                output = model(batch)
                loss, _, _ = tft_loss.get_quantiles_loss_and_q_risk(
                    outputs=output["predicted_quantiles"],
                    targets=batch["target"],
                    desired_quantiles=desired_quantiles
                )
                validation_losses.append(loss.item())

        mean_validation_loss = float(
            np.mean(validation_losses)
        )
        if mean_validation_loss < best_validation_loss:
            best_validation_loss = mean_validation_loss
            best_epoch = epoch + 1
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    artifact = {
        "model": model,
        "joint_scaler": joint_scaler,
        "model_cols": joint_columns,
        "val_loader": validation_loader,
        "val_arrays": validation_arrays
    }
    prediction_frame, metrics = _predict_tft_split_from_artifact(
        artifact,
        temporary_cfg,
        "val"
    )
    target_times = pd.DatetimeIndex(
        validation_arrays["tgt_times"]
    )

    return {
        "model_family": "TFT",
        "lookback_hours": lookback,
        "forecast_horizon_hours": horizon,
        "train_months": SENSITIVITY_TRAIN_MONTHS,
        "hidden_units": hidden_units,
        "batch_size": batch_size,
        "num_heads": num_heads,
        "best_epoch": best_epoch,
        "validation_rows": len(prediction_frame),
        "validation_target_start": target_times.min(),
        "validation_target_end": target_times.max(),
        **metrics
    }


def run_lookback_horizon_sensitivity(df):
    """
    Run 12 validation-only trials: 3 univariate families x 4 window tasks.

    Horizon is treated as a separate forecasting task. The winning lookback is
    selected independently within each family and horizon. No holdout data is
    constructed, predicted, scored, or used for selection.
    """
    ensure_output_dir(
        SENSITIVITY_OUTPUT_DIR
    )
    results = []

    univariate_frames = {
        "LSTM": build_model_df_for_config(df, cfg_uni_lstm),
        "ANN": build_model_df_for_config(df, cfg_uni_ann),
        "TFT": build_model_df_for_config(df, cfg_uni_tft)
    }

    runners = {
        "LSTM": fit_sensitivity_lstm,
        "ANN": fit_sensitivity_ann,
        "TFT": fit_sensitivity_tft
    }

    for family in ["LSTM", "ANN", "TFT"]:
        for horizon in SENSITIVITY_HORIZONS:
            for lookback in SENSITIVITY_LOOKBACKS:
                print(
                    f"Sensitivity: family={family}, "
                    f"lookback={lookback}h, horizon={horizon}h"
                )
                row = runners[family](
                    univariate_frames[family],
                    lookback=lookback,
                    horizon=horizon
                )
                results.append(row)
                tf.keras.backend.clear_session()
                gc.collect()

    results_df = pd.DataFrame(
        results
    ).sort_values([
        "model_family",
        "forecast_horizon_hours",
        "lookback_hours"
    ]).reset_index(drop=True)

    # Select lookback only within a fixed model family and forecast horizon.
    selected_lookbacks_df = (
        results_df
        .sort_values(
            [
                "model_family",
                "forecast_horizon_hours",
                "RMSE",
                "MAE"
            ]
        )
        .groupby(
            [
                "model_family",
                "forecast_horizon_hours"
            ],
            as_index=False
        )
        .head(1)
        .reset_index(drop=True)
    )

    results_df.to_csv(
        os.path.join(
            SENSITIVITY_OUTPUT_DIR,
            "lookback_horizon_validation_results.csv"
        ),
        index=False
    )

    selected_lookbacks_df.to_csv(
        os.path.join(
            SENSITIVITY_OUTPUT_DIR,
            "selected_lookback_within_each_horizon.csv"
        ),
        index=False
    )

    for family in results_df["model_family"].unique():
        family_results = results_df.loc[
            results_df["model_family"] == family
        ]
        family_results.to_csv(
            os.path.join(
                SENSITIVITY_OUTPUT_DIR,
                f"{family.lower()}_validation_results.csv"
            ),
            index=False
        )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(16, 5),
        sharey=False
    )

    for axis, family in zip(
        axes,
        ["ANN", "LSTM", "TFT"]
    ):
        family_results = results_df.loc[
            results_df["model_family"] == family
        ]

        for horizon in SENSITIVITY_HORIZONS:
            horizon_results = family_results.loc[
                family_results["forecast_horizon_hours"] == horizon
            ].sort_values("lookback_hours")

            axis.plot(
                horizon_results["lookback_hours"],
                horizon_results["RMSE"],
                marker="o",
                label=f"{horizon}-hour horizon"
            )

        axis.set_title(f"{family} validation sensitivity")
        axis.set_xlabel("Lookback window (hours)")
        axis.set_ylabel("Validation RMSE (mg/L)")
        axis.set_xticks(SENSITIVITY_LOOKBACKS)
        axis.grid(alpha=0.3)
        axis.legend()

    fig.suptitle(
        "Validation-Only Lookback and Forecast-Horizon Sensitivity"
    )
    fig.tight_layout()
    fig.savefig(
        os.path.join(
            SENSITIVITY_OUTPUT_DIR,
            "lookback_horizon_validation_rmse.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)

    print("\nAll sensitivity results:")
    display(results_df)
    print("\nSelected lookback within each family and horizon:")
    display(selected_lookbacks_df)

    return results_df, selected_lookbacks_df


# # Run this optional screen before the production grids.
# sensitivity_results_df, selected_sensitivity_lookbacks_df = (
#     run_lookback_horizon_sensitivity(df)
# )


#Baseline Statistical Models

#Exponential Smoothing Models

In [ ]:
# Exponential Smoothing

smoothing_fixed_cfg = BaselineConfig(
    model_family="smoothing",
    target_col="totalchlorine",
    output_dir=f"{BASE_OUTPUT}/Baselines/ExponentialSmoothing",
    holdout_start="2025-05-01",
    smoothing_models=("ses", "des", "holt_winters"),
    show_plots=True
)

smoothing_fixed_outputs = run_baseline_experiment(
    df=df,
    cfg=smoothing_fixed_cfg
)


#ARIMA

In [ ]:
# ARIMA

arima_fixed_cfg = BaselineConfig(
    model_family="arima",
    target_col="totalchlorine",
    output_dir=f"{BASE_OUTPUT}/Baselines/ARIMA",
    holdout_start="2025-05-01",
    candidate_models=ARIMA_CANDIDATE_MODELS,
    n_splits=5,
    cv_test_size=720,
    maxiter=700,
    show_plots=True
)

arima_fixed_outputs = run_baseline_experiment(
    df=df,
    cfg=arima_fixed_cfg
)

#SARIMA

In [ ]:
# SARIMA

sarima_fixed_cfg = BaselineConfig(
    model_family="sarima",
    target_col="totalchlorine",
    output_dir=f"{BASE_OUTPUT}/Baselines/SARIMA",
    holdout_start="2025-05-01",
    candidate_models=SARIMA_CANDIDATE_MODELS,
    n_splits=5,
    cv_test_size=720,
    maxiter=700,
    show_plots=True
)

sarima_fixed_outputs = run_baseline_experiment(
    df=df,
    cfg=sarima_fixed_cfg
)

#SARIMAX (All Predictors)

In [ ]:
# SARIMAX All Predictors

sarimax_fixed_cfg = BaselineConfig(
    model_family="sarimax",
    target_col="totalchlorine",
    output_dir=f"{BASE_OUTPUT}/Baselines/SARIMAX/full_model",
    holdout_start="2025-05-01",
    exog_cols=[
        "turbidity",
        "finishwater",
        "chlorine_conversion",
    ],
    candidate_models=SARIMAX_CANDIDATE_MODELS,
    n_splits=5,
    cv_test_size=720,
    maxiter=700,
    show_plots=True
)

sarimax_fixed_outputs = run_baseline_experiment(
    df=df,
    cfg=sarimax_fixed_cfg
)

#SARIMAX (Turbidity)

In [ ]:
# SARIMAX Turbidity

sarimax_fixed_cfg = BaselineConfig(
    model_family="sarimax",
    target_col="totalchlorine",
    output_dir=f"{BASE_OUTPUT}/Baselines/SARIMAX/turbidity",
    holdout_start="2025-05-01",
    exog_cols=[
        "turbidity",
        # "finishwater",
        # "chlorine_conversion",
    ],
    candidate_models=SARIMAX_CANDIDATE_MODELS,
    n_splits=5,
    cv_test_size=720,
    maxiter=1000,
    show_plots=True
)

sarimax_fixed_outputs = run_baseline_experiment(
    df=df,
    cfg=sarimax_fixed_cfg
)

#SARIMAX (Flowrate)

In [ ]:
# SARIMAX Flowrate

sarimax_fixed_cfg = BaselineConfig(
    model_family="sarimax",
    target_col="totalchlorine",
    output_dir=f"{BASE_OUTPUT}/Baselines/SARIMAX/flowrate",
    holdout_start="2025-05-01",
    exog_cols=[
        # "turbidity",
        "finishwater",
        # "chlorine_conversion",
    ],
    candidate_models=SARIMAX_CANDIDATE_MODELS,
    n_splits=5,
    cv_test_size=720,
    maxiter=1000,
    show_plots=True
)

sarimax_fixed_outputs = run_baseline_experiment(
    df=df,
    cfg=sarimax_fixed_cfg
)

#SARIMAX (Turbidiy + Flowrate)

In [ ]:
# SARIMAX Flowrate and Turbidity

sarimax_fixed_cfg = BaselineConfig(
    model_family="sarimax",
    target_col="totalchlorine",
    output_dir=f"{BASE_OUTPUT}/Baselines/SARIMAX/flowrate_turbidity",
    holdout_start="2025-05-01",
    exog_cols=[
        "turbidity",
        "finishwater",
        # "chlorine_conversion",
    ],
    candidate_models=SARIMAX_CANDIDATE_MODELS,
    n_splits=5,
    cv_test_size=720,
    maxiter=1000,
    show_plots=True
)

sarimax_fixed_outputs = run_baseline_experiment(
    df=df,
    cfg=sarimax_fixed_cfg
)

#AI/ML Models

# *Deep Learning Models*

#LSTM Univariate

#Chunk 1-Winter

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_uni_lstm,
    chunk_name="chunk_1_winter",
    config_name="Univariate"
)


#Chunk 2-Spring

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_uni_lstm,
    chunk_name="chunk_2_spring",
    config_name="Univariate"
)


#Chunk 3-Summer

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_uni_lstm,
    chunk_name="chunk_3_summer",
    config_name="Univariate"
)


#Chunk 4-Autumn

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_uni_lstm,
    chunk_name="chunk_4_autumn",
    config_name="Univariate"
)


#LSTM Multivariable (Full Model: Chlorine Conversion, Turbidity, and Flowrate)

#Chunk 1-Winter

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_lstm,
    chunk_name="chunk_1_winter",
    config_name="FullModel"
)


#Chunk 2-Spring

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_lstm,
    chunk_name="chunk_2_spring",
    config_name="FullModel"
)


#Chunk 3-Summer

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_lstm,
    chunk_name="chunk_3_summer",
    config_name="FullModel"
)


#Chunk 4-Autumn

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_lstm,
    chunk_name="chunk_4_autumn",
    config_name="FullModel"
)


#LSTM Multivariable (Turbidity)

#Chunk 1-Winter

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_lstm,
    chunk_name="chunk_1_winter",
    config_name="Turbidity"
)

#Chunk 2-Spring

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_lstm,
    chunk_name="chunk_2_spring",
    config_name="Turbidity"
)

#Chunk 3-Summer

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_lstm,
    chunk_name="chunk_3_summer",
    config_name="Turbidity"
)

#Chunk 4-Autumn

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_lstm,
    chunk_name="chunk_4_autumn",
    config_name="Turbidity"
)

#LSTM Multivariable (Flowrate)

#Chunk 1-Winter

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_lstm,
    chunk_name="chunk_1_winter",
    config_name="Flowrate"
)

#Chunk 2-Spring

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_lstm,
    chunk_name="chunk_2_spring",
    config_name="Flowrate"
)

#Chunk 3-Summer

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_lstm,
    chunk_name="chunk_3_summer",
    config_name="Flowrate"
)

#Chunk 4-Autumn

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_lstm,
    chunk_name="chunk_4_autumn",
    config_name="Flowrate"
)

#LSTM Multivariable (Flowrate + Turbidity)

#Chunk 1-Winter

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_lstm,
    chunk_name="chunk_1_winter",
    config_name="Flowrate_Turbidity"
)

#Chunk 2-Spring

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_lstm,
    chunk_name="chunk_2_spring",
    config_name="Flowrate_Turbidity"
)

#Chunk 3-Summer

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_lstm,
    chunk_name="chunk_3_summer",
    config_name="Flowrate_Turbidity"
)

#Chunk 4-Autumn

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_lstm,
    chunk_name="chunk_4_autumn",
    config_name="Flowrate_Turbidity"
)

#ANN Univariate

#Chunk 1-Winter

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_uni_ann,
    chunk_name="chunk_1_winter",
    config_name="Univariate"
)


#Chunk 2-Spring

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_uni_ann,
    chunk_name="chunk_2_spring",
    config_name="Univariate"
)


#Chunk 3-Summer

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_uni_ann,
    chunk_name="chunk_3_summer",
    config_name="Univariate"
)


#Chunk 4-Autumn (2025)

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_uni_ann,
    chunk_name="chunk_4_autumn",
    config_name="Univariate"
)


#ANN Multivariable (Full Model: Chlorine Conversion, Turbidity, and Flowrate)

#Chunk 1-Winter

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_ann,
    chunk_name="chunk_1_winter",
    config_name="FullModel"
)


#Chunk 2-Spring

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_ann,
    chunk_name="chunk_2_spring",
    config_name="FullModel"
)


#Chunk 3-Summer

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_ann,
    chunk_name="chunk_3_summer",
    config_name="FullModel"
)


#Chunk 4-Autumn

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_ann,
    chunk_name="chunk_4_autumn",
    config_name="FullModel"
)


#ANN Multivariable (Turbidity)

#Chunk 1-Winter

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_ann,
    chunk_name="chunk_1_winter",
    config_name="Turbidity"
)

#Chunk 2-Spring

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_ann,
    chunk_name="chunk_2_spring",
    config_name="Turbidity"
)

#Chunk 3-Summer

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_ann,
    chunk_name="chunk_3_summer",
    config_name="Turbidity"
)

#Chunk 4-Autumn

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_ann,
    chunk_name="chunk_4_autumn",
    config_name="Turbidity"
)

#ANN Multivariable (Flowrate)

#Chunk 1-Winter

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_ann,
    chunk_name="chunk_1_winter",
    config_name="Flowrate"
)

#Chunk 2-Spring

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_ann,
    chunk_name="chunk_2_spring",
    config_name="Flowrate"
)

#Chunk 3-Summer

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_ann,
    chunk_name="chunk_3_summer",
    config_name="Flowrate"
)

#Chunk 4-Autumn

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_ann,
    chunk_name="chunk_4_autumn",
    config_name="Flowrate"
)

#ANN Multivariable (Flowrate + Turbidity)

#Chunk 1-Winter

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_ann,
    chunk_name="chunk_1_winter",
    config_name="Flowrate_Turbidity"
)

#Chunk 2-Spring

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_ann,
    chunk_name="chunk_2_spring",
    config_name="Flowrate_Turbidity"
)

#Chunk 3-Summer

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_ann,
    chunk_name="chunk_3_summer",
    config_name="Flowrate_Turbidity"
)

#Chunk 4-Autumn

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_ann,
    chunk_name="chunk_4_autumn",
    config_name="Flowrate_Turbidity"
)

#TFT Univariate

#Chunk 1-Winter

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_uni_tft,
    chunk_name="chunk_1_winter",
    config_name="Univariate"
)


#Chunk 2-Spring

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_uni_tft,
    chunk_name="chunk_2_spring",
    config_name="Univariate"
)


#Chunk 3-Summer

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_uni_tft,
    chunk_name="chunk_3_summer",
    config_name="Univariate"
)


#Chunk 4-Autumn

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_uni_tft,
    chunk_name="chunk_4_autumn",
    config_name="Univariate"
)


#TFT Multivariable (Full Model: Chlorine Conversion, Turbidity, and Flowrate)

#Chunk 1-Winter

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_tft,
    chunk_name="chunk_1_winter",
    config_name="FullModel"
)


#Chunk 2-Spring

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_tft,
    chunk_name="chunk_2_spring",
    config_name="FullModel"
)


#Chunk 3-Summer

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_tft,
    chunk_name="chunk_3_summer",
    config_name="FullModel"
)


#Chunk 4-Autumn

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_tft,
    chunk_name="chunk_4_autumn",
    config_name="FullModel"
)


#TFT Multivariable (Turbidity)

#Chunk 1-Winter

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_tft,
    chunk_name="chunk_1_winter",
    config_name="Turbidity"
)


#Chunk 2-Spring

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_tft,
    chunk_name="chunk_2_spring",
    config_name="Turbidity"
)

#Chunk 4-Summer

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_tft,
    chunk_name="chunk_3_summer",
    config_name="Turbidity"
)

#Chunk 5-Autumn

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_tft,
    chunk_name="chunk_4_autumn",
    config_name="Turbidity"
)

#TFT Multivariable (Flowrate)

#Chunk 1-Winter

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_tft,
    chunk_name="chunk_1_winter",
    config_name="Flowrate"
)

#Chunk 2-Spring

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_tft,
    chunk_name="chunk_2_spring",
    config_name="Flowrate"
)

#Chunk 3-Summer

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_tft,
    chunk_name="chunk_3_summer",
    config_name="Flowrate"
)

#Chunk 4-Autumn

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_tft,
    chunk_name="chunk_4_autumn",
    config_name="Flowrate"
)

#TFT Multivariable (Flowrate + Turbidity)

#Chunk 1-Winter

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_tft,
    chunk_name="chunk_1_winter",
    config_name="Flowrate_Turbidity"
)

#Chunk 2-Spring

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_tft,
    chunk_name="chunk_2_spring",
    config_name="Flowrate_Turbidity"
)

#Chunk 3-Summer

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_tft,
    chunk_name="chunk_3_summer",
    config_name="Flowrate_Turbidity"
)

#Chunk 4-Autumn

In [ ]:
# Run this model configuration for one seasonal validation/holdout chunk.
(
    selection_results_df,
    config_summary,
    best_config_per_season,
    final_holdout_results_df,
    final_forecasts_df
) = run_and_save_dl_chunk(
    df=df,
    cfg=cfg_multi_tft,
    chunk_name="chunk_4_autumn",
    config_name="Flowrate_Turbidity"
)

# Percentile-normalized cross-plant generalization exploration

This optional experiment is separate from every raw-scale model above. It
re-expresses continuous plant measurements as empirical percentiles in `[0, 1]`
so that models learn relative operating position rather than plant-specific
measurement ranges.

Important interpretation rules:

- Percentile transformers are fitted using the model's training window only.
- Validation and holdout observations never fit the transformations.
- Raw ANN, LSTM, TFT, baseline and classification results remain unchanged.
- The new source models forecast percentile-normalized residual chlorine.
- Predictions are also converted back to mg/L for operational interpretation.
- Applying a Sergio training-distribution transform is distribution adaptation,
  not strict zero-shot domain generalization.
- Regulatory conclusions must continue to use reconstructed mg/L values.


In [ ]:
# Percentile-normalization settings and reusable source-model helpers.
PERCENTILE_SOURCE_OUTPUT = os.path.join(
    BASE_OUTPUT,
    "Generalization",
    "Percentile_Normalized",
    "Alexander_Orr"
)

PERCENTILE_SOURCE_MODEL_DIR = os.path.join(
    SAVED_MODEL_DIR,
    "Percentile_Normalized",
    "DeepLearning"
)

PERCENTILE_CONTINUOUS_COLUMNS = [
    "totalchlorine",
    "turbidity",
    "finishwater"
]

os.makedirs(PERCENTILE_SOURCE_OUTPUT, exist_ok=True)
os.makedirs(PERCENTILE_SOURCE_MODEL_DIR, exist_ok=True)


def percentile_columns_for_config(df_in, cfg):
    # Transform only continuous plant measurements used by this configuration.
    # Binary indicators, UTC calendar variables and static identifiers retain
    # their original meaning.
    if cfg.model_family in {"ann", "lstm"}:
        configured_columns = list(cfg.numeric_feature_cols) + [cfg.target_col]
    elif cfg.model_family == "tft":
        configured_columns = (
            list(cfg.historical_feature_cols)
            + list(cfg.future_known_feature_cols)
            + [cfg.target_col]
        )
    else:
        raise ValueError("Percentile exploration supports ANN, LSTM, and TFT.")

    return [
        column
        for column in PERCENTILE_CONTINUOUS_COLUMNS
        if column in configured_columns and column in df_in.columns
    ]


def fit_percentile_transformers(training_frame, columns):
    # Fit one empirical distribution per continuous measurement using training
    # rows only. Uniform output represents approximate percentile rank.
    transformers = {}

    for column in columns:
        values = pd.to_numeric(
            training_frame[column],
            errors="coerce"
        ).dropna().to_numpy().reshape(-1, 1)

        if len(values) < 2:
            raise ValueError(
                f"At least two training observations are required for {column}."
            )

        transformer = QuantileTransformer(
            n_quantiles=min(1000, len(values)),
            output_distribution="uniform",
            subsample=None,
            random_state=SEED
        )
        transformer.fit(values)
        transformers[column] = transformer

    return transformers


def apply_percentile_transformers(frame, transformers):
    transformed = frame.copy()

    for column, transformer in transformers.items():
        numeric_values = pd.to_numeric(
            transformed[column],
            errors="coerce"
        ).to_numpy().reshape(-1, 1)

        transformed[column] = transformer.transform(
            numeric_values
        ).reshape(-1)

    return transformed


def inverse_percentile_values(values, transformer):
    # Neural predictions can fall slightly outside [0, 1]. Clip only for the
    # inverse empirical-CDF mapping and preserve unclipped percentile outputs.
    values = np.asarray(values, dtype=float).reshape(-1)
    clipped = np.clip(values, 0.0, 1.0)

    return transformer.inverse_transform(
        clipped.reshape(-1, 1)
    ).reshape(-1)

def percentile_band_metrics(actual_percentiles, forecast_percentiles):
    # Report error separately in low, typical, and high operating regions.
    frame = pd.DataFrame({
        "actual_percentile": np.asarray(actual_percentiles, dtype=float),
        "forecast_percentile": np.asarray(forecast_percentiles, dtype=float)
    }).dropna()

    band_edges = [0.0, 0.10, 0.30, 0.70, 0.90, 1.0]
    band_labels = [
        "0th-10th",
        "10th-30th",
        "30th-70th",
        "70th-90th",
        "90th-100th"
    ]
    frame["PercentileBand"] = pd.cut(
        frame["actual_percentile"],
        bins=band_edges,
        labels=band_labels,
        include_lowest=True
    )

    rows = []
    for band in band_labels:
        part = frame.loc[frame["PercentileBand"] == band]
        if part.empty:
            rows.append({
                "PercentileBand": band,
                "Observations": 0,
                "RMSE_Percentile": np.nan,
                "MAE_Percentile": np.nan,
                "Bias_Percentile": np.nan
            })
            continue

        errors = (
            part["forecast_percentile"]
            - part["actual_percentile"]
        )
        rows.append({
            "PercentileBand": band,
            "Observations": int(len(part)),
            "RMSE_Percentile": float(np.sqrt(np.mean(errors ** 2))),
            "MAE_Percentile": float(np.mean(np.abs(errors))),
            "Bias_Percentile": float(np.mean(errors))
        })

    return pd.DataFrame(rows)



def load_raw_winning_configuration(model_family, config_name, season):
    configuration_path = os.path.join(
        SAVED_MODEL_DIR,
        "DeepLearning",
        model_family.upper(),
        config_name,
        season,
        "winning_configuration.json"
    )

    if not os.path.exists(configuration_path):
        raise FileNotFoundError(
            "Run and save the corresponding raw-scale winner first: "
            f"{configuration_path}"
        )

    with open(configuration_path, encoding="utf-8") as file:
        return json.load(file)


def restrict_config_to_saved_winner(cfg, winning_row):
    # Reuse the raw model's selected architecture and training-window length so
    # the exploration changes the data representation rather than retuning a
    # larger hyperparameter grid on the same validation month.
    restricted = copy.deepcopy(cfg)
    restricted.train_length_options = (int(winning_row["train_months"]),)
    restricted.hidden_units_options = (int(winning_row["hidden_units"]),)
    restricted.batch_size_options = (int(winning_row["batch_size"]),)

    if restricted.model_family == "tft":
        restricted.num_heads_options = (int(winning_row["num_heads"]),)

    return restricted


# Save the fitted percentile model, transformations, predictions and metrics.
def save_percentile_source_artifact(
    artifact,
    cfg,
    config_name,
    season,
    transformers,
    transformed_columns,
    winning_row,
    percentile_predictions,
    mg_l_predictions,
    percentile_metrics,
    mg_l_metrics
):
    # Identify the model family and create its output folder.
    family = cfg.model_family.lower()

    save_dir = os.path.join(
        PERCENTILE_SOURCE_MODEL_DIR,
        family.upper(),
        config_name,
        season
    )

    os.makedirs(
        save_dir,
        exist_ok=True
    )

    # Save the plant-specific percentile transformations.
    joblib.dump(
        transformers,
        os.path.join(
            save_dir,
            "plant_percentile_transformers.joblib"
        )
    )

    # Save ANN-specific components.
    if family == "ann":
        artifact["model"].save(
            os.path.join(
                save_dir,
                "frozen_model.keras"
            )
        )

        joblib.dump(
            artifact["preprocessor"],
            os.path.join(
                save_dir,
                "fitted_preprocessor.joblib"
            )
        )

        joblib.dump(
            artifact["y_scaler"],
            os.path.join(
                save_dir,
                "fitted_target_scaler.joblib"
            )
        )

        numeric_cols, all_feature_cols = (
            get_ann_feature_lists(
                cfg.target_col,
                cfg.numeric_feature_cols,
                cfg.categorical_feature_cols,
                cfg.lag_steps
            )
        )

        metadata = {
            "model_family": "ann",
            "target_col": cfg.target_col,
            "numeric_base_cols": list(
                cfg.numeric_feature_cols
            ),
            "categorical_cols": list(
                cfg.categorical_feature_cols
            ),
            "numeric_cols": list(
                numeric_cols
            ),
            "all_feature_cols": list(
                all_feature_cols
            ),
            "lag_steps": [
                int(value)
                for value in cfg.lag_steps
            ],
            "horizon_steps": int(
                cfg.horizon_steps
            ),
            "hidden_units": int(
                winning_row["hidden_units"]
            ),
            "batch_size": int(
                winning_row["batch_size"]
            )
        }

    # Save LSTM-specific components.
    elif family == "lstm":
        artifact["model"].save(
            os.path.join(
                save_dir,
                "frozen_model.keras"
            )
        )

        joblib.dump(
            artifact["scaler"],
            os.path.join(
                save_dir,
                "fitted_scaler.joblib"
            )
        )

        metadata = {
            "model_family": "lstm",
            "target_col": cfg.target_col,
            "input_cols": list(
                cfg.numeric_feature_cols
            ),
            "model_cols": list(
                artifact["model_cols"]
            ),
            "lookback": int(
                cfg.lookback
            ),
            "horizon_steps": int(
                cfg.horizon_steps
            ),
            "hidden_units": int(
                winning_row["hidden_units"]
            ),
            "batch_size": int(
                winning_row["batch_size"]
            )
        }

    # Save TFT-specific components.
    elif family == "tft":
        torch.save(
            artifact["model"].state_dict(),
            os.path.join(
                save_dir,
                "frozen_model_state_dict.pt"
            )
        )

        joblib.dump(
            artifact["joint_scaler"],
            os.path.join(
                save_dir,
                "fitted_joint_scaler.joblib"
            )
        )

        joblib.dump(
            artifact["static_scaler"],
            os.path.join(
                save_dir,
                "fitted_static_scaler.joblib"
            )
        )

        metadata = {
            "model_family": "tft",
            "target_col": cfg.target_col,
            "historical_feature_cols": list(
                cfg.historical_feature_cols
            ),
            "future_known_feature_cols": list(
                cfg.future_known_feature_cols
            ),
            "static_numeric_cols": list(
                cfg.static_numeric_cols
            ),
            "model_cols": list(
                artifact["model_cols"]
            ),
            "lookback": int(
                cfg.lookback
            ),
            "horizon_steps": int(
                cfg.horizon_steps
            ),
            "hidden_units": int(
                winning_row["hidden_units"]
            ),
            "num_heads": int(
                winning_row["num_heads"]
            ),
            "batch_size": int(
                winning_row["batch_size"]
            ),
            "quantiles": [
                float(value)
                for value in cfg.quantiles
            ],
            "learning_rate": float(
                cfg.learning_rate
            ),
            "dropout": 0.0,
            "lstm_layers": 1
        }

    else:
        raise ValueError(
            "Percentile artifact saving supports ANN, LSTM, and TFT."
        )

    # Add metadata shared by ANN, LSTM and TFT.
    metadata.update({
        "season": season,
        "config_name": config_name,
        "train_months": int(
            winning_row["train_months"]
        ),
        "selected_train_months": int(
            winning_row["train_months"]
        ),
        "data_representation": (
            "within_plant_training_percentiles"
        ),
        "percentile_output_distribution": (
            "uniform_0_1"
        ),
        "percentile_transformed_columns": list(
            transformed_columns
        ),
        "percentile_transform_fit_start": str(
            winning_row["train_start"]
        ),
        "percentile_transform_fit_end_exclusive": str(
            winning_row["train_end"]
        ),
        "timeline_timezone": "UTC",
        "target_timestamp_alignment": "t_plus_horizon",
        "strict_zero_shot": False,
        "interpretation": (
            "Percentile-normalized source model for "
            "distribution-adapted cross-plant transfer exploration"
        )
    })

    # Save the metadata needed to reload the fitted model.
    with open(
        os.path.join(
            save_dir,
            "model_metadata.json"
        ),
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            metadata,
            file,
            indent=4
        )

    # Save the hyperparameter configuration used for this model.
    with open(
        os.path.join(
            save_dir,
            "winning_configuration.json"
        ),
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            winning_row,
            file,
            indent=4,
            default=str
        )

    # Save the training history when it is available.
    if "history_dict" in artifact:
        pd.DataFrame(
            artifact["history_dict"]
        ).to_csv(
            os.path.join(
                save_dir,
                "training_history.csv"
            ),
            index=False
        )

    # Save holdout predictions on the percentile-normalized scale.
    percentile_predictions.to_csv(
        os.path.join(
            save_dir,
            "alexander_holdout_predictions_percentile.csv"
        )
    )

    # Locate the actual and predicted percentile columns.
    actual_column = next(
        column
        for column in percentile_predictions.columns
        if column.startswith("actual_")
    )

    forecast_column = next(
        column
        for column in percentile_predictions.columns
        if column.startswith("predicted_")
    )

    # Calculate and save metrics for each percentile band.
    percentile_band_results = percentile_band_metrics(
        percentile_predictions[actual_column],
        percentile_predictions[forecast_column]
    )

    percentile_band_results.to_csv(
        os.path.join(
            save_dir,
            "alexander_holdout_metrics_by_percentile_band.csv"
        ),
        index=False
    )

    # Save holdout predictions converted back to mg/L.
    mg_l_predictions.to_csv(
        os.path.join(
            save_dir,
            "alexander_holdout_predictions_mg_L.csv"
        )
    )

    # Combine percentile-scale and original-scale metrics.
    metrics_row = {
        "ModelFamily": family.upper(),
        "Configuration": config_name,
        "Season": season,
        **{
            f"Percentile_{key}": value
            for key, value in percentile_metrics.items()
        },
        **{
            f"mg_L_{key}": value
            for key, value in mg_l_metrics.items()
        }
    }

    # Save the combined metrics for this completed run.
    pd.DataFrame([
        metrics_row
    ]).to_csv(
        os.path.join(
            save_dir,
            "alexander_holdout_metrics_both_scales.csv"
        ),
        index=False
    )

    # Display the metrics and create both forecast plots immediately.
    display_and_save_percentile_run_outputs(
        percentile_predictions=percentile_predictions,
        mg_l_predictions=mg_l_predictions,
        metrics_row=metrics_row,
        save_dir=save_dir
    )

    print(
        "Saved percentile-normalized source model:",
        save_dir
    )

    # Return both objects expected by run_percentile_source_experiment().
    return save_dir, metrics_row

# Display and save the metrics and forecast plots for each completed percentile run.
def display_and_save_percentile_run_outputs(
    percentile_predictions,
    mg_l_predictions,
    metrics_row,
    save_dir
):
    # Convert the completed run's metrics into a one-row table.
    metrics_df = pd.DataFrame([metrics_row])

    print("\n" + "=" * 70)
    print(
        f"PERCENTILE HOLDOUT RESULTS: "
        f"{metrics_row['ModelFamily']} | "
        f"{metrics_row['Configuration']} | "
        f"{metrics_row['Season']}"
    )
    print("=" * 70)

    display(metrics_df)

    # Locate the actual and predicted columns because their complete names
    # depend on the model and forecast horizon.
    percentile_actual_col = next(
        column
        for column in percentile_predictions.columns
        if column.startswith("actual_")
    )

    percentile_forecast_col = next(
        column
        for column in percentile_predictions.columns
        if column.startswith("predicted_")
    )

    # Plot forecast versus actual on the percentile-normalized scale.
    fig, ax = plt.subplots(figsize=(14, 6))

    ax.plot(
        percentile_predictions.index,
        percentile_predictions[percentile_actual_col],
        label="Actual percentile"
    )

    ax.plot(
        percentile_predictions.index,
        percentile_predictions[percentile_forecast_col],
        label="Forecast percentile"
    )

    # Use confidence intervals when available. TFT may instead provide
    # explicitly named q10 and q90 quantile forecasts.
    percentile_lower = (
        "lower_ci"
        if "lower_ci" in percentile_predictions.columns
        else "q10"
        if "q10" in percentile_predictions.columns
        else None
    )

    percentile_upper = (
        "upper_ci"
        if "upper_ci" in percentile_predictions.columns
        else "q90"
        if "q90" in percentile_predictions.columns
        else None
    )

    if (
        percentile_lower is not None
        and percentile_upper is not None
    ):
        ax.fill_between(
            percentile_predictions.index,
            percentile_predictions[percentile_lower],
            percentile_predictions[percentile_upper],
            alpha=0.20,
            label="Forecast interval"
        )

    ax.set_title(
        f"{metrics_row['ModelFamily']} "
        f"{metrics_row['Configuration']} "
        f"{metrics_row['Season']} Forecast — Percentile Scale"
    )
    ax.set_xlabel("Forecast Timestamp (UTC)")
    ax.set_ylabel("Residual Chlorine Percentile")
    ax.legend()
    ax.grid(alpha=0.3)

    fig.autofmt_xdate()
    fig.tight_layout()

    fig.savefig(
        os.path.join(
            save_dir,
            "alexander_holdout_forecast_percentile.png"
        ),
        dpi=600,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    # Plot the same forecast after converting the target back to mg/L.
    fig, ax = plt.subplots(figsize=(14, 6))

    ax.plot(
        mg_l_predictions.index,
        mg_l_predictions["actual"],
        label="Actual"
    )

    ax.plot(
        mg_l_predictions.index,
        mg_l_predictions["forecast"],
        label="Forecast"
    )

    # Plot the inverse-transformed interval when one is available.
    if (
        "lower_ci" in mg_l_predictions.columns
        and "upper_ci" in mg_l_predictions.columns
    ):
        ax.fill_between(
            mg_l_predictions.index,
            mg_l_predictions["lower_ci"],
            mg_l_predictions["upper_ci"],
            alpha=0.20,
            label="Forecast interval"
        )

    elif (
        "q10" in mg_l_predictions.columns
        and "q90" in mg_l_predictions.columns
    ):
        ax.fill_between(
            mg_l_predictions.index,
            mg_l_predictions["q10"],
            mg_l_predictions["q90"],
            alpha=0.20,
            label="Forecast interval"
        )

    ax.set_title(
        f"{metrics_row['ModelFamily']} "
        f"{metrics_row['Configuration']} "
        f"{metrics_row['Season']} Forecast — Original Scale"
    )
    ax.set_xlabel("Forecast Timestamp (UTC)")
    ax.set_ylabel("Residual Chlorine (mg/L)")
    ax.legend()
    ax.grid(alpha=0.3)

    fig.autofmt_xdate()
    fig.tight_layout()

    fig.savefig(
        os.path.join(
            save_dir,
            "alexander_holdout_forecast_mg_L.png"
        ),
        dpi=600,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

def run_percentile_source_experiment(
    df,
    cfg,
    config_name,
    season
):
    # Load the raw model's already selected architecture and split information.
    architecture_source = getattr(
        cfg,
        "percentile_architecture_source",
        config_name
    )
    raw_winner = load_raw_winning_configuration(
        cfg.model_family,
        architecture_source,
        season
    )
    restricted_cfg = restrict_config_to_saved_winner(cfg, raw_winner)
    model_df = build_model_df_for_config(df, restricted_cfg)

    train_start = ensure_same_tz(raw_winner["train_start"], model_df.index.tz)
    train_end = ensure_same_tz(raw_winner["train_end"], model_df.index.tz)
    transform_training_frame = model_df.loc[
        (model_df.index >= train_start)
        & (model_df.index < train_end)
    ].copy()

    transformed_columns = percentile_columns_for_config(
        model_df,
        restricted_cfg
    )
    if restricted_cfg.target_col not in transformed_columns:
        raise ValueError("The target must be percentile transformed.")

    transformers = fit_percentile_transformers(
        transform_training_frame,
        transformed_columns
    )
    percentile_df = apply_percentile_transformers(
        model_df,
        transformers
    )

    registry_key = _selection_key(cfg.model_family, season)
    previous_artifact = FITTED_SEASONAL_WINNERS.get(registry_key)

    try:
        run_one_eval_month(
            percentile_df,
            restricted_cfg,
            raw_winner["eval_month_start"],
            chunk_name=f"percentile_{config_name}_{season}"
        )
        percentile_artifact = FITTED_SEASONAL_WINNERS[registry_key]
    finally:
        # Preserve the original raw-scale retained winner in memory.
        if previous_artifact is None:
            FITTED_SEASONAL_WINNERS.pop(registry_key, None)
        else:
            FITTED_SEASONAL_WINNERS[registry_key] = previous_artifact

    if cfg.model_family == "ann":
        percentile_predictions, percentile_metrics = (
            _make_ann_holdout_from_artifact(
                percentile_artifact,
                restricted_cfg
            )
        )
    elif cfg.model_family == "lstm":
        percentile_predictions, percentile_metrics = (
            _make_lstm_holdout_from_artifact(
                percentile_artifact,
                restricted_cfg
            )
        )
    else:
        percentile_predictions, percentile_metrics = (
            _predict_tft_split_from_artifact(
                percentile_artifact,
                restricted_cfg,
                "test"
            )
        )

    actual_column = next(
        column for column in percentile_predictions.columns
        if column.startswith("actual_")
    )
    forecast_column = next(
        column for column in percentile_predictions.columns
        if column.startswith("predicted_")
    )
    target_transformer = transformers[restricted_cfg.target_col]

    mg_l_predictions = pd.DataFrame(
        index=percentile_predictions.index
    )
    mg_l_predictions["actual"] = inverse_percentile_values(
        percentile_predictions[actual_column],
        target_transformer
    )
    mg_l_predictions["forecast"] = inverse_percentile_values(
        percentile_predictions[forecast_column],
        target_transformer
    )

    for interval_column in ["lower_ci", "upper_ci", "q10", "q50", "q90"]:
        if interval_column in percentile_predictions.columns:
            mg_l_predictions[interval_column] = inverse_percentile_values(
                percentile_predictions[interval_column],
                target_transformer
            )

    mg_l_metrics = evaluate_predictions(
        mg_l_predictions["actual"],
        mg_l_predictions["forecast"]
    )

    save_dir, metrics_row = save_percentile_source_artifact(
        artifact=percentile_artifact,
        cfg=restricted_cfg,
        config_name=config_name,
        season=season,
        transformers=transformers,
        transformed_columns=transformed_columns,
        winning_row=percentile_artifact["row"],
        percentile_predictions=percentile_predictions,
        mg_l_predictions=mg_l_predictions,
        percentile_metrics=percentile_metrics,
        mg_l_metrics=mg_l_metrics
    )

    return {
        "artifact": percentile_artifact,
        "percentile_predictions": percentile_predictions,
        "mg_l_predictions": mg_l_predictions,
        "metrics": metrics_row,
        "save_dir": save_dir
    }


def make_percentile_source_configurations():
    """Build every feature configuration used for cross-plant transfer.

    The three intermediate configurations reuse the architecture selected for
    the matching FullModel family/season. They differ only in their predictor
    inputs, so no additional validation-based architecture search is added.
    """
    configurations = []

    family_specs = [
        (cfg_uni_ann, "Univariate", "Univariate"),
        (cfg_multi_ann, "Turbidity", "FullModel"),
        (cfg_multi_ann, "Flowrate", "FullModel"),
        (cfg_multi_ann, "Turbidity_Flowrate", "FullModel"),
        (cfg_multi_ann, "FullModel", "FullModel"),
        (cfg_uni_lstm, "Univariate", "Univariate"),
        (cfg_multi_lstm, "Turbidity", "FullModel"),
        (cfg_multi_lstm, "Flowrate", "FullModel"),
        (cfg_multi_lstm, "Turbidity_Flowrate", "FullModel"),
        (cfg_multi_lstm, "FullModel", "FullModel"),
        (cfg_uni_tft, "Univariate", "Univariate"),
        (cfg_multi_tft, "Turbidity", "FullModel"),
        (cfg_multi_tft, "Flowrate", "FullModel"),
        (cfg_multi_tft, "Turbidity_Flowrate", "FullModel"),
        (cfg_multi_tft, "FullModel", "FullModel")
    ]

    feature_map = {
        "Turbidity": ["turbidity"],
        "Flowrate": ["finishwater"],
        "Turbidity_Flowrate": ["turbidity", "finishwater"]
    }

    for base_cfg, config_name, architecture_source in family_specs:
        cfg = copy.deepcopy(base_cfg)

        if config_name in feature_map:
            selected_features = feature_map[config_name]

            if cfg.model_family == "ann":
                # Chlorine history is supplied by ANN lag features.
                cfg.numeric_feature_cols = list(selected_features)
                cfg.categorical_feature_cols = []
            elif cfg.model_family == "lstm":
                cfg.numeric_feature_cols = [
                    cfg.target_col,
                    *selected_features
                ]
                cfg.categorical_feature_cols = []
            elif cfg.model_family == "tft":
                cfg.historical_feature_cols = [
                    cfg.target_col,
                    *selected_features
                ]

        cfg.output_dir = os.path.join(
            PERCENTILE_SOURCE_OUTPUT,
            cfg.model_family.upper(),
            config_name
        )
        configurations.append((cfg, config_name, architecture_source))

    return configurations


def run_all_percentile_source_experiments(df):
    # Train every ANN, LSTM and TFT source configuration for every season.
    # Intermediate feature subsets reuse the matching FullModel winner's
    # architecture and training length, then save as distinct artifacts.
    configurations = make_percentile_source_configurations()
    seasons = ["Winter", "Spring", "Summer", "Autumn"]
    result_rows = []

    for cfg, config_name, architecture_source in configurations:
        for season in seasons:
            raw_configuration_path = os.path.join(
                SAVED_MODEL_DIR,
                "DeepLearning",
                cfg.model_family.upper(),
                architecture_source,
                season,
                "winning_configuration.json"
            )
            if not os.path.exists(raw_configuration_path):
                print(
                    "Percentile source model skipped because its architecture "
                    "source winner has not been saved:",
                    cfg.model_family,
                    config_name,
                    season,
                    "required winner:",
                    architecture_source
                )
                continue

            try:
                # Load architecture metadata from Univariate or FullModel while
                # saving the trained feature-subset artifact under config_name.
                winning_row = load_raw_winning_configuration(
                    cfg.model_family,
                    architecture_source,
                    season
                )
                restricted_cfg = restrict_config_to_saved_winner(
                    cfg,
                    winning_row
                )

                # run_percentile_source_experiment normally loads metadata by
                # config name. Temporarily provide the architecture source as
                # an explicit attribute consumed by the updated loader below.
                restricted_cfg.percentile_architecture_source = (
                    architecture_source
                )
                result = run_percentile_source_experiment(
                    df=df,
                    cfg=restricted_cfg,
                    config_name=config_name,
                    season=season
                )
                result_rows.append(result["metrics"])
            except Exception as error:
                print(
                    "Percentile source model failed:",
                    cfg.model_family,
                    config_name,
                    season,
                    "-",
                    error
                )

    results = pd.DataFrame(result_rows)
    results.to_csv(
        os.path.join(
            PERCENTILE_SOURCE_OUTPUT,
            "all_alexander_percentile_source_metrics.csv"
        ),
        index=False
    )
    return results


## Run percentile-normalized Alexander Orr source models

Run the raw-scale seasonal configuration first and save its winner. Then run
the matching percentile experiment below. Each call trains a new model using
the same selected architecture and split, while fitting percentile mappings on
that model's training window only.


In [ ]:
# Run every percentile-normalized source configuration.
#
# Configurations saved separately for ANN, LSTM and TFT:
# Univariate, Turbidity, Flowrate, Turbidity_Flowrate and FullModel.
# Each configuration runs for Winter, Spring, Summer and Autumn whenever its
# required raw Univariate or FullModel seasonal winner is available.
RUN_ALL_PERCENTILE_SOURCE_EXPERIMENTS = True

if RUN_ALL_PERCENTILE_SOURCE_EXPERIMENTS:
    all_percentile_source_results = (
        run_all_percentile_source_experiments(df)
    )
    display(all_percentile_source_results)

#Re-run last TFT portion

In [ ]:
# Rerun only:
# TFT Turbidity_Flowrate and TFT FullModel
# for Winter, Spring, Summer and Autumn.

RUN_SELECTED_PERCENTILE_SOURCE_EXPERIMENTS = True

SELECTED_MODEL_FAMILY = "tft"

SELECTED_CONFIGURATIONS = [
    "Turbidity_Flowrate",
    "FullModel"
]

SELECTED_SEASONS = [
    "Winter",
    "Spring",
    "Summer",
    "Autumn"
]

if RUN_SELECTED_PERCENTILE_SOURCE_EXPERIMENTS:

    available_configurations = (
        make_percentile_source_configurations()
    )

    selected_result_rows = []

    for (
        cfg,
        config_name,
        architecture_source
    ) in available_configurations:

        # Skip ANN and LSTM.
        if cfg.model_family.lower() != SELECTED_MODEL_FAMILY:
            continue

        # Skip Univariate, Turbidity-only and Flowrate-only.
        if config_name not in SELECTED_CONFIGURATIONS:
            continue

        for season in SELECTED_SEASONS:

            print("\n" + "=" * 80)
            print(
                f"RUNNING: {cfg.model_family.upper()} | "
                f"{config_name} | {season}"
            )
            print("=" * 80)

            raw_configuration_path = os.path.join(
                SAVED_MODEL_DIR,
                "DeepLearning",
                cfg.model_family.upper(),
                architecture_source,
                season,
                "winning_configuration.json"
            )

            if not os.path.exists(raw_configuration_path):
                print(
                    "SKIPPED — required raw winner not found:",
                    raw_configuration_path
                )
                continue

            try:
                winning_row = load_raw_winning_configuration(
                    cfg.model_family,
                    architecture_source,
                    season
                )

                restricted_cfg = restrict_config_to_saved_winner(
                    cfg,
                    winning_row
                )

                restricted_cfg.percentile_architecture_source = (
                    architecture_source
                )

                result = run_percentile_source_experiment(
                    df=df,
                    cfg=restricted_cfg,
                    config_name=config_name,
                    season=season
                )

                selected_result_rows.append(
                    result["metrics"]
                )

                print(
                    f"COMPLETED: {cfg.model_family.upper()} | "
                    f"{config_name} | {season}"
                )

            except Exception as error:
                print(
                    f"FAILED: {cfg.model_family.upper()} | "
                    f"{config_name} | {season}"
                )
                print("Reason:", error)

    selected_percentile_source_results = pd.DataFrame(
        selected_result_rows
    )

    display(selected_percentile_source_results)

    selected_percentile_source_results.to_csv(
        os.path.join(
            PERCENTILE_SOURCE_OUTPUT,
            "rerun_tft_turbidity_flowrate_fullmodel_metrics.csv"
        ),
        index=False
    )

#Statistical Comparison of Models with Hypothesis Testing-Friedman

#Compares Models per Season

In [ ]:
BASE_PATH = Path("/content/drive/MyDrive/Forecasting_AlexanderOrr")
DL_PATH = BASE_PATH / "DeepLearning"
BASELINE_PATH = BASE_PATH / "Baselines"
OUTPUT_PATH = BASE_PATH / "Statistical_Comparisons"

In [ ]:
DATE_COL = "date"
RESIDUAL_COL = "residual"
ALPHA = 0.05

SEASONS = ["Winter", "Spring", "Summer", "Autumn"]
DL_MODELS = ["ANN", "LSTM", "TFT"]
EXPONENTIAL_SMOOTHING_MODELS = [
    "SES",
    "DES",
    "TES"
]

ARIMA_FAMILY_MODELS = [
    "ARIMA",
    "SARIMA",
    "SARIMAX"
]

if not BASE_PATH.exists():
    raise FileNotFoundError(
        f"Drive folder was not found: {BASE_PATH}\n"
        "Confirm that Google Drive is mounted and that BASE_PATH is correct."
    )

print(f"Residual output root: {BASE_PATH}")

In [ ]:
# Comparison labels used in the output tables
DL_CONFIG_LABELS = {
    "DL_univariate": "Residual Chlorine",
    "DL_flowrate": "Residual Chlorine + Flowrate",
    "DL_turbidity": "Residual Chlorine + Turbidity",
    "DL_flowrate_turbidity": "Residual Chlorine + Flowrate + Turbidity",
    "DL_full_model": (
        "Residual Chlorine + Flowrate + Turbidity + Chlorine Conversion"
    ),
}

DL_CONFIG_PURPOSES = {
    "DL_univariate": "Compare ANN, LSTM, and TFT univariate models",
    "DL_flowrate": "Compare ANN, LSTM, and TFT flowrate models",
    "DL_turbidity": "Compare ANN, LSTM, and TFT turbidity models",
    "DL_flowrate_turbidity": (
        "Compare ANN, LSTM, and TFT flowrate-plus-turbidity models"
    ),
    "DL_full_model": "Compare ANN, LSTM, and TFT full multivariate models",
}

DL_GROUPS = list(DL_CONFIG_LABELS.keys())

In [ ]:
# Automatically discover and classify DL residual CSV files

def residual_csvs(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(
        p for p in folder.rglob("*.csv")
        if "residual" in p.name.lower()
    )


def identify_season_fr(path):
    text = str(path).lower()
    for season in SEASONS:
        if season.lower() in text:
            return season
    return None


def identify_dl_group(path):
    text = str(path).lower().replace("-", "_").replace(" ", "_")

    # Test the most specific multivariate categories first.
    if any(token in text for token in [
        "full_model", "fullmodel", "model4", "chlorine_conversion"
    ]):
        return "DL_full_model"

    if any(token in text for token in [
        "flowrate_turbidity", "turbidity_flowrate", "flow_turb",
        "model3"
    ]) or ("flowrate" in text and "turbidity" in text):
        return "DL_flowrate_turbidity"

    if any(token in text for token in ["model1", "flowrate_only"]):
        return "DL_flowrate"

    if any(token in text for token in ["model2", "turbidity_only"]):
        return "DL_turbidity"

    if "flowrate" in text and "turbidity" not in text:
        return "DL_flowrate"

    if "turbidity" in text and "flowrate" not in text:
        return "DL_turbidity"

    if any(token in text for token in [
        "univariate", "_uni_", "/uni/", "\\uni\\"
    ]):
        return "DL_univariate"

    # Existing univariate filenames generally contain no multivariate token.
    return "DL_univariate"


def file_priority(path):
    """Prefer final/winning holdout files when duplicates exist."""
    name = path.name.lower()
    priority_tokens = [
        "true_holdout", "final_holdout", "winning", "winner",
        "best", "fixed_holdout", "holdout", "residuals"
    ]
    score = sum((len(priority_tokens) - i) for i, token in enumerate(priority_tokens)
                if token in name)
    return (score, path.stat().st_mtime, path.name)


def discover_dl_file_specs():
    candidates = []

    for model in DL_MODELS:
        model_folder = DL_PATH / model

        for path in residual_csvs(model_folder):
            season = identify_season_fr(path)
            if season is None:
                print(f"Skipped DL file with no season in its name/path: {path}")
                continue

            group = identify_dl_group(path)
            candidates.append({
                "path": path,
                "model": model,
                "model_class": "DL",
                "input_type": (
                    "univariate" if group == "DL_univariate" else "multivariate"
                ),
                "season": season,
                "comparison_group": group,
                "predictors": DL_CONFIG_LABELS[group],
                "purpose": DL_CONFIG_PURPOSES[group],
            })

    # Keep one residual file for each architecture/category/season slot.
    selected = []
    grouped = {}
    for item in candidates:
        key = (item["comparison_group"], item["season"], item["model"])
        grouped.setdefault(key, []).append(item)

    for key, items in sorted(grouped.items()):
        items = sorted(items, key=lambda x: file_priority(x["path"]), reverse=True)
        selected.append(items[0])

        if len(items) > 1:
            print(f"Multiple files found for {key}. Selected: {items[0]['path'].name}")
            for ignored in items[1:]:
                print(f"  Ignored duplicate candidate: {ignored['path'].name}")

    return selected


dl_file_specs = discover_dl_file_specs()
print(f"Discovered {len(dl_file_specs)} DL residual files for comparison.")

In [ ]:
# Discover baseline residual files recursively from the shared Baselines folder.

def identify_baseline_family(path):
    """Identify the baseline family from its filename and folder path."""

    text = (
        str(path)
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    # Check the most specific ARIMA names first.
    if "sarimax" in text:
        return "SARIMAX"

    if "sarima" in text:
        return "SARIMA"

    if "arima" in text:
        return "ARIMA"

    # Holt-Winters is the triple exponential-smoothing model.
    if any(
        token in text
        for token in [
            "holt_winters",
            "holtwinters",
            "triple_exponential",
            "tes"
        ]
    ):
        return "TES"

    if any(
        token in text
        for token in [
            "double_exponential",
            "des"
        ]
    ):
        return "DES"

    if any(
        token in text
        for token in [
            "simple_exponential",
            "ses"
        ]
    ):
        return "SES"

    return None


def identify_sarimax_configuration(path):
    """Identify the SARIMAX predictor configuration from its path."""

    text = (
        str(path)
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    # Test the most specific configuration first.
    if any(
        token in text
        for token in [
            "flowrate_turbidity",
            "turbidity_flowrate",
            "flow_turb"
        ]
    ):
        return "Flowrate + Turbidity"

    if any(
        token in text
        for token in [
            "full_model",
            "fullmodel",
            "all_predictors",
            "allvariables"
        ]
    ):
        return "Full Model"

    if "flowrate" in text and "turbidity" not in text:
        return "Flowrate"

    if "turbidity" in text and "flowrate" not in text:
        return "Turbidity"

    if "chlorine_conversion" in text:
        return "Chlorine Conversion"

    return "Unspecified"


def discover_baseline_file_specs():
    """Discover fixed-holdout baseline residual files recursively."""

    all_files = residual_csvs(BASELINE_PATH)
    candidates = []

    for path in all_files:
        family = identify_baseline_family(path)

        if family is None:
            print(
                "Skipped unidentified baseline residual file:",
                path
            )
            continue

        if family in EXPONENTIAL_SMOOTHING_MODELS:
            candidates.append({
                "path": path,
                "model": family,
                "model_class": "Exponential Smoothing",
                "input_type": "fixed_holdout",
                "season": "Hard Partition",
                "comparison_group": "exponential_smoothing",
                "predictors": (
                    "Residual-chlorine history"
                ),
                "purpose": (
                    "Compare SES, DES, and TES on the common "
                    "fixed holdout"
                ),
            })

        elif family in ARIMA_FAMILY_MODELS:
            if family == "SARIMAX":
                configuration = (
                    identify_sarimax_configuration(path)
                )

                model_label = (
                    f"SARIMAX — {configuration}"
                )

                predictors = configuration

            else:
                model_label = family
                predictors = (
                    "Residual-chlorine history"
                )

            candidates.append({
                "path": path,
                "model": model_label,
                "model_class": "ARIMA Family",
                "input_type": "fixed_holdout",
                "season": "Hard Partition",
                "comparison_group": "arima_family",
                "predictors": predictors,
                "purpose": (
                    "Compare ARIMA, SARIMA, and SARIMAX "
                    "specifications on the common fixed holdout"
                ),
            })

    # Keep one file for every unique comparison-group/model slot.
    grouped = {}

    for item in candidates:
        key = (
            item["comparison_group"],
            item["model"]
        )
        grouped.setdefault(key, []).append(item)

    selected = []

    for key, items in sorted(grouped.items()):
        ranked = sorted(
            items,
            key=lambda item: file_priority(item["path"]),
            reverse=True
        )

        selected.append(
            ranked[0]
        )

        if len(ranked) > 1:
            print(
                f"Multiple files found for {key}. "
                f"Selected: {ranked[0]['path']}"
            )

            for ignored in ranked[1:]:
                print(
                    "  Ignored duplicate:",
                    ignored["path"]
                )

    return selected


baseline_file_specs = discover_baseline_file_specs()

print(
    f"Discovered {len(baseline_file_specs)} "
    "fixed-holdout baseline residual files."
)

In [ ]:
# Combined file table: only files that currently exist are included
file_specs = pd.DataFrame(
    dl_file_specs + baseline_file_specs
)

if file_specs.empty:
    raise ValueError(
        "No residual CSV files were discovered under Forecasting_AlexanderOrr."
    )

file_specs["exists"] = file_specs["path"].apply(lambda p: Path(p).exists())

print("FILE CHECK")
print("----------")
print(
    file_specs[[
        "comparison_group", "season", "model", "exists", "path"
    ]].to_string(index=False)
)

In [ ]:
# Load all currently available residual files

def read_residual_file(path):
    path = Path(path)

    if path.suffix.lower() in [
        ".xlsx",
        ".xls"
    ]:
        return pd.read_excel(path)

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)

    raise ValueError(
        f"Unsupported file type: {path}"
    )


def standardize_residual_columns(df, path):
    """Standardize timestamp and residual names across saved outputs."""

    df = df.copy()

    timestamp_candidates = [
        "date",
        "Date",
        "timestamp_utc",
        "timestamp",
        "Timestamp",
        "forecast_timestamp",
        "ForecastTimestamp",
        "target_timestamp",
        "TargetTimestamp",
        "datetime",
        "Datetime",
        "index",
        "Unnamed: 0"
    ]

    residual_candidates = [
        "residual",
        "Residual",
        "error",
        "Error",
        "forecast_error"
    ]

    timestamp_column = next(
        (
            column
            for column in timestamp_candidates
            if column in df.columns
        ),
        None
    )

    residual_column = next(
        (
            column
            for column in residual_candidates
            if column in df.columns
        ),
        None
    )

    # If the residual was not saved directly, reconstruct it when possible.
    actual_candidates = [
        "actual",
        "Actual",
        "observed",
        "y_true"
    ]

    forecast_candidates = [
        "forecast",
        "Forecast",
        "predicted",
        "prediction",
        "y_pred"
    ]

    actual_column = next(
        (
            column
            for column in actual_candidates
            if column in df.columns
        ),
        None
    )

    forecast_column = next(
        (
            column
            for column in forecast_candidates
            if column in df.columns
        ),
        None
    )

    if (
        residual_column is None
        and actual_column is not None
        and forecast_column is not None
    ):
        df["residual"] = (
            pd.to_numeric(
                df[actual_column],
                errors="coerce"
            )
            - pd.to_numeric(
                df[forecast_column],
                errors="coerce"
            )
        )

        residual_column = "residual"

    missing = []

    if timestamp_column is None:
        missing.append(
            "recognizable timestamp column"
        )

    if residual_column is None:
        missing.append(
            "residual or actual/forecast columns"
        )

    if missing:
        print(
            f"Skipped {path.name}; missing: {missing}. "
            f"Available columns: {df.columns.tolist()}"
        )
        return None

    df = df.rename(
        columns={
            timestamp_column: DATE_COL,
            residual_column: RESIDUAL_COL
        }
    )

    return df


def load_all_residuals(file_specs):
    dfs = []

    for _, row in file_specs.iterrows():
        path = Path(
            row["path"]
        )

        if not path.exists():
            print(
                f"Skipped missing file: {path}"
            )
            continue

        df = read_residual_file(
            path
        )

        df = standardize_residual_columns(
            df,
            path
        )

        if df is None:
            continue

        temp = df[
            [
                DATE_COL,
                RESIDUAL_COL
            ]
        ].copy()

        temp[DATE_COL] = (
            pd.to_datetime(
                temp[DATE_COL],
                utc=True,
                errors="coerce"
            )
            .dt.tz_localize(None)
        )

        temp["residual"] = pd.to_numeric(
            temp[RESIDUAL_COL],
            errors="coerce"
        )

        temp["absolute_residual"] = (
            temp["residual"].abs()
        )

        metadata_cols = [
            "model",
            "model_class",
            "input_type",
            "season",
            "comparison_group",
            "predictors",
            "purpose"
        ]

        for column in metadata_cols:
            temp[column] = row[column]

        temp["source_file"] = str(
            path
        )

        temp = temp.dropna(
            subset=[
                DATE_COL,
                "absolute_residual"
            ]
        )

        if temp.empty:
            print(
                "Skipped empty or unusable residual file:",
                path
            )
            continue

        dfs.append(
            temp
        )

        print(
            f"Loaded: {row['comparison_group']} | "
            f"{row['model']} | "
            f"{len(temp):,} rows"
        )

    if not dfs:
        raise ValueError(
            "No usable residual files were loaded."
        )

    return pd.concat(
        dfs,
        ignore_index=True
    )


residuals_long = load_all_residuals(
    file_specs
)

In [ ]:
# Existing Friedman + Wilcoxon/Holm statistical-test logic

def holm_correction(p_values):
    p_values = np.asarray(p_values)
    order = np.argsort(p_values)
    sorted_p = p_values[order]
    m = len(sorted_p)

    adjusted_sorted = np.array([
        min((m - i) * p, 1.0)
        for i, p in enumerate(sorted_p)
    ])
    adjusted_sorted = np.maximum.accumulate(adjusted_sorted)

    adjusted = np.empty(m)
    adjusted[order] = adjusted_sorted
    return adjusted


def run_friedman_wilcoxon(data, comparison_group, season=None, alpha=0.05):
    df = data[data["comparison_group"] == comparison_group].copy()

    if season is not None:
        df = df[df["season"] == season].copy()
        comparison_label = f"{comparison_group}_{season}"
        season_label = season
    else:
        comparison_label = f"{comparison_group}_Overall"
        season_label = "Overall"

    models = sorted(df["model"].unique())

    # Friedman requires at least three related groups.
    if len(models) < 3:
        print(
            f"Skipping {comparison_label}: only {len(models)} available model(s); "
            "Friedman requires at least 3."
        )
        return None, None

    wide = (
        df.pivot_table(
            index=DATE_COL,
            columns="model",
            values="absolute_residual",
            aggfunc="mean"
        )
        .dropna(subset=models)
    )

    if wide.empty:
        print(f"Skipping {comparison_label}: no paired timestamps.")
        return None, None

    friedman_stat, friedman_p = friedmanchisquare(
        *[wide[model] for model in models]
    )

    predictors = df["predictors"].iloc[0]
    purpose = df["purpose"].iloc[0]

    friedman_result = pd.DataFrame([{
        "comparison": comparison_label,
        "comparison_group": comparison_group,
        "season": season_label,
        "predictors": predictors,
        "purpose": purpose,
        "models_compared": ", ".join(models),
        "n_models": len(models),
        "n_paired_observations": len(wide),
        "friedman_statistic": friedman_stat,
        "friedman_p_value": friedman_p,
        "significant_at_0_05": friedman_p < alpha
    }])

    posthoc_result = None

    if friedman_p < alpha:
        rows = []

        for model_1, model_2 in combinations(models, 2):
            differences = wide[model_1] - wide[model_2]

            if np.allclose(differences, 0):
                statistic, p_value = 0.0, 1.0
            else:
                statistic, p_value = wilcoxon(
                    wide[model_1],
                    wide[model_2],
                    zero_method="wilcox",
                    alternative="two-sided"
                )

            rows.append({
                "comparison": comparison_label,
                "comparison_group": comparison_group,
                "season": season_label,
                "predictors": predictors,
                "pairwise_test": f"{model_1} vs {model_2}",
                "model_1": model_1,
                "model_2": model_2,
                "wilcoxon_statistic": statistic,
                "raw_p_value": p_value
            })

        posthoc_result = pd.DataFrame(rows)
        posthoc_result["holm_adjusted_p"] = holm_correction(
            posthoc_result["raw_p_value"]
        )
        posthoc_result["significant_after_holm"] = (
            posthoc_result["holm_adjusted_p"] < alpha
        )
        posthoc_result = posthoc_result.sort_values(
            ["holm_adjusted_p", "raw_p_value"]
        ).reset_index(drop=True)

    return friedman_result, posthoc_result

In [ ]:
# Run three families of statistical comparisons.

all_friedman = []
all_posthoc = []

# 1. Deep-learning comparisons
# ANN versus LSTM versus TFT within the same predictor
# configuration and seasonal holdout.

for group in DL_GROUPS:
    for season in SEASONS:

        friedman_result, posthoc_result = (
            run_friedman_wilcoxon(
                residuals_long,
                comparison_group=group,
                season=season,
                alpha=ALPHA
            )
        )

        if friedman_result is not None:
            all_friedman.append(
                friedman_result
            )

        if posthoc_result is not None:
            all_posthoc.append(
                posthoc_result
            )


# 2. Exponential-smoothing comparison
# SES versus DES versus TES on their common fixed holdout.


friedman_result, posthoc_result = (
    run_friedman_wilcoxon(
        residuals_long,
        comparison_group="exponential_smoothing",
        season="Hard Partition",
        alpha=ALPHA
    )
)

if friedman_result is not None:
    all_friedman.append(
        friedman_result
    )

if posthoc_result is not None:
    all_posthoc.append(
        posthoc_result
    )


# 3. ARIMA-family comparison
# ARIMA, SARIMA and available SARIMAX configurations on
# their common fixed-holdout timestamps.

friedman_result, posthoc_result = (
    run_friedman_wilcoxon(
        residuals_long,
        comparison_group="arima_family",
        season="Hard Partition",
        alpha=ALPHA
    )
)

if friedman_result is not None:
    all_friedman.append(
        friedman_result
    )

if posthoc_result is not None:
    all_posthoc.append(
        posthoc_result
    )


friedman_summary = (
    pd.concat(
        all_friedman,
        ignore_index=True
    )
    if all_friedman
    else pd.DataFrame()
)

posthoc_summary = (
    pd.concat(
        all_posthoc,
        ignore_index=True
    )
    if all_posthoc
    else pd.DataFrame()
)

print("\nFRIEDMAN SUMMARY")
print(
    friedman_summary
    if not friedman_summary.empty
    else "No eligible comparison."
)

print("\nWILCOXON/HOLM POST HOC SUMMARY")
print(
    posthoc_summary
    if not posthoc_summary.empty
    else "No significant Friedman result."
)


In [ ]:
# Create descriptive residual summary before selecting best models.

descriptive_summary = (
    residuals_long
    .groupby(
        [
            "comparison_group",
            "season",
            "predictors",
            "purpose",
            "model",
            "model_class",
            "input_type"
        ],
        dropna=False
    )["absolute_residual"]
    .agg(
        n="count",
        mean_absolute_residual="mean",
        median_absolute_residual="median",
        std_absolute_residual="std",
        min_absolute_residual="min",
        max_absolute_residual="max"
    )
    .reset_index()
    .sort_values(
        [
            "comparison_group",
            "season",
            "mean_absolute_residual"
        ]
    )
    .reset_index(drop=True)
)

print("\nDESCRIPTIVE RESIDUAL SUMMARY")
display(descriptive_summary)

In [ ]:
#Best models
best_models = (
    descriptive_summary
    .sort_values([
        "comparison_group",
        "season",
        "mean_absolute_residual"
    ])
    .groupby(["comparison_group", "season"])
    .head(1)
    .reset_index(drop=True)
)

In [ ]:
# Save statistical-comparison outputs back to Google Drive
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

friedman_summary.to_csv(
    OUTPUT_PATH / "friedman_summary.csv",
    index=False
)
posthoc_summary.to_csv(
    OUTPUT_PATH / "wilcoxon_posthoc_summary.csv",
    index=False
)
descriptive_summary.to_csv(
    OUTPUT_PATH / "absolute_residual_descriptive_summary.csv",
    index=False
)
best_models.to_csv(
    OUTPUT_PATH / "best_models_by_comparison.csv",
    index=False
)
file_specs.to_csv(
    OUTPUT_PATH / "files_used_for_statistical_comparisons.csv",
    index=False
)

print(f"Saved outputs to: {OUTPUT_PATH}")

#Compares Models per Univariate/Multivariable Configuration

In [ ]:

# ADDITIONAL FRIEDMAN ANALYSIS
# UNIVARIATE VERSUS MULTIVARIABLE CONFIGURATIONS

DL_CONFIGURATION_ORDER = [
    "DL_univariate",
    "DL_turbidity",
    "DL_flowrate",
    "DL_flowrate_turbidity",
    "DL_full_model"
]

DL_CONFIGURATION_NAMES = {
    "DL_univariate": "Univariate",
    "DL_turbidity": "Turbidity",
    "DL_flowrate": "Flowrate",
    "DL_flowrate_turbidity": (
        "Flowrate + Turbidity"
    ),
    "DL_full_model": "Full Model"
}


def run_configuration_friedman(
    data,
    architecture,
    season,
    alpha=0.05
):
    """
    Compare univariate and multivariable configurations
    for one DL architecture and one seasonal holdout.
    """

    subset = data.loc[
        (data["model"] == architecture)
        & (data["season"] == season)
        & (
            data["comparison_group"]
            .isin(DL_CONFIGURATION_ORDER)
        )
    ].copy()

    available_configurations = [
        configuration
        for configuration in DL_CONFIGURATION_ORDER
        if configuration
        in subset["comparison_group"].unique()
    ]

    comparison_name = (
        f"{architecture}_{season}_"
        "Predictor_Configurations"
    )

    if len(available_configurations) < 3:
        print(
            f"Skipping {comparison_name}: "
            f"only {len(available_configurations)} "
            "configurations are available."
        )

        return None, None, None

    wide = (
        subset
        .pivot_table(
            index=DATE_COL,
            columns="comparison_group",
            values="absolute_residual",
            aggfunc="mean"
        )
        .dropna(
            subset=available_configurations
        )
    )

    if wide.empty:
        print(
            f"Skipping {comparison_name}: "
            "no common paired timestamps."
        )

        return None, None, None

    friedman_statistic, friedman_p_value = (
        friedmanchisquare(
            *[
                wide[configuration].to_numpy()
                for configuration
                in available_configurations
            ]
        )
    )

    friedman_result = pd.DataFrame([
        {
            "comparison": comparison_name,
            "architecture": architecture,
            "season": season,
            "comparison_type": (
                "Univariate versus "
                "multivariable configurations"
            ),
            "configurations_compared": ", ".join(
                DL_CONFIGURATION_NAMES[
                    configuration
                ]
                for configuration
                in available_configurations
            ),
            "n_configurations": len(
                available_configurations
            ),
            "n_paired_observations": len(wide),
            "friedman_statistic": (
                friedman_statistic
            ),
            "friedman_p_value": (
                friedman_p_value
            ),
            "significant_at_0_05": (
                friedman_p_value < alpha
            )
        }
    ])

    descriptive_rows = []

    for configuration in available_configurations:
        configuration_values = wide[
            configuration
        ]

        descriptive_rows.append(
            {
                "comparison": comparison_name,
                "architecture": architecture,
                "season": season,
                "configuration": (
                    DL_CONFIGURATION_NAMES[
                        configuration
                    ]
                ),
                "configuration_code": configuration,
                "n_paired_observations": len(
                    configuration_values
                ),
                "mean_absolute_residual": (
                    configuration_values.mean()
                ),
                "median_absolute_residual": (
                    configuration_values.median()
                ),
                "std_absolute_residual": (
                    configuration_values.std()
                )
            }
        )

    descriptive_result = pd.DataFrame(
        descriptive_rows
    ).sort_values(
        "mean_absolute_residual"
    )

    posthoc_result = None

    if friedman_p_value < alpha:
        pairwise_rows = []

        configuration_pairs = combinations(
            available_configurations,
            2
        )

        for configuration_1, configuration_2 in (
            configuration_pairs
        ):
            values_1 = wide[
                configuration_1
            ]

            values_2 = wide[
                configuration_2
            ]

            differences = (
                values_1 - values_2
            )

            if np.allclose(
                differences,
                0
            ):
                wilcoxon_statistic = 0.0
                raw_p_value = 1.0

            else:
                (
                    wilcoxon_statistic,
                    raw_p_value
                ) = wilcoxon(
                    values_1,
                    values_2,
                    zero_method="wilcox",
                    alternative="two-sided"
                )

            pairwise_rows.append(
                {
                    "comparison": comparison_name,
                    "architecture": architecture,
                    "season": season,
                    "configuration_1": (
                        DL_CONFIGURATION_NAMES[
                            configuration_1
                        ]
                    ),
                    "configuration_2": (
                        DL_CONFIGURATION_NAMES[
                            configuration_2
                        ]
                    ),
                    "wilcoxon_statistic": (
                        wilcoxon_statistic
                    ),
                    "raw_p_value": raw_p_value
                }
            )

        posthoc_result = pd.DataFrame(
            pairwise_rows
        )

        posthoc_result[
            "holm_adjusted_p"
        ] = holm_correction(
            posthoc_result[
                "raw_p_value"
            ].to_numpy()
        )

        posthoc_result[
            "significant_after_holm"
        ] = (
            posthoc_result[
                "holm_adjusted_p"
            ]
            < alpha
        )

        posthoc_result = (
            posthoc_result
            .sort_values(
                [
                    "holm_adjusted_p",
                    "raw_p_value"
                ]
            )
            .reset_index(drop=True)
        )

    return (
        friedman_result,
        posthoc_result,
        descriptive_result
    )


configuration_friedman_results = []
configuration_posthoc_results = []
configuration_descriptive_results = []


for architecture in DL_MODELS:
    for season in SEASONS:
        (
            friedman_result,
            posthoc_result,
            descriptive_result
        ) = run_configuration_friedman(
            data=residuals_long,
            architecture=architecture,
            season=season,
            alpha=ALPHA
        )

        if friedman_result is not None:
            configuration_friedman_results.append(
                friedman_result
            )

        if posthoc_result is not None:
            configuration_posthoc_results.append(
                posthoc_result
            )

        if descriptive_result is not None:
            configuration_descriptive_results.append(
                descriptive_result
            )


configuration_friedman_summary = (
    pd.concat(
        configuration_friedman_results,
        ignore_index=True
    )
    if configuration_friedman_results
    else pd.DataFrame()
)

configuration_posthoc_summary = (
    pd.concat(
        configuration_posthoc_results,
        ignore_index=True
    )
    if configuration_posthoc_results
    else pd.DataFrame()
)

configuration_descriptive_summary = (
    pd.concat(
        configuration_descriptive_results,
        ignore_index=True
    )
    if configuration_descriptive_results
    else pd.DataFrame()
)


print(
    "\nUNIVARIATE VERSUS MULTIVARIABLE "
    "FRIEDMAN RESULTS"
)

display(
    configuration_friedman_summary
)

print(
    "\nWILCOXON POST HOC RESULTS"
)

if configuration_posthoc_summary.empty:
    print(
        "No post hoc results were required."
    )
else:
    display(
        configuration_posthoc_summary
    )

print(
    "\nPAIRED CONFIGURATION DESCRIPTIVE SUMMARY"
)

display(
    configuration_descriptive_summary
)

In [ ]:
# SARIMA VERSUS SARIMAX PREDICTOR CONFIGURATIONS


SARIMAX_COMPARISON_MODELS = [
    "SARIMA",
    "SARIMAX — Turbidity",
    "SARIMAX — Flowrate",
    "SARIMAX — Flowrate + Turbidity",
    "SARIMAX — Full Model"
]


def run_sarima_sarimax_configuration_friedman(
    data,
    alpha=0.05
):
    """
    Compare SARIMA with SARIMAX predictor configurations
    using absolute residuals from their common fixed-holdout
    timestamps.
    """

    subset = data.loc[
        (data["comparison_group"] == "arima_family")
        & (data["season"] == "Hard Partition")
        & (
            data["model"].isin(
                SARIMAX_COMPARISON_MODELS
            )
        )
    ].copy()

    available_models = [
        model
        for model in SARIMAX_COMPARISON_MODELS
        if model in subset["model"].unique()
    ]

    comparison_name = (
        "SARIMA_vs_SARIMAX_"
        "Predictor_Configurations"
    )

    print(
        "Baseline configurations found:",
        available_models
    )

    missing_models = [
        model
        for model in SARIMAX_COMPARISON_MODELS
        if model not in available_models
    ]

    if missing_models:
        print(
            "Baseline configurations not found:",
            missing_models
        )

    if len(available_models) < 3:
        print(
            f"Skipping {comparison_name}: "
            f"only {len(available_models)} "
            "eligible configurations were found."
        )

        return None, None, None

    wide = (
        subset
        .pivot_table(
            index=DATE_COL,
            columns="model",
            values="absolute_residual",
            aggfunc="mean"
        )
        .dropna(
            subset=available_models
        )
    )

    if wide.empty:
        print(
            f"Skipping {comparison_name}: "
            "the configurations have no common "
            "paired holdout timestamps."
        )

        return None, None, None

    friedman_statistic, friedman_p_value = (
        friedmanchisquare(
            *[
                wide[model].to_numpy()
                for model in available_models
            ]
        )
    )

    friedman_result = pd.DataFrame([
        {
            "comparison": comparison_name,
            "architecture": (
                "SARIMA and SARIMAX"
            ),
            "season": "Hard Partition",
            "comparison_type": (
                "Univariate seasonal model versus "
                "SARIMAX predictor configurations"
            ),
            "configurations_compared": ", ".join(
                available_models
            ),
            "n_configurations": len(
                available_models
            ),
            "n_paired_observations": len(wide),
            "friedman_statistic": (
                friedman_statistic
            ),
            "friedman_p_value": (
                friedman_p_value
            ),
            "significant_at_0_05": (
                friedman_p_value < alpha
            )
        }
    ])

    descriptive_rows = []

    for model in available_models:
        model_values = wide[model]

        descriptive_rows.append(
            {
                "comparison": comparison_name,
                "architecture": (
                    "SARIMA and SARIMAX"
                ),
                "season": "Hard Partition",
                "configuration": model,
                "configuration_code": model,
                "n_paired_observations": len(
                    model_values
                ),
                "mean_absolute_residual": (
                    model_values.mean()
                ),
                "median_absolute_residual": (
                    model_values.median()
                ),
                "std_absolute_residual": (
                    model_values.std()
                )
            }
        )

    descriptive_result = (
        pd.DataFrame(
            descriptive_rows
        )
        .sort_values(
            "mean_absolute_residual"
        )
        .reset_index(drop=True)
    )

    posthoc_result = None

    if friedman_p_value < alpha:
        pairwise_rows = []

        for model_1, model_2 in combinations(
            available_models,
            2
        ):
            values_1 = wide[model_1]
            values_2 = wide[model_2]

            differences = (
                values_1 - values_2
            )

            if np.allclose(
                differences,
                0
            ):
                wilcoxon_statistic = 0.0
                raw_p_value = 1.0

            else:
                (
                    wilcoxon_statistic,
                    raw_p_value
                ) = wilcoxon(
                    values_1,
                    values_2,
                    zero_method="wilcox",
                    alternative="two-sided"
                )

            pairwise_rows.append(
                {
                    "comparison": comparison_name,
                    "architecture": (
                        "SARIMA and SARIMAX"
                    ),
                    "season": "Hard Partition",
                    "configuration_1": model_1,
                    "configuration_2": model_2,
                    "wilcoxon_statistic": (
                        wilcoxon_statistic
                    ),
                    "raw_p_value": raw_p_value
                }
            )

        posthoc_result = pd.DataFrame(
            pairwise_rows
        )

        posthoc_result[
            "holm_adjusted_p"
        ] = holm_correction(
            posthoc_result[
                "raw_p_value"
            ].to_numpy()
        )

        posthoc_result[
            "significant_after_holm"
        ] = (
            posthoc_result[
                "holm_adjusted_p"
            ]
            < alpha
        )

        posthoc_result = (
            posthoc_result
            .sort_values(
                [
                    "holm_adjusted_p",
                    "raw_p_value"
                ]
            )
            .reset_index(drop=True)
        )

    return (
        friedman_result,
        posthoc_result,
        descriptive_result
    )


(
    baseline_configuration_friedman,
    baseline_configuration_posthoc,
    baseline_configuration_descriptive
) = run_sarima_sarimax_configuration_friedman(
    data=residuals_long,
    alpha=ALPHA
)


if baseline_configuration_friedman is not None:
    configuration_friedman_summary = (
        pd.concat(
            [
                configuration_friedman_summary,
                baseline_configuration_friedman
            ],
            ignore_index=True
        )
    )


if baseline_configuration_posthoc is not None:
    configuration_posthoc_summary = (
        pd.concat(
            [
                configuration_posthoc_summary,
                baseline_configuration_posthoc
            ],
            ignore_index=True
        )
    )


if baseline_configuration_descriptive is not None:
    configuration_descriptive_summary = (
        pd.concat(
            [
                configuration_descriptive_summary,
                baseline_configuration_descriptive
            ],
            ignore_index=True
        )
    )


print(
    "\nUPDATED UNIVARIATE VERSUS "
    "MULTIVARIABLE FRIEDMAN RESULTS"
)

display(
    configuration_friedman_summary
)


print(
    "\nUPDATED WILCOXON POST HOC RESULTS"
)

if configuration_posthoc_summary.empty:
    print(
        "No post hoc comparisons were required."
    )
else:
    display(
        configuration_posthoc_summary
    )


print(
    "\nUPDATED CONFIGURATION "
    "DESCRIPTIVE SUMMARY"
)

display(
    configuration_descriptive_summary
)

In [ ]:
OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

configuration_friedman_summary.to_csv(
    OUTPUT_PATH
    / (
        "friedman_univariate_vs_"
        "multivariable.csv"
    ),
    index=False
)

configuration_posthoc_summary.to_csv(
    OUTPUT_PATH
    / (
        "wilcoxon_univariate_vs_"
        "multivariable.csv"
    ),
    index=False
)

configuration_descriptive_summary.to_csv(
    OUTPUT_PATH
    / (
        "descriptive_univariate_vs_"
        "multivariable.csv"
    ),
    index=False
)

print(
    "Additional configuration-comparison "
    f"results saved to: {OUTPUT_PATH}"
)

#Model Comparison Metrics: Summary Radar/Spider Plots

In [ ]:
OUTPUT_ROOT = Path("/content/drive/MyDrive/Forecasting_AlexanderOrr")
DEEP_LEARNING_DIR = OUTPUT_ROOT / "DeepLearning"
BASELINES_DIR = OUTPUT_ROOT / "Baselines"

RADAR_OUTPUT_DIR = OUTPUT_ROOT / "Radar_Plots"
RADAR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SARIMAX_DIR = (
    Path("/content/drive/MyDrive/Forecasting_AlexanderOrr")
    / "Baselines"
    / "SARIMAX"
)

if not DEEP_LEARNING_DIR.exists():
    raise FileNotFoundError(f"Deep-learning folder not found: {DEEP_LEARNING_DIR}")

if not BASELINES_DIR.exists():
    raise FileNotFoundError(f"Baselines folder not found: {BASELINES_DIR}")

print(f"Deep-learning metrics folder: {DEEP_LEARNING_DIR}")
print(f"Baseline metrics folder:      {BASELINES_DIR}")
print(f"Radar plots will be saved in: {RADAR_OUTPUT_DIR}")

In [ ]:
#Settings
categories = ["Winter", "Summer", "Spring", "Autumn"]

metrics_to_plot = [
    "RMSE",
    "R2",
    "MAPE",
    "MAE"
]

plt.rcParams.update({
    "font.family": "serif",
    "figure.dpi": 300,
    "savefig.dpi": 1200,
    "font.size": 18
})

In [ ]:
#Locate proper file in Google Drive
DL_ARCHITECTURES = [
    "ANN",
    "LSTM",
    "TFT"
]

DL_CONFIGURATIONS = [
    "univariate",
    "flowrate",
    "turbidity",
    "flowrate_turbidity",
    "full_model"
]

DL_CONFIGURATION_LABELS = {
    "univariate": "Univariate Models",
    "flowrate": "Flowrate Models",
    "turbidity": "Turbidity Models",
    "flowrate_turbidity": "Flowrate + Turbidity Models",
    "full_model": "Full Models"
}

BASELINE_MODELS = [
    "ARIMA",
    "DES",
    "SARIMA",
    "SARIMAX",
    "SES",
    "TES"
]


def normalize_path_text(file_path):
    """
    Converts the full path to lowercase normalized text so that
    folder names and filenames can both be inspected.
    """

    return (
        str(file_path)
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


def identify_season_rd(file_path):
    """
    Identifies the season from either the filename or folder path.
    """

    text = normalize_path_text(file_path)

    for season in categories:
        if season.lower() in text:
            return season

    return None


def identify_dl_configuration(file_path):
    """
    Identifies the predictor configuration from the complete path.

    Supported configurations:
    - univariate
    - flowrate
    - turbidity
    - flowrate_turbidity
    - full_model
    """

    text = normalize_path_text(file_path)

    # Full model must be checked first because it can also contain
    # flowrate and turbidity terms.
    full_model_tokens = [
        "full_model",
        "fullmodel",
        "all_variables",
        "allvariables",
        "all_predictors",
        "chlorine_conversion",
        "chlorine_condition",
        "model4"
    ]

    if any(token in text for token in full_model_tokens):
        return "full_model"

    flowrate_turbidity_tokens = [
        "flowrate_turbidity",
        "flowrate_and_turbidity",
        "flow_turbidity",
        "turbidity_flowrate",
        "flowrate+turbidity",
        "model3"
    ]

    if any(
        token in text
        for token in flowrate_turbidity_tokens
    ):
        return "flowrate_turbidity"

    flowrate_tokens = [
        "flowrate_only",
        "flow_only",
        "finishwater_only",
        "model1"
    ]

    if any(token in text for token in flowrate_tokens):
        return "flowrate"

    turbidity_tokens = [
        "turbidity_only",
        "model2"
    ]

    if any(token in text for token in turbidity_tokens):
        return "turbidity"

    if "univariate" in text:
        return "univariate"

    # Generic identification when folders do not use "_only".
    has_flowrate = (
        "flowrate" in text
        or "finishwater" in text
    )

    has_turbidity = "turbidity" in text

    has_multivariate_label = "multivariate" in text

    if has_flowrate and has_turbidity:
        return "flowrate_turbidity"

    if has_flowrate:
        return "flowrate"

    if has_turbidity:
        return "turbidity"

    # Existing univariate filenames may contain only:
    # architecture + season + hyperparameters + metrics.
    if not has_multivariate_label:
        return "univariate"

    return None


def is_metrics_csv(file_path):
    """
    Keeps CSV files that appear to contain model metrics.
    Excludes residual files.
    """

    name = file_path.name.lower()

    return (
        file_path.suffix.lower() == ".csv"
        and "metric" in name
        and "residual" not in name
    )


def metrics_file_priority(file_path):
    """
    Gives priority to final/true holdout files.

    A lower number indicates stronger preference.
    """

    text = normalize_path_text(file_path)

    priority_terms = [
        "true_holdout_metrics",
        "final_holdout_metrics",
        "true_holdout",
        "final_holdout",
        "winning",
        "winner",
        "best",
        "fixed_holdout_metrics",
        "fixed_holdout",
        "holdout_metrics",
        "holdout",
        "metrics"
    ]

    for priority, term in enumerate(priority_terms):
        if term in text:
            return priority

    return len(priority_terms)


def select_metrics_file(candidate_files, description):
    """
    Selects one metrics file when multiple files match.

    Final/true holdout files receive priority.
    The most recently modified file breaks ties.
    """

    if not candidate_files:
        print(f"Missing: {description}")
        return None

    candidate_files = sorted(
        candidate_files,
        key=lambda file_path: (
            metrics_file_priority(file_path),
            -file_path.stat().st_mtime,
            str(file_path)
        )
    )

    selected_file = candidate_files[0]

    print(f"Selected {description}:")
    print(f"  {selected_file}")

    if len(candidate_files) > 1:
        print("  Other matching files not used:")

        for extra_file in candidate_files[1:]:
            print(f"    {extra_file}")

    return selected_file


def discover_dl_model_files():
    """
    Creates this structure:

    DL_MODEL_FILES[configuration][architecture][season]
    """

    discovered_files = {
        configuration: {
            architecture: {
                season: None
                for season in categories
            }
            for architecture in DL_ARCHITECTURES
        }
        for configuration in DL_CONFIGURATIONS
    }

    for architecture in DL_ARCHITECTURES:

        architecture_folder = (
            DEEP_LEARNING_DIR / architecture
        )

        if not architecture_folder.exists():
            print(
                f"Architecture folder not found: "
                f"{architecture_folder}"
            )
            continue

        metrics_files = [
            file_path
            for file_path
            in architecture_folder.rglob("*.csv")
            if is_metrics_csv(file_path)
        ]

        for configuration in DL_CONFIGURATIONS:

            for season in categories:

                matching_files = [
                    file_path
                    for file_path in metrics_files
                    if (
                        identify_season_rd(file_path) == season
                        and identify_dl_configuration(
                            file_path
                        ) == configuration
                    )
                ]

                discovered_files[
                    configuration
                ][
                    architecture
                ][
                    season
                ] = select_metrics_file(
                    candidate_files=matching_files,
                    description=(
                        f"{architecture} | "
                        f"{configuration} | "
                        f"{season}"
                    )
                )

    return discovered_files


def discover_baseline_model_files():
    """
    Finds one hard-partition metrics file per baseline model.
    """

    discovered_files = {}

    for model_name in BASELINE_MODELS:

        model_folder = BASELINES_DIR / model_name

        if not model_folder.exists():
            print(
                f"Baseline folder not found: "
                f"{model_folder}"
            )

            discovered_files[model_name] = None
            continue

        matching_files = [
            file_path
            for file_path in model_folder.rglob("*.csv")
            if is_metrics_csv(file_path)
        ]

        discovered_files[model_name] = (
            select_metrics_file(
                candidate_files=matching_files,
                description=(
                    f"{model_name} hard-partition metrics"
                )
            )
        )

    return discovered_files


DL_MODEL_FILES = discover_dl_model_files()

BASELINE_MODEL_FILES = (
    discover_baseline_model_files()
)

In [ ]:
#Reading metrics
def normalize_metric_name(metric_name):
    """
    Normalizes common variations such as R² versus R2.
    """

    return (
        str(metric_name)
        .strip()
        .upper()
        .replace("²", "2")
        .replace("_", "")
        .replace(" ", "")
    )


def read_metric_value(file_path, metric):
    """
    Reads one metric from one metrics CSV.
    """

    df = pd.read_csv(file_path)

    normalized_columns = {
        normalize_metric_name(column): column
        for column in df.columns
    }

    normalized_metric = normalize_metric_name(metric)

    if normalized_metric not in normalized_columns:
        raise ValueError(
            f"{metric} not found in:\n"
            f"{file_path}\n\n"
            f"Available columns:\n"
            f"{list(df.columns)}"
        )

    original_column = normalized_columns[
        normalized_metric
    ]

    return float(df[original_column].iloc[0])


def collect_seasonal_metric_values(
    model_files,
    metric
):
    """
    Preserves the existing seasonal radar data structure:

    {
        "ANN": [Winter, Summer, Spring, Autumn],
        "LSTM": [Winter, Summer, Spring, Autumn],
        "TFT": [Winter, Summer, Spring, Autumn]
    }

    A model is plotted only when all four seasonal files
    are currently available.
    """

    values = {}

    for model_name, season_files in model_files.items():

        missing_seasons = [
            season
            for season in categories
            if season_files.get(season) is None
        ]

        if missing_seasons:
            print(
                f"Skipping {model_name} for {metric}; "
                f"missing seasons: "
                f"{', '.join(missing_seasons)}"
            )
            continue

        model_values = []

        for season in categories:

            file_path = season_files[season]

            model_values.append(
                read_metric_value(
                    file_path=file_path,
                    metric=metric
                )
            )

        values[model_name] = model_values

    return values


def collect_baseline_metric_values(metric):
    """
    Returns one hard-partition value per baseline model.

    Baselines are not seasonal.
    """

    values = {}

    for model_name, file_path in (
        BASELINE_MODEL_FILES.items()
    ):

        if file_path is None:
            print(
                f"Skipping {model_name} for {metric}; "
                f"no hard-partition metrics file found."
            )
            continue

        values[model_name] = read_metric_value(
            file_path=file_path,
            metric=metric
        )

    return values

In [ ]:
#Spider plot function
def make_spider_plot(
    metric,
    metric_values,
    title_label,
    output_prefix
):

    N = len(categories)

    angles = np.linspace(
        0,
        2 * np.pi,
        N,
        endpoint=False
    ).tolist()

    angles += angles[:1]

    fig = plt.figure(figsize=(18, 18))
    ax = plt.subplot(111, polar=True)

    ax.set_theta_offset(pi / 2)
    ax.set_theta_direction(-1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([])

    all_values = [
        value
        for model_values in metric_values.values()
        for value in model_values
    ]

    min_val = min(all_values)
    max_val = max(all_values)

    if metric == "R2" and min_val < 0:

        radial_min = min_val * 1.10

        radial_limit = (
            max_val * 1.20
            if max_val > 0
            else max_val * 0.90
        )

    else:
        radial_min = 0
        radial_limit = max_val * 1.25

    if radial_limit == radial_min:
        radial_limit = radial_min + 1

    ax.set_ylim(
        radial_min,
        radial_limit
    )

    label_radius = radial_limit * 1.15

    ax.text(
        0,
        label_radius,
        "Winter",
        ha="center",
        va="center",
        fontsize=30,
        fontweight="bold"
    )

    ax.text(
        np.pi / 2,
        label_radius * 1.04,
        "Summer",
        ha="center",
        va="center",
        fontsize=30,
        fontweight="bold"
    )

    ax.text(
        np.pi,
        label_radius,
        "Spring",
        ha="center",
        va="center",
        fontsize=30,
        fontweight="bold"
    )

    ax.text(
        3 * np.pi / 2,
        label_radius * 1.04,
        "Autumn",
        ha="center",
        va="center",
        fontsize=30,
        fontweight="bold"
    )

    yticks = np.linspace(
        radial_min,
        radial_limit,
        6
    )[1:]

    ax.set_yticks(yticks)

    ax.set_yticklabels(
        [f"{x:.4f}" for x in yticks],
        fontsize=16
    )

    ax.set_rlabel_position(20)

    ax.grid(
        True,
        linewidth=1.3,
        alpha=0.30
    )

    ax.spines["polar"].set_linewidth(1.5)

    for model_name, model_values in (
        metric_values.items()
    ):

        model_plot = (
            model_values
            + model_values[:1]
        )

        ax.plot(
            angles,
            model_plot,
            marker="o",
            linewidth=3.5,
            markersize=9,
            label=model_name
        )

        ax.fill(
            angles,
            model_plot,
            alpha=0.05
        )

    plt.title(
        f"Seasonal Model Performance "
        f"Comparison ({metric})\n"
        f"{title_label}",
        fontsize=40,
        fontweight="bold",
        pad=125
    )

    ax.legend(
        loc="upper left",
        bbox_to_anchor=(0.97, 1.10),
        frameon=False,
        fontsize=22
    )

    table_data = []

    for i, season in enumerate(categories):

        row = [season]

        for model_name in metric_values.keys():

            row.append(
                f"{metric_values[model_name][i]:.4f}"
            )

        table_data.append(row)

    table = plt.table(
        cellText=table_data,
        colLabels=(
            ["Season"]
            + list(metric_values.keys())
        ),
        cellLoc="center",
        colLoc="center",
        loc="bottom",
        bbox=[-0.10, -0.44, 1.18, 0.30]
    )

    table.auto_set_font_size(False)

    if len(metric_values.keys()) <= 3:
        table.set_fontsize(20)

    elif len(metric_values.keys()) <= 5:
        table.set_fontsize(14)

    else:
        table.set_fontsize(12)

    plt.subplots_adjust(
        bottom=0.34,
        top=0.72,
        left=0.08,
        right=0.82
    )

    clean_title = (
        output_prefix
        .lower()
        .replace("/", "_")
        .replace(" ", "_")
    )

    output_file = (
        RADAR_OUTPUT_DIR
        / (
            f"{clean_title}_spider_chart_"
            f"{metric.lower()}.png"
        )
    )

    plt.savefig(
        output_file,
        dpi=1200,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    print(f"Saved: {output_file}")

In [ ]:
#Baseline model radar function

def make_baseline_spider_plot(
    metric,
    metric_values,
    title_label,
    output_prefix
):
    """
    Creates a nonseasonal radar plot.

    Each axis is one baseline model:
    ARIMA, DES, SARIMA, SARIMAX, SES, TES.
    """

    model_names = list(metric_values.keys())
    model_values = list(metric_values.values())

    N = len(model_names)

    if N < 3:
        print(
            f"At least three baseline models are "
            f"required for a radar plot. "
            f"Only {N} available for {metric}."
        )
        return

    angles = np.linspace(
        0,
        2 * np.pi,
        N,
        endpoint=False
    ).tolist()

    angles += angles[:1]

    plot_values = model_values + model_values[:1]

    fig = plt.figure(figsize=(18, 18))
    ax = plt.subplot(111, polar=True)

    ax.set_theta_offset(pi / 2)
    ax.set_theta_direction(-1)

    ax.set_xticks(angles[:-1])

    ax.set_xticklabels(
        model_names,
        fontsize=24,
        fontweight="bold"
    )

    min_val = min(model_values)
    max_val = max(model_values)

    if metric == "R2" and min_val < 0:

        radial_min = min_val * 1.10

        radial_limit = (
            max_val * 1.20
            if max_val > 0
            else max_val * 0.90
        )

    else:
        radial_min = 0
        radial_limit = max_val * 1.25

    if radial_limit == radial_min:
        radial_limit = radial_min + 1

    ax.set_ylim(
        radial_min,
        radial_limit
    )

    yticks = np.linspace(
        radial_min,
        radial_limit,
        6
    )[1:]

    ax.set_yticks(yticks)

    ax.set_yticklabels(
        [f"{x:.4f}" for x in yticks],
        fontsize=16
    )

    ax.set_rlabel_position(20)

    ax.grid(
        True,
        linewidth=1.3,
        alpha=0.30
    )

    ax.spines["polar"].set_linewidth(1.5)

    ax.plot(
        angles,
        plot_values,
        marker="o",
        linewidth=3.5,
        markersize=9,
        label=metric
    )

    ax.fill(
        angles,
        plot_values,
        alpha=0.05
    )

    plt.title(
        f"Baseline Model Performance "
        f"Comparison ({metric})\n"
        f"{title_label}",
        fontsize=40,
        fontweight="bold",
        pad=125
    )

    table_data = [
        [
            model_name,
            f"{metric_values[model_name]:.4f}"
        ]
        for model_name in model_names
    ]

    table = plt.table(
        cellText=table_data,
        colLabels=["Model", metric],
        cellLoc="center",
        colLoc="center",
        loc="bottom",
        bbox=[0.20, -0.44, 0.60, 0.30]
    )

    table.auto_set_font_size(False)
    table.set_fontsize(18)

    plt.subplots_adjust(
        bottom=0.34,
        top=0.72,
        left=0.08,
        right=0.92
    )

    clean_title = (
        output_prefix
        .lower()
        .replace("/", "_")
        .replace(" ", "_")
    )

    output_file = (
        RADAR_OUTPUT_DIR
        / (
            f"{clean_title}_spider_chart_"
            f"{metric.lower()}.png"
        )
    )

    plt.savefig(
        output_file,
        dpi=1200,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    print(f"Saved: {output_file}")


for metric in metrics_to_plot:

    baseline_metric_values = (
        collect_baseline_metric_values(
            metric=metric
        )
    )

    if len(baseline_metric_values) < 3:
        print(
            f"Skipping baseline radar for {metric}; "
            f"only {len(baseline_metric_values)} "
            f"baseline models are available."
        )
        continue

    make_baseline_spider_plot(
        metric=metric,
        metric_values=baseline_metric_values,
        title_label=(
            "Baseline Models — Hard Partition"
        ),
        output_prefix=(
            "baseline_models_hard_partition"
        )
    )

In [ ]:
# AI/DL univariate models
for metric in metrics_to_plot:

    dl_metric_values = (
        collect_seasonal_metric_values(
            model_files=(
                DL_MODEL_FILES["univariate"]
            ),
            metric=metric
        )
    )

    if not dl_metric_values:
        print(
            f"No complete univariate DL models "
            f"are available for {metric}."
        )
        continue

    make_spider_plot(
        metric=metric,
        metric_values=dl_metric_values,
        title_label="AI/ML Univariate Models",
        output_prefix="ai_dl_univariate_models"
    )

In [ ]:
#DL multivariable plots
multivariate_configurations = [
    "flowrate",
    "turbidity",
    "flowrate_turbidity",
    "full_model"
]


for configuration in multivariate_configurations:

    configuration_label = (
        DL_CONFIGURATION_LABELS[
            configuration
        ]
    )

    for metric in metrics_to_plot:

        experiment_metric_values = (
            collect_seasonal_metric_values(
                model_files=(
                    DL_MODEL_FILES[
                        configuration
                    ]
                ),
                metric=metric
            )
        )

        if not experiment_metric_values:
            print(
                f"No complete models available for "
                f"{configuration_label} — {metric}."
            )
            continue

        make_spider_plot(
            metric=metric,
            metric_values=(
                experiment_metric_values
            ),
            title_label=configuration_label,
            output_prefix=(
                f"ai_dl_{configuration}"
            )
        )

In [ ]:
#Metrics of SARIMAX Models
SARIMAX_DIR = BASELINES_DIR / "SARIMAX"


def find_sarimax_configuration_files():
    """
    Searches each immediate subfolder inside Baselines/SARIMAX.

    Expected folders may include:
        Flowrate
        Turbidity
        Flowrate_Turbidity
        Full_Model

    Each subfolder is treated as one SARIMAX configuration.
    One metrics CSV is selected from each configuration folder.
    """

    if not SARIMAX_DIR.exists():
        raise FileNotFoundError(
            f"SARIMAX folder not found:\n{SARIMAX_DIR}"
        )

    configuration_files = {}

    for configuration_folder in sorted(
        SARIMAX_DIR.iterdir()
    ):

        if not configuration_folder.is_dir():
            continue

        candidate_files = [
            file_path
            for file_path
            in configuration_folder.rglob("*.csv")
            if is_metrics_csv(file_path)
        ]

        if not candidate_files:
            print(
                f"No metrics CSV found in "
                f"{configuration_folder.name}"
            )
            continue

        selected_file = select_metrics_file(
            candidate_files=candidate_files,
            description=(
                f"SARIMAX configuration "
                f"{configuration_folder.name}"
            )
        )

        if selected_file is None:
            continue

        configuration_label = (
            configuration_folder.name
            .replace("_", " ")
            .replace("-", " ")
            .title()
        )

        configuration_files[
            configuration_label
        ] = selected_file

    print("\nSARIMAX configuration files selected:")

    for configuration_name, file_path in (
        configuration_files.items()
    ):
        print(
            f"{configuration_name}: "
            f"{file_path}"
        )

    return configuration_files


def collect_sarimax_configuration_values(
    configuration_files,
    metric
):
    """
    Reads one hard-partition metric value for each
    SARIMAX configuration.
    """

    metric_values = {}

    for configuration_name, file_path in (
        configuration_files.items()
    ):

        try:
            metric_values[
                configuration_name
            ] = read_metric_value(
                file_path=file_path,
                metric=metric
            )

        except Exception as error:
            print(
                f"Skipping {configuration_name} "
                f"for {metric}: {error}"
            )

    return metric_values


def make_sarimax_configuration_spider_plot(
    metric,
    metric_values
):
    """
    Creates one nonseasonal radar chart.

    Each radar axis represents one SARIMAX predictor
    configuration.
    """

    configuration_names = list(
        metric_values.keys()
    )

    configuration_values = list(
        metric_values.values()
    )

    number_of_configurations = len(
        configuration_names
    )

    if number_of_configurations < 3:
        print(
            f"Skipping SARIMAX radar for {metric}. "
            f"At least three configurations are required, "
            f"but only {number_of_configurations} "
            f"are available."
        )
        return

    angles = np.linspace(
        0,
        2 * np.pi,
        number_of_configurations,
        endpoint=False
    ).tolist()

    angles += angles[:1]

    closed_values = (
        configuration_values
        + configuration_values[:1]
    )

    fig = plt.figure(
        figsize=(18, 18)
    )

    ax = plt.subplot(
        111,
        polar=True
    )

    ax.set_theta_offset(
        pi / 2
    )

    ax.set_theta_direction(
        -1
    )

    ax.set_xticks(
        angles[:-1]
    )

    ax.set_xticklabels(
        configuration_names,
        fontsize=18,
        fontweight="bold"
    )

    minimum_value = min(
        configuration_values
    )

    maximum_value = max(
        configuration_values
    )

    if metric == "R2" and minimum_value < 0:

        radial_minimum = (
            minimum_value * 1.10
        )

        radial_maximum = (
            maximum_value * 1.20
            if maximum_value > 0
            else maximum_value * 0.90
        )

    else:
        radial_minimum = 0
        radial_maximum = (
            maximum_value * 1.25
        )

    if radial_maximum == radial_minimum:
        radial_maximum = (
            radial_minimum + 1
        )

    ax.set_ylim(
        radial_minimum,
        radial_maximum
    )

    radial_ticks = np.linspace(
        radial_minimum,
        radial_maximum,
        6
    )[1:]

    ax.set_yticks(
        radial_ticks
    )

    ax.set_yticklabels(
        [
            f"{value:.4f}"
            for value in radial_ticks
        ],
        fontsize=16
    )

    ax.set_rlabel_position(
        20
    )

    ax.grid(
        True,
        linewidth=1.3,
        alpha=0.30
    )

    ax.spines[
        "polar"
    ].set_linewidth(
        1.5
    )

    ax.plot(
        angles,
        closed_values,
        marker="o",
        linewidth=3.5,
        markersize=9,
        label=metric
    )

    ax.fill(
        angles,
        closed_values,
        alpha=0.05
    )

    plt.title(
        f"SARIMAX Configuration Performance "
        f"Comparison ({metric})\n"
        f"Hard-Partition Results",
        fontsize=40,
        fontweight="bold",
        pad=125
    )

    table_data = [
        [
            configuration_name,
            f"{metric_values[configuration_name]:.4f}"
        ]
        for configuration_name
        in configuration_names
    ]

    table = plt.table(
        cellText=table_data,
        colLabels=[
            "SARIMAX Configuration",
            metric
        ],
        cellLoc="center",
        colLoc="center",
        loc="bottom",
        bbox=[
            0.10,
            -0.48,
            0.80,
            0.32
        ]
    )

    table.auto_set_font_size(
        False
    )

    if number_of_configurations <= 5:
        table.set_fontsize(17)

    elif number_of_configurations <= 8:
        table.set_fontsize(14)

    else:
        table.set_fontsize(11)

    plt.subplots_adjust(
        bottom=0.38,
        top=0.72,
        left=0.08,
        right=0.92
    )

    output_file = (
        RADAR_OUTPUT_DIR
        / (
            "sarimax_configurations_"
            f"spider_chart_{metric.lower()}.png"
        )
    )

    plt.savefig(
        output_file,
        dpi=1200,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    print(f"Saved: {output_file}")



# Find all available SARIMAX configuration folders


SARIMAX_CONFIGURATION_FILES = (
    find_sarimax_configuration_files()
)



# Create one SARIMAX radar plot for each metric


for metric in metrics_to_plot:

    sarimax_configuration_values = (
        collect_sarimax_configuration_values(
            configuration_files=(
                SARIMAX_CONFIGURATION_FILES
            ),
            metric=metric
        )
    )

    if len(sarimax_configuration_values) < 3:
        print(
            f"Skipping SARIMAX configuration radar "
            f"for {metric}; only "
            f"{len(sarimax_configuration_values)} "
            f"configurations are available."
        )
        continue

    make_sarimax_configuration_spider_plot(
        metric=metric,
        metric_values=(
            sarimax_configuration_values
        )
    )

#Save a local copy of all outputs (if needed)

In [ ]:
# import os

# for item in os.listdir('/content'):
#     full_path = os.path.join('/content', item)
#     if os.path.isdir(full_path) and item.lower().startswith('output'):
#         print(full_path)

In [ ]:
# import os
# import shutil

# #Create temporary staging folder
# staging_dir = '/content/AlexanderOrr_All_Outputs'
# os.makedirs(staging_dir, exist_ok=True)

# #Folders to include
# folders_to_copy = [
#     # '/content/outputs_uni_lstm_corrected',
#     # '/content/outputs_uni_tft_corrected'
#     # '/content/outputs_uni_ann_corrected'
#     # '/content/outputs_smoothing_fixed',
#     # '/content/outputs_arima_fixed',
#     # '/content/outputs_sarima_fixed',
#     '/content/outputs_sarimax_fixed',
#     # '/content/ml_classification_outputs',

# ]

# #Copy folders into staging area
# for folder in folders_to_copy:
#     destination = os.path.join(staging_dir, os.path.basename(folder))

#     #Remove old copy if exists
#     if os.path.exists(destination):
#         shutil.rmtree(destination)

#     shutil.copytree(folder, destination)

# print("All folders copied.")

In [ ]:
# import shutil

# zip_base = '/content/AlexanderOrr_Forecasting_Outputs'

# shutil.make_archive(
#     base_name=zip_base,
#     format='zip',
#     root_dir=staging_dir
# )

# print(f'ZIP created: {zip_base}.zip')

In [ ]:
# from google.colab import files

# files.download('/content/AlexanderOrr_Forecasting_Outputs.zip')

#Classic ML Algorithm Pipeline to Detect Threshold Exceedances

In [ ]:
# Classical ML forecasting settings

# Classify residual chlorine as an exceedance when it is above
# upper threshold of 4.0 mg/L.
THRESHOLD_CHLORINE = 4.0

# Predict whether an exceedance will occur 12 hours after the forecast origin.
FORECAST_HORIZON = 12

# Use measurements from the previous 24 hourly observations as lagged inputs.
N_LAGS = 24


# Reserve observations beginning on May 1, 2025 for final holdout evaluation.
# The boundary is explicitly UTC to match the canonical modeling index.
HOLDOUT_START = pd.Timestamp(
    "2025-05-01",
    tz="UTC"
)


# Use five chronological folds for time-series cross-validation.
N_SPLITS = 5

# Reserve the final 90 days, or 2,160 hourly observations, of the pre-holdout
# period for classification-threshold calibration.
CALIBRATION_HOURS = 90 * 24

# Select the classification threshold using F-beta with beta=2.
# This weights recall more heavily than precision, which emphasizes detecting
# chlorine exceedances rather than minimizing false alarms.
THRESHOLD_BETA = 2.0


In [ ]:
# # Threshold grid searched on the independent pre-holdout calibration period only
# THRESHOLD_GRID = np.round(np.arange(0.05, 0.96, 0.05), 2)

# # Threshold objective: recall-weighted F2; ties favor recall, then precision
# PRIMARY_THRESHOLD_OBJECTIVE = "F2"

# # OUTPUT_DIR = "ml_classification_outputs"
# # os.makedirs(OUTPUT_DIR, exist_ok=True)

# # Save all classification outputs directly to Google Drive
# BASE_OUTPUT = "/content/drive/MyDrive/Forecasting_AlexanderOrr"

# OUTPUT_DIR = os.path.join(
#     BASE_OUTPUT,
#     "MachineLearning",
#     "Classification"
# )

# os.makedirs(OUTPUT_DIR, exist_ok=True)

# print(f"Classification output folder: {OUTPUT_DIR}")

# Classification-threshold search and ML output settings

# Search candidate probability thresholds from 0.05 through 0.95 using
# increments of 0.05. This search must use the independent pre-holdout
# calibration period only.
THRESHOLD_GRID = np.round(
    np.arange(
        0.05,
        0.96,
        0.05
    ),
    2
)


# Select the probability threshold using recall-weighted F2.
# When multiple thresholds have the same F2 score, later selection logic should
# favor higher recall first and then higher precision.
PRIMARY_THRESHOLD_OBJECTIVE = "F2"
# Using F2 says that missing an exceedance is more serious than producing an unnecessary alert. It will generally select a lower probability threshold, detect more exceedances, and tolerate more false alarms.

# Confirm that the shared forecasting output root was defined earlier.
if "BASE_OUTPUT" not in globals():
    raise NameError(
        "BASE_OUTPUT must be defined before configuring ML outputs."
    )


# Save all classification outputs under:
# Forecasting_AlexanderOrr/MachineLearning/Classification
OUTPUT_DIR = os.path.join(
    BASE_OUTPUT,
    "MachineLearning",
    "Classification"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# Display the resolved Google Drive folder before running the ML pipeline.
print(
    f"Classification output folder: {OUTPUT_DIR}"
)



In [ ]:
# Validate the variables required by the ML classification pipeline.

# Define the target, plant measurements, and operational indicator expected by
# the active classification models.
required_cols = [
    "totalchlorine",
    "turbidity",
    "finishwater",
    "chlorine_conversion"
]


# Identify any required columns that are absent from the prepared DataFrame.
missing_cols = [
    column
    for column in required_cols
    if column not in df.columns
]


# Stop the pipeline before feature engineering if required variables are missing.
if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )


# Convert every required variable to numeric values.
# Values that cannot be converted are replaced with NaN.
for column in required_cols:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# Confirm that the prepared DataFrame uses timestamps as its index.
if not isinstance(
    df.index,
    pd.DatetimeIndex
):
    raise ValueError(
        "Prepared dataframe must have a DatetimeIndex."
    )


In [ ]:
# Create the binary 12-hour-ahead classification target.

# Mark the current observation as an exceedance when residual chlorine is
# strictly greater than the 4.0 mg/L upper threshold.
# A value exactly equal to 4.0 mg/L is at the limit, not above it.
df["exceedance_now"] = (
    df["totalchlorine"]
    > THRESHOLD_CHLORINE
).astype(int)


# Shift the current exceedance indicator backward by the forecast horizon.
# The row at forecast origin t now contains the class that occurs at t+12 hours.
df["target"] = df["exceedance_now"].shift(
    -FORECAST_HORIZON
)


# Preserve the exact UTC timestamp corresponding to each future target.
# This keeps holdout predictions, plots, and metrics aligned to t+12 rather
# than incorrectly labeling them with the forecast-origin timestamp.
df["target_timestamp"] = (
    pd.Series(
        df.index,
        index=df.index
    )
    .shift(-FORECAST_HORIZON)
)

In [ ]:
# ML feature-engineering helpers

# Create historical lag features for the selected measurement columns.
# With hourly data, lag 1 represents one hour before forecast origin t and
# lag 24 represents 24 hours before t.
def create_lag_features(
    data,
    columns,
    n_lags
):
    data = data.copy()

    for column in columns:
        for lag in range(
            1,
            n_lags + 1
        ):
            data[
                f"{column}_lag_{lag}"
            ] = data[column].shift(lag)

    return data


# Create rolling summary statistics using measurements available at or before
# forecast origin t. The current observation is included intentionally because
# it is assumed to be known when the 12-hour-ahead forecast is issued.
def create_rolling_features(data):
    data = data.copy()

    # Residual-chlorine rolling features
    data["chlorine_roll_mean_6"] = (
        data["totalchlorine"]
        .rolling(6)
        .mean()
    )

    data["chlorine_roll_std_6"] = (
        data["totalchlorine"]
        .rolling(6)
        .std()
    )

    data["chlorine_roll_max_12"] = (
        data["totalchlorine"]
        .rolling(12)
        .max()
    )

    data["chlorine_roll_min_12"] = (
        data["totalchlorine"]
        .rolling(12)
        .min()
    )

    # Turbidity rolling features
    data["turbidity_roll_mean_6"] = (
        data["turbidity"]
        .rolling(6)
        .mean()
    )

    data["turbidity_roll_std_6"] = (
        data["turbidity"]
        .rolling(6)
        .std()
    )

    # Flow-rate rolling features
    data["finishwater_roll_mean_6"] = (
        data["finishwater"]
        .rolling(6)
        .mean()
    )

    data["finishwater_roll_std_6"] = (
        data["finishwater"]
        .rolling(6)
        .std()
    )

    return data


# Create recent-change features using current and historical measurements only.
def create_change_features(data):
    data = data.copy()

    # Change in residual chlorine over the previous one and three hours.
    data["chlorine_diff_1"] = (
        data["totalchlorine"]
        .diff(1)
    )

    data["chlorine_diff_3"] = (
        data["totalchlorine"]
        .diff(3)
    )

    # One-hour changes in turbidity and flow rate.
    data["turbidity_diff_1"] = (
        data["turbidity"]
        .diff(1)
    )

    data["finishwater_diff_1"] = (
        data["finishwater"]
        .diff(1)
    )

    return data


# Create explicit UTC calendar features from the canonical UTC index.
# Existing UTC features are preserved instead of recalculated.
def create_time_features(data):
    data = data.copy()

    # Confirm that time features are derived from a UTC DatetimeIndex.
    if not isinstance(
        data.index,
        pd.DatetimeIndex
    ):
        raise TypeError(
            "Time features require a DatetimeIndex."
        )

    if str(data.index.tz) != "UTC":
        raise ValueError(
            "ML time features must be created from a UTC index."
        )

    # Create raw UTC calendar features when they are not already available.
    if "utc_hour" not in data.columns:
        data["utc_hour"] = data.index.hour

    if "utc_day_of_week" not in data.columns:
        data["utc_day_of_week"] = (
            data.index.dayofweek
        )

    if "utc_month" not in data.columns:
        data["utc_month"] = data.index.month

    # Represent UTC hour cyclically so hour 23 is close to hour 0.
    if "utc_hour_sin" not in data.columns:
        data["utc_hour_sin"] = np.sin(
            2
            * np.pi
            * data["utc_hour"]
            / 24.0
        )

    if "utc_hour_cos" not in data.columns:
        data["utc_hour_cos"] = np.cos(
            2
            * np.pi
            * data["utc_hour"]
            / 24.0
        )

    # Represent UTC day of week cyclically so Sunday is close to Monday.
    if "utc_dow_sin" not in data.columns:
        data["utc_dow_sin"] = np.sin(
            2
            * np.pi
            * data["utc_day_of_week"]
            / 7.0
        )

    if "utc_dow_cos" not in data.columns:
        data["utc_dow_cos"] = np.cos(
            2
            * np.pi
            * data["utc_day_of_week"]
            / 7.0
        )

    return data


# Create lag features for residual chlorine, turbidity, and flow rate.
lag_cols = [
    "totalchlorine",
    "turbidity",
    "finishwater"
]

df = create_lag_features(
    data=df,
    columns=lag_cols,
    n_lags=N_LAGS
)

df = create_rolling_features(df)
df = create_change_features(df)
df = create_time_features(df)


# Identify the engineered columns expected to contain initial missing values
# because of lagging, rolling calculations, differencing, or target shifting.
lag_feature_cols = [
    f"{column}_lag_{lag}"
    for column in lag_cols
    for lag in range(
        1,
        N_LAGS + 1
    )
]

rolling_feature_cols = [
    "chlorine_roll_mean_6",
    "chlorine_roll_std_6",
    "chlorine_roll_max_12",
    "chlorine_roll_min_12",
    "turbidity_roll_mean_6",
    "turbidity_roll_std_6",
    "finishwater_roll_mean_6",
    "finishwater_roll_std_6"
]

change_feature_cols = [
    "chlorine_diff_1",
    "chlorine_diff_3",
    "turbidity_diff_1",
    "finishwater_diff_1"
]

required_engineered_cols = (
    ["target", "target_timestamp"]
    + lag_feature_cols
    + rolling_feature_cols
    + change_feature_cols
)


# Remove only rows that are incomplete because of the engineered predictors or
# unavailable future targets. Missing values in unrelated EDA columns will not
# unnecessarily remove valid ML observations.
df_model = df.dropna(
    subset=required_engineered_cols
).copy()


# Store the binary future target as an integer after removing its final missing
# values created by the 12-hour shift.
df_model["target"] = (
    df_model["target"]
    .astype(int)
)


print(
    "ML feature engineering completed."
)

print(
    "Modeling rows:",
    len(df_model)
)

print(
    "Modeling period:",
    df_model.index.min(),
    "to",
    df_model.index.max()
)

In [ ]:
# Define the predictor columns used by the ML classification pipeline.

# Include the current plant measurements and operating-condition indicator
# available at forecast origin t.
base_features = [
    "turbidity",
    "finishwater",
    "chlorine_conversion",
    "totalchlorine"
]


# Include rolling summaries, recent changes, and explicit UTC calendar features.
engineered_features = [
    # Residual-chlorine rolling features
    "chlorine_roll_mean_6",
    "chlorine_roll_std_6",
    "chlorine_roll_max_12",
    "chlorine_roll_min_12",

    # Turbidity rolling features
    "turbidity_roll_mean_6",
    "turbidity_roll_std_6",

    # Flow-rate rolling features
    "finishwater_roll_mean_6",
    "finishwater_roll_std_6",

    # Recent measurement changes
    "chlorine_diff_1",
    "chlorine_diff_3",
    "turbidity_diff_1",
    "finishwater_diff_1",

    # Raw and cyclical UTC calendar features
    "utc_hour",
    "utc_hour_sin",
    "utc_hour_cos",
    "utc_day_of_week",
    "utc_month",
    "utc_dow_sin",
    "utc_dow_cos"
]


# Begin with the current and engineered predictors.
feature_cols = (
    base_features
    + engineered_features
)


# Add the previous 24 hourly values of residual chlorine, turbidity, and flow
# rate. The column names must match those created by create_lag_features().
for column in lag_cols:
    for lag in range(
        1,
        N_LAGS + 1
    ):
        feature_cols.append(
            f"{column}_lag_{lag}"
        )


# Remove duplicate feature names while preserving their intended order.
feature_cols = list(
    dict.fromkeys(feature_cols)
)


# Confirm that all requested predictors exist.
missing_feature_cols = [
    column
    for column in feature_cols
    if column not in df_model.columns
]

if missing_feature_cols:
    raise ValueError(
        "Required ML features are missing:\n"
        f"{missing_feature_cols}"
    )


# Confirm that future labels and timestamps are not included as predictors.
forbidden_predictors = {
    "target",
    "target_timestamp",
    "exceedance_now"
}

leaking_predictors = sorted(
    set(feature_cols)
    & forbidden_predictors
)

if leaking_predictors:
    raise ValueError(
        "Target leakage detected in feature list: "
        f"{leaking_predictors}"
    )


# Create the predictor matrix and 12-hour-ahead binary target.
X = df_model[
    feature_cols
].copy()

y = df_model[
    "target"
].copy()


# Confirm that predictors and targets remain perfectly aligned by forecast
# origin before chronological splitting.
assert X.index.equals(y.index), (
    "Predictor and target indexes are not aligned."
)

assert not X.isna().any().any(), (
    "The predictor matrix contains missing values."
)

assert not y.isna().any(), (
    "The target contains missing values."
)

assert y.isin([0, 1]).all(), (
    "The classification target contains values other than 0 and 1."
)


# Display the final modeling dimensions and class distribution.
print("=" * 70)
print("FINAL ML DATASET")
print("=" * 70)

print(
    "df_model shape:",
    df_model.shape
)

print(
    "X shape:",
    X.shape
)

print(
    "Number of predictors:",
    len(feature_cols)
)

print(
    "Positive class rate (overall):",
    round(y.mean(), 4)
)

print(
    "Positive count:",
    int(y.sum())
)

print(
    "Negative count:",
    int((y == 0).sum())
)


# Export the feature-engineered ML dataset under the central classification
# output directory.
ml_output_path = os.path.join(
    OUTPUT_DIR,
    "alexanderorrml.xlsx"
)


# Create an export-only copy so removing timezone metadata for Excel does not
# modify the modeling DataFrame used by Python.
df_export = df_model.copy()


# Label the index clearly as a UTC forecast-origin timestamp.
df_export.index.name = (
    "forecast_origin_timestamp_utc"
)


# Excel does not support timezone-aware datetime indexes. Remove only the
# timezone metadata from the export copy; the displayed values remain UTC.
if (
    isinstance(
        df_export.index,
        pd.DatetimeIndex
    )
    and df_export.index.tz is not None
):
    df_export.index = (
        df_export.index
        .tz_localize(None)
    )


# Remove timezone metadata from datetime columns such as target_timestamp.
# These values also continue to represent UTC after export.
timezone_columns = (
    df_export
    .select_dtypes(
        include=["datetimetz"]
    )
    .columns
)

for column in timezone_columns:
    df_export[column] = (
        df_export[column]
        .dt
        .tz_localize(None)
    )


# Save the complete feature-engineered classification dataset.
df_export.to_excel(
    ml_output_path,
    index=True
)


print(
    "\nMachine-learning dataset exported to:"
)

print(
    ml_output_path
)

print(
    "Dataset exported successfully. "
    "The index represents UTC forecast-origin time, and "
    "target_timestamp represents the corresponding UTC t+12 outcome. "
    "Timezone metadata was removed only for Excel compatibility."
)

In [ ]:
# Chronological development, calibration, and final-holdout split

# Calculate the beginning of the independent 90-day calibration period.
# Both calibration and holdout boundaries refer to future target timestamps.
CALIBRATION_START = (
    HOLDOUT_START
    - pd.Timedelta(
        hours=CALIBRATION_HOURS
    )
)


# Confirm that every ML row predicts exactly 12 hours into the future.
target_time_difference = (
    df_model["target_timestamp"]
    - df_model.index
)

assert target_time_difference.eq(
    pd.Timedelta(
        hours=FORECAST_HORIZON
    )
).all(), (
    "Target timestamps are not aligned with the forecast horizon."
)


# Reserve every row whose predicted outcome occurs before the holdout boundary
# as pre-holdout data.
train_df = df_model.loc[
    df_model["target_timestamp"]
    < HOLDOUT_START
].copy()


# Reserve every row whose predicted outcome occurs at or after May 1, 2025 for
# final holdout evaluation.
test_df = df_model.loc[
    df_model["target_timestamp"]
    >= HOLDOUT_START
].copy()


# Create the complete pre-holdout and final-holdout predictor matrices.
X_train = train_df[
    feature_cols
].copy()

y_train = train_df[
    "target"
].copy()

X_test = test_df[
    feature_cols
].copy()

y_test = test_df[
    "target"
].copy()


# Confirm that target periods do not overlap.
assert (
    train_df["target_timestamp"].max()
    < test_df["target_timestamp"].min()
), "Training and holdout target periods overlap."

assert (
    test_df["target_timestamp"].min()
    >= HOLDOUT_START
), "The holdout contains targets before HOLDOUT_START."


# Report both forecast-origin times and future target times.
print("=" * 70)
print("PRE-HOLDOUT / FINAL-HOLDOUT SPLIT")
print("=" * 70)

print(
    "Pre-holdout rows:",
    len(train_df)
)

print(
    "Holdout rows:",
    len(test_df)
)

print(
    "Final training target:",
    train_df["target_timestamp"].max()
)

print(
    "First holdout target:",
    test_df["target_timestamp"].min()
)

print(
    "Final holdout target:",
    test_df["target_timestamp"].max()
)

print(
    "First holdout forecast origin:",
    test_df.index.min()
)

print(
    "Final holdout forecast origin:",
    test_df.index.max()
)


# Report the relative sizes of the retained pre-holdout and holdout periods.
retained_rows = (
    len(train_df)
    + len(test_df)
)

print(
    "Training percentage of retained rows: "
    f"{100 * len(train_df) / retained_rows:.2f}%"
)

print(
    "Holdout percentage of retained rows: "
    f"{100 * len(test_df) / retained_rows:.2f}%"
)


# Divide the pre-holdout period into model-development and independent
# threshold-calibration periods using future target timestamps.
model_train_df = train_df.loc[
    train_df["target_timestamp"]
    < CALIBRATION_START
].copy()

calibration_df = train_df.loc[
    (
        train_df["target_timestamp"]
        >= CALIBRATION_START
    )
    & (
        train_df["target_timestamp"]
        < HOLDOUT_START
    )
].copy()


# Stop if either period is empty.
if model_train_df.empty:
    raise ValueError(
        "The model-development period is empty."
    )

if calibration_df.empty:
    raise ValueError(
        "The threshold-calibration period is empty."
    )


# Create the development and calibration predictor matrices.
X_model_train = model_train_df[
    feature_cols
].copy()

y_model_train = model_train_df[
    "target"
].copy()

X_calibration = calibration_df[
    feature_cols
].copy()

y_calibration = calibration_df[
    "target"
].copy()


# Confirm strict chronological separation using target timestamps.
assert (
    model_train_df["target_timestamp"].max()
    < calibration_df["target_timestamp"].min()
), "Development and calibration target periods overlap."

assert (
    calibration_df["target_timestamp"].max()
    < test_df["target_timestamp"].min()
), "Calibration and holdout target periods overlap."


# Confirm that every split contains both classification outcomes.
for split_name, split_target in {
    "development": y_model_train,
    "calibration": y_calibration,
    "holdout": y_test
}.items():
    if split_target.nunique() < 2:
        raise ValueError(
            f"The {split_name} period does not contain both classes."
        )


print("\n" + "=" * 70)
print("DEVELOPMENT / CALIBRATION DETAILS")
print("=" * 70)

print(
    "Development rows:",
    len(model_train_df)
)

print(
    "Calibration rows:",
    len(calibration_df)
)

print(
    "Development target period:",
    model_train_df["target_timestamp"].min(),
    "to",
    model_train_df["target_timestamp"].max()
)

print(
    "Calibration target period:",
    calibration_df["target_timestamp"].min(),
    "to",
    calibration_df["target_timestamp"].max()
)

print(
    "Development positive rate:",
    round(y_model_train.mean(), 4)
)

print(
    "Calibration positive rate:",
    round(y_calibration.mean(), 4)
)

print(
    "Holdout positive rate:",
    round(y_test.mean(), 4)
)


In [ ]:
# #Pre-processing
# numeric_features = feature_cols

# numeric_transformer = Pipeline(steps=[
#     ("imputer", SimpleImputer(strategy="median")),
#     ("scaler", StandardScaler())
# ])

# preprocessor = ColumnTransformer(
#     transformers=[
#         ("num", numeric_transformer, numeric_features)
#     ],
#     remainder="drop"
# )

#August 6

# Define the preprocessing applied to every ML predictor.

# Treat all selected predictors as numeric, including the binary
# chlorine-conversion indicator.
numeric_features = feature_cols.copy()


# Replace any unexpected missing predictor values with the training-period
# median and standardize each feature using training-period statistics.
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# Apply the numeric preprocessing pipeline only to the selected feature columns.
# Any columns not explicitly included in feature_cols are excluded, preventing
# target, target_timestamp, and other metadata from entering the models.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        )
    ],
    remainder="drop"
)

In [ ]:
# Define the candidate classification algorithms and hyperparameter grids.

# Store unfitted base estimators. Each estimator will later be combined with the
# preprocessing object inside a model Pipeline.
base_models = {
    # Linear probability classifier with class balancing for rare exceedances.
    "LogisticRegression": LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=SEED
    ),

    # Nonlinear rule-based classifier with complexity tuned below.
    "DecisionTree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=SEED
    ),

    # Ensemble of decision trees with parallel CPU processing.
    "RandomForest": RandomForestClassifier(
        class_weight="balanced",
        n_jobs=-1,
        random_state=SEED
    ),

    # Nonlinear radial-basis SVM with probability estimates enabled for
    # threshold calibration and precision-recall evaluation.
    "SVM": SVC(
        probability=True,
        class_weight="balanced",
        kernel="rbf",
        random_state=SEED
    ),

    # Distance-based classifier. Feature standardization in the preprocessing
    # pipeline is especially important for KNN.
    "KNN": KNeighborsClassifier()
}


# Define the hyperparameter combinations tested using chronological
# cross-validation within the model-development period only.
param_grids = {
    "LogisticRegression": {
        # Control the strength of L2 regularization.
        "C": [
            0.01,
            0.1,
            1.0,
            10.0
        ],
        "penalty": ["l2"],
        "solver": ["lbfgs"]
    },

    "DecisionTree": {
        # Control tree complexity and the minimum size of terminal leaves.
        "max_depth": [
            4,
            6,
            8,
            10,
            12
        ],
        "min_samples_leaf": [
            5,
            10,
            20,
            40
        ],
        "criterion": [
            "gini",
            "entropy"
        ]
    },

    "RandomForest": {
        # Compare forest size, tree depth, leaf size, and feature sampling.
        "n_estimators": [
            300,
            800
        ],
        "max_depth": [
            8,
            12,
            16
        ],
        "min_samples_leaf": [
            5,
            10,
            20
        ],
        "max_features": [
            "sqrt",
            "log2"
        ]
    },

    "SVM": {
        # Tune the penalty strength and radial-basis kernel scale.
        "C": [
            0.1,
            1.0,
            10.0
        ],
        "gamma": [
            "scale",
            "auto"
        ]
    },

    "KNN": {
        # Tune neighborhood size, vote weighting, and distance metric.
        # p=1 uses Manhattan distance; p=2 uses Euclidean distance.
        "n_neighbors": [
            5,
            10,
            15,
            25,
            35
        ],
        "weights": [
            "uniform",
            "distance"
        ],
        "p": [
            1,
            2
        ]
    }
}


# Convert one hyperparameter dictionary into a stable readable string for
# result tables and exported CSV files.
def params_to_string(params):
    if not params:
        return "default"

    return "; ".join(
        f"{parameter}={params[parameter]}"
        for parameter in sorted(params)
    )


# Rank hyperparameter configurations using development-period CV metrics.
# Average precision is the primary criterion because the positive exceedance
# class is uncommon. Remaining metrics are used only as tie-breakers.
def select_best_hyperparams(
    hyperparam_summary_df
):
    if hyperparam_summary_df.empty:
        raise ValueError(
            "No successful hyperparameter results are available."
        )

    required_metric_cols = [
        "AveragePrecision",
        "Recall",
        "F1",
        "Precision",
        "ROC_AUC"
    ]

    missing_metric_cols = [
        column
        for column in required_metric_cols
        if column not in hyperparam_summary_df.columns
    ]

    if missing_metric_cols:
        raise ValueError(
            "Hyperparameter summary is missing metrics: "
            f"{missing_metric_cols}"
        )

    # Rank higher metric values first.
    ranked = (
        hyperparam_summary_df
        .sort_values(
            by=[
                "AveragePrecision",
                "Recall",
                "F1",
                "Precision",
                "ROC_AUC"
            ],
            ascending=[
                False,
                False,
                False,
                False,
                False
            ]
        )
        .reset_index(drop=True)
    )

    # Return the winning row and the complete ranked comparison table.
    return ranked.iloc[0], ranked

In [ ]:
# Metric utilities
def safe_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return cm, tn, fp, fn, tp

def evaluate_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    f2 = fbeta_score(y_true, y_pred, beta=THRESHOLD_BETA, zero_division=0)

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        roc_auc = np.nan

    try:
        ap = average_precision_score(y_true, y_prob)
    except ValueError:
        ap = np.nan

    cm, tn, fp, fn, tp = safe_confusion_matrix(y_true, y_pred)

    return {
        "threshold": threshold,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "F2": f2,
        "ROC_AUC": roc_auc,
        "AveragePrecision": ap,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "ConfusionMatrix": cm,
        "y_pred": y_pred
    }

# def select_best_threshold(threshold_perf_df):
#     """
#     Select threshold using CV summary.
#     Primary objective: maximize recall.
#     Tie-breakers: AveragePrecision, F1, Precision.
#     """
#     ranked = threshold_perf_df.sort_values(
#         by=["Recall", "AveragePrecision", "F1", "Precision"],
#         ascending=[False, False, False, False]
#     ).reset_index(drop=True)

#     return ranked.iloc[0]["Threshold"], ranked

def select_best_threshold(threshold_perf_df):
    """Select on independent calibration data using recall-weighted F2."""
    ranked = threshold_perf_df.sort_values(
        by=["F2", "Recall", "Precision"],
        ascending=[False, False, False]
    ).reset_index(drop=True)
    return ranked.iloc[0]["Threshold"], ranked

def get_probabilities(pipe, X_data, model_name):
    if hasattr(pipe.named_steps["model"], "predict_proba"):
        return pipe.predict_proba(X_data)[:, 1]
    raise ValueError(f"{model_name} does not support predict_proba.")

In [ ]:
# Chronological hyperparameter tuning and independent threshold calibration

# Use expanding-window time-series cross-validation.
# The 12-row gap ensures that every training target occurs before the first
# validation forecast origin, preventing horizon overlap.
tscv = TimeSeriesSplit(
    n_splits=N_SPLITS,
    gap=FORECAST_HORIZON
)


# Store candidate summaries, selected-candidate fold results, failures, and
# final hyperparameter choices.
hyperparameter_tuning_results = []
selected_cv_fold_results = []
hyperparameter_tuning_failures = []
best_hyperparameter_results = []
best_hyperparameters = {}
tuned_models = {}


print("\n" + "=" * 70)
print("HYPERPARAMETER TUNING WITH TIME-SERIES CROSS-VALIDATION")
print("=" * 70)


# Tune each classification algorithm independently.
for model_name, base_model in base_models.items():
    print(
        f"\nModel: {model_name}"
    )

    candidate_summaries = []

    # Preserve the original parameter dictionaries and fold-level results so
    # DataFrame type conversion does not alter integer hyperparameters.
    candidate_param_lookup = {}
    candidate_fold_lookup = {}

    for params in ParameterGrid(
        param_grids[model_name]
    ):
        params_string = params_to_string(
            params
        )

        fold_rows = []
        failed_candidate = False

        print(
            f"  Testing params: {params_string}"
        )

        # Evaluate the candidate on chronological expanding-window folds.
        for fold, (
            train_indices,
            validation_indices
        ) in enumerate(
            tscv.split(X_model_train),
            start=1
        ):
            X_fold_train = X_model_train.iloc[
                train_indices
            ]

            y_fold_train = y_model_train.iloc[
                train_indices
            ]

            X_fold_validation = X_model_train.iloc[
                validation_indices
            ]

            y_fold_validation = y_model_train.iloc[
                validation_indices
            ]

            # Recover the future target timestamps corresponding to each fold.
            fold_train_target_times = (
                model_train_df.iloc[
                    train_indices
                ]["target_timestamp"]
            )

            fold_validation_target_times = (
                model_train_df.iloc[
                    validation_indices
                ]["target_timestamp"]
            )

            try:
                # Confirm that the gap prevents training outcomes from extending
                # into the validation forecast-origin period.
                assert (
                    fold_train_target_times.max()
                    < X_fold_validation.index.min()
                ), (
                    "Training targets overlap the validation "
                    "forecast-origin period."
                )

                # Both classes are required for fitting and evaluating the
                # classification models consistently.
                if y_fold_train.nunique() < 2:
                    raise ValueError(
                        "Training fold does not contain both classes."
                    )

                if y_fold_validation.nunique() < 2:
                    raise ValueError(
                        "Validation fold does not contain both classes."
                    )

                # Clone the estimator so every candidate and fold starts from
                # an unfitted model.
                model = (
                    clone(base_model)
                    .set_params(**params)
                )

                # Keep preprocessing inside the pipeline so medians and scaling
                # parameters are fitted using this training fold only.
                pipe = Pipeline(
                    steps=[
                        (
                            "preprocessor",
                            clone(preprocessor)
                        ),
                        (
                            "model",
                            model
                        )
                    ]
                )

                pipe.fit(
                    X_fold_train,
                    y_fold_train
                )

                # Obtain positive-class probabilities or decision scores using
                # the fitted fold-specific pipeline.
                y_validation_probability = get_probabilities(
                    pipe,
                    X_fold_validation,
                    model_name
                )

                # Use 0.50 only to compare hyperparameter configurations.
                # The final operating threshold is selected later using the
                # independent calibration period.
                metrics = evaluate_at_threshold(
                    y_true=y_fold_validation,
                    y_prob=y_validation_probability,
                    threshold=0.50
                )

                fold_rows.append({
                    "Model": model_name,
                    "Params": params_string,
                    "Fold": fold,
                    "TrainingRows": len(
                        X_fold_train
                    ),
                    "ValidationRows": len(
                        X_fold_validation
                    ),
                    "TrainingTargetEnd": (
                        fold_train_target_times.max()
                    ),
                    "ValidationTargetStart": (
                        fold_validation_target_times.min()
                    ),
                    "Accuracy": metrics["Accuracy"],
                    "Precision": metrics["Precision"],
                    "Recall": metrics["Recall"],
                    "F1": metrics["F1"],
                    "ROC_AUC": metrics["ROC_AUC"],
                    "AveragePrecision": (
                        metrics["AveragePrecision"]
                    )
                })

            except Exception as error:
                failed_candidate = True

                hyperparameter_tuning_failures.append({
                    "Model": model_name,
                    "Params": params_string,
                    "Fold": fold,
                    "Error": str(error)
                })

                print(
                    f"    Fold {fold} failed: {error}"
                )

                # A candidate must succeed on every fold to remain eligible.
                break

        # Retain only candidates that completed every chronological fold.
        if (
            not failed_candidate
            and len(fold_rows) == N_SPLITS
        ):
            fold_df = pd.DataFrame(
                fold_rows
            )

            candidate_summary = {
                "Model": model_name,
                "Params": params_string,
                "Accuracy": fold_df[
                    "Accuracy"
                ].mean(),
                "Precision": fold_df[
                    "Precision"
                ].mean(),
                "Recall": fold_df[
                    "Recall"
                ].mean(),
                "F1": fold_df[
                    "F1"
                ].mean(),
                "ROC_AUC": fold_df[
                    "ROC_AUC"
                ].mean(),
                "AveragePrecision": fold_df[
                    "AveragePrecision"
                ].mean()
            }

            # Retain separate parameter columns for analysis and exporting.
            for parameter, value in params.items():
                candidate_summary[
                    f"param_{parameter}"
                ] = value

            candidate_summaries.append(
                candidate_summary
            )

            hyperparameter_tuning_results.append(
                candidate_summary
            )

            candidate_param_lookup[
                params_string
            ] = params.copy()

            candidate_fold_lookup[
                params_string
            ] = fold_rows.copy()


    # At least one configuration must complete every fold successfully.
    candidate_summary_df = pd.DataFrame(
        candidate_summaries
    )

    if candidate_summary_df.empty:
        raise RuntimeError(
            f"All hyperparameter candidates failed for {model_name}. "
            "Review the recorded tuning failures."
        )


    # Select the candidate with the highest mean average precision.
    # Recall, F1, precision, and ROC AUC are used as tie-breakers.
    (
        best_row,
        ranked_candidates
    ) = select_best_hyperparams(
        candidate_summary_df
    )

    best_params_string = best_row[
        "Params"
    ]

    # Recover the original parameter types from the lookup dictionary.
    best_params = candidate_param_lookup[
        best_params_string
    ].copy()

    best_hyperparameters[
        model_name
    ] = best_params

    # Preserve one unfitted estimator with the selected hyperparameters.
    tuned_models[
        model_name
    ] = (
        clone(base_model)
        .set_params(**best_params)
    )

    # Store only the selected candidate's fold-level results for later
    # cross-model statistical comparisons.
    selected_cv_fold_results.extend(
        candidate_fold_lookup[
            best_params_string
        ]
    )

    # Store the selected candidate's average CV metrics.
    best_hyperparameter_results.append(
        best_row.to_dict()
    )

    # Save the complete ranked candidate table.
    ranked_candidates.to_csv(
        os.path.join(
            OUTPUT_DIR,
            (
                f"{model_name}_"
                "hyperparameter_tuning_results.csv"
            )
        ),
        index=False
    )

    print(
        f"\n  Best params for {model_name}: "
        f"{best_params}"
    )

    print(
        f"  Best CV AP="
        f"{best_row['AveragePrecision']:.4f}, "
        f"Recall={best_row['Recall']:.4f}, "
        f"F1={best_row['F1']:.4f}, "
        f"Precision={best_row['Precision']:.4f}"
    )


# Replace the base estimators with unfitted estimators containing the selected
# hyperparameters. Preprocessing will still be fitted inside later pipelines.
models = tuned_models


# Create complete tuning, selected-fold, best-configuration, and failure tables.
hyperparameter_tuning_results_df = pd.DataFrame(
    hyperparameter_tuning_results
)

cv_fold_results_df = pd.DataFrame(
    selected_cv_fold_results
)

best_hyperparameter_results_df = pd.DataFrame(
    best_hyperparameter_results
)

best_hyperparameters_df = pd.DataFrame([
    {
        "Model": model_name,
        "BestParams": params_to_string(
            params
        ),
        **{
            f"param_{parameter}": value
            for parameter, value
            in params.items()
        }
    }
    for model_name, params
    in best_hyperparameters.items()
])

hyperparameter_tuning_failures_df = pd.DataFrame(
    hyperparameter_tuning_failures
)


print("\n" + "=" * 70)
print("BEST HYPERPARAMETERS")
print("=" * 70)

print(
    best_hyperparameters_df
)


# Independent threshold calibration

# Store threshold-level results, frozen operating thresholds, fitted development
# pipelines, and calibration probabilities.
cv_threshold_results = []
locked_thresholds = {}
final_model_objects = {}
calibration_curve_objects = {}


# Fit each selected model on the development period only.
for model_name, model in models.items():
    pipe = Pipeline(
        steps=[
            (
                "preprocessor",
                clone(preprocessor)
            ),
            (
                "model",
                clone(model)
            )
        ]
    )

    pipe.fit(
        X_model_train,
        y_model_train
    )

    # Generate probabilities for the independent pre-holdout calibration set.
    y_calibration_probability = get_probabilities(
        pipe,
        X_calibration,
        model_name
    )

    if y_calibration.nunique() < 2:
        raise ValueError(
            f"Calibration set for {model_name} "
            "does not contain both classes."
        )

    threshold_rows = []

    # Evaluate every candidate threshold using calibration observations only.
    for threshold in THRESHOLD_GRID:
        metrics = evaluate_at_threshold(
            y_true=y_calibration,
            y_prob=y_calibration_probability,
            threshold=threshold
        )

        row = {
            "Model": model_name,
            "Threshold": float(threshold),
            "BestParams": params_to_string(
                best_hyperparameters[
                    model_name
                ]
            ),
            **{
                metric: metrics[metric]
                for metric in [
                    "Accuracy",
                    "Precision",
                    "Recall",
                    "F1",
                    "F2",
                    "ROC_AUC",
                    "AveragePrecision"
                ]
            }
        }

        threshold_rows.append(row)
        cv_threshold_results.append(row)

    threshold_summary_df = pd.DataFrame(
        threshold_rows
    )

    # Select the threshold with the highest calibration F2.
    # Ties should favor recall and then precision.
    (
        best_threshold,
        ranked_thresholds
    ) = select_best_threshold(
        threshold_summary_df
    )

    # Freeze the threshold before final holdout evaluation.
    locked_thresholds[
        model_name
    ] = float(best_threshold)

    # Retain the development-fitted pipeline.
    # It is not refitted on calibration or holdout observations.
    final_model_objects[
        model_name
    ] = pipe

    # Preserve calibration predictions for later curve plotting.
    calibration_curve_objects[
        model_name
    ] = {
        "y_true": y_calibration.copy(),
        "y_prob": (
            y_calibration_probability.copy()
        ),
        "target_timestamp": (
            calibration_df[
                "target_timestamp"
            ].copy()
        )
    }

    ranked_thresholds.to_csv(
        os.path.join(
            OUTPUT_DIR,
            (
                f"{model_name}_"
                "calibration_threshold_summary.csv"
            )
        ),
        index=False
    )

    print(
        f"{model_name}: independently calibrated "
        f"threshold={best_threshold:.2f}"
    )


# Create combined threshold-calibration result tables.
cv_threshold_results_df = pd.DataFrame(
    cv_threshold_results
)


# Report default-threshold CV performance for the selected configuration of
# each model, not the average across all rejected candidates.
cv_summary_050 = (
    best_hyperparameter_results_df
    .set_index("Model")[
        [
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "ROC_AUC",
            "AveragePrecision"
        ]
    ]
    .sort_index()
)


# Create a compact table containing the frozen calibration thresholds.
locked_thresholds_df = (
    pd.DataFrame({
        "Model": list(
            locked_thresholds.keys()
        ),
        "LockedThreshold": list(
            locked_thresholds.values()
        )
    })
    .sort_values("Model")
    .reset_index(drop=True)
)


print(
    "\nLOCKED THRESHOLDS FROM "
    "INDEPENDENT CALIBRATION"
)

print(
    locked_thresholds_df
)

print(
    "\nCROSS-VALIDATION PERFORMANCE "
    "AT THE DEFAULT 0.50 THRESHOLD"
)

print(
    "USING SELECTED HYPERPARAMETERS"
)

print("=" * 70)

print(
    cv_summary_050.round(4)
)

In [ ]:
# Statistical comparison of the selected ML model configurations.
# Each model is evaluated on the same development TimeSeriesSplit folds.
# Average Precision is used because exceedance events are highly imbalanced
# and it was the primary hyperparameter-selection metric.
#
# Calibration and final holdout observations are not used in this analysis.

ML_STATISTICAL_ALPHA = 0.05

ml_selected_fold_results = []


# Re-evaluate only the selected hyperparameter configuration for each model.
for model_name, base_model in base_models.items():

    selected_params = best_hyperparameters[model_name]

    print(
        f"Collecting paired CV results for {model_name}: "
        f"{params_to_string(selected_params)}"
    )


    # Use the same chronological folds for every model so that fold results
    # form valid paired observations for the Friedman test.
    for fold, (train_indices, validation_indices) in enumerate(
        tscv.split(X_model_train),
        start=1
    ):

        X_fold_train = X_model_train.iloc[
            train_indices
        ]

        y_fold_train = y_model_train.iloc[
            train_indices
        ]

        X_fold_validation = X_model_train.iloc[
            validation_indices
        ]

        y_fold_validation = y_model_train.iloc[
            validation_indices
        ]


        # Confirm that the forecast outcomes in the training fold occur before
        # the validation forecast origins.
        fold_train_target_times = (
            model_train_df
            .iloc[train_indices]["target_timestamp"]
        )

        fold_validation_origins = (
            model_train_df
            .iloc[validation_indices]
            .index
        )

        assert (
            fold_train_target_times.max()
            < fold_validation_origins.min()
        ), (
            f"Temporal overlap detected for {model_name}, "
            f"fold {fold}."
        )


        # Both classes are required to calculate classification metrics.
        if (
            y_fold_train.nunique() < 2
            or y_fold_validation.nunique() < 2
        ):
            raise ValueError(
                f"Fold {fold} for {model_name} does not "
                "contain both outcome classes."
            )


        # Fit preprocessing and the selected classifier using only the
        # training portion of the current fold.
        selected_model = (
            clone(base_model)
            .set_params(**selected_params)
        )

        fold_pipeline = Pipeline(steps=[
            (
                "preprocessor",
                clone(preprocessor)
            ),
            (
                "model",
                selected_model
            )
        ])

        fold_pipeline.fit(
            X_fold_train,
            y_fold_train
        )


        # Generate validation probabilities for threshold-independent metrics.
        validation_probabilities = get_probabilities(
            fold_pipeline,
            X_fold_validation,
            model_name
        )

        fold_average_precision = average_precision_score(
            y_fold_validation,
            validation_probabilities
        )

        fold_roc_auc = roc_auc_score(
            y_fold_validation,
            validation_probabilities
        )


        # Store one paired observation for each model and fold.
        ml_selected_fold_results.append({
            "Model": model_name,
            "Fold": fold,
            "AveragePrecision": fold_average_precision,
            "ROC_AUC": fold_roc_auc,
            "Parameters": params_to_string(
                selected_params
            )
        })


# Combine and save the actual fold-level results.
ml_selected_fold_results_df = pd.DataFrame(
    ml_selected_fold_results
)

ml_selected_fold_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ml_selected_models_fold_results.csv"
    ),
    index=False
)


# Verify that every model has exactly one result for every CV fold.
expected_models = sorted(
    base_models.keys()
)

expected_folds = list(
    range(1, N_SPLITS + 1)
)

fold_counts = (
    ml_selected_fold_results_df
    .groupby("Model")["Fold"]
    .nunique()
)

assert set(fold_counts.index) == set(expected_models), (
    "Not every ML model is represented in the fold results."
)

assert (
    fold_counts == N_SPLITS
).all(), (
    "Not every ML model has results for all CV folds."
)


# Reshape the Average Precision values so rows are paired CV folds and
# columns are model families.
ml_ap_by_fold = (
    ml_selected_fold_results_df
    .pivot(
        index="Fold",
        columns="Model",
        values="AveragePrecision"
    )
    .reindex(
        index=expected_folds,
        columns=expected_models
    )
)

if ml_ap_by_fold.isna().any().any():
    raise ValueError(
        "Missing paired fold results prevent the Friedman test."
    )


# Run the global Friedman test.
# Null hypothesis: all model families have the same performance distribution.
friedman_statistic, friedman_p_value = friedmanchisquare(
    *[
        ml_ap_by_fold[model_name].to_numpy()
        for model_name in expected_models
    ]
)

friedman_results_df = pd.DataFrame([{
    "Metric": "AveragePrecision",
    "NumberOfModels": len(expected_models),
    "NumberOfFolds": len(ml_ap_by_fold),
    "FriedmanStatistic": friedman_statistic,
    "PValue": friedman_p_value,
    "Alpha": ML_STATISTICAL_ALPHA,
    "Significant": (
        friedman_p_value < ML_STATISTICAL_ALPHA
    )
}])

friedman_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ml_friedman_test_average_precision.csv"
    ),
    index=False
)


print("\n" + "=" * 70)
print("FRIEDMAN TEST: ML AVERAGE PRECISION")
print("=" * 70)

print(friedman_results_df.round(6))


# Calculate descriptive fold-level summaries for interpretation.
ml_fold_summary_df = (
    ml_selected_fold_results_df
    .groupby("Model", as_index=False)
    .agg(
        MeanAveragePrecision=(
            "AveragePrecision",
            "mean"
        ),
        MedianAveragePrecision=(
            "AveragePrecision",
            "median"
        ),
        StdAveragePrecision=(
            "AveragePrecision",
            "std"
        ),
        MeanROCAUC=(
            "ROC_AUC",
            "mean"
        )
    )
    .sort_values(
        "MeanAveragePrecision",
        ascending=False
    )
)

ml_fold_summary_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ml_selected_models_fold_summary.csv"
    ),
    index=False
)

print("\nPaired CV summary:")
print(
    ml_fold_summary_df.round(6)
)


# Run paired Wilcoxon post-hoc comparisons only when the global Friedman
# test rejects the equal-performance null hypothesis.
ml_posthoc_results_df = pd.DataFrame()

if friedman_p_value < ML_STATISTICAL_ALPHA:

    posthoc_rows = []

    for model_a, model_b in combinations(
        expected_models,
        2
    ):

        scores_a = ml_ap_by_fold[
            model_a
        ].to_numpy()

        scores_b = ml_ap_by_fold[
            model_b
        ].to_numpy()

        score_differences = (
            scores_a - scores_b
        )


        # If all paired differences are zero, the models are identical on
        # every fold and the Wilcoxon p-value is defined here as 1.
        if np.allclose(
            score_differences,
            0
        ):
            wilcoxon_statistic = 0.0
            raw_p_value = 1.0

        else:
            wilcoxon_result = wilcoxon(
                scores_a,
                scores_b,
                alternative="two-sided",
                zero_method="wilcox",
                method="auto"
            )

            wilcoxon_statistic = (
                wilcoxon_result.statistic
            )

            raw_p_value = (
                wilcoxon_result.pvalue
            )


        posthoc_rows.append({
            "ModelA": model_a,
            "ModelB": model_b,
            "MeanAP_ModelA": scores_a.mean(),
            "MeanAP_ModelB": scores_b.mean(),
            "MeanDifference_A_minus_B": (
                score_differences.mean()
            ),
            "MedianDifference_A_minus_B": (
                np.median(score_differences)
            ),
            "WilcoxonStatistic": wilcoxon_statistic,
            "RawPValue": raw_p_value
        })


    ml_posthoc_results_df = pd.DataFrame(
        posthoc_rows
    )


    # Apply Holm correction to control family-wise error across all pairwise
    # model comparisons.
    reject_null, adjusted_p_values, _, _ = multipletests(
        ml_posthoc_results_df["RawPValue"],
        alpha=ML_STATISTICAL_ALPHA,
        method="holm"
    )

    ml_posthoc_results_df[
        "HolmAdjustedPValue"
    ] = adjusted_p_values

    ml_posthoc_results_df[
        "SignificantAfterHolm"
    ] = reject_null


    ml_posthoc_results_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "ml_wilcoxon_posthoc_holm.csv"
        ),
        index=False
    )


    print("\n" + "=" * 70)
    print("WILCOXON POST-HOC TESTS WITH HOLM CORRECTION")
    print("=" * 70)

    print(
        ml_posthoc_results_df.round(6)
    )

else:
    print(
        "\nThe Friedman test was not statistically significant. "
        "Pairwise Wilcoxon tests were not performed."
    )

In [ ]:
# Evaluate the untouched final holdout using the fitted development models
# and thresholds locked during the independent calibration period.
#
# No fitting, threshold adjustment, hyperparameter tuning, or model selection
# is performed using the final holdout.

test_results = []
holdout_curve_objects = {}

print("\n" + "=" * 70)
print("FINAL HOLDOUT TEST RESULTS (LOCKED THRESHOLDS)")
print("=" * 70)


# Evaluate each model retained after development and threshold calibration.
for model_name in models:

    # Retrieve the pipeline fitted only on the development period.
    # Do not refit it on X_train because X_train includes calibration data.
    fitted_pipeline = final_model_objects[model_name]

    # Generate holdout probabilities without modifying the fitted model.
    y_test_prob = get_probabilities(
        fitted_pipeline,
        X_test,
        model_name
    )

    # Apply the threshold selected exclusively from calibration data.
    locked_threshold = locked_thresholds[model_name]

    metrics = evaluate_at_threshold(
        y_true=y_test,
        y_prob=y_test_prob,
        threshold=locked_threshold
    )

    # Store one final, untouched-holdout result per model.
    test_results.append({
        "Model": model_name,
        "LockedThreshold": locked_threshold,
        "Accuracy": metrics["Accuracy"],
        "Precision": metrics["Precision"],
        "Recall": metrics["Recall"],
        "F1": metrics["F1"],
        "F2": metrics["F2"],
        "ROC_AUC": metrics["ROC_AUC"],
        "AveragePrecision": metrics["AveragePrecision"],
        "TN": metrics["TN"],
        "FP": metrics["FP"],
        "FN": metrics["FN"],
        "TP": metrics["TP"]
    })

    # Calculate holdout precision-recall and ROC curves for reporting only.
    # These curves must not be used to retune thresholds or select models.
    precision_curve_values, recall_curve_values, pr_thresholds = (
        precision_recall_curve(
            y_test,
            y_test_prob
        )
    )

    fpr, tpr, roc_thresholds = roc_curve(
        y_test,
        y_test_prob
    )

    holdout_curve_objects[model_name] = {
        "y_true": y_test.to_numpy(),
        "y_prob": y_test_prob,
        "target_timestamps": test_df["target_timestamp"].to_numpy(),
        "precision_curve": precision_curve_values,
        "recall_curve": recall_curve_values,
        "pr_thresholds": pr_thresholds,
        "fpr": fpr,
        "tpr": tpr,
        "roc_thresholds": roc_thresholds,
        "average_precision": metrics["AveragePrecision"],
        "roc_auc": metrics["ROC_AUC"],
        "locked_threshold": locked_threshold
    }

    print(
        f"{model_name}: "
        f"threshold={locked_threshold:.2f}, "
        f"Precision={metrics['Precision']:.4f}, "
        f"Recall={metrics['Recall']:.4f}, "
        f"F1={metrics['F1']:.4f}, "
        f"F2={metrics['F2']:.4f}, "
        f"AP={metrics['AveragePrecision']:.4f}, "
        f"ROC_AUC={metrics['ROC_AUC']:.4f}, "
        f"TN={metrics['TN']}, "
        f"FP={metrics['FP']}, "
        f"FN={metrics['FN']}, "
        f"TP={metrics['TP']}"
    )


# Combine the untouched-holdout results without ranking them.
test_results_df = pd.DataFrame(test_results)


# Select the primary model using only cross-validation performance.
# For each model family, retrieve the row corresponding to its previously
# selected best hyperparameter configuration.
selected_cv_rows = []

for model_name, best_params in best_hyperparameters.items():

    best_params_string = params_to_string(
        best_params
    )

    matching_rows = hyperparameter_tuning_results_df.loc[
        (
            hyperparameter_tuning_results_df["Model"]
            == model_name
        )
        & (
            hyperparameter_tuning_results_df["Params"]
            == best_params_string
        )
    ]

    if len(matching_rows) != 1:
        raise ValueError(
            f"Expected exactly one selected CV result for "
            f"{model_name}, but found {len(matching_rows)}."
        )

    selected_cv_rows.append(
        matching_rows.iloc[0].to_dict()
    )


# This table contains one CV-selected configuration per model family.
selected_cv_model_summary = pd.DataFrame(
    selected_cv_rows
)


# Preselect the primary model using CV performance only.
# Average Precision is primary; the remaining metrics break ties.
selected_model_name = (
    selected_cv_model_summary
    .sort_values(
        by=[
            "AveragePrecision",
            "Recall",
            "F1",
            "Precision",
            "ROC_AUC"
        ],
        ascending=[
            False,
            False,
            False,
            False,
            False
        ]
    )
    .iloc[0]["Model"]
)


# Mark the CV-preselected primary model in the holdout results.
# Holdout performance is not used to choose the model.
test_results_df["PreselectedPrimaryModel"] = (
    test_results_df["Model"]
    == selected_model_name
)


print("\n" + "=" * 70)
print("CV-PRESELECTED PRIMARY MODEL")
print("=" * 70)

print(
    selected_cv_model_summary[
        [
            "Model",
            "Params",
            "AveragePrecision",
            "Recall",
            "F1",
            "Precision",
            "ROC_AUC"
        ]
    ]
    .sort_values(
        "AveragePrecision",
        ascending=False
    )
    .round(4)
)

print(
    "\nPrimary model selected using CV:",
    selected_model_name
)


print("\n" + "=" * 70)
print("FINAL HOLDOUT RESULTS")
print("=" * 70)

print(
    "The holdout results below are reported for final evaluation only."
)

print(
    test_results_df.round(4)
)

In [ ]:
# Plot final-holdout precision-recall curves for descriptive comparison.
# The primary model was selected using development cross-validation before
# holdout evaluation. These curves do not change or repeat model selection.

fig, ax = plt.subplots(figsize=(10, 7))


# The prevalence baseline represents the precision expected from a classifier
# making alerts without useful discrimination.
holdout_prevalence = y_test.mean()

ax.axhline(
    y=holdout_prevalence,
    linestyle="--",
    color="black",
    label=(
        f"Prevalence baseline = "
        f"{holdout_prevalence:.3f}"
    )
)


# Plot one precision-recall curve for each fitted classifier.
# Average Precision summarizes performance across all possible thresholds.
for model_name, curve_data in holdout_curve_objects.items():

    ax.plot(
        curve_data["recall_curve"],
        curve_data["precision_curve"],
        label=(
            f"{model_name} "
            f"(AP={curve_data['average_precision']:.3f})"
        )
    )


# Configure and save the figure.
ax.set_title(
    "Final Holdout Precision-Recall Curves — Descriptive Comparison"
)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()

fig.savefig(
    os.path.join(
        OUTPUT_DIR,
        "holdout_precision_recall_curves.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# Plot final-holdout ROC curves as a secondary descriptive comparison.
# The primary model was already selected using development cross-validation.
# These curves do not change the model selection or locked thresholds.

fig, ax = plt.subplots(figsize=(10, 7))


# Plot one ROC curve for each classifier.
# Each curve shows the true-positive rate against the false-positive rate
# across all possible classification thresholds.
for model_name, curve_data in holdout_curve_objects.items():

    ax.plot(
        curve_data["fpr"],
        curve_data["tpr"],
        label=(
            f"{model_name} "
            f"(ROC AUC={curve_data['roc_auc']:.3f})"
        )
    )


# Add the performance expected from a classifier with no discrimination.
ax.plot(
    [0, 1],
    [0, 1],
    color="black",
    linestyle="--",
    label="Chance"
)


# Configure and save the figure.
ax.set_title(
    "Final Holdout ROC Curves — Descriptive Comparison"
)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()

fig.savefig(
    os.path.join(
        OUTPUT_DIR,
        "holdout_roc_curves.png"
    ),
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# Examine decision-threshold sensitivity using the independent calibration set.
# The final holdout is intentionally excluded from this analysis.
# This analysis describes how precision, recall, F1, and F2 change as the
# classification threshold changes.

threshold_sensitivity_rows = []


# Evaluate every candidate threshold for each tuned classifier.
for model_name, calibration_data in calibration_curve_objects.items():

    for threshold in THRESHOLD_GRID:

        metrics = evaluate_at_threshold(
            y_true=y_calibration,
            y_prob=calibration_data["y_prob"],
            threshold=threshold
        )

        threshold_sensitivity_rows.append({
            "Model": model_name,
            "Threshold": float(threshold),
            "Precision": metrics["Precision"],
            "Recall": metrics["Recall"],
            "F1": metrics["F1"],
            "F2": metrics["F2"]
        })


# Combine and save all calibration threshold results.
threshold_sensitivity_df = pd.DataFrame(
    threshold_sensitivity_rows
)

threshold_sensitivity_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "calibration_threshold_sensitivity.csv"
    ),
    index=False
)


# Plot the calibration sensitivity of the most relevant threshold-dependent
# metrics. F2 is emphasized because it is the threshold-selection objective.
for metric in ["Recall", "Precision", "F2"]:

    fig, ax = plt.subplots(figsize=(10, 7))

    for model_name in threshold_sensitivity_df["Model"].unique():

        model_results = threshold_sensitivity_df.loc[
            threshold_sensitivity_df["Model"] == model_name
        ].sort_values("Threshold")

        line = ax.plot(
            model_results["Threshold"],
            model_results[metric],
            label=model_name
        )[0]

        # Mark the threshold that was locked for this model using calibration
        # performance only.
        locked_threshold = locked_thresholds[model_name]

        locked_result = model_results.loc[
            np.isclose(
                model_results["Threshold"],
                locked_threshold
            )
        ]

        if not locked_result.empty:

            ax.scatter(
                locked_result["Threshold"].iloc[0],
                locked_result[metric].iloc[0],
                color=line.get_color(),
                edgecolor="black",
                s=80,
                zorder=3
            )


    # Configure and save the calibration sensitivity figure.
    ax.set_title(
        f"Calibration {metric} by Decision Threshold"
    )
    ax.set_xlabel("Decision Threshold")
    ax.set_ylabel(metric)
    ax.legend()
    ax.grid(alpha=0.3)

    fig.tight_layout()

    fig.savefig(
        os.path.join(
            OUTPUT_DIR,
            f"calibration_{metric.lower()}_by_threshold.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)


In [ ]:
# Extract impurity-based feature importance from the fitted Random Forest.
# The model was fitted on the development period only. This analysis does not
# use holdout outcomes and does not modify the trained model.

rf_importance_df = None


# Continue only if a fitted Random Forest pipeline is available.
if "RandomForest" in final_model_objects:

    rf_pipeline = final_model_objects["RandomForest"]

    # Retrieve the fitted classifier and preprocessing objects.
    rf_model = rf_pipeline.named_steps["model"]
    rf_preprocessor = rf_pipeline.named_steps["preprocessor"]

    # Retrieve feature names in the exact order received by the classifier.
    transformed_feature_names = (
        rf_preprocessor.get_feature_names_out()
    )

    # Remove the ColumnTransformer prefix, such as "num__", to make the
    # feature names easier to read in the table and figure.
    readable_feature_names = [
        feature_name.split("__", 1)[-1]
        for feature_name in transformed_feature_names
    ]

    # Match each transformed feature with its Random Forest importance.
    rf_importance_df = (
        pd.DataFrame({
            "Feature": readable_feature_names,
            "Importance": rf_model.feature_importances_
        })
        .sort_values(
            by="Importance",
            ascending=False
        )
        .reset_index(drop=True)
    )


    # Save the complete feature-importance table.
    rf_importance_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "random_forest_feature_importance.csv"
        ),
        index=False
    )


    # Display the 25 most important predictors.
    print("\n" + "=" * 70)
    print("TOP 25 RANDOM FOREST FEATURE IMPORTANCES")
    print("=" * 70)

    print(
        rf_importance_df
        .head(25)
        .round(6)
    )


    # Plot the 25 most important predictors.
    top_n = 25

    top_rf = (
        rf_importance_df
        .head(top_n)
        .sort_values(
            "Importance",
            ascending=True
        )
    )

    fig, ax = plt.subplots(figsize=(10, 8))

    ax.barh(
        top_rf["Feature"],
        top_rf["Importance"]
    )

    ax.set_title(
        "Random Forest Feature Importance (Top 25)"
    )
    ax.set_xlabel("Impurity-Based Importance")
    ax.set_ylabel("Feature")

    fig.tight_layout()

    fig.savefig(
        os.path.join(
            OUTPUT_DIR,
            "random_forest_feature_importance_top25.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

else:
    print(
        "A fitted Random Forest pipeline was not found. "
        "Feature importance was not calculated."
    )

In [ ]:
# Save all classification results, diagnostics, and reporting tables.
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# Save the averaged CV results for every hyperparameter candidate.
hyperparameter_tuning_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "development_cv_all_hyperparameter_candidates.csv"
    ),
    index=False
)


# Save the best hyperparameter configuration selected for each model family.
best_hyperparameters_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "best_hyperparameters.csv"
    ),
    index=False
)


# Save one selected CV result per model family.
selected_cv_model_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "development_cv_selected_model_summary.csv"
    ),
    index=False
)


# Save tuning failures only when at least one failure occurred.
if not hyperparameter_tuning_failures_df.empty:

    hyperparameter_tuning_failures_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "hyperparameter_tuning_failures.csv"
        ),
        index=False
    )


# Save threshold performance across the independent calibration period.
cv_threshold_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "calibration_threshold_results.csv"
    ),
    index=False
)


# Save the final threshold locked for each classifier.
locked_thresholds_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "locked_thresholds_from_calibration.csv"
    ),
    index=False
)


# Save the final untouched-holdout metrics.
test_results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "final_holdout_test_results_locked_thresholds.csv"
    ),
    index=False
)


# Save Random Forest impurity-based feature importance when available.
if rf_importance_df is not None:

    rf_importance_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "random_forest_feature_importance.csv"
        ),
        index=False
    )


# Save holdout curve coordinates and prediction-level results for each model.
for model_name, curve_data in holdout_curve_objects.items():

    # Precision-recall thresholds contain one fewer value than the precision
    # and recall arrays. Add a final missing value to align the table lengths.
    pr_threshold_values = np.append(
        curve_data["pr_thresholds"],
        np.nan
    )

    pr_df = pd.DataFrame({
        "Recall": curve_data["recall_curve"],
        "Precision": curve_data["precision_curve"],
        "Threshold": pr_threshold_values
    })

    pr_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"{model_name}_holdout_pr_curve.csv"
        ),
        index=False
    )


    # Save ROC coordinates and their corresponding thresholds.
    roc_df = pd.DataFrame({
        "FPR": curve_data["fpr"],
        "TPR": curve_data["tpr"],
        "Threshold": curve_data["roc_thresholds"]
    })

    roc_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"{model_name}_holdout_roc_curve.csv"
        ),
        index=False
    )


    # Apply the already locked threshold to create the final binary decisions.
    locked_threshold = curve_data["locked_threshold"]

    holdout_predictions_df = pd.DataFrame({
        "target_timestamp_utc": curve_data["target_timestamps"],
        "actual_exceedance": curve_data["y_true"].astype(int),
        "predicted_probability": curve_data["y_prob"],
        "locked_threshold": locked_threshold,
        "predicted_exceedance": (
            curve_data["y_prob"] >= locked_threshold
        ).astype(int)
    })

    holdout_predictions_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"{model_name}_final_holdout_predictions.csv"
        ),
        index=False
    )


# Report the saved outputs.
print("\n" + "=" * 70)
print("CLASSIFICATION FILES SAVED")
print("=" * 70)

print(
    f"Output folder: {OUTPUT_DIR}"
)

print("\nCore result files:")
print("- development_cv_all_hyperparameter_candidates.csv")
print("- development_cv_selected_model_summary.csv")
print("- best_hyperparameters.csv")

if not hyperparameter_tuning_failures_df.empty:
    print("- hyperparameter_tuning_failures.csv")

print("- calibration_threshold_results.csv")
print("- locked_thresholds_from_calibration.csv")
print("- final_holdout_test_results_locked_thresholds.csv")

print("\nFigures and supporting files:")
print("- holdout_precision_recall_curves.png")
print("- holdout_roc_curves.png")
print("- calibration_threshold_sensitivity.csv")
print("- calibration_recall_by_threshold.png")
print("- calibration_precision_by_threshold.png")
print("- calibration_f2_by_threshold.png")

if rf_importance_df is not None:
    print("- random_forest_feature_importance.csv")
    print("- random_forest_feature_importance_top25.png")

print("- Per-model final holdout prediction CSV files")
print("- Per-model holdout precision-recall curve CSV files")
print("- Per-model holdout ROC curve CSV files")
print("- Per-model hyperparameter tuning CSV files")
print("- Per-model calibration threshold summary CSV files")

In [ ]:
# Save the locked classical-ML configuration for future reuse
# This package preserves:
# - Selected hyperparameters for every classifier
# - Locked probability threshold for every classifier
# - Calibration performance at each locked threshold
# - Hyperparameter and threshold-selection rules
# - Exact ordered feature list
# - Candidate hyperparameter grids
# - Fitted preprocessing + model pipelines
#
# The final holdout is not used to make or change these decisions.


# Create a dedicated Google Drive folder inside the existing
# classification output directory.
LOCKED_ML_DIR = os.path.join(
    OUTPUT_DIR,
    "Locked_Configuration"
)

os.makedirs(
    LOCKED_ML_DIR,
    exist_ok=True
)


def make_json_safe(value):
    """
    Convert NumPy, pandas, timestamp, and Path values into types
    that can be written safely to JSON.
    """
    if isinstance(value, dict):
        return {
            str(key): make_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple, set, np.ndarray, pd.Index)
    ):
        return [
            make_json_safe(item)
            for item in value
        ]

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        if np.isnan(value):
            return None
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()

    if isinstance(value, Path):
        return str(value)

    return value

# Build one reusable record per classic ML model

locked_model_records = []

for model_name in sorted(best_hyperparameters):

    locked_threshold = float(
        locked_thresholds[model_name]
    )

    # Find the calibration result corresponding to the exact
    # threshold selected and locked for this model.
    locked_threshold_result = cv_threshold_results_df.loc[
        (
            cv_threshold_results_df["Model"]
            == model_name
        )
        &
        np.isclose(
            cv_threshold_results_df[
                "Threshold"
            ].astype(float),
            locked_threshold
        )
    ]

    if len(locked_threshold_result) != 1:
        raise ValueError(
            f"Expected exactly one calibration result for "
            f"{model_name} at threshold {locked_threshold}, "
            f"but found {len(locked_threshold_result)}."
        )

    locked_threshold_result = (
        locked_threshold_result.iloc[0]
    )

    # Retrieve the CV result corresponding to the selected
    # hyperparameter configuration.
    selected_cv_result = selected_cv_model_summary.loc[
        selected_cv_model_summary["Model"]
        == model_name
    ]

    if len(selected_cv_result) != 1:
        raise ValueError(
            f"Expected exactly one selected CV result for "
            f"{model_name}, but found {len(selected_cv_result)}."
        )

    selected_cv_result = selected_cv_result.iloc[0]

    model_record = {
        "model": model_name,

        # Locked model settings
        "locked_hyperparameters": (
            best_hyperparameters[model_name]
        ),
        "locked_probability_threshold": (
            locked_threshold
        ),

        # Hyperparameter-selection decision
        "hyperparameter_selection_metric": (
            "AveragePrecision"
        ),
        "selected_cv_average_precision": (
            selected_cv_result["AveragePrecision"]
        ),
        "selected_cv_recall_at_0_50": (
            selected_cv_result["Recall"]
        ),
        "selected_cv_precision_at_0_50": (
            selected_cv_result["Precision"]
        ),
        "selected_cv_f1_at_0_50": (
            selected_cv_result["F1"]
        ),
        "selected_cv_roc_auc": (
            selected_cv_result["ROC_AUC"]
        ),

        # Threshold-selection decision
        "threshold_selection_metric": (
            PRIMARY_THRESHOLD_OBJECTIVE
        ),
        "threshold_beta": float(
            THRESHOLD_BETA
        ),
        "calibration_f2_at_locked_threshold": (
            locked_threshold_result["F2"]
        ),
        "calibration_recall_at_locked_threshold": (
            locked_threshold_result["Recall"]
        ),
        "calibration_precision_at_locked_threshold": (
            locked_threshold_result["Precision"]
        ),
        "calibration_f1_at_locked_threshold": (
            locked_threshold_result["F1"]
        ),
        "calibration_average_precision": (
            locked_threshold_result["AveragePrecision"]
        ),

        # File containing the fitted preprocessing/model pipeline
        "fitted_pipeline_file": (
            f"{model_name}_locked_pipeline.joblib"
        )
    }

    locked_model_records.append(
        model_record
    )


# Create the complete locked-configuration package

locked_ml_configuration = {
    "schema_version": "1.0",

    "created_utc": (
        datetime.now(timezone.utc).isoformat()
    ),

    "purpose": (
        "Reusable Alexander Orr classical-ML configuration "
        "locked before final-holdout evaluation"
    ),

    "target_definition": {
        "source_variable": "totalchlorine",
        "positive_class_rule": (
            f"totalchlorine(t+{FORECAST_HORIZON}) "
            f"> {THRESHOLD_CHLORINE} mg/L"
        ),
        "regulatory_threshold_mg_L": float(
            THRESHOLD_CHLORINE
        ),
        "forecast_horizon_hours": int(
            FORECAST_HORIZON
        )
    },

    "data_and_validation": {
        "time_zone": "UTC",
        "frequency": "hourly",
        "holdout_start": str(
            HOLDOUT_START
        ),
        "calibration_hours": int(
            CALIBRATION_HOURS
        ),
        "time_series_cv_splits": int(
            N_SPLITS
        ),
        "cv_gap_hours": int(
            FORECAST_HORIZON
        ),
        "random_seed": int(
            SEED
        ),
        "holdout_used_for_selection": False
    },

    "hyperparameter_selection_rule": {
        "primary_metric": "AveragePrecision",
        "tie_breakers_in_order": [
            "Recall",
            "F1",
            "Precision",
            "ROC_AUC"
        ],
        "classification_threshold_during_cv": 0.50,
        "selection_data": (
            "Development-period TimeSeriesSplit only"
        )
    },

    "probability_threshold_selection_rule": {
        "primary_metric": (
            PRIMARY_THRESHOLD_OBJECTIVE
        ),
        "beta": float(
            THRESHOLD_BETA
        ),
        "tie_breakers_in_order": [
            "Recall",
            "Precision"
        ],
        "candidate_thresholds": (
            THRESHOLD_GRID.tolist()
        ),
        "selection_data": (
            "Independent pre-holdout calibration period only"
        )
    },

    "primary_model_selection": {
        "selected_model": (
            selected_model_name
        ),
        "primary_metric": (
            "AveragePrecision"
        ),
        "selection_data": (
            "Development cross-validation only"
        ),
        "holdout_used_for_model_selection": False
    },

    "feature_schema": {
        "feature_count": len(
            feature_cols
        ),
        "ordered_feature_names": list(
            feature_cols
        ),
        "base_features": list(
            base_features
        ),
        "engineered_features": list(
            engineered_features
        ),
        "lag_source_columns": list(
            lag_cols
        ),
        "maximum_lag_hours": int(
            N_LAGS
        )
    },

    # This preserves not only the winners but also the range
    # of hyperparameters that was originally explored.
    "candidate_hyperparameter_grids": (
        param_grids
    ),

    "models": locked_model_records,

    "software_versions": {
        "python": (
            __import__("sys").version.split()[0]
        ),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__
    }
}



# Save the authoritative JSON configuration

LOCKED_CONFIGURATION_PATH = os.path.join(
    LOCKED_ML_DIR,
    "locked_ml_configuration.json"
)

with open(
    LOCKED_CONFIGURATION_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        make_json_safe(
            locked_ml_configuration
        ),
        file,
        indent=4
    )

# Save a compact, human-readable CSV summary

locked_model_summary_df = pd.json_normalize(
    make_json_safe(
        locked_model_records
    ),
    sep="."
)

locked_model_summary_df.to_csv(
    os.path.join(
        LOCKED_ML_DIR,
        "locked_model_summary.csv"
    ),
    index=False
)


# Save the exact feature names and required feature order

with open(
    os.path.join(
        LOCKED_ML_DIR,
        "feature_schema.json"
    ),
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        make_json_safe(
            locked_ml_configuration[
                "feature_schema"
            ]
        ),
        file,
        indent=4
    )


# Save the exact fitted pipelines
#
# Each joblib file contains:
# - The fitted preprocessing operations
# - Scaling and transformations
# - The fitted classifier
#
# These pipelines were fitted before final-holdout evaluation.

for model_name, fitted_pipeline in (
    final_model_objects.items()
):

    pipeline_path = os.path.join(
        LOCKED_ML_DIR,
        f"{model_name}_locked_pipeline.joblib"
    )

    joblib.dump(
        fitted_pipeline,
        pipeline_path
    )

# Confirm saved files

print("\n" + "=" * 70)
print("LOCKED CLASSICAL-ML REUSE PACKAGE SAVED")
print("=" * 70)

print(
    f"Google Drive folder: {LOCKED_ML_DIR}"
)

print("\nCore reuse files:")
print("- locked_ml_configuration.json")
print("- locked_model_summary.csv")
print("- feature_schema.json")

print("\nFitted pipelines:")

for model_name in final_model_objects:
    print(
        f"- {model_name}_locked_pipeline.joblib"
    )

#Commit Current Notebook to Github (LFS due to file size)

In [ ]:
import os
import shutil
import stat
import subprocess
from pathlib import Path

from google.colab import drive, userdata



# SETTINGS

REPO_OWNER = "angelicadatascience"
REPO_NAME = "watertreatment"
REPO_BRANCH = "main"

REPO_URL = (
    f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

REPO_PATH = Path(
    f"/content/{REPO_NAME}"
)



# MOUNT GOOGLE DRIVE

drive.mount(
    "/content/drive",
    force_remount=False
)


# READ GITHUB TOKEN FROM COLAB SECRETS

GITHUB_TOKEN = userdata.get("github_token")

if not GITHUB_TOKEN:
    raise ValueError(
        "The Colab Secret named 'github_token' was not found. "
        "Open the key icon in Colab, add the GitHub token using "
        "the exact name github_token, and enable notebook access."
    )



# CONFIGURE SECURE NONINTERACTIVE AUTHENTICATION

askpass_path = Path("/tmp/git_askpass.sh")

askpass_path.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) echo "$GITHUB_USERNAME" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
""",
    encoding="utf-8"
)

askpass_path.chmod(
    askpass_path.stat().st_mode | stat.S_IEXEC
)

git_env = os.environ.copy()
git_env["GIT_ASKPASS"] = str(askpass_path)
git_env["GIT_TERMINAL_PROMPT"] = "0"
git_env["GITHUB_USERNAME"] = REPO_OWNER
git_env["GITHUB_TOKEN"] = GITHUB_TOKEN



# CLONE OR UPDATE THE REPOSITORY

git_folder = REPO_PATH / ".git"

if git_folder.is_dir():
    print("Existing repository found. Updating it...")

    subprocess.run(
        [
            "git",
            "-C",
            str(REPO_PATH),
            "fetch",
            "origin"
        ],
        env=git_env,
        check=True
    )

    subprocess.run(
        [
            "git",
            "-C",
            str(REPO_PATH),
            "pull",
            "--ff-only",
            "origin",
            REPO_BRANCH
        ],
        env=git_env,
        check=True
    )

else:
    # Remove only an incomplete clone at this exact path.
    if REPO_PATH.exists():
        print(
            "Removing incomplete repository folder:",
            REPO_PATH
        )
        shutil.rmtree(REPO_PATH)

    print("Cloning repository...")

    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            REPO_BRANCH,
            REPO_URL,
            str(REPO_PATH)
        ],
        env=git_env,
        check=True
    )



# CONFIRM RESULT

branch_result = subprocess.run(
    [
        "git",
        "-C",
        str(REPO_PATH),
        "branch",
        "--show-current"
    ],
    text=True,
    capture_output=True,
    check=True
)

commit_result = subprocess.run(
    [
        "git",
        "-C",
        str(REPO_PATH),
        "log",
        "-1",
        "--oneline",
        "--decorate"
    ],
    text=True,
    capture_output=True,
    check=True
)

print("\nRepository ready:", REPO_PATH)
print("Current branch:", branch_result.stdout.strip())
print("Latest commit:", commit_result.stdout.strip())

In [ ]:
import shutil
import subprocess
from pathlib import Path


REPO_PATH = Path("/content/watertreatment")

NOTEBOOK_NAME = (
    "AlexanderOrr_ResChlorine_Official.ipynb"
)

DRIVE_ROOT = Path("/content/drive/MyDrive")

REPO_NOTEBOOK = (
    REPO_PATH / NOTEBOOK_NAME
)


def find_latest_drive_notebook(
    notebook_name,
    drive_root
):
    matches = list(
        drive_root.rglob(notebook_name)
    )

    matches = [
        path
        for path in matches
        if path.is_file()
    ]

    if not matches:
        raise FileNotFoundError(
            f"Could not find {notebook_name} "
            f"inside {drive_root}"
        )

    matches.sort(
        key=lambda path: path.stat().st_mtime,
        reverse=True
    )

    print(
        f"Found {len(matches)} matching "
        "notebook(s):"
    )

    for index, path in enumerate(
        matches,
        start=1
    ):
        size = (
            path.stat().st_size
            / 1024**2
        )

        print(
            f"{index}. {path} "
            f"({size:.2f} MiB)"
        )

    return matches[0]


DRIVE_NOTEBOOK = (
    find_latest_drive_notebook(
        NOTEBOOK_NAME,
        DRIVE_ROOT
    )
)

shutil.copy2(
    DRIVE_NOTEBOOK,
    REPO_NOTEBOOK
)

print("\nSelected Drive notebook:")
print(DRIVE_NOTEBOOK)

print("\nUpdated repository notebook:")
print(REPO_NOTEBOOK)

print(
    "\nCopied size:",
    f"{REPO_NOTEBOOK.stat().st_size / 1024**2:.2f} MiB"
)


status_result = subprocess.run(
    [
        "git",
        "-C",
        str(REPO_PATH),
        "status",
        "--short",
        "--",
        NOTEBOOK_NAME
    ],
    text=True,
    capture_output=True,
    check=True
)

print("\nGit status:")

if status_result.stdout.strip():
    print(status_result.stdout)
else:
    print(
        "No change detected. The Drive and "
        "repository copies may already match."
    )

In [ ]:
import os
import stat
import subprocess
from pathlib import Path

from google.colab import userdata


REPO_PATH = Path("/content/watertreatment")

NOTEBOOK_NAME = (
    "AlexanderOrr_ResChlorine_Official.ipynb"
)

GITHUB_USERNAME = "angelicadatascience"

GITHUB_TOKEN = userdata.get("github_token")

if not GITHUB_TOKEN:
    raise ValueError(
        "The Colab Secret named github_token "
        "was not found."
    )


askpass_path = Path("/tmp/git_askpass.sh")

askpass_path.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) echo "$GITHUB_USERNAME" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
""",
    encoding="utf-8"
)

askpass_path.chmod(
    askpass_path.stat().st_mode
    | stat.S_IEXEC
)

git_env = os.environ.copy()

git_env["GIT_ASKPASS"] = str(
    askpass_path
)

git_env["GIT_TERMINAL_PROMPT"] = "0"

git_env["GITHUB_USERNAME"] = (
    GITHUB_USERNAME
)

git_env["GITHUB_TOKEN"] = (
    GITHUB_TOKEN
)


subprocess.run(
    [
        "git",
        "-C",
        str(REPO_PATH),
        "config",
        "user.name",
        GITHUB_USERNAME
    ],
    check=True
)

subprocess.run(
    [
        "git",
        "-C",
        str(REPO_PATH),
        "config",
        "user.email",
        (
            "angelicadatascience"
            "@users.noreply.github.com"
        )
    ],
    check=True
)


lfs_result = subprocess.run(
    [
        "git",
        "-C",
        str(REPO_PATH),
        "lfs",
        "version"
    ],
    text=True,
    capture_output=True
)

if lfs_result.returncode != 0:
    raise RuntimeError(
        "Git LFS is not available in this runtime."
    )

print(lfs_result.stdout.strip())


subprocess.run(
    [
        "git",
        "-C",
        str(REPO_PATH),
        "lfs",
        "track",
        NOTEBOOK_NAME
    ],
    check=True
)


subprocess.run(
    [
        "git",
        "-C",
        str(REPO_PATH),
        "add",
        ".gitattributes",
        NOTEBOOK_NAME
    ],
    check=True
)


staged_result = subprocess.run(
    [
        "git",
        "-C",
        str(REPO_PATH),
        "diff",
        "--cached",
        "--quiet"
    ]
)

if staged_result.returncode == 0:
    print(
        "Nothing new to commit. "
        "The repository is already current."
    )

elif staged_result.returncode == 1:
    commit_result = subprocess.run(
        [
            "git",
            "-C",
            str(REPO_PATH),
            "commit",
            "-m",
            "Update Alexander Orr official notebook"
        ],
        text=True,
        capture_output=True
    )

    print(commit_result.stdout)
    print(commit_result.stderr)

    if commit_result.returncode != 0:
        raise RuntimeError(
            "The Git commit failed."
        )

    push_result = subprocess.run(
        [
            "git",
            "-C",
            str(REPO_PATH),
            "push",
            "origin",
            "main"
        ],
        env=git_env,
        text=True,
        capture_output=True
    )

    print(push_result.stdout)
    print(push_result.stderr)

    if push_result.returncode != 0:
        raise RuntimeError(
            "The GitHub push failed."
        )

    print("Notebook committed and pushed successfully.")

else:
    raise RuntimeError(
        "Git could not inspect the staged changes."
    )


subprocess.run(
    [
        "git",
        "-C",
        str(REPO_PATH),
        "log",
        "-1",
        "--oneline",
        "--decorate"
    ],
    check=True
)

subprocess.run(
    [
        "git",
        "-C",
        str(REPO_PATH),
        "status",
        "--short"
    ],
    check=True
)

##Checking Google Drive Output Folder Size


In [ ]:
from pathlib import Path

OUTPUTS = Path("/content/drive/MyDrive/Forecasting_AlexanderOrr")

total_size = sum(
    f.stat().st_size
    for f in OUTPUTS.rglob("*")
    if f.is_file()
)

print(f"Total size: {total_size / 1024**3:.2f} GB")

#Copy Model Outputs from Google Drive Folder into GitHub


In [ ]:
import shutil
from pathlib import Path

SOURCE = Path("/content/drive/MyDrive/Forecasting_AlexanderOrr")
DEST = Path("/content/watertreatment/outputs/Forecasting_AlexanderOrr")

DEST.parent.mkdir(parents=True, exist_ok=True)

shutil.copytree(
    SOURCE,
    DEST,
    dirs_exist_ok=True
)

print("Outputs copied successfully.")


In [ ]:
#Verification
!find /content/watertreatment/outputs | head -30

In [ ]:
!git -C /content/watertreatment add outputs

In [ ]:
!git -C /content/watertreatment status

In [ ]:
#Commit
import datetime

timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

!git -C /content/watertreatment commit -m "Add complete dissertation outputs - {timestamp}"

In [ ]:
#Push
import subprocess

subprocess.run(
    [
        "git",
        "-C",
        "/content/watertreatment",
        "push",
        "origin",
        "main",
    ],
    env=git_env,
    check=True,
)

print("Outputs pushed successfully.")

#Uploading datasets to Github

In [ ]:
!pip -q install gdown

import shutil
import tempfile
from pathlib import Path

import gdown

REPO_PATH = Path("/content/watertreatment").resolve()
DATA_PATH = (REPO_PATH / "data").resolve()

# Exact Google Drive file IDs from your shared folder.
DATASETS = {
    "alexanderorr.xlsx":
        "1Sk-sZCbEEAOHMWivwzM8n3LEhlxjAKo-",

    "covadonga_hourly_data.xlsx":
        "1WUMC4tWsG_05mf9uAyUgKD1FTP367uFe",

    "losfiltros_hourly_data_caguas.xlsx":
        "1rlvgEONq9THUh0H5S2IM7-JI1HhN9Xhm",
}

# Safety checks
if not REPO_PATH.is_dir():
    raise FileNotFoundError(
        f"Repository not found: {REPO_PATH}"
    )

if not (REPO_PATH / ".git").exists():
    raise RuntimeError(
        f"This is not the expected Git repository: {REPO_PATH}"
    )

if DATA_PATH != REPO_PATH / "data":
    raise RuntimeError(
        f"Unexpected data-folder path: {DATA_PATH}"
    )

# Download into a temporary folder first.
temporary_directory = Path(
    tempfile.mkdtemp(prefix="watertreatment_datasets_")
)

print("Downloading and validating datasets...")

try:
    for filename, file_id in DATASETS.items():
        temporary_file = temporary_directory / filename

        downloaded = gdown.download(
            id=file_id,
            output=str(temporary_file),
            quiet=False,
        )

        if downloaded is None or not temporary_file.is_file():
            raise RuntimeError(
                f"Download failed: {filename}"
            )

        if temporary_file.stat().st_size == 0:
            raise RuntimeError(
                f"Downloaded file is empty: {filename}"
            )

        # XLSX files are ZIP-based and should begin with PK.
        with temporary_file.open("rb") as file:
            signature = file.read(2)

        if signature != b"PK":
            raise RuntimeError(
                f"{filename} does not appear to be a valid XLSX file."
            )

        print(
            f"Validated: {filename} "
            f"({temporary_file.stat().st_size / 1024**2:.2f} MiB)"
        )

    downloaded_names = {
        path.name
        for path in temporary_directory.iterdir()
        if path.is_file()
    }

    if downloaded_names != set(DATASETS):
        raise RuntimeError(
            "The downloaded file list does not match the expected datasets."
        )

    # All three downloads passed validation.
    # Now replace only the contents of the repository's data folder.
    DATA_PATH.mkdir(parents=True, exist_ok=True)

    print("\nRemoving old data-folder contents:")

    for existing_item in DATA_PATH.iterdir():
        print(f"Removing: {existing_item.name}")

        if existing_item.is_dir() and not existing_item.is_symlink():
            shutil.rmtree(existing_item)
        else:
            existing_item.unlink()

    # Copy the three validated files.
    for filename in DATASETS:
        shutil.copy2(
            temporary_directory / filename,
            DATA_PATH / filename,
        )

finally:
    shutil.rmtree(
        temporary_directory,
        ignore_errors=True,
    )

print("\nFinal repository data folder:")

for path in sorted(DATA_PATH.iterdir()):
    print(
        f"- {path.name}: "
        f"{path.stat().st_size / 1024**2:.2f} MiB"
    )

In [ ]:
import subprocess
from pathlib import Path

REPO_PATH = Path("/content/watertreatment")
DATA_PATH = REPO_PATH / "data"

expected_files = {
    "alexanderorr.xlsx",
    "covadonga_hourly_data.xlsx",
    "losfiltros_hourly_data_caguas.xlsx",
}

actual_files = {
    path.name
    for path in DATA_PATH.iterdir()
    if path.is_file()
}

if actual_files != expected_files:
    raise RuntimeError(
        "The data folder does not contain exactly the three expected files.\n"
        f"Expected: {sorted(expected_files)}\n"
        f"Found: {sorted(actual_files)}"
    )

if "git_env" not in globals():
    raise RuntimeError(
        "git_env is not defined. Run your GitHub authentication cell first."
    )

# Ensure the Git LFS hook is correctly installed.
subprocess.run(
    [
        "git", "-C", str(REPO_PATH),
        "lfs", "update", "--force",
    ],
    check=True,
)

# Track Excel datasets using Git LFS.
subprocess.run(
    [
        "git", "-C", str(REPO_PATH),
        "lfs", "track", "data/*.xlsx",
    ],
    check=True,
)

# Configure the repository-specific commit identity.
subprocess.run(
    [
        "git", "-C", str(REPO_PATH),
        "config", "user.name",
        "angelicadatascience",
    ],
    check=True,
)

subprocess.run(
    [
        "git", "-C", str(REPO_PATH),
        "config", "user.email",
        "angelicadatascience@users.noreply.github.com",
    ],
    check=True,
)

# Stage .gitattributes plus all additions and deletions in data.
subprocess.run(
    [
        "git", "-C", str(REPO_PATH),
        "add", ".gitattributes",
    ],
    check=True,
)

subprocess.run(
    [
        "git", "-C", str(REPO_PATH),
        "add", "-A", "--", "data",
    ],
    check=True,
)

print("STAGED CHANGES")
subprocess.run(
    [
        "git", "-C", str(REPO_PATH),
        "status", "--short",
    ],
    check=True,
)

commit_result = subprocess.run(
    [
        "git", "-C", str(REPO_PATH),
        "commit", "-m",
        "Replace data folder with three plant datasets",
    ],
    text=True,
    capture_output=True,
)

print(commit_result.stdout)
print(commit_result.stderr)

if commit_result.returncode != 0:
    if "nothing to commit" in (
        commit_result.stdout + commit_result.stderr
    ).lower():
        print("The repository already contains these exact changes.")
    else:
        raise RuntimeError("Git commit failed.")

subprocess.run(
    [
        "git", "-C", str(REPO_PATH),
        "push", "origin", "main",
    ],
    env=git_env,
    check=True,
)

print("\nLATEST COMMIT")
subprocess.run(
    [
        "git", "-C", str(REPO_PATH),
        "log", "-1", "--oneline", "--decorate",
    ],
    check=True,
)

print("\nFINAL STATUS")
subprocess.run(
    [
        "git", "-C", str(REPO_PATH),
        "status", "--short",
    ],
    check=True,
)

#Upload Public Notebook Version

In [ ]:
import json
import os
import re
import stat
import subprocess
import tempfile
from pathlib import Path

from google.colab import drive, userdata

# ============================================================
# SETTINGS
# ============================================================

drive.mount("/content/drive")

SOURCE_NOTEBOOK = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "AlexanderOrr_ResChlorine_Official_Public_Clean.ipynb"
)

GITHUB_USERNAME = "angelicadatascience"
REPOSITORY_NAME = "watertreatment_public"

REPOSITORY_URL = (
    f"https://github.com/{GITHUB_USERNAME}/{REPOSITORY_NAME}.git"
)

GITHUB_FILENAME = (
    "AlexanderOrr_ResChlorine_Official_Public.ipynb"
)

# ============================================================
# LOCATE AND READ YOUR EXACT DRIVE NOTEBOOK
# ============================================================

if not SOURCE_NOTEBOOK.is_file():
    matches = list(
        Path("/content/drive/MyDrive").rglob(
            "AlexanderOrr_ResChlorine_Official_Public_Clean.ipynb"
        )
    )

    if not matches:
        raise FileNotFoundError(
            "The selected clean notebook was not found in My Drive."
        )

    SOURCE_NOTEBOOK = max(
        matches,
        key=lambda path: path.stat().st_mtime
    )

print("Using this exact Drive notebook:")
print(SOURCE_NOTEBOOK)

with SOURCE_NOTEBOOK.open("r", encoding="utf-8") as file:
    public_notebook = json.load(file)

# ============================================================
# REMOVE SAVED RUNS FROM THE GITHUB VERSION
# ============================================================

removed_outputs = 0
removed_execution_counts = 0
removed_attachments = 0

for cell in public_notebook.get("cells", []):
    if cell.get("cell_type") == "code":
        removed_outputs += len(cell.get("outputs", []))
        cell["outputs"] = []

        if cell.get("execution_count") is not None:
            removed_execution_counts += 1

        cell["execution_count"] = None

    if "attachments" in cell:
        removed_attachments += len(cell.get("attachments", {}))
        cell.pop("attachments", None)

public_notebook.setdefault("metadata", {}).pop("widgets", None)

# Convert the cleaned notebook to text for verification.
cleaned_notebook_text = json.dumps(
    public_notebook,
    ensure_ascii=False,
    indent=1
) + "\n"

# ============================================================
# CHECK FOR LITERAL CREDENTIALS
# ============================================================

credential_patterns = {
    "GitHub token": r"github_pat_[A-Za-z0-9_]{20,}|gh[pousr]_[A-Za-z0-9]{20,}",
    "OpenAI key": r"sk-[A-Za-z0-9_-]{20,}",
    "Google API key": r"AIza[A-Za-z0-9_-]{20,}",
    "AWS access key": r"AKIA[0-9A-Z]{16}",
    "Private key": r"-----BEGIN [A-Z ]*PRIVATE KEY-----",
    "Credential in GitHub URL": (
        r"https://[^\s/:@]+:[^\s/@]+@github\.com"
    ),
}

credential_matches = [
    name
    for name, pattern in credential_patterns.items()
    if re.search(
        pattern,
        cleaned_notebook_text,
        flags=re.IGNORECASE
    )
]

if credential_matches:
    raise RuntimeError(
        "Possible credentials detected. Nothing was uploaded: "
        f"{credential_matches}"
    )

# Verify the in-memory GitHub version.
remaining_outputs = sum(
    len(cell.get("outputs", []))
    for cell in public_notebook.get("cells", [])
)

remaining_execution_counts = sum(
    cell.get("execution_count") is not None
    for cell in public_notebook.get("cells", [])
    if cell.get("cell_type") == "code"
)

if remaining_outputs != 0:
    raise RuntimeError("Saved outputs remain.")

if remaining_execution_counts != 0:
    raise RuntimeError("Execution counts remain.")

print()
print("Safety check passed.")
print("Outputs removed:", removed_outputs)
print("Execution counts removed:", removed_execution_counts)
print("Attachments removed:", removed_attachments)
print("Credential matches: 0")

# ============================================================
# SECURE GITHUB AUTHENTICATION
# ============================================================

GITHUB_TOKEN = userdata.get("github_token")

if not GITHUB_TOKEN:
    raise ValueError(
        "The Colab Secret named github_token was not found."
    )

askpass_path = Path("/tmp/github_public_askpass.sh")

askpass_path.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) echo "$GITHUB_USERNAME" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
""",
    encoding="utf-8"
)

askpass_path.chmod(
    askpass_path.stat().st_mode | stat.S_IEXEC
)

git_env = os.environ.copy()
git_env["GIT_ASKPASS"] = str(askpass_path)
git_env["GIT_TERMINAL_PROMPT"] = "0"
git_env["GITHUB_USERNAME"] = GITHUB_USERNAME
git_env["GITHUB_TOKEN"] = GITHUB_TOKEN

# ============================================================
# CLONE INTO A NEW TEMPORARY DIRECTORY
# ============================================================

temporary_directory = Path(
    tempfile.mkdtemp(prefix="watertreatment_public_")
)

repository_path = temporary_directory / REPOSITORY_NAME

subprocess.run(
    [
        "git",
        "clone",
        REPOSITORY_URL,
        str(repository_path),
    ],
    env=git_env,
    check=True,
)

# ============================================================
# WRITE ONLY THE SANITIZED VERSION INTO THE REPOSITORY
# ============================================================

repository_notebook = repository_path / GITHUB_FILENAME

repository_notebook.write_text(
    cleaned_notebook_text,
    encoding="utf-8"
)

subprocess.run(
    [
        "git",
        "-C",
        str(repository_path),
        "config",
        "user.name",
        GITHUB_USERNAME,
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        str(repository_path),
        "config",
        "user.email",
        "angelicadatascience@users.noreply.github.com",
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        str(repository_path),
        "add",
        "--",
        GITHUB_FILENAME,
    ],
    check=True,
)

has_changes = (
    subprocess.run(
        [
            "git",
            "-C",
            str(repository_path),
            "diff",
            "--cached",
            "--quiet",
        ]
    ).returncode
    != 0
)

if has_changes:
    subprocess.run(
        [
            "git",
            "-C",
            str(repository_path),
            "commit",
            "-m",
            "Add sanitized Alexander Orr notebook",
        ],
        check=True,
    )

    subprocess.run(
        [
            "git",
            "-C",
            str(repository_path),
            "push",
            "origin",
            "main",
        ],
        env=git_env,
        check=True,
    )

    print()
    print("UPLOAD COMPLETED SUCCESSFULLY")
else:
    print()
    print("GitHub already contains this exact cleaned notebook.")

print(
    f"https://github.com/{GITHUB_USERNAME}/"
    f"{REPOSITORY_NAME}/blob/main/{GITHUB_FILENAME}"
)